# Nature Water reviewer revision — canonical transfer analysis

This is the analysis notebook used for the revised Nature Water transfer. The public copy removes local absolute paths and stored execution outputs while retaining the analysis logic, locked configuration, canonical facility × scenario workflow, validation contracts, and publication-figure generation used for the transfer.

The licensed Baxtel facility records are not distributed. Place lawful local inputs under `data/raw/` as documented in `REPRO.md` before running the empirical workflow.


In [ ]:
from pathlib import Path
import sys, json
PAYLOAD = {'main.tex': "\\documentclass[11pt]{article}\n\n\\usepackage[margin=1in]{geometry}\n\\usepackage[T1]{fontenc}\n\\usepackage[utf8]{inputenc}\n\\usepackage{lmodern}\n\\usepackage{microtype}\n\\usepackage{graphicx}\n\\usepackage{amsmath,amssymb}\n\\usepackage[numbers,sort&compress]{natbib}\n\\usepackage{booktabs,longtable,array}\n\\usepackage{tabularx}\n\\usepackage{adjustbox}\n\\usepackage{placeins}\n\\usepackage{xcolor}\n\\usepackage[hidelinks]{hyperref}\n\\usepackage[switch]{lineno}\n\\usepackage{tikz}\n\\usetikzlibrary{arrows.meta,positioning,fit,calc}\n\\graphicspath{{images/}{upload/}{./}}\n\n\\newcommand{\\GLyr}{\\ensuremath{\\mathrm{GL\\,yr^{-1}}}}\n\\newcommand{\\TWhyr}{\\ensuremath{\\mathrm{TWh\\,yr^{-1}}}}\n\\newcommand{\\LkWh}{\\ensuremath{\\mathrm{L\\,kWh^{-1}}}}\n\\newcommand{\\InventoryProvider}{Baxtel}\n\\newcommand{\\GridDataset}{EPA eGRID2023 Revision 2}\n\\newcommand{\\safegraphic}[2][\\textwidth]{%\n  \\IfFileExists{images/#2}{\\includegraphics[width=#1]{images/#2}}{%\n    \\IfFileExists{#2}{\\includegraphics[width=#1]{#2}}{%\n      \\ifNWFinal\\PackageError{NWrevision}{Missing figure #2}{Run the notebook and include its images folder.}\\else\n      \\fbox{\\parbox{0.88\\linewidth}{\\textbf{DRAFT:} Figure not yet generated:\\par\\path{#2}}}\\fi}}}\n\n\\title{Pathway and indicator choices alter water-screening priorities for U.S. hyperscale data centers}\n\\author{Gianluca Guidi$^{1,*}$ and Francesca Dominici$^{1,*}$}\n\\date{}\n\n\n% REBUILT revision template. Finalization switches this flag after the local run.\n\\newif\\ifNWFinal\n\\NWFinalfalse\n\\newcommand{\\NWpending}[1]{\\par\\smallskip\\noindent\\textbf{DRAFT -- #1}\\par\\smallskip}\n\\newcommand{\\NWinput}[1]{\\IfFileExists{#1}{\\input{#1}}{\\ifNWFinal\n  \\PackageError{NWrevision}{Missing generated file #1}{Run the revision notebook before finalization.}\n  \\else\\NWpending{Generated results not yet available: \\texttt{\\detokenize{#1}}}\\fi}}\n\\newcommand{\\FundingStatement}{\\NWpending{Author funding statement required.}}\n\\newcommand{\\ConflictsStatement}{\\NWpending{Author competing-interest confirmation required.}}\n\\newcommand{\\RelatedManuscriptStatement}{\\NWpending{Related manuscript status and supplied copy require author confirmation.}}\n\\newcommand{\\ReviewerArchive}{\\textbf{DRAFT: verified reviewer-access archive required}}\n\\IfFileExists{generated/author_statements.tex}{\\input{generated/author_statements.tex}}{}\n\\AtBeginDocument{\\ifNWFinal\\else\n\\begin{center}\\fbox{\\parbox{0.90\\textwidth}{\\textbf{REBUILT REVISION DRAFT -- NOT FOR SUBMISSION.}\nThe new regional and sensitivity outputs require the local empirical run and author sign-off.\nAny retained baseline numbers are checked against that run before finalization.}}\n\\end{center}\\fi}\n\n\\begin{document}\n\\maketitle\n\n\\noindent$^{1}$Harvard T.H. Chan School of Public Health, Harvard University, Boston, MA, USA.\\\\\n$^{*}$Correspondence to: ggianluca@hsph.harvard.edu; fdominic@hsph.harvard.edu.\n\n\\linenumbers\n\n\\begin{abstract}\nNational data-center water footprints are often reported as single values, although their pathways and risk indicators prioritize different places. We link a 15 December 2025 inventory of 472 U.S. hyperscale facility records to HydroBASINS level-6 hosting basins, annual Aqueduct indicators and balancing authorities. A reference scenario yields 299.89 gigalitres per year (GL yr$^{-1}$; one GL is one billion litres): 74.16 GL on site and 225.74 GL attributed to electricity. Electricity-related consumption remains larger when hydropower evaporation is assigned zero. Half of on-site consumption is reached in 10 of 63 hosting basins (15.9\\%), whereas half of electricity-related consumption is reached in 3 of 24 balancing authorities (12.5\\%). Withdrawal-based stress identifies 25 high-context hosting basins, compared with seven under consumption-based depletion. Pathway and indicator choices therefore redirect the geography and evidence prioritized for water management.\n\\end{abstract}\n\nData centers consume water through their own operations and through the electricity systems associated with their load. On-site consumption is primarily associated with evaporative heat rejection, cooling-tower operation and water-quality management, whereas electricity-related consumption arises from thermoelectric cooling and, under some attribution conventions, hydropower-reservoir evaporation \\citep{mytton2021,lei2022,macknick2012,meldrum2013,grubert2016}. Rapid growth and limited standardized reporting make these demands difficult to evaluate, especially when national totals conceal the places where local water systems and regional electricity systems face different questions \\citep{cooley2026}.\n\nPrior work established both pathways, introduced site-and-source Water Usage Effectiveness (WUE) concepts and located data-center water burdens in physical watersheds \\citep{greengrid2011,siddik2021}. Electricity--water research has likewise shown that power-system transfers redistribute water dependencies across regions \\citep{chini2018,grubert2018,peer2019}. The unresolved decision problem is therefore not whether two pathways exist. It is whether the locations prioritized for follow-up remain the same when a current hyperscale inventory is represented in pathway-appropriate geographies and screened with alternative water-context indicators.\n\nWe address that question with a dated, facility-resolved U.S. hyperscale inventory. The analysis makes three advances. First, it quantifies on-site and electricity-related consumption for a reported-current-capacity snapshot and preserves the pathways in distinct analytical geographies. Second, it evaluates concentration as a share of each pathway's own hosting regions and reports within-partition distributional indices rather than inferring physical concentration from raw region counts alone. Third, it tests how the hosting-basin priority set changes when withdrawal-based water stress is replaced by consumption-based depletion, while separately bounding the influential hydropower-attribution convention. All national and regional outputs derive from one validated facility-by-scenario table, so the scientific claims and displayed aggregates share the same accounting basis.\n\nWe use ``hyperscale'' as the source inventory's company-type classification rather than a universal engineering threshold. The analytic population comprises 472 contiguous-U.S. records with positive reported current capacity and complete required spatial assignments in a licensed \\InventoryProvider{} inventory \\citep{baxtel2025}. We interpret the provider's \\texttt{current\\_mw} field as facility-input nameplate power for modeling. On-site water is estimated from information-technology (IT) electricity, power usage effectiveness (PUE), WUE and an explicit consumptive fraction. Electricity-related water is attributed using average balancing-authority (BA) generation shares and technology-average operational consumption factors. The physical labels ``on-site'' and ``electricity-related'' are used throughout; greenhouse-gas Scope terminology is an analogy only \\citep{ghgscope2}.\n\n\\begin{figure}[t]\n\\centering\n\\safegraphic[\\textwidth]{fig_1_NEW.pdf}\n\\caption{\\textbf{Study design and analytical workflow.}\n\\textbf{a}, Data inputs include the dated hyperscale facility inventory, electricity-system attributes, hydrologic context and prespecified model parameters.\n\\textbf{b}, Reported current capacity is converted to annualized facility-input electricity under the utilization scenario. IT electricity is obtained using power usage effectiveness (PUE) and combined with water usage effectiveness (WUE) and the on-site consumptive fraction to estimate the on-site pathway. Electricity-related consumption is estimated from facility-input electricity, balancing-authority generation shares and technology-specific operational water-consumption factors. The pathways remain separate in basin and BA inference; their sum is reported nationally and in explicitly labelled host-location profiles.\n\\textbf{c}, Outputs include national pathway totals, separate spatial attribution to facility-hosting HydroBASINS level-6 basins and hosting balancing authorities, within-partition concentration metrics, and hydrologic-context analyses using Aqueduct baseline water stress (BWS) and baseline water depletion (BWD). All reported outputs derive from a single validated facility-by-scenario table subject to conservation and subset checks. On-site and electricity-related pathways are retained separately in spatial analyses because they represent different analytical geographies and require different evidence for follow-up.}\n\\label{fig:design}\n\\end{figure}\n\n\n\\section*{Results}\n\n\\subsection*{The electricity-related pathway remains larger across the specified cases}\n\nThe 472 inventory records totalled 20,041 MW of reported current power capacity, interpreted as facility-input nameplate capacity. At reference utilization ($u=0.66$), this corresponds to 115.87 \\TWhyr{} of annualized facility electricity. The reference factor set yields 299.89 \\GLyr{} of operational-water consumption attributable to the inventory: 74.16 \\GLyr{} on site and 225.74 \\GLyr{} through electricity supply (Table~\\ref{tab:national}). Electricity-related water therefore represents 75.27\\% of the reference estimate. These are annualized scenario outputs, not metered facility totals.\n\nThe pathway ordering persists across the specified cases while its magnitude changes. The lower-load/lower-water case yields 204.91 \\GLyr{} and the higher-load/water-intensive case 450.60 \\GLyr{}. Holding reference activity and cooling assumptions fixed while assigning no reservoir evaporation to hydropower reduces electricity-related consumption from 225.74 to 128.35 \\GLyr{} and the total to 202.50 \\GLyr{}. Electricity-related water remains the larger pathway at this endpoint (63.38\\%). Substituting the U.S.-average net freshwater estimate of 1.7 m$^3$ GJ$^{-1}$ reported by Grubert (approximately 6.1 L kWh$^{-1}$) for the selected 8.0 L kWh$^{-1}$ factor yields approximately 202.6 \\GLyr{} electricity related and 276.8 \\GLyr{} in total. Thus, the qualitative pathway ordering is stable across the stated cases, whereas absolute values and regional electricity rankings remain factor-set dependent.\n\nTwo external comparisons provide order-of-magnitude checks rather than validation. The reference electricity estimate is 65.8\\% of the 176 TWh reported for the entire U.S. data-center sector in 2023 by Lawrence Berkeley National Laboratory (LBNL), although the populations and dates differ \\citep{lbnl2024}. The reference WUE of 0.80 L kWh$^{-1}_{\\mathrm{IT}}$ is deliberately above the approximately 0.32--0.40 L kWh$^{-1}_{\\mathrm{IT}}$ aggregate hyperscale values represented in recent LBNL scenarios. In an additional consumptive-fraction sensitivity motivated by the Pacific Institute's synthesis of typical facility withdrawals, interpreting the same numerical WUE as withdrawal based and setting $c=0.74$ would lower on-site consumption to 54.9 \\GLyr{} and the total to 280.6 \\GLyr{} without changing on-site regional ranks \\citep{cooley2026}. The Supplementary Information gives the comparison boundaries.\n\n\\begin{table}[!htbp]\n\\centering\n\\small\n\\begin{adjustbox}{max width=\\textwidth}\n\\begin{tabular}{lrrrrrrr}\n\\toprule\nScenario & Electricity & On-site & Electricity-related & Total & On-site & Electricity-related & Intensity \\\\\n & (\\TWhyr) & (\\GLyr) & (\\GLyr) & (\\GLyr) & (\\%) & (\\%) & (\\LkWh) \\\\\n\\midrule\nLower-load/lower-water & 96.56 & 16.79 & 188.11 & 204.91 & 8.20 & 91.80 & 2.12 \\\\\nBaseline/reference & 115.87 & 74.16 & 225.74 & 299.89 & 24.73 & 75.27 & 2.59 \\\\\nHigher-load/water-intensive & 149.22 & 159.88 & 290.72 & 450.60 & 35.48 & 64.52 & 3.02 \\\\\nNo hydropower attribution & 115.87 & 74.16 & 128.35 & 202.50 & 36.62 & 63.38 & 1.75 \\\\\n\\bottomrule\n\\end{tabular}\n\\end{adjustbox}\n\\caption{\\textbf{National annualized scenario results.} Electricity is facility-input electricity. The first and third rows jointly change utilization, PUE and WUE and therefore do not isolate a single parameter. The reference row is a selected central case, not an empirical median or maximum-likelihood estimate. The no-hydropower case differs from it only by setting the hydropower consumption factor to zero. Rows are mutually exclusive; displayed values may not add exactly because of rounding.}\n\\label{tab:national}\n\\end{table}\n\n\\subsection*{Pathway choice changes the geography of follow-up}\n\nOn-site consumption was aggregated to the HydroBASINS level-6 polygon containing each facility. Sixty-three hosting basins have positive modeled on-site burden; 62 also have valid annual Aqueduct context. Among those context-covered basins, 25 meet the withdrawal-based high-context threshold and 15 combine high context with at-least-median on-site burden (Fig.~\\ref{fig:geographies}a). Because common utilization, PUE, WUE and consumptive-fraction values are applied within each scenario, this map shows where modeled capacity-based burden intersects hydrologic context; it does not estimate spatial variation in cooling performance.\n\nElectricity-related consumption was attributed to the BA polygon containing each facility using the BA's annual production mix. Figure~\\ref{fig:geographies}b maps burden continuously and outlines BAs above the median modeled grid-water intensity. The display distinguishes hosted electricity demand from factor-set intensity without using coal-plus-gas share or hosted-site stress as water-impact criteria. Polygon containment defines the attribution region; it does not identify a facility's retail utility, the generating plants or their source-water basins.\n\n\\begin{figure}[t]\n\\centering\n\\safegraphic[\\textwidth]{Figure2_Pathway_Burden_Maps.png}\n\\caption{\\textbf{Pathway-specific burden and context.} \\textbf{a}, On-site consumption in facility-hosting HydroBASINS level-6 basins. Pink outlines identify annual Aqueduct baseline water stress of at least 3; black outlines identify its intersection with at-least-median positive on-site burden. Sixty-two of the 63 positive-burden hosting basins have valid annual Aqueduct context. \\textbf{b}, Electricity-related consumption attributed with facility-hosting-BA annual production mixes. Fill shows continuous burden and outlines identify grid-water intensity above the median among hosting BAs under the reference factor set. Both panels are screening displays rather than measured impact classes or physical source-water maps.}\n\\label{fig:geographies}\n\\end{figure}\n\nThe continuous BA diagnostic reinforces that siting context and electricity attribution answer different questions (Extended Data Fig.~\\ref{fig:ed_quadrant}). Across 24 hosting BAs, capacity-weighted hosted-site stress spans almost the full Aqueduct scale, while modeled grid-water intensity ranges from below 1 to about 8 \\LkWh{}. Their Spearman association is weak ($\\rho=-0.13$), consistent with one axis describing the context of hosted data-center sites and the other the attributed production mix.\n\n\\subsection*{Half of each pathway is concentrated in fewer than one in six hosting regions}\n\nWithin the reference model, 10 of 63 positive-burden hosting basins (15.9\\%) are required to reach half of national on-site consumption, whereas 3 of 24 hosting BAs (12.5\\%) reach half of electricity-related consumption (Fig.~\\ref{fig:concentration}). The corresponding within-partition Gini coefficients are 0.539 and 0.666, and the five leading regions account for 34.7\\% and 72.4\\% of the respective pathway totals. These measures consistently identify compact priority sets within both analytical systems, with a steeper electricity-attribution profile under the selected factor set.\n\nThese compact regional sets identify where additional facility, utility or plant evidence could cover a large share of modeled consumption. Common-partition and without-PJM diagnostics distinguish pathway concentration from the choice of reporting boundaries (Supplementary Information).\n\n\\begin{figure}[t]\n\\centering\n\\safegraphic[0.84\\textwidth]{Figure3_Normalized_Concentration.png}\n\\caption{\\textbf{Within-partition concentration profiles of the two operational pathways.} Positive-burden hosting regions are ranked from largest to smallest. The horizontal axis is the percentage of each pathway's hosting regions included and the vertical axis is cumulative pathway share; the diagonal denotes equal distribution. Ten of 63 hosting basins reach at least half of on-site consumption and 3 of 24 hosting BAs reach the same threshold for electricity-related consumption. The corresponding within-partition Gini coefficients are 0.539 and 0.666. Because the underlying partitions are non-equivalent, the curves support prioritization within each geography rather than a scale-free comparison of physical clustering.}\n\\label{fig:concentration}\n\\end{figure}\n\n\\subsection*{Indicator choice changes the high-context basin set by 72\\%}\n\nAqueduct baseline water stress (\\texttt{bws\\_score}) compares gross demand, including consumptive and non-consumptive demand, with available renewable supply. Baseline water depletion (\\texttt{bwd\\_score}) instead compares consumptive demand with available supply \\citep{kuzma2023}. Among the 62 hosting basins with valid annual context, withdrawal-based stress identifies 25 above the common score-3 threshold, whereas consumption-based depletion identifies seven---a 72\\% reduction in the high-context set (Extended Data Fig.~\\ref{fig:ed_stress}). The subset also meeting the on-site burden criterion changes from 15 to six, a 60\\% reduction. Indicator choice therefore redirects categorical follow-up even though it does not alter modeled water volumes or concentration.\n\n\\subsection*{Capacity alone does not determine host-location burden}\n\nHost-location profiles show that similar amounts of reported current capacity can be associated with different modeled pathway balances (Fig.~\\ref{fig:profiles}). Most plotted states and hosting BAs are electricity-related dominant under the reference factor set, while a smaller set is on-site dominant or within 10\\% between pathways. Scenario ranges are widest where hosted activity and modeled intensity are both high. These profiles demonstrate why capacity is an incomplete screening variable, but they are not physical state water accounts and their electricity-side ordering is not assumed invariant to hydropower or other factor choices.\n\n\\begin{figure}[p]\n\\centering\n\\begin{minipage}{0.49\\textwidth}\n\\centering\n\\safegraphic[\\textwidth]{Figure4_State_Pathway_Dominance.png}\\\\[-0.3em]\n\\textbf{a} Host-state pathway balance\n\\end{minipage}\\hfill\n\\begin{minipage}{0.49\\textwidth}\n\\centering\n\\safegraphic[\\textwidth]{Figure4_State_Scenario_Intervals.png}\\\\[-0.3em]\n\\textbf{b} Host-state scenario range\n\\end{minipage}\n\n\\vspace{0.8em}\n\n\\begin{minipage}{0.49\\textwidth}\n\\centering\n\\safegraphic[\\textwidth]{Figure4_BA_Pathway_Dominance.png}\\\\[-0.3em]\n\\textbf{c} Hosting-BA pathway balance\n\\end{minipage}\\hfill\n\\begin{minipage}{0.49\\textwidth}\n\\centering\n\\safegraphic[\\textwidth]{Figure4_BA_Scenario_Intervals.png}\\\\[-0.3em]\n\\textbf{d} Hosting-BA scenario range\n\\end{minipage}\n\\caption{\\textbf{Host-location pathway profiles after reconciliation to the national total.} \\textbf{a,c}, Reference-factor-set pathway balance for states and BAs. Electricity-related dominant means that electricity-related consumption exceeds on-site consumption by more than 10\\%; on-site dominant is defined analogously; similar denotes a difference within 10\\%. \\textbf{b,d}, Lower-load/lower-water, reference and higher-load/water-intensive totals for the 15 states and BAs with the largest hosted capacity. Each point is a mutually exclusive scenario. State and BA values are host-location attributions, not verified physical source-water totals; electricity-side rankings may change under alternative generation-water factors.}\n\\label{fig:profiles}\n\\end{figure}\n\n\n\\subsection*{Named regions and the robustness of their priority status}\n\\NWinput{generated/region_results.tex}\n\\NWinput{generated/regions_basins.tex}\n\\NWinput{generated/regions_ba.tex}\n\\NWinput{generated/pjm_results.tex}\n\\NWinput{generated/hydro_results.tex}\n\\NWinput{generated/threshold_results.tex}\n\n\\subsection*{Cooling heterogeneity and inventory-stage coverage}\n\\NWinput{generated/climate_results.tex}\n\\NWinput{generated/stage_results.tex}\n\n\\section*{Discussion}\n\nThe central finding is that the water-screening geography of hyperscale digital infrastructure is conditional on how the problem is represented. A single national footprint combines two pathways that belong to non-nested spatial systems, while a single ``water-stressed'' label conceals a large difference between withdrawal-based stress and consumption-based depletion. In this current-capacity screen, each pathway concentrates half of its modeled total in fewer than one in six of its own hosting regions, yet the regions and evidence relevant to follow-up differ. Replacing BWS with BWD reduces the high-context hosting-basin set from 25 to seven.\n\nThis distinction is operationally consequential. On-site estimates direct attention to cooling architecture, water source, return flows, utility service areas, permits and drought planning. Electricity-related estimates direct attention to regional generation, plant cooling and source water, utility resource planning, transmission, procurement and load management \\citep{scott2011}. A combined total can describe magnitude, but it cannot by itself identify which evidence or actor is relevant to a reduction. Recent policy synthesis has similarly emphasized separate reporting of direct and indirect water and the need for standardized facility data \\citep{cooley2026}; the present analysis supplies a facility-resolved structure for organizing that follow-up across the U.S. hyperscale inventory.\n\nThe conclusions have different levels of robustness. Across all specified cases, including zero hydropower attribution, electricity-related consumption remains larger than on-site consumption. The dependence of the high-context basin set on indicator choice is also direct and does not affect modeled volumes. Exact national totals and reference-case regional rankings are more conditional: they depend on the facility-capacity boundary, common utilization/PUE/WUE assumptions, BA production mixes and generation-water factors. Hydropower accounts for 43.1\\% of the displayed reference electricity-related estimate, making the zero- and 6.1-L kWh$^{-1}$ comparisons essential rather than ancillary.\n\nThe analysis remains a national screening study rather than a set of facility measurements. The inventory is a provider-defined reported-current-capacity cross-section; common operating parameters do not resolve facility-specific cooling, ramp-up or source water; and the BA model does not trace electricity to plants or source basins. Company-level disclosures were not treated as direct validation because their organizational, geographic, temporal and withdrawal-versus-consumption boundaries are not consistently aligned with this inventory. These limitations constrain absolute magnitude and local interpretation, but they do not alter the principal decision insight: pathway and indicator choices change which places and data are prioritized. Decision-grade assessment of an individual project would require metered electricity and water, cooling configuration, utility intake sources, return flows, seasonal operations, electricity contracts and plant-level generation-water data.\n\n\\section*{Methods}\n\n\\subsection*{Study population and accounting boundary}\n\nThe source inventory was a licensed \\InventoryProvider{} facility file delivered to Harvard and frozen on 15 December 2025 \\citep{baxtel2025}. Records were eligible when the provider's company-type field matched a case-insensitive, whole-field expression for \\texttt{hyperscale}, \\texttt{hyperscaler} or \\texttt{hyper scale}; composite colocation labels were not included by substring matching; a unique provider record identifier, valid coordinates and positive \\texttt{current\\_mw} were present; and the assigned state was in the contiguous United States. Alaska, Hawaii and U.S. territories were excluded. The final analytic population contained 472 records with 20,041 MW. We did not independently redefine or merge the provider's campus and building units beyond enforcing identifier uniqueness; the analysis therefore inherits the provider's record segmentation and possible dependence among co-located records. We did not infer duplicate capacity from spatial proximity alone, and no records were merged solely because coordinates were identical or nearby.\n\nWe interpret \\texttt{current\\_mw} as facility-input nameplate power for the model. Announced, construction and planned capacity fields were not added. Because stage labels were insufficiently complete and standardized for a defensible operating-only subset, the primary analysis retained all eligible positive-current-capacity records and did not infer operating status from age or free text. The estimand is consequently annualized operational-water consumption under current-capacity scenarios, not observed water consumption by a verified operating census. The provider capacity boundary is not independently metered by the authors; a conditional IT-capacity interpretation is reported in the Supplement to show which conclusions depend on this boundary.\n\nThe accounting boundary includes WUE-derived on-site operational water and water attributed to grid electricity generation. It excludes gross withdrawals not consumed, district energy, behind-the-meter generation without metered flows, construction, equipment manufacturing, semiconductor supply chains, fuel-cycle water and other embodied uses. Scope analogues refer only to whether modeled operational consumption occurs at the facility or is attributed to electricity; they do not claim formal conformance with greenhouse-gas reporting.\n\n\\subsection*{Spatial assignment and environmental context}\n\nCoordinates were converted to points in EPSG:4326 and assigned by point-in-polygon joins. The BA boundary input was the project-archived WattTime grid-region layer stored as \\path{balancing_authorities_polygons/balancing_authorities_EPA.shp}, joined on the \\texttt{region\\_B\\_1} identifier; the exact file used in the analysis is identified by its checksum in the run manifest \\citep{watttimeBA}. Each included record was required to receive no more than one state, BA and HydroBASINS level-6 identifier. Ambiguous joins, missing required assignments, duplicated facility--scenario rows and BAs without a valid generation mix caused the workflow to stop rather than impute values. Polygon identifiers were normalized before joining, and BA generation shares were required to sum to one within a 0.98--1.02 pre-normalization tolerance. Assignment to a BA polygon defines the hosting-BA production-mix attribution only; it does not verify the retail utility, interconnection or physical flow serving a facility.\n\nHydroBASINS polygons derive from the HydroSHEDS drainage network \\citep{lehner2013}. Level 6 was used because the annual Aqueduct 4.0 indicators in the analysis are keyed to that geography. The primary field was annual \\texttt{bws\\_score}; negative no-data sentinels were set to missing and finite values were required to lie on the documented 0--5 scale. Annual \\texttt{bwd\\_score} was evaluated as the consumption-based sensitivity. High context was defined as a score of at least 3. Neither field entered the volumetric water equations.\n\n\\subsection*{Facility electricity}\n\nFor facility $i$ and scenario $s$, annualized facility-input electricity was\n\\begin{equation}\nE^{\\mathrm{fac}}_{i,s}=P_i\\times8760\\times u_s,\n\\end{equation}\nwhere $P_i$ is reported current power capacity interpreted as facility-input nameplate MW and $u_s$ is annual-average utilization. The equation yields MWh yr$^{-1}$; before applying litre-per-kWh intensities, we used\n\\begin{equation}\nE^{\\mathrm{fac,kWh}}_{i,s}=1000E^{\\mathrm{fac}}_{i,s}.\n\\end{equation}\nBecause $P_i$ is interpreted as facility-input rather than IT-only power, PUE was not multiplied into this equation. IT electricity was recovered as\n\\begin{equation}\nE^{\\mathrm{IT,kWh}}_{i,s}=\\frac{E^{\\mathrm{fac,kWh}}_{i,s}}{\\mathrm{PUE}_s}.\n\\end{equation}\n\n\\subsection*{On-site operational consumption}\n\nISO/IEC 30134-9 defines WUE as site water use per unit IT electricity \\citep{iso30134}. On-site consumption was estimated as\n\\begin{equation}\nW^{(1)}_{i,s}=10^{-3}E^{\\mathrm{IT,kWh}}_{i,s}\\,\\mathrm{WUE}_s\\,c_s,\n\\end{equation}\nwhere WUE is in litres per kWh of IT electricity, $c_s$ is the assumed consumptive fraction and $10^{-3}$ converts litres to cubic metres; dividing cubic metres by $10^6$ yields GL. The primary WUE settings are author-selected site-water-use scenarios, not extracted LBNL observations; $c_s=1$ assigns all modeled site water to consumption. For any input already defined as consumption per IT kWh, $c_s$ must equal one. A smaller $c_s$ is applicable only to an explicitly withdrawal-based input, not to a consumption-based WUE. Because $W^{(1)}$ scales linearly with $c_s$, a lower fraction would lower on-site volume without changing the basin ranking when the same value is applied nationally.\n\nThe lower-load/lower-water, reference and higher-load/water-intensive settings were, respectively, $u=0.55,0.66,0.85$; PUE$=1.15,1.25,1.40$; and WUE$=0.20,0.80,1.50$ \\LkWh{} of IT electricity. These are joint, literature-informed scenario settings rather than empirical quantiles, confidence limits or observed facility values \\citep{mytton2021,lei2022,lbnl2024}. The reference WUE of 0.80 L kWh$^{-1}_{\\mathrm{IT}}$ is deliberately conservative for on-site volume relative to the approximately 0.32--0.40 L kWh$^{-1}_{\\mathrm{IT}}$ aggregate hyperscale values reported in recent LBNL scenarios; it is not claimed as a measured fleet mean. The Supplementary Information documents the evidentiary role and non-claims of each setting.\n\n\\subsection*{Electricity-related operational consumption}\n\nEach facility inherited the average annual generation shares $s_{r,f}$ of its hosting BA $r$ from the \\GridDataset{} BA-level production-mix table \\citep{usepaegrid2023}. The seven mutually exclusive classes were coal, gas, nuclear, hydropower, wind, solar and other. The source mix represents calendar-year 2023 generation rather than an import-adjusted consumption mix, and is temporally offset from the December 2025 facility snapshot.\n\nBA grid-water intensity was\n\\begin{equation}\nI_{r,s}=\\sum_f s_{r,f}w_{f,s},\n\\end{equation}\nand electricity-related consumption was\n\\begin{equation}\nW^{(2)}_{i,s}=10^{-3}E^{\\mathrm{fac,kWh}}_{i,s}I_{r(i),s}.\n\\end{equation}\nSelected screening attribution factors (\\LkWh) were 1.9 for coal, 0.7 for gas, 2.5 for nuclear, 8.0 for hydropower, 0 for wind, 0.1 for solar and 1.0 for other generation. The non-hydropower values were selected from harmonized operational-factor literature \\citep{macknick2012,meldrum2013}; the hydropower value is a manuscript-selected reservoir-evaporation attribution convention informed by the hydropower literature rather than a claimed national best estimate \\citep{grubert2016,mekonnen2012}. ``Other'' is a residual simplifying class. Reporting methods and plant configuration contribute additional uncertainty \\citep{sjostedt2025}. These values are not contemporary plant-, prime-mover-, cooling-system- or water-source-specific coefficients. The no-hydropower sensitivity changed only the hydropower factor to zero while retaining reference activity and on-site assumptions. A second analytical comparison uses 6.1 L kWh$^{-1}$, the rounded conversion of Grubert's U.S.-average net freshwater estimate of 1.7 m$^3$ GJ$^{-1}$; neither factor is treated as reservoir specific.\n\nTotal operational consumption was\n\\begin{equation}\nW^{\\mathrm{tot}}_{i,s}=W^{(1)}_{i,s}+W^{(2)}_{i,s}.\n\\end{equation}\nThe pathways remain separate in physical interpretation. Their sum is reported nationally and in host-location profiles, which are attributed accounts rather than physical source-water totals.\n\n\\subsection*{Burden, context and concentration analyses}\n\nFor the primary on-site screen, facility values were summed within hosting basins. Positive-burden basins at or above the median were labelled high burden; basins with annual \\texttt{bws\\_score}$\\geq3$ were labelled high context; the intersection met both criteria. The primary model used the hosting basin's own score, not touching-neighbour stress. We repeated the screen using annual \\texttt{bwd\\_score}. These thresholds create descriptive classes only; no ecological or regulatory validation is claimed.\n\nFor BA context, electricity-related burden and modeled grid-water intensity were reported separately. The main BA map uses continuous burden and an above-median intensity outline only as a descriptive decomposition; no composite hotspot or impact class is defined. Capacity-weighted hosted-site stress is retained as a separate diagnostic:\n\\begin{equation}\n\\bar S_r=\\frac{\\sum_{i:r(i)=r}P_iS_i}{\\sum_{i:r(i)=r}P_i}.\n\\end{equation}\nThis quantity describes the Aqueduct context of hosted data-center sites and is not stress at generating plants. Coal-plus-gas share is retained, if shown, only as energy-transition context and does not enter the water classification or inference.\n\nSpatial concentration was calculated independently for on-site basin totals and electricity-related BA totals. Regions were ranked in descending order, shares of the corresponding national pathway were calculated, and cumulative shares were summed. The reported count is the smallest rank at which cumulative share reached or exceeded 0.5. Rank was normalized by the number of positive-burden regions, and within-partition Gini coefficients were calculated from regional pathway shares. The concentration denominator contains 63 positive-burden hosting basins and 24 hosting BAs; 62 of those basins have valid annual Aqueduct context. None of these normalizations makes the spatial partitions physically commensurate, so cross-pathway contrasts are interpreted as within-partition screening profiles.\n\n\\subsection*{Scenario and sensitivity interpretation}\n\nThe lower-load/lower-water, reference and higher-load/water-intensive cases jointly vary utilization, PUE and WUE and are interpreted as structured cases rather than confidence limits. Electricity-related water scales linearly with utilization for a fixed generation mix, while on-site water scales with $u\\,\\mathrm{WUE}\\,c/\\mathrm{PUE}$. The Supplementary Information reports one-at-a-time sensitivities, including $c=0.74$, and the zero- and 6.1-L kWh$^{-1}$ hydropower comparisons. Regional electricity rankings are reported for the three stated hydropower coefficients; their stability outside those specified cases is not assumed.\n\n\n\\subsection*{Regional robustness diagnostics}\nFor hydropower coefficients 0, 6.1 and 8.0 L kWh$^{-1}$, the revised workflow recomputes every hosting-BA intensity and burden while retaining the reference facility electricity and all non-hydropower coefficients. It reports the complete rankings, minimum sets reaching half the pathway, top-five membership, Spearman rank correlations, Gini and HHI. Rank correlations use average ranks for ties; displayed ranked sets use BA code as the deterministic tie-break. All hosting BAs remain in the common support, including zero-water BAs under zero hydropower. Positive-burden counts are reported separately. The without-PJM diagnostic removes PJM and renormalizes shares to the remaining BAs; the full-fleet result remains primary.\n\nOn-site and electricity-related burdens are also aggregated to identical hosting-state and hosting-BA partitions. These diagnostics hold the statistical partition constant without treating the electricity component as physical water use in the host region. Aqueduct comparisons use only basins with both valid scores, hold the high-burden median at the value from all positive-burden hosting basins, and sweep context thresholds 2, 2.5, 3, 3.5 and 4. Counts, intersections, set inclusion, Jaccard overlap and captured national on-site shares are exported for every threshold. Missing context is not assigned zero.\n\nA paired climate/topology sensitivity uses documented, facility-assigned PUE and WUE inputs from the same coefficient case, with complete coverage required for every case. It holds facility-input electricity fixed, recomputes IT electricity using the paired PUE, and then applies the paired WUE with its explicit withdrawal or consumption basis. It reports basin totals, concentration, top-ten overlap and rank agreement, and repeats the burden--context screens. These are conditional cooling cases, not estimates of installed cooling-system prevalence. The coefficient and assignment provenance accompany the aggregate outputs. Incomplete coefficients or unverified water bases cannot enter a final build.\n\nProvider stage labels are reported as descriptive record and capacity shares. Missing labels remain missing; mapped operating shares require an explicit, documented label crosswalk. Basin labels likewise require a documented PFAF-ID crosswalk and are not inferred from a nearby city or river. Geographic labels refer to hosting regions rather than verified intake basins.\n\n\\subsection*{Validation and reproducibility}\n\nThe workflow built one canonical facility-by-scenario table. Automated contracts required unique facility--scenario rows, complete required assignments, normalized generation shares, non-negative water quantities, $W^{\\mathrm{tot}}=W^{(1)}+W^{(2)}$, reconciliation of state, BA and basin aggregates with national totals, and ranked subsets no greater than their parent totals. Input hashes, a configuration hash, validation outputs and aggregate figure-source tables were written with each run. All externally supplied figure files must be generated from the same archived run and manifest.\n\n\\section*{Data availability}\n\nThe source facility inventory was supplied under a commercial third-party licence by \\InventoryProvider{}. The authors are not permitted to redistribute provider-supplied facility identifiers, coordinates or other restricted record-level fields. Independent access to the underlying provider records requires lawful access obtained directly from Baxtel under its applicable commercial terms. Requests to the corresponding authors for non-restricted derived tables, validation outputs and methodological documentation will receive a response within two weeks. Where permitted by the licence, the authors will provide editors and referees with the applicable data-use documentation and non-restricted materials needed to audit the reported analysis.\n\nThe non-restricted materials supporting the aggregate results---including scenario configuration, aggregate figure-source tables, non-restricted spatial crosswalks, validation reports, input checksums, the configuration hash and a synthetic-input workflow---are maintained with the analysis code at \\ReviewerArchive{}. A versioned public release corresponding to the accepted manuscript will be frozen no later than publication; the identified reviewer archive provides the assessment copy. The figure-source tables permit numerical reproduction of the published figures and tables from released derived values, while the synthetic workflow tests code paths and invariants. Neither can reconstruct or validate the proprietary facility records. Full end-to-end reconstruction of the empirical facility-to-geography joins requires lawful access to the licensed Baxtel file. Public inputs are available from the U.S. Environmental Protection Agency (eGRID), World Resources Institute (Aqueduct 4.0), HydroSHEDS/HydroBASINS, WattTime and the U.S. Census Bureau.\n\n\\section*{Code availability}\n\nAnalysis code, configuration templates, validation tests, environment specifications and execution instructions are maintained at \\ReviewerArchive{}. The reviewer-access archive identified above contains the code and non-restricted outputs used for assessment; the accepted version will be preserved publicly. The public workflow excludes restricted facility records and includes a synthetic-input example.\n\n\\section*{Use of generative AI}\n\nGenerative-AI assistance was used during revision for language editing and code drafting. It was not used as a scientific data source or to determine reported numerical results. The authors inspected all code, citations, calculations, outputs and interpretations and take responsibility for the work. A generative image model was used to explore an earlier conceptual draft, which is not used in this build. Scientific maps and plots are generated programmatically from the documented analysis; Figure~\\ref{fig:design} is a deterministic text-and-equation flow chart and contains no generated imagery.\n\n\\section*{Funding and acknowledgements}\n\\FundingStatement{}\n\n\\section*{Author contributions}\n\nG.G. conceived and implemented the analysis. G.G. and F.D. designed the study, interpreted the results and wrote the manuscript. Both authors reviewed and approved the submitted version.\n\n\\section*{Competing interests}\n\n\\ConflictsStatement{}\n\n\\clearpage\n\\section*{Extended Data}\n\n\\setcounter{figure}{0}\n\\renewcommand{\\figurename}{Extended Data Fig.}\n\\renewcommand{\\theHfigure}{ED.\\arabic{figure}}\n\n\\begin{figure}[!htbp]\n\\centering\n\\safegraphic[0.95\\textwidth]{ExtendedData_Threshold_Sweep.pdf}\n\\caption{\\textbf{Context-threshold sensitivity.} High-context and joint burden--context basin counts at thresholds 2, 2.5, 3, 3.5 and 4. Both indicators use the same complete-context basin set; the burden median is defined across all positive-burden hosting basins.}\n\\label{fig:ed_threshold}\n\\end{figure}\n\n\\begin{figure}[!htbp]\n\\centering\n\\safegraphic[0.82\\textwidth]{ExtendedData_Aqueduct_BWS_vs_BWD.png}\n\\caption{\\textbf{Sensitivity to annual Aqueduct indicator.} Number of hosting basins classified as high background context and as both high burden and high context. Withdrawal-based baseline stress uses annual \\texttt{bws\\_score}; consumption-based depletion uses annual \\texttt{bwd\\_score}. The common context threshold is 3 on the 0--5 scale and the common burden threshold is the median among positive-burden hosting basins. These are descriptive classes, not impact categories.}\n\\label{fig:ed_stress}\n\\end{figure}\n\n\\begin{figure}[!htbp]\n\\centering\n\\safegraphic[0.78\\textwidth]{Figure3_Scope2_Quadrant.png}\n\\caption{\\textbf{BA grid-water intensity versus hosted-site stress.} Each point is a hosting BA. The horizontal axis is capacity-weighted annual baseline water stress at hosted data-center sites; the vertical axis is BA-average grid-water-consumption intensity. Point area represents hosted capacity and colour represents electricity-related water. Dashed lines mark Aqueduct score 3 and the median BA grid-water intensity. The plot compares an attribution intensity with a siting-context indicator; it does not locate physical generating plants or their source-water basins.}\n\\label{fig:ed_quadrant}\n\\end{figure}\n\n\n\\begin{figure}[p]\n\\centering\n\\safegraphic[0.88\\textwidth]{ExtendedData_Hydro_Ranks.pdf}\n\\caption{\\textbf{Hydropower-attribution sensitivity of hosting-BA priorities.} Electricity-related ranks under coefficients 0, 6.1 and 8.0 L kWh$^{-1}$ with fixed reference electricity and non-hydropower factors. All hosting BAs are retained; deterministic code ordering resolves equal burden for the display.}\n\\label{fig:ed_hydrorank}\n\\end{figure}\n\\begin{figure}[p]\n\\centering\n\\safegraphic[0.88\\textwidth]{ExtendedData_Climate_Sensitivity.pdf}\n\\caption{\\textbf{Conditional cooling heterogeneity.} Basin on-site water under the uniform reference and documented paired PUE/WUE cases. Points are hosting basins, not individual facilities; the diagonal indicates equality. Cases hold facility electricity fixed and do not establish installed cooling-system prevalence.}\n\\label{fig:ed_climate}\n\\end{figure}\n\n\\clearpage\n\\begin{thebibliography}{99}\n\n\\bibitem[Mytton(2021)]{mytton2021}\nD. Mytton. Data centre water consumption. \\emph{npj Clean Water} \\textbf{4}, 11 (2021). doi:10.1038/s41545-021-00101-w.\n\n\\bibitem[Lei and Masanet(2022)]{lei2022}\nN. Lei and E. Masanet. Climate- and technology-specific PUE and WUE estimations for U.S. data centers using open data. \\emph{Resources, Conservation and Recycling} \\textbf{182}, 106323 (2022). doi:10.1016/j.resconrec.2022.106323.\n\n\\bibitem[Macknick et al.(2012)]{macknick2012}\nJ. Macknick, R. Newmark, G. Heath and K. C. Hallett. Operational water consumption and withdrawal factors for electricity generating technologies: a review of existing literature. \\emph{Environmental Research Letters} \\textbf{7}, 045802 (2012). doi:10.1088/1748-9326/7/4/045802.\n\n\\bibitem[Meldrum et al.(2013)]{meldrum2013}\nJ. Meldrum, S. Nettles-Anderson, G. Heath and J. Macknick. Life cycle water use for electricity generation: a review and harmonization of literature estimates. \\emph{Environmental Research Letters} \\textbf{8}, 015031 (2013). doi:10.1088/1748-9326/8/1/015031.\n\n\\bibitem[Grubert(2016)]{grubert2016}\nE. Grubert. Water consumption from hydroelectricity in the United States. \\emph{Advances in Water Resources} \\textbf{96}, 88--94 (2016). doi:10.1016/j.advwatres.2016.07.004.\n\n\\bibitem[Cooley and Shimabuku(2026)]{cooley2026}\nH. Cooley and M. Shimabuku. \\emph{How Much Water Do Data Centers Use? And Why Does It Matter?} Pacific Institute Issue Brief (2026). \\url{https://pacinst.org/publication/how-much-water-do-data-centers-use-and-why-does-it-matter/}.\n\n\\bibitem[The Green Grid(2011)]{greengrid2011}\nThe Green Grid. \\emph{Water Usage Effectiveness (WUE): A Green Grid Data Center Sustainability Metric}. White Paper 35 (2011).\n\n\\bibitem[Siddik et al.(2021)]{siddik2021}\nM. A. B. Siddik, A. Shehabi and L. Marston. The environmental footprint of data centers in the United States. \\emph{Environmental Research Letters} \\textbf{16}, 064017 (2021). doi:10.1088/1748-9326/abfba1.\n\n\\bibitem[Chini et al.(2018)]{chini2018}\nC. M. Chini, L. A. Djehdian, W. N. Lubega and A. S. Stillwell. Virtual water transfers of the U.S. electric grid. \\emph{Nature Energy} \\textbf{3}, 1115--1123 (2018). doi:10.1038/s41560-018-0266-1.\n\n\\bibitem[Grubert and Sanders(2018)]{grubert2018}\nE. Grubert and K. T. Sanders. Water use in the United States energy system: a national assessment and unit process inventory of water consumption and withdrawals. \\emph{Environmental Science \\& Technology} \\textbf{52}, 6695--6703 (2018). doi:10.1021/acs.est.8b00139.\n\n\\bibitem[Peer et al.(2019)]{peer2019}\nR. A. M. Peer, E. Grubert and K. T. Sanders. A regional assessment of the water embedded in the U.S. electricity system. \\emph{Environmental Research Letters} \\textbf{14} (2019). doi:10.1088/1748-9326/ab2daa.\n\n\\bibitem[Baxtel(2025)]{baxtel2025}\nBaxtel. \\emph{United States data-center facility inventory: licensed snapshot delivered to Harvard, 15 December 2025} (proprietary dataset, 2025).\n\n\\bibitem[WRI and WBCSD(2015)]{ghgscope2}\nWorld Resources Institute and World Business Council for Sustainable Development. \\emph{GHG Protocol Scope 2 Guidance} (2015).\n\n\\bibitem[Shehabi et al.(2024)]{lbnl2024}\nA. Shehabi \\emph{et al.} \\emph{2024 United States Data Center Energy Usage Report}. Lawrence Berkeley National Laboratory, LBNL-2001637 (2024). doi:10.71468/P1WC7Q.\n\n\\bibitem[Kuzma et al.(2023)]{kuzma2023}\nS. Kuzma \\emph{et al.} \\emph{Aqueduct 4.0: Updated Decision-Relevant Global Water Risk Indicators}. World Resources Institute Technical Note (2023). doi:10.46830/writn.23.00061.\n\n\\bibitem[Scott et al.(2011)]{scott2011}\nC. A. Scott, S. A. Pierce, M. J. Pasqualetti, A. L. Jones, B. E. Montz and J. H. Hoover. Policy and institutional dimensions of the water-energy nexus. \\emph{Energy Policy} \\textbf{39}, 6622--6630 (2011). doi:10.1016/j.enpol.2011.08.013.\n\n\\bibitem[WattTime(2026)]{watttimeBA}\nWattTime. \\emph{Grid Region Map Geometry and balancing-authority lookup documentation}. \\url{https://legacy-docs.watttime.org/} (accessed 10 September 2026).\n\n\\bibitem[Lehner and Grill(2013)]{lehner2013}\nB. Lehner and G. Grill. Global river hydrography and network routing: baseline data and new approaches to study the world's large river systems. \\emph{Hydrological Processes} \\textbf{27}, 2171--2186 (2013). doi:10.1002/hyp.9740.\n\n\\bibitem[ISO and IEC(2022)]{iso30134}\nInternational Organization for Standardization and International Electrotechnical Commission. \\emph{ISO/IEC 30134-9: Data centres---Key performance indicators---Part 9: Water usage effectiveness (WUE)} (2022).\n\n\\bibitem[U.S. EPA(2025)]{usepaegrid2023}\nU.S. Environmental Protection Agency. \\emph{eGRID2023 Data, Revision 2} (released 12 June 2025).\n\n\\bibitem[Mekonnen and Hoekstra(2012)]{mekonnen2012}\nM. M. Mekonnen and A. Y. Hoekstra. The blue water footprint of electricity from hydropower. \\emph{Hydrology and Earth System Sciences} \\textbf{16}, 179--187 (2012). doi:10.5194/hess-16-179-2012.\n\n\\bibitem[Sjostedt et al.(2025)]{sjostedt2025}\nE. Sjostedt \\emph{et al.} Impact of thermoelectric power plant operations and water use reporting methods on thermoelectric power plant water use. \\emph{Environmental Science \\& Technology} \\textbf{59}, 4482--4492 (2025). doi:10.1021/acs.est.4c02024.\n\\end{thebibliography}\n\n\\end{document}\n", 'REVIEW_COVERAGE_AND_SIGNOFF.md': "# Review coverage and author sign-off\n\nThis checklist separates an implemented method from a completed empirical result. 'Implemented' below does not mean the national analysis was rerun in ChatGPT. No raw facility or spatial input data were available for such a run.\n\n| Review issue | Rebuilt implementation | Required evidence before submission |\n|---|---|---|\n| Named priority basins and BAs | Top-ten basin and top-five reference-BA tables plus complete aggregate CSVs | Run-specific values; exact basin-label provenance; approval to release regional data |\n| Single Aqueduct threshold | Five-threshold sweep on the paired-context basin set | Inspect counts, water coverage and dependence on missing context |\n| Whether BWD set is nested in BWS | Set intersection, subset flags, entered/exited IDs and Jaccard | Report actual set identity, not just count contraction |\n| Hydropower spatial robustness | All hosting BA amounts/ranks at 0, 6.1 and 8.0; rank and set agreement | Inspect top sets and changes; no assumption that the original three BAs persist |\n| Partition comparison | Both pathways and capacity summarized on identical BA/state partitions | Interpret these as host attribution, not physical source-water geography |\n| PJM dependence | Remove PJM and normalize to remaining regions | Keep full-fleet result primary; report denominator explicitly |\n| Climate-specific on-site information | Fully paired, complete-coverage PUE/WUE case engine and basin comparisons | Documented coefficient table and geographic assignment; no fabricated climate values |\n| Stage/operation ambiguity | Raw-label records and capacity plus optional documented classification | Verify dictionary and labels; missing status is not operating |\n| WUE consumption vs withdrawal | Input-basis validation; consumption requires fraction 1 | Verify each coefficient's numerator; 0.74 is not applied twice |\n| Unit consistency | Canonical equation checks and explicit MWh/kWh/m3/GL definitions | Validate model capacity boundary with provider evidence where possible |\n| Figures | Programmatic Figure 1; new sensitivity figures; source plotting patches | Inspect real-data layouts, annotation collisions, colorbar ticks, image provenance |\n| Citation order and wording | First-appearance bibliography ordering; reduced repeated caveats | Independently verify references, factor provenance and final conclusion wording |\n| Preprint | Research Square DOI disclosed in cover-letter template | Confirm identity and related versions |\n| Funding and conflicts | Explicit author-supplied statements, not invented 'none' | All authors confirm factual declarations |\n| Reviewer access | A concrete reviewer-access URL is required by finalization | Actually upload allowed code/outputs; test access as a reviewer |\n| Related manuscript | Author-supplied cover-letter statement | Attach current manuscript and overlap statement; verify status |\n| NC response letter | Not automatically regenerated | Update the earlier response to match completed analyses and actual figure method |\n| Reproducibility | Run identity, source hashes, output hashes, baseline checks | Local raw-data run and independent author examination |\n\n## Deliberately not inferred\n\nIntermediate provider collection counts, plant-specific coefficients, physical intake basins, installed cooling technologies, licence permissions, funding, competing interests and editorial acceptance are not inferable from rounded manuscript numbers. Missing climate coefficients and missing hydrologic labels remain explicit input requirements.\n\n## Submission gate\n\nDo not describe this delivery as submission-ready before the empirical run. Do not use synthetic smoke-test outputs in a paper. A final source ZIP is made only after the empirical checks and author confirmations. Finalization enforces workflow checks; it does not certify scientific validity, regulatory permissions, journal compliance or acceptance.\n", 'START_HERE_DELIVERY.md': "# Nature Water revision: rebuilt replacement package\n\nThis package was newly reconstructed on 21 September 2026. It is NOT a recovered byte-for-byte copy of the unavailable earlier delivery. It contains a real revision implementation and revised LaTeX templates, not merely renamed backups. The raw empirical datasets are not included.\n\n## Start here\n\nPut the unzipped folder inside the analysis project used by your previous notebook, or download the new notebook directly into that project. The notebook embeds the supporting files and creates the bundle when they are absent. Existing editable inputs are not overwritten.\n\nOpen `Nature_Water_Reviewer_Revision.ipynb` and select Restart Kernel and Run All Cells. Its original facility CSV, BA polygon, generation-mix, HydroBASINS and annual-Aqueduct paths are retained. The old analysis engine remains the source of `result['canonical']`; the new layer checks its equations and adds the review diagnostics.\n\nOutputs go to `nature_water_outputs/reviewer_revision/`. They include aggregate CSVs, generated LaTeX tables and result paragraphs, vector and raster diagnostics, a draft Overleaf project, a numerical cross-check and a run-identity record. The new notebook should be run once rather than running both the old and new notebooks in sequence.\n\n**Do not upload the template `main.tex` or `supplement.tex` alone as a finished submission.** They deliberately display a draft notice and identify missing generated inputs until the local run is complete. A successful syntax test is not verification of the empirical findings.\n\n## New input tables\n\nThe header-only files under `inputs/` contain no invented coefficients or names.\n\n- `climate_pairs.csv`: one row per analytic facility and cooling case. Required fields: `facility_id`, `case`, paired `pue` and `wue_L_per_kWh_IT`, `water_basis` (`consumption` or `withdrawal`), `consumptive_fraction`, `coefficient_source`, and `assignment_source`. Every case requires full coverage. For consumption-based WUE, the fraction must be exactly 1. A withdrawal-based fraction must be documented. Coefficients and assignments must be sourced; the code does not fetch, invent or extrapolate a Lei-Masanet dataset. Keep this table private because it contains facility identifiers.\n- `basin_labels.csv`: exact PFAF ID, recognizable hosting-basin label and label provenance. Never identify a utility intake basin from a facility point or a nearby river name. The code exports IDs and hosted-state context even before this crosswalk is completed.\n- `stage_mapping.csv`: exact provider stage label, a reviewed class (use `operating` for that class) and mapping provenance. Unknown labels remain unknown. No operating status is inferred from positive current capacity.\n\nThese inputs are optional for generating an exploratory draft but required, where relevant, for the final all-review-points build. An absent climate table produces a clear 'not run' status; it never produces a fabricated result.\n\n## What is implemented\n\nNamed regional tables; BWS/BWD threshold sweeps at 2, 2.5, 3, 3.5 and 4; complete-context coverage; set inclusion and overlap; hydropower amounts and ranks at 0, 6.1 and 8.0 L/kWh; same-partition Gini/HHI and minimum half-total sets; without-PJM diagnostics; raw stage coverage and explicit stage classification; paired cooling sensitivities; exact-result tables and prose; and a genuinely programmatic Figure 1. The previous plot code is retained with sparse colorbar ticks and reduced annotation collisions; actual empirical layouts still need visual review.\n\nThe new layer writes regional or national outputs only. A region with few records can still permit inference: aggregate release permissions and disclosure risk require author review before sharing.\n\n## Finalizing the local build\n\n1. Inspect `diagnostics/manuscript_number_checks.csv`. It checks the manuscript's retained baseline numbers, not just equation identities. A failed check must be resolved from the data/version history, not bypassed to keep familiar numbers.\n2. Inspect every generated table and figure and compile the draft manuscript and supplement. Check the new narrative against the outputs. The climate inputs, basin naming, stage mapping and source literature require scientific author review.\n3. Copy `author_signoff.template.json` to `author_signoff.json` in the output folder. Enter factual funding, competing-interest, related-manuscript and verified reviewer-access statements, and make only the confirmations actually supported. The code does not create or upload a repository, grant a licence or issue a DOI.\n4. Use the notebook's finalization cell. Finalization refuses synthetic runs, mismatching baseline numbers, missing climate results, missing basin labels, unmapped stage labels, changed outputs, changed source inputs or an unmatched sign-off identity.\n5. It creates `Nature_Water_Overleaf_after_local_verification.zip` containing the source, generated tables and figures. Compile each document separately in Overleaf with pdfLaTeX. The package does not submit anything to the journal. Review the final PDFs before upload.\n\nThe preliminary cover-letter template is included as a convenience. The existing Nature Communications response letter must also be aligned with the actual new results and the final Figure 1 production method; it is not automatically rewritten by this package.\n\n## Tests and limitations\n\nRun `python -m unittest discover -s tests -v` from the bundle. Tests use explicitly synthetic fixtures, verify equations and rejection behavior, and cannot validate Baxtel data, spatial joins or publication eligibility. A local run on the licensed dataset, coefficient provenance checks, funding/conflict confirmation and actual reviewer access remain necessary.\n\nThe code uses the supplied reference bibliography and model conventions. It is not a fresh bibliographic or journal-policy verification. In particular, each literature-derived coefficient needs an exact source and water-accounting basis. The wider plant/cooling electricity model, electricity-transfer tracing and eGRID-subregion rerun are not implemented here.\n", 'requirements.txt': 'numpy\npandas\nscipy\nmatplotlib\ngeopandas\nshapely\npyyaml\nopenpyxl\nnbformat\nipykernel\n', 'cover_letter.tex': "\\documentclass[11pt]{letter}\n\\usepackage[margin=1in]{geometry}\n\\usepackage[T1]{fontenc}\n\\usepackage[utf8]{inputenc}\n\\usepackage{lmodern}\n\\usepackage{lineno}\n\\usepackage[hidelinks]{hyperref}\n\n% REBUILT revision template. Finalization switches this flag after the local run.\n\\newif\\ifNWFinal\n\\NWFinalfalse\n\\newcommand{\\NWpending}[1]{\\par\\smallskip\\noindent\\textbf{DRAFT -- #1}\\par\\smallskip}\n\\newcommand{\\NWinput}[1]{\\IfFileExists{#1}{\\input{#1}}{\\ifNWFinal\n  \\PackageError{NWrevision}{Missing generated file #1}{Run the revision notebook before finalization.}\n  \\else\\NWpending{Generated results not yet available: \\texttt{\\detokenize{#1}}}\\fi}}\n\\newcommand{\\FundingStatement}{\\NWpending{Author funding statement required.}}\n\\newcommand{\\ConflictsStatement}{\\NWpending{Author competing-interest confirmation required.}}\n\\newcommand{\\RelatedManuscriptStatement}{\\NWpending{Related manuscript status and supplied copy require author confirmation.}}\n\\newcommand{\\ReviewerArchive}{\\textbf{DRAFT: verified reviewer-access archive required}}\n\\IfFileExists{generated/author_statements.tex}{\\input{generated/author_statements.tex}}{}\n\\AtBeginDocument{\\ifNWFinal\\else\n\\begin{center}\\fbox{\\parbox{0.90\\textwidth}{\\textbf{REBUILT REVISION DRAFT -- NOT FOR SUBMISSION.}\nThe new regional and sensitivity outputs require the local empirical run and author sign-off.\nAny retained baseline numbers are checked against that run before finalization.}}\n\\end{center}\\fi}\n\n\\signature{Gianluca Guidi and Francesca Dominici\\\\Harvard T.H. Chan School of Public Health}\n\\address{Harvard T.H. Chan School of Public Health\\\\Boston, Massachusetts, USA}\n\\date{\\today}\n\\begin{document}\n\\begin{letter}{Editors\\\\\\emph{Nature Water}}\n\\opening{Dear Editors,}\nWe submit ``Pathway and indicator choices alter water-screening priorities for U.S. hyperscale data centers\\,'' for consideration as an Analysis. The study separates the hydrologic hosting context of on-site water from electricity-system attribution, identifies regional priorities explicitly, and evaluates their dependence on context thresholds, hydropower allocation and documented paired cooling coefficients.\n\nThis is a revised transfer following peer review at \\emph{Nature Communications}. The corrected workflow derives national and regional displays from one validated facility-by-scenario table. New analyses examine named priority regions, threshold and set agreement, hydropower reranking, common-partition concentration and dependence on PJM. Plant/cooling-system-resolved electricity attribution and a BA-versus-eGRID-subregion rerun remain outside this study and are not claimed as completed.\n\n\\textbf{Preprint.} An earlier version was posted on Research Square, DOI \\url{https://doi.org/10.21203/rs.3.rs-10021458/v1}. The present version incorporates the revisions described in the accompanying response.\n\n\\textbf{Data and code.} The facility inventory is licensed from Baxtel. The non-restricted reviewer materials and execution instructions are available at \\ReviewerArchive{}. The provider's redistribution restrictions are explained in the manuscript.\n\n\\textbf{Related work.} \\RelatedManuscriptStatement{}\n\n\\textbf{Author declarations.} Funding: \\FundingStatement{} Competing interests: \\ConflictsStatement{}\n\nThank you for considering the manuscript.\n\\closing{Sincerely,}\n\\end{letter}\n\\end{document}\n", 'supplement.tex': "\\documentclass[11pt]{article}\n\n\\usepackage[margin=1in]{geometry}\n\\usepackage[T1]{fontenc}\n\\usepackage[utf8]{inputenc}\n\\usepackage{lmodern}\n\\usepackage{amsmath,amssymb}\n\\usepackage[numbers,sort&compress]{natbib}\n\\usepackage{booktabs,longtable,tabularx,array,adjustbox}\n\\usepackage{xcolor}\n\\usepackage[hidelinks]{hyperref}\n\\usepackage[switch]{lineno}\n\\usepackage{microtype}\n\n\\newcommand{\\GLyr}{\\ensuremath{\\mathrm{GL\\,yr^{-1}}}}\n\\newcommand{\\TWhyr}{\\ensuremath{\\mathrm{TWh\\,yr^{-1}}}}\n\\newcommand{\\LkWh}{\\ensuremath{\\mathrm{L\\,kWh^{-1}}}}\n\\newcommand{\\mthree}{\\ensuremath{\\mathrm{m^3}}}\n\\newcommand{\\InventoryProvider}{Baxtel}\n\\newcommand{\\GridDataset}{EPA eGRID2023 Revision 2}\n\\newcolumntype{Y}{>{\\raggedright\\arraybackslash}X}\n\\newcolumntype{C}{>{\\centering\\arraybackslash}X}\n\n\\renewcommand{\\theequation}{S\\arabic{equation}}\n\\renewcommand{\\thetable}{S\\arabic{table}}\n\\renewcommand{\\thefigure}{S\\arabic{figure}}\n\n\\title{Supplementary Information for\\\\\n\\textit{Pathway and indicator choices alter water-screening priorities for U.S. hyperscale data centers}}\n\\author{Gianluca Guidi$^{1,*}$ and Francesca Dominici$^{1,*}$}\n\\date{}\n\n\n% REBUILT revision template. Finalization switches this flag after the local run.\n\\newif\\ifNWFinal\n\\NWFinalfalse\n\\newcommand{\\NWpending}[1]{\\par\\smallskip\\noindent\\textbf{DRAFT -- #1}\\par\\smallskip}\n\\newcommand{\\NWinput}[1]{\\IfFileExists{#1}{\\input{#1}}{\\ifNWFinal\n  \\PackageError{NWrevision}{Missing generated file #1}{Run the revision notebook before finalization.}\n  \\else\\NWpending{Generated results not yet available: \\texttt{\\detokenize{#1}}}\\fi}}\n\\newcommand{\\FundingStatement}{\\NWpending{Author funding statement required.}}\n\\newcommand{\\ConflictsStatement}{\\NWpending{Author competing-interest confirmation required.}}\n\\newcommand{\\RelatedManuscriptStatement}{\\NWpending{Related manuscript status and supplied copy require author confirmation.}}\n\\newcommand{\\ReviewerArchive}{\\textbf{DRAFT: verified reviewer-access archive required}}\n\\IfFileExists{generated/author_statements.tex}{\\input{generated/author_statements.tex}}{}\n\\AtBeginDocument{\\ifNWFinal\\else\n\\begin{center}\\fbox{\\parbox{0.90\\textwidth}{\\textbf{REBUILT REVISION DRAFT -- NOT FOR SUBMISSION.}\nThe new regional and sensitivity outputs require the local empirical run and author sign-off.\nAny retained baseline numbers are checked against that run before finalization.}}\n\\end{center}\\fi}\n\n\\begin{document}\n\\maketitle\n\n\\noindent$^{1}$Harvard T.H. Chan School of Public Health, Harvard University, Boston, MA, USA.\\\\\n$^{*}$Correspondence to: ggianluca@hsph.harvard.edu; fdominic@hsph.harvard.edu.\n\n\\linenumbers\n\n\\section*{Overview}\n\nThis Supplementary Information documents the derivation, parameterization, spatial-data rules, sensitivity definitions, validation contracts and result tables underlying the main \\emph{Analysis}. The study asks how pathway and indicator choices alter the places prioritized by a national screen of U.S. hyperscale data-center water. It estimates annualized operational-water consumption for a 15 December 2025 reported-current-capacity snapshot of 472 facility records, retaining WUE-derived on-site water and water attributed to electricity supply as separate pathways.\n\nAll national and regional products originate from one canonical facility-by-scenario table. Regional values are generated directly as machine-readable outputs rather than transcribed into reporting tables. The Supplement distinguishes modeled volume, hosting-basin context and hosting-BA attribution; it also separates completed analyses from extensions that were not executed for the submitted real-data run.\n\n\\section{Evidentiary boundary}\n\nThe analysis combines third-party facility records with public hydrologic and electricity-system data. A ``priority'' in this study means a region contributing a large modeled pathway share or meeting a named contextual threshold; it is not a measured ecological impact, regulatory designation or engineering recommendation. Table~\\ref{tab:evidence_boundary} states the strongest interpretation supported by each layer.\n\n\\begin{table}[t]\n\\centering\n\\small\n\\begin{tabularx}{\\textwidth}{@{}p{0.18\\textwidth}Y Y Y@{}}\n\\toprule\n\\textbf{Layer} & \\textbf{Input or modeled quantity} & \\textbf{Interpretation supported} & \\textbf{Not established} \\\\\n\\midrule\nFacility inventory & Provider-defined facility record, coordinates and reported \\texttt{current\\_mw} & Dated current-capacity cross-section under explicit inclusion rules & Metered load, fully operational status, or independent campus/building re-segmentation \\\\\nOn-site pathway & IT electricity multiplied by WUE and consumptive fraction & Annualized screening estimate at the facility and hosting-basin aggregation & Utility withdrawal, intake basin, cooling system, return flow or metered consumption \\\\\nElectricity pathway & Facility-input electricity multiplied by BA-average resource shares and class-average factors & Hosting-BA annual production-mix attribution under a specified factor set & Electron tracing, plant dispatch, physical plant/source basin or market-based portfolio account \\\\\nAqueduct context & Annual BWS or BWD score for the hosting HydroBASINS level-6 polygon & Chronic background context under the named indicator and threshold & Seasonal drought coincidence, local ecological damage or scarcity-weighted impact \\\\\nBenchmarks & Difference from fixed-facility-electricity on-site or empirical grid-intensity references & Transparent comparative gap under stated assumptions & Feasibility, achievable savings, cost, policy authority or forecast \\\\\n\\bottomrule\n\\end{tabularx}\n\\caption{Evidentiary boundary. Each output is interpreted only within the data and attribution rule that produced it.}\n\\label{tab:evidence_boundary}\n\\end{table}\n\n\\section{Study population and estimand}\n\n\\subsection{Inventory snapshot and operational definition of hyperscale}\n\nThe source inventory was a licensed \\InventoryProvider{} facility file delivered to Harvard and frozen on 15 December 2025 \\citep{baxtel2025}. ``Hyperscale'' is the provider's company-type classification rather than a universal power, floor-area or server-count threshold. Records were eligible when the company-type field matched a case-insensitive, whole-field expression for \\texttt{hyperscale}, \\texttt{hyperscaler} or \\texttt{hyper scale}; composite colocation labels were not included by substring matching; a unique provider record identifier, valid geographic coordinates and positive \\texttt{current\\_mw} were present; and the assigned state was in the contiguous United States. Alaska, Hawaii and U.S. territories were excluded. The final analytic population contained 472 records with 20,041 MW.\n\nThe source unit is the provider-defined record. We enforced identifier uniqueness but did not independently merge or redefine campus and building records. The analysis therefore inherits the provider's facility segmentation and possible dependence among co-located records. Spatial proximity alone was not treated as evidence of duplicate electrical capacity, and records were not merged solely because their coordinates were identical or nearby. The provider also performed data collection and classification steps outside the authors' control; the public release documents the fields received, the filters applied and the authors' validation contracts, but cannot independently reconstruct the provider's source collection process.\n\nThe analysis is a reported-current-capacity cross-section, not a census of metered operating load. We interpret \\texttt{current\\_mw} as facility-input nameplate power for modeling. Announced, construction and planned capacity fields are not added. Stage labels were insufficiently complete and standardized for a defensible operating-only national subset, so the primary analysis retains all eligible positive-current-capacity records and does not infer operating status from facility age or free text. The phrase ``operational water'' refers to the operational life-cycle boundary of the modeled water, not proof that every record was fully operating at the snapshot. The provider capacity boundary is not independently metered by the authors; Section~\\ref{sec:capacity_boundary} reports the exact conditional scaling if the field were instead interpreted as IT capacity.\n\n\\begin{table}[t]\n\\centering\n\\small\n\\begin{tabularx}{\\textwidth}{@{}p{0.21\\textwidth}Y Y@{}}\n\\toprule\n\\textbf{Inventory element} & \\textbf{Rule used} & \\textbf{Interpretive consequence} \\\\\n\\midrule\nProvider and snapshot & Licensed \\InventoryProvider{} file frozen 15 December 2025 & Provider identity and access restrictions must be disclosed; estimates refer to this dated cross-section \\\\\nRecord unit & Unique provider facility identifier; no independent campus/building regrouping & Results inherit provider segmentation and possible dependence among co-located records \\\\\nHyperscale classification & Provider company-type label matching the configured expression & Population should not be generalized to enterprise, colocation or edge facilities outside that classification \\\\\nCapacity & Positive \\texttt{current\\_mw}, interpreted as facility-input nameplate MW & Utilization converts the reported capacity to annual electricity; this interpretation is an explicit model assumption \\\\\nGeography & Valid coordinates and contiguous-U.S. state assignment & Alaska, Hawaii and territories are outside the estimand \\\\\nStage & All eligible positive-current-capacity rows; no imputed operational status & Results are a current-capacity scenario, not a metered operating-only census \\\\\nPlanned capacity & Separate announced/construction/planned fields excluded & Study is not a forecast of announced development \\\\\nFinal population & 472 records; 20,041 MW & These denominators apply to every primary scenario \\\\\n\\bottomrule\n\\end{tabularx}\n\\caption{Inventory definition, eligibility rules and inherited limitations.}\n\\label{tab:inventory}\n\\end{table}\n\nThe analytic sample statistics describe the records available for this analysis. Provider collection-stage counts cannot be inferred from the final denominator. The revision exports the actual author-side filtering diagnostics separately when the original pipeline supplies them, and reports the complete descriptive stage distribution below.\n\\NWinput{generated/stages.tex}\n\n\n\\subsection{Annualized scenario estimand}\n\nThe estimand is annualized operational-water consumption under a specified utilization, PUE, WUE, on-site consumptive fraction and generation-water factor set. It is not a sum of observed utility bills. Every scenario applies one national parameter set to all records, enabling transparent comparisons while avoiding false facility-level precision.\n\nUtilization and PUE describe different quantities. Utilization, $u_s$, is the assumed ratio of annual-average facility-input power to the reported capacity interpreted as nameplate power. PUE is total facility electricity divided by IT-equipment electricity. Thus, utilization determines annual facility electricity, whereas PUE partitions that electricity between IT equipment and supporting infrastructure. A uniform $u_s$ collapses both heterogeneity across facilities and maturation within a facility; it must not be interpreted as a measured load factor.\n\nThe labels ``lower-load/lower-water'' and ``higher-load/water-intensive'' are used because those cases jointly change utilization, PUE and WUE. They do not isolate a single efficiency effect and are not probability bounds, confidence intervals or empirical quantiles.\n\n\\subsection{Conditional capacity-boundary sensitivity}\\label{sec:capacity_boundary}\n\nThe primary equations interpret the provider's \\texttt{current\\_mw} field as total facility-input nameplate power. Because the authors do not have facility meters that independently verify this boundary, we also state the algebraic consequence of the alternative interpretation that the same field represents IT nameplate power. Under that alternative, total facility power is $P_i\\mathrm{PUE}_s$, IT electricity is $P_i\\times8760\\times u_s$, and both modeled pathways are multiplied by the scenario PUE relative to the primary calculation. For the baseline/reference PUE of 1.25, the conditional values would therefore be 144.84 TWh yr$^{-1}$ of facility electricity, 92.70 GL yr$^{-1}$ on site, 282.18 GL yr$^{-1}$ electricity related and 374.86 GL yr$^{-1}$ in total. Because the same positive factor multiplies both pathways and every record within a scenario, pathway shares, host-region ranks and concentration curves are unchanged. This calculation does not determine which capacity boundary is correct; that question must be resolved from the provider data dictionary or facility evidence.\n\n\\section{Accounting boundary and pathway terminology}\n\n\\subsection{Consumption, site water use and withdrawals}\n\nThe reported outcome is operational water \\emph{consumption}: water treated as removed from immediate local availability, principally through evaporation. Gross withdrawals are not estimated. WUE is a site-water-use metric per unit of IT electricity under ISO/IEC 30134-9 and its open methodological precursors \\citep{iso30134,greengrid2011}. To translate WUE-reported site water into the modeled consumptive outcome, the equations include an explicit on-site consumptive fraction $c_s$.\n\nThe primary WUE settings are author-selected site-water-use scenarios rather than measured or extracted LBNL coefficients. Setting $c_s=1$ assigns all modeled site water to consumption. For consumption-based WUE, $c_s=1$ is required, not an additional conservative correction; values below one apply only to an explicitly withdrawal-based WUE. This is not a claim that every litre entering a cooling system is physically consumed. Because $W^{(1)}$ is linear in $c_s$, any common value below one would reduce all on-site volumes proportionally, leave within-pathway basin ranks unchanged, and increase the electricity-related share under the same grid-factor set.\n\n\\subsection{Relationship to greenhouse-gas scopes}\n\nThe GHG Protocol defines Scope 2 greenhouse-gas emissions to include acquired electricity, steam, heat and cooling \\citep{ghgscope2}. This study quantifies only a hosting-BA production-mix electricity attribution. It is not presented as formal GHG Protocol location-based accounting, for which eGRID subregions or utility-specific information are conventional reporting bases. District energy, purchased cooling, combined heat and power, export or reuse of waste heat and behind-the-meter generation require facility-specific flow data and are outside the boundary. The physical labels on-site and electricity-related are therefore preferred.\n\nNo market-based water result is presented. A defensible market-based account would require contract- and generator-specific electricity quantities, temporal matching, attribute ownership, treatment of unbundled certificates and a residual-mix intensity for unmatched consumption. Public renewable-energy claims or PPA percentages alone are not converted to zero water. Water embodied in construction, servers, semiconductors, fuels and other supply-chain activities is excluded; the result is not a complete organizational or life-cycle water footprint.\n\n\\begin{table}[t]\n\\centering\n\\small\n\\begin{tabularx}{\\textwidth}{@{}p{0.16\\textwidth}Y Y Y@{}}\n\\toprule\n\\textbf{Pathway} & \\textbf{Modeled quantity} & \\textbf{Analytical geography and relevant follow-up} & \\textbf{What the result does not establish} \\\\\n\\midrule\nOn-site & WUE-derived site water multiplied by $c_s$; primary $c_s=1$ & Facility-hosting HydroBASINS level-6 basin; cooling architecture, water source, permits, reclaimed water and drought planning & Verified utility intake basin, withdrawal volume, cooling system or metered facility consumption \\\\\nElectricity-related & Facility-input electricity multiplied by BA-average operational generation-water intensity & Hosting BA attribution; follow-up may involve utilities, generators, system operators, regulators, procurement and load management & Physical generating plant, generator source basin, electron tracing or contract-specific market-based water \\\\\nExcluded value chain & Not quantified & Construction and equipment supply chains would require product- and supplier-resolved life-cycle data & A complete organizational water footprint or Scope 3 analogue \\\\\n\\bottomrule\n\\end{tabularx}\n\\caption{Accounting boundary, analytical geography and interpretation of the two operational pathways.}\n\\label{tab:pathways}\n\\end{table}\n\n\\section{Water-accounting model}\n\n\\subsection{Facility electricity}\n\nFor facility $i$ and scenario $s$, annual facility-input electricity is\n\\begin{equation}\nE^{\\mathrm{fac}}_{i,s}=P_i\\times8760\\times u_s,\n\\label{eq:facility_energy}\n\\end{equation}\nwhere $P_i$ is facility-input nameplate power in MW, 8760 is the number of hours in a non-leap year and $u_s$ is dimensionless utilization. Equation~\\ref{eq:facility_energy} yields MWh yr$^{-1}$. Before applying intensities in L kWh$^{-1}$, energy is converted explicitly:\n\\begin{equation}\nE^{\\mathrm{fac,kWh}}_{i,s}=1000E^{\\mathrm{fac}}_{i,s}.\n\\label{eq:kwh}\n\\end{equation}\nBecause $P_i$ is interpreted as facility-input rather than IT-only power, multiplying equation~\\ref{eq:facility_energy} by PUE would double count non-IT overhead. Instead, IT electricity is recovered as\n\\begin{equation}\nE^{\\mathrm{IT,kWh}}_{i,s}=\\frac{E^{\\mathrm{fac,kWh}}_{i,s}}{\\mathrm{PUE}_s}.\n\\label{eq:it_energy}\n\\end{equation}\n\n\\subsection{On-site operational consumption}\n\nOn-site operational consumption is\n\\begin{equation}\nW^{(1)}_{i,s}=10^{-3}E^{\\mathrm{IT,kWh}}_{i,s}\\,\\mathrm{WUE}_s\\,c_s\n=10^{-3}E^{\\mathrm{fac,kWh}}_{i,s}\\frac{\\mathrm{WUE}_s c_s}{\\mathrm{PUE}_s},\n\\label{eq:onsite}\n\\end{equation}\nwhere WUE is in litres of site water per kWh of IT electricity, $c_s$ is the assumed consumptive fraction and $10^{-3}$ converts litres to cubic metres. Evaporative cooling can lower electricity demand for heat rejection but consumes water when liquid water is evaporated to carry heat away; cycles of concentration, blowdown, climate and water-treatment practice also affect site water requirements \\citep{mytton2021,lei2022,epa2012}. The scenario WUE values therefore describe system-level operating regimes rather than a claim that all facilities use the same cooling architecture.\n\n\\subsection{Electricity-related operational consumption}\n\nEach facility is assigned the average annual generation shares $s_{r,f}$ of hosting BA $r$ from the \\GridDataset{} BA-level resource-mix table \\citep{usepaegrid2023}. The seven mutually exclusive classes are coal, natural gas, nuclear, hydropower, wind, solar and residual ``other'' generation. The residual class combines the source-table oil, biomass, geothermal, other-fossil and other-unknown/purchased-fuel shares that are not parameterized separately. Shares must be finite and non-negative and sum to one after only small rounding correction. The resource mix represents calendar-year 2023 generation, not an import-adjusted consumption mix, and is temporally offset from the December 2025 facility snapshot.\n\nBA grid-water intensity is\n\\begin{equation}\nI_{r,s}=\\sum_f s_{r,f}w_{f,s},\n\\label{eq:grid_intensity}\n\\end{equation}\nwhere $w_{f,s}$ is an operational water-consumption factor in L kWh$^{-1}$. Facility electricity-related consumption is\n\\begin{equation}\nW^{(2)}_{i,s}=10^{-3}E^{\\mathrm{fac,kWh}}_{i,s}I_{r(i),s}.\n\\label{eq:electricity_water}\n\\end{equation}\nWind, solar and hydropower shares are not omitted. The non-hydropower factors are selected screening values from harmonized operational-factor literature \\citep{macknick2012,meldrum2013}; they are not estimates for the sampled fleet or for an identified current plant, prime mover, cooling system or water source. The 8.0 L kWh$^{-1}$ hydropower value is a manuscript-selected reservoir-evaporation attribution convention, not a claimed U.S. national best estimate; it is isolated against a zero-attribution endpoint \\citep{grubert2016,mekonnen2012}. The 1.0 L kWh$^{-1}$ residual-``other'' value is a simplifying assumption applied to heterogeneous source categories. Plant configuration, operating conditions and reporting method generate substantial within-class variation \\citep{sjostedt2025}. A plant/cooling-system-resolved sensitivity and source-documented probability distribution were not executed for the submitted real-data run. Accordingly, BA-level values are interpreted as factor-set-dependent hosting-BA production-mix attributions rather than plant-level water estimates.\n\n\\subsection{Total and pathway separation}\n\nThe national operational total is\n\\begin{equation}\nW^{\\mathrm{tot}}_{i,s}=W^{(1)}_{i,s}+W^{(2)}_{i,s}.\n\\label{eq:total}\n\\end{equation}\nThe two terms are added for national accounting and explicitly labelled host-location profiles, not to relocate electricity-generation water to the facility basin. They remain separate in spatial outputs because a facility-hosting basin is not necessarily the facility's source-water basin, and a BA is an attribution region rather than a physical generation-water basin.\n\n\\begin{table}[t]\n\\centering\n\\small\n\\begin{tabularx}{\\textwidth}{@{}p{0.20\\textwidth}p{0.18\\textwidth}Y p{0.20\\textwidth}@{}}\n\\toprule\n\\textbf{Quantity} & \\textbf{Units} & \\textbf{Operation} & \\textbf{Resulting units} \\\\\n\\midrule\n$P_i\\times8760\\times u_s$ & MW $\\times$ h yr$^{-1}$ & Utilization is dimensionless & MWh yr$^{-1}$ \\\\\n$1000E^{\\mathrm{fac}}$ & 1000 kWh MWh$^{-1}$ & Explicit MWh-to-kWh conversion & kWh yr$^{-1}$ \\\\\n$E^{\\mathrm{fac,kWh}}/\\mathrm{PUE}$ & kWh yr$^{-1}$ & PUE is dimensionless & IT kWh yr$^{-1}$ \\\\\n$E^{\\mathrm{IT,kWh}}\\times\\mathrm{WUE}\\times c_s$ & kWh yr$^{-1}$ $\\times$ L kWh$^{-1}$ & $c_s$ is dimensionless & L yr$^{-1}$ \\\\\n$\\sum_f s_{r,f}w_f$ & dimensionless $\\times$ L kWh$^{-1}$ & Generation shares sum to one & L kWh$^{-1}$ \\\\\n$E^{\\mathrm{fac,kWh}}\\times I_r$ & kWh yr$^{-1}$ $\\times$ L kWh$^{-1}$ & Electricity-related pathway & L yr$^{-1}$ \\\\\n$10^{-3}\\times$ litres & $10^{-3}$ m$^3$ L$^{-1}$ & Volume conversion & m$^3$ yr$^{-1}$ \\\\\n$10^{-6}\\times$ cubic metres & $10^{-6}$ GL m$^{-3}$ & Reporting conversion & GL yr$^{-1}$ \\\\\n\\bottomrule\n\\end{tabularx}\n\\caption{Dimensional audit of the accounting equations.}\n\\label{tab:dimensions}\n\\end{table}\n\n\\section{Scenario construction and parameterization}\n\n\\subsection{Operating scenarios}\n\nThe lower-load/lower-water, baseline and higher-load/water-intensive cases jointly vary utilization, PUE and WUE. They are deterministic cases rather than statistical confidence bounds or empirical quantiles. The no-hydropower-attribution case retains baseline activity and cooling parameters and changes only the hydropower factor. Rows are mutually exclusive and must not be summed.\n\n\\begin{table}[t]\n\\centering\n\\small\n\\begin{tabularx}{\\textwidth}{@{}p{0.27\\textwidth}CCCCC@{}}\n\\toprule\n\\textbf{Scenario} & $u_s$ & \\textbf{PUE} & \\textbf{WUE} & $c_s$ & $w_{\\mathrm{hydro}}$ \\\\\n & & & \\textbf{(L kWh$^{-1}_{\\mathrm{IT}}$)} & & \\textbf{(L kWh$^{-1}$)} \\\\\n\\midrule\nLower-load/lower-water & 0.55 & 1.15 & 0.20 & 1.00 & 8.00 \\\\\nBaseline/reference & 0.66 & 1.25 & 0.80 & 1.00 & 8.00 \\\\\nHigher-load/water-intensive & 0.85 & 1.40 & 1.50 & 1.00 & 8.00 \\\\\nNo hydropower attribution & 0.66 & 1.25 & 0.80 & 1.00 & 0.00 \\\\\n\\bottomrule\n\\end{tabularx}\n\\caption{Scenario parameters. WUE is defined per unit IT electricity; generation-water factors are defined per unit facility electricity attributed through the BA mix. The first and third rows jointly change activity and cooling assumptions.}\n\\label{tab:scenarios}\n\\end{table}\n\nThe PUE and WUE settings span deliberately broad, literature-informed operating regimes \\citep{mytton2021,lei2022,lbnl2024}. They are not estimated percentiles of the sampled facilities because nationally consistent facility-level measurements are unavailable. The 0.80 L kWh$^{-1}_{\\mathrm{IT}}$ reference WUE is intentionally conservative for on-site volume relative to the approximately 0.32--0.40 L kWh$^{-1}_{\\mathrm{IT}}$ aggregate hyperscale values reported in recent LBNL scenarios; it is not a measured fleet average. Common national parameters preserve within-pathway rank order induced by capacity and geography unless a pathway-specific factor changes.\n\n\\begin{table}[t]\n\\centering\n\\footnotesize\n\\begin{tabularx}{\\textwidth}{@{}p{0.18\\textwidth}Y Y@{}}\n\\toprule\n\\textbf{Setting} & \\textbf{Evidence and analytical role} & \\textbf{What is not claimed} \\\\\n\\midrule\n$u=0.55,0.66,0.85$ & Author-selected annual load-scaling cases spanning lower, reference and higher utilization; applied uniformly to reported capacity & Not measured facility load factors, an empirical distribution or a ramp-up model \\\\\nPUE$=1.15,1.25,1.40$ & Broad range from highly efficient to higher-overhead operation; recent aggregate hyperscale/colocation values are generally below 1.4 \\citep{lbnl2024} & Not paired facility observations and not an independent causal cooling intervention \\\\\nWUE$=0.20$ & Lower-water case; values around 0.1--0.3 L kWh$^{-1}_{\\mathrm{IT}}$ are reported for some hyperscale systems \\citep{lbnl2024} & Not a national lower confidence limit \\\\\nWUE$=0.80$ & Conservative reference for on-site volume, above recent LBNL aggregate hyperscale scenarios of approximately 0.32--0.40 L kWh$^{-1}_{\\mathrm{IT}}$ \\citep{lbnl2024} & Not a fleet mean, median or maximum-likelihood estimate \\\\\nWUE$=1.50$ & Water-intensive stress case informed by broad climate and cooling-technology variation \\citep{mytton2021,lei2022} & Not an upper quantile or probability bound \\\\\n$c=1$ & Conservative primary translation of WUE-reported site water to consumption when return-flow data are unavailable & Not a claim that every litre entering a facility is consumed \\\\\n$c=0.74$ & Illustrative sensitivity using the typical withdrawal-to-consumption fraction summarized by the Pacific Institute \\citep{cooley2026} & Not a facility-specific estimate or a direct calibration of WUE-reported site water \\\\\n\\bottomrule\n\\end{tabularx}\n\\caption{Evidence and non-claims for the common scenario settings. The scenarios are structured cases, not a fitted uncertainty distribution.}\n\\label{tab:parameter_evidence}\n\\end{table}\n\n\\subsection{Parameter separability and interpretation}\n\nFor a fixed generation mix, the electricity-related pathway scales as $W^{(2)}\\propto u$, whereas the on-site pathway scales as\n\\begin{equation}\nW^{(1)}\\propto u\\,\\frac{\\mathrm{WUE}}{\\mathrm{PUE}}\\,c.\n\\label{eq:separability}\n\\end{equation}\nThe joint scenarios therefore cannot identify which single parameter caused a change in the national total. Fixed-facility-electricity comparisons in Section~\\ref{sec:fixed_activity} are used only to display pathway-specific reference differences. The local elasticities are exact: $\\partial\\ln W^{(1)}/\\partial\\ln u=1$, $\\partial\\ln W^{(1)}/\\partial\\ln\\mathrm{WUE}=1$, $\\partial\\ln W^{(1)}/\\partial\\ln c=1$ and $\\partial\\ln W^{(1)}/\\partial\\ln\\mathrm{PUE}=-1$; for a fixed regional intensity, $\\partial\\ln W^{(2)}/\\partial\\ln u=1$. These identities make clear which uncertainties rescale a pathway and which can change regional ordering.\n\n\\begin{table}[t]\n\\centering\n\\small\n\\begin{tabularx}{0.90\\textwidth}{@{}Y C C@{}}\n\\toprule\n\\textbf{One-at-a-time setting at reference facility electricity} & \\textbf{On-site (GL yr$^{-1}$)} & \\textbf{Total (GL yr$^{-1}$)} \\\\\n\\midrule\nReference: WUE 0.80, PUE 1.25, $c=1$ & 74.2 & 299.9 \\\\\nWUE 0.20 & 18.5 & 244.3 \\\\\nWUE 0.40 & 37.1 & 262.8 \\\\\nWUE 1.50 & 139.1 & 364.8 \\\\\n$c=0.74$ & 54.9 & 280.6 \\\\\n$c=0.50$ & 37.1 & 262.8 \\\\\nPUE 1.15 & 80.6 & 306.3 \\\\\nPUE 1.40 & 66.2 & 292.0 \\\\\n\\bottomrule\n\\end{tabularx}\n\\caption{Analytical one-at-a-time sensitivity of the on-site pathway. Electricity-related consumption is held at 225.74 GL yr$^{-1}$. Values are rounded rescalings of the reference equations, not additional observations. The $c=0.74$ row is an illustrative withdrawal-basis comparison informed by the Pacific Institute synthesis; it is not applicable when WUE already measures consumption and does not calibrate the reference WUE. The PUE direction is conditional on fixed facility-input electricity: a higher PUE implies less IT electricity in equation~\\ref{eq:it_energy}; this is not an engineering water-saving claim.}\n\\label{tab:one_at_a_time}\n\\end{table}\n\nAt the displayed precision, assigning zero reservoir evaporation to hydropower lowers the baseline electricity-related estimate by 97.39 GL yr$^{-1}$, or 43.1\\% of its baseline value. This large contribution is why hydropower is reported separately rather than absorbed into a generic scenario range. It also demonstrates that the baseline electricity total is factor-set dependent. At baseline activity and cooling assumptions, the zero-hydropower electricity pathway is 1.108 L per facility-input kWh and the on-site pathway is 0.640 L per facility-input kWh. A uniform reduction greater than 42.2\\% across all non-hydropower electricity factors would be required to reverse their ordering. This break-even calculation is deterministic and does not represent a probability bound or achievable-reduction estimate.\n\n\\subsection{Generation-water factors}\n\n\\begin{table}[t]\n\\centering\n\\small\n\\begin{tabularx}{0.86\\textwidth}{@{}p{0.28\\textwidth}C Y@{}}\n\\toprule\n\\textbf{Technology class} & \\textbf{Selected screening factor (L kWh$^{-1}$)} & \\textbf{Boundary note} \\\\\n\\midrule\nCoal & 1.9 & Selected harmonized operational value; not a fleet-specific estimate \\\\\nNatural gas & 0.7 & Selected harmonized operational value; configurations differ materially \\\\\nNuclear & 2.5 & Selected harmonized operational value; cooling and source water dominate variation \\\\\nHydropower & 8.0 & Selected reservoir-evaporation attribution convention; compared with zero attribution \\\\\nWind & 0.0 & Operational water consumption represented as zero in the factor set \\\\\nSolar & 0.1 & Selected operational screening value \\\\\nOther & 1.0 & Simplifying factor for oil, biomass, geothermal, other fossil and unknown/purchased-fuel shares \\\\\n\\bottomrule\n\\end{tabularx}\n\\caption{Selected screening factors used in the submitted attribution. They should not be read as current plant-specific measurements or comprehensive uncertainty bounds.}\n\\label{tab:factors}\n\\end{table}\n\nThe hydropower factor is influential because reservoir evaporation can be large and its allocation among electricity, water supply, flood control, navigation and recreation is both empirical and normative \\citep{grubert2016}. The no-hydropower case is therefore reported as a zero-attribution endpoint, not as a physical lower bound on reservoir evaporation. Grubert's U.S.-average net freshwater estimate of 1.7 m$^3$ GJ$^{-1}$ converts to approximately 6.1 L kWh$^{-1}$ and supplies an additional deterministic comparison. Likewise, assigning a coal-plus-gas share to a BA provides transition context but is neither necessary nor sufficient for high water consumption; nuclear and hydropower may also be water intensive under the selected conventions.\n\n\\begin{table}[t]\n\\centering\n\\small\n\\begin{tabularx}{0.88\\textwidth}{@{}Y C C Y@{}}\n\\toprule\n\\textbf{Hydropower factor} & \\textbf{Electricity-related (GL yr$^{-1}$)} & \\textbf{Total (GL yr$^{-1}$)} & \\textbf{Interpretation} \\\\\n\\midrule\n0 L kWh$^{-1}$ & 128.35 & 202.50 & Zero attribution of reservoir evaporation \\\\\n6.1 L kWh$^{-1}$ & 202.6 & 276.8 & Rounded U.S.-average net freshwater factor from 1.7 m$^3$ GJ$^{-1}$ \\citep{grubert2016} \\\\\n8.0 L kWh$^{-1}$ & 225.74 & 299.89 & Selected manuscript convention used in the reference scenario \\\\\n\\bottomrule\n\\end{tabularx}\n\\caption{Hydropower-only sensitivity at reference facility electricity and on-site assumptions. Values change only the hydropower attribution factor and do not resolve reservoir purpose, allocation method or site-specific evaporation.}\n\\label{tab:hydro_sensitivity}\n\\end{table}\n\nFor any alternative technology factor $w_f'$, equation~\\ref{eq:grid_intensity} can be recomputed without altering facility electricity or spatial assignments. The national sensitivity to a change in one factor is linear:\n\\begin{equation}\n\\frac{\\partial W^{(2)}_s}{\\partial w_f}\n=10^{-3}\\sum_i E^{\\mathrm{fac,kWh}}_{i,s}s_{r(i),f}.\n\\label{eq:factor_sensitivity}\n\\end{equation}\nThe submitted outputs do not present a Monte Carlo interval because validated probability bounds and a complete current plant/cooling-system input layer were unavailable for the real-data run. The joint scenario endpoints therefore represent selected cases rather than comprehensive uncertainty. This unresolved factor uncertainty is listed explicitly in Table~\\ref{tab:status}.\n\n\\subsection{External plausibility checks and comparability limits}\n\nExternal quantities are used only as order-of-magnitude checks because no public comparator shares the same facility population, date, capacity boundary, geographic scope and consumption definition. Table~\\ref{tab:plausibility} makes those differences explicit rather than treating agreement as validation.\n\n\\begin{table}[t]\n\\centering\n\\footnotesize\n\\begin{tabularx}{\\textwidth}{@{}p{0.18\\textwidth}p{0.34\\textwidth}Y@{}}\n\\toprule\n\\textbf{Comparison} & \\textbf{Observed relationship} & \\textbf{Interpretation} \\\\\n\\midrule\nLBNL all-sector electricity, 2023 & Reference hyperscale electricity is 115.87 TWh yr$^{-1}$, or 65.8\\% of the 176 TWh all-U.S.-data-center estimate \\citep{lbnl2024} & Broad magnitude check only; populations and dates differ \\\\\nLBNL hyperscale WUE scenarios & Reference WUE is 0.80 L kWh$^{-1}_{\\mathrm{IT}}$, above the approximately 0.32--0.40 L kWh$^{-1}_{\\mathrm{IT}}$ aggregate values represented in recent scenarios \\citep{lbnl2024} & The reference on-site case is intentionally conservative; it is not a fitted fleet mean \\\\\nSiddik et al. pathway structure & Electricity-related water is 75.27\\% of the reference total, consistent with an approximately three-quarter electricity-related share in the broader historical U.S. sector \\citep{siddik2021} & Reasonableness check; the split is explicitly not claimed as novel \\\\\nIllustrative consumptive fraction & Only under a withdrawal-basis reinterpretation of WUE, applying the illustrative $c=0.74$ gives 54.9 GL yr$^{-1}$ on site, 280.6 GL yr$^{-1}$ total and an 80.4\\% electricity-related share \\citep{cooley2026} & Cross-boundary sensitivity only; facility withdrawals and WUE-reported site water are not universally equivalent \\\\\n\\bottomrule\n\\end{tabularx}\n\\caption{External plausibility checks and their non-comparability boundaries. These comparisons test order of magnitude and direction; they do not validate individual facilities or substitute for aligned observed data.}\n\\label{tab:plausibility}\n\\end{table}\n\nCompany sustainability disclosures were not pooled as a validation dataset because most report global or organizational totals, use varying withdrawal/consumption and freshwater boundaries, and do not consistently identify the U.S. facilities represented in the licensed inventory. A future validation would require facility- or region-matched metered water and electricity under a common reporting boundary.\n\n\\section{Spatial linkage and environmental context}\n\n\\subsection{One-to-one spatial assignments}\n\nFacility coordinates are represented as points in EPSG:4326. Point-in-polygon joins assign each facility to a U.S. state, one BA and one HydroBASINS level-6 polygon. The BA boundary input was the project-archived WattTime grid-region layer stored as \\path{balancing_authorities_polygons/balancing_authorities_EPA.shp}, joined on \\texttt{region\\_B\\_1}; the exact project copy is identified by its checksum in the run manifest \\citep{watttimeBA}. A spatial join that produces more than one assignment per facility is treated as an error rather than silently duplicated. Maps are projected in EPSG:5070 for the contiguous United States. Required spatial coverage must be complete before the canonical model is run. BA polygon containment specifies the hosting-BA production-mix attribution region; it does not verify retail utility service, a direct interconnection or the transmission path serving the facility.\n\n\\begin{table}[t]\n\\centering\n\\small\n\\begin{tabularx}{\\textwidth}{@{}p{0.22\\textwidth}p{0.22\\textwidth}Y@{}}\n\\toprule\n\\textbf{Input} & \\textbf{Analytic role} & \\textbf{Contract} \\\\\n\\midrule\nFacility inventory & IDs, coordinates and facility-input capacity & Unique IDs; valid coordinates; positive capacity; configured hyperscale label and geographic exclusions \\\\\nBA polygons & Hosting electricity-system attribution region & At most one polygon per facility; non-missing assignment for included rows \\\\\nBA generation mix & Seven technology shares & Matching BA key; finite, non-negative shares; pre-normalization sum between 0.98 and 1.02 \\\\\nHydroBASINS level 6 & Facility-hosting hydrologic context & Unique \\texttt{PFAF\\_ID}; at most one host polygon per facility \\\\\nAqueduct 4.0 annual baseline & Annual stress and depletion context & Annual \\texttt{bws\\_score} and \\texttt{bwd\\_score}; unique basin key; negative no-data sentinels removed; finite scores within 0--5 \\\\\nState boundaries & State reporting & At most one state per facility; contiguous-U.S. filter \\\\\n\\bottomrule\n\\end{tabularx}\n\\caption{Required spatial and tabular inputs and the contracts applied before analysis.}\n\\label{tab:input_contracts}\n\\end{table}\n\n\\subsection{HydroBASINS and Aqueduct indicators}\n\nHydroBASINS is derived from the HydroSHEDS drainage network and provides globally consistent nested basin polygons \\citep{lehner2013}. Level 6 is used because the Aqueduct 4.0 annual indicators in this analysis are keyed to that geography \\citep{kuzma2023}. It is a screening geography, not evidence that a water utility withdraws from the polygon containing a facility. Assigning the facility to a finer U.S. Hydrologic Unit Code (HUC) polygon without service-area and intake provenance would add siting precision but would not identify source water.\n\nThe primary contextual indicator is annual baseline water stress, \\texttt{bws\\_score}, which compares total gross demand, including consumptive and non-consumptive demand, with available renewable surface and groundwater supply. Annual baseline water depletion, \\texttt{bwd\\_score}, compares consumptive demand with available supply and is used as a sensitivity. Both are scores on the documented 0--5 scale; high background stress or depletion is defined as a score of at least 3. Monthly fields such as \\texttt{bws\\_01\\_score} are not used as annual indicators.\n\nThe modeled water volume is not multiplied by either Aqueduct score. The indicators supply background hydrologic context and are not treated as scarcity-weighted damage factors. AWARE is a distinct consumption-based scarcity characterization model \\citep{boulay2018}; it was not substituted for Aqueduct because a licensed, validated and spatially matched AWARE layer was not configured for this run.\n\n\\subsection{BA attribution and physical generation geography}\n\nA balancing authority is an operational region in which supply and demand are balanced, but BAs differ institutionally and are not unitary water-governance jurisdictions. We use the facility-hosting BA as a transparent production-mix attribution and coordination proxy. This is not formal GHG Protocol location-based accounting. The resulting average intensity neither traces electrons nor identifies the plants and watersheds where electricity-generation water is consumed. Actual decisions can reside with vertically integrated utilities, generators, regional transmission organizations or independent system operators, public utility commissions, federal power administrations and large buyers.\n\nEPA commonly recommends eGRID subregions for location-based greenhouse-gas inventories, whereas the present primary question concerns BA-scale attribution of hosted load. Spatial convention can change regional values. A complete BA-versus-eGRID-subregion water rerun was not executed because a validated subregion-polygon and complete matching resource-mix input were not configured for this analysis. We therefore do not claim equivalence between BA and eGRID-subregion results. Prior electricity--water research demonstrates the importance of interregional transfers and plant-resolved geography \\citep{chini2018,grubert2018,peer2019}; those physical flows remain outside the data boundary.\n\n\\section{Burden, context and concentration metrics}\n\n\\subsection{On-site basin screen}\n\nFor each scenario, on-site consumption is summed within the HydroBASINS level-6 polygon hosting the facility. Let $B_b=\\sum_{i:b(i)=b}W^{(1)}_i$ be on-site burden in hosting basin $b$. Because $u$, PUE, WUE and $c$ are common across records within a scenario, $B_b$ is exactly proportional to the summed reported capacity assigned to basin $b$; the map does not estimate spatial variation in cooling performance. A basin is labelled high burden when $B_b$ is at or above the median among positive-burden hosting basins. A separate high-context flag is assigned when the basin's own annual Aqueduct score is at least 3. The primary intersection class requires both flags. The median and score-3 thresholds are transparent display conventions rather than empirically optimized or regulatory cut-offs; continuous burden and score values remain the underlying quantities.\n\nThe primary analysis does not propagate stress from touching neighbours. Adjacency neither changes the burden calculation nor proves a shared source. A touching-neighbour helper exists in the codebase, but its result was not used in the submitted figures. The map is described as an intersection screen, not a measured impact class.\n\n\\subsection{BA descriptors}\n\nElectricity-related consumption and modeled grid-water intensity are summed or calculated by hosting BA. The main map displays continuous burden and, for descriptive decomposition only, an outline for intensity above the median among positive-burden hosting BAs. Neither the burden median nor the intensity median is a validated policy threshold, and no composite electricity-water hotspot is defined. Hosted-site stress is retained as a separate diagnostic and is the nameplate-capacity-weighted mean Aqueduct stress of data-center sites in BA $r$:\n\\begin{equation}\n\\bar S_r^{\\mathrm{host}}=\n\\frac{\\sum_{i:r(i)=r}P_iS_i}{\\sum_{i:r(i)=r}P_i}.\n\\label{eq:host_stress}\n\\end{equation}\nIt answers where a BA's hosted data-center capacity is sited and is not stress at generating plants. Coal-plus-gas share may be retained as a machine-readable energy-transition descriptor, but it is neither necessary nor sufficient for high water consumption, does not enter a water classification and is not used as an inferential outcome.\n\n\\subsection{Concentration}\n\nConcentration is calculated independently for on-site basin totals and electricity-related BA totals. For pathway $p$, regions are ranked by burden $B^{(p)}_{(1)}\\geq\\cdots\\geq B^{(p)}_{(R)}$. The cumulative share through rank $k$ is\n\\begin{equation}\nC^{(p)}_k=\\frac{\\sum_{j=1}^{k}B^{(p)}_{(j)}}{\\sum_{j=1}^{R}B^{(p)}_{(j)}}.\n\\label{eq:concentration}\n\\end{equation}\nThe reported concentration count is the smallest $k$ for which $C^{(p)}_k\\geq0.5$. Normalized rank $q_k=k/R$ is plotted so each curve spans 0--100\\% of its own positive-burden hosting regions. We additionally calculate the within-partition Gini coefficient from regional pathway shares. The denominator contains 63 positive-burden hosting basins and 24 hosting BAs; 62 of those basins have valid annual Aqueduct context. These metrics reduce dependence on raw region counts but do not make the partitions physically commensurate because they differ in area, topology and function.\n\n\\subsection{Fixed-facility-electricity reference comparisons}\\label{sec:fixed_activity}\n\nThese comparisons retain reference-scenario facility-input electricity and change the pathway-specific reference parameters. They do not hold delivered computing work or IT electricity fixed. For on-site water, the comparison uses the lower-water PUE, WUE and $c$ values without changing facility-input electricity:\n\\begin{equation}\n\\Delta W^{(1)}_i=\\max\\left[0,\nW^{(1)}_{i,\\mathrm{base}}-\n10^{-3}E^{\\mathrm{fac,kWh}}_{i,\\mathrm{base}}\n\\frac{\\mathrm{WUE}_{\\mathrm{low}}c_{\\mathrm{low}}}{\\mathrm{PUE}_{\\mathrm{low}}}\n\\right].\n\\label{eq:cooling_benchmark}\n\\end{equation}\nFor electricity-related water, baseline facility electricity is retained and BA intensity is capped at the empirical 25th percentile among the 24 hosting BAs:\n\\begin{align}\nI_r^{\\mathrm{ref}}&=\\min\\{I_r,Q_{0.25}(I)\\},\\\\\n\\Delta W^{(2)}_i&=10^{-3}E^{\\mathrm{fac,kWh}}_{i,\\mathrm{base}}\n\\left(I_{r(i)}-I_{r(i)}^{\\mathrm{ref}}\\right).\n\\label{eq:grid_benchmark}\n\\end{align}\nBAs already below the reference receive a zero difference. The percentile is recomputed from the corrected hosting-BA distribution rather than hard-coded. In the on-site comparison, changing PUE at fixed facility-input electricity changes inferred IT electricity, and applying common parameters nationally makes the regional difference proportional to reference on-site burden. It is therefore a combined parameter comparison rather than a pure cooling-technology counterfactual or an independent spatial prioritization result. Both quantities are modeled differences to transparent references; they are not forecasts, achievable savings, engineering-feasibility estimates, marginal-dispatch models, policy targets or cost curves.\n\n\\subsection{Illustrative ramp-up diagnostic}\n\nWhere commissioning year $y_i$ is available, the workflow can apply an explicitly illustrative three-year capacity ramp:\n\\begin{equation}\nr_i=\\min\\left\\{1,\\max\\left[0,\\frac{2025-y_i+1}{3}\\right]\\right\\},\n\\qquad P_i^{\\mathrm{ramp}}=r_iP_i.\n\\label{eq:ramp}\n\\end{equation}\nMissing commissioning years retain $r_i=1$ and future commissioning years receive zero effective capacity. This optional diagnostic is defined to expose sensitivity to facility maturation, but it is not used in the primary totals because commissioning year is not observed utilization and present-day capacity cannot be back-cast into a historical time series. Stage and commissioning-cohort summaries are descriptive only; no announced-facility water forecast or temporal-persistence claim is made.\n\n\\section{Supplementary results}\n\n\\subsection{National scenarios}\n\nThe national scenario outputs are reported in Table~\\ref{tab:national}. Baseline facility electricity is 115.87 \\TWhyr{}. Baseline operational consumption is 299.89 \\GLyr{}, with 74.16 \\GLyr{} on site and 225.74 \\GLyr{} electricity related. The electricity-related component represents 75.27\\% of the baseline total. The approximately three-to-one split is consistent with the structure reported for a broader historical U.S. data-center population \\citep{siddik2021}; it is not claimed as the novelty of this study.\n\nThe higher-load/water-intensive case yields 450.60 \\GLyr{}, whereas the lower-load/lower-water case yields 204.91 \\GLyr{}. Setting only the hydropower factor to zero reduces electricity-related consumption by 97.39 \\GLyr{} and total consumption to 202.50 \\GLyr{}. Electricity-related consumption remains larger than on-site consumption in that case, at 63.38\\% of the total. Values may not add exactly at the displayed precision because pathway values and totals are rounded independently from unrounded facility results.\n\n\\begin{table}[t]\n\\centering\n\\small\n\\begin{adjustbox}{max width=\\textwidth}\n\\begin{tabular}{lrrrrrrr}\n\\toprule\nScenario & Electricity & On-site & Electricity-related & Total & On-site & Electricity-related & Intensity \\\\\n & (TWh yr$^{-1}$) & (GL yr$^{-1}$) & (GL yr$^{-1}$) & (GL yr$^{-1}$) & (\\%) & (\\%) & (L kWh$^{-1}$) \\\\\n\\midrule\nLower-load/lower-water & 96.56 & 16.79 & 188.11 & 204.91 & 8.20 & 91.80 & 2.12 \\\\\nBaseline/reference & 115.87 & 74.16 & 225.74 & 299.89 & 24.73 & 75.27 & 2.59 \\\\\nHigher-load/water-intensive & 149.22 & 159.88 & 290.72 & 450.60 & 35.48 & 64.52 & 3.02 \\\\\nNo hydropower attribution & 115.87 & 74.16 & 128.35 & 202.50 & 36.62 & 63.38 & 1.75 \\\\\n\\bottomrule\n\\end{tabular}\n\\end{adjustbox}\n\\caption{National annualized results across mutually exclusive scenarios. Intensity is total operational consumption per unit facility-input electricity.}\n\\label{tab:national}\n\\end{table}\n\n\\subsection{Aqueduct-indicator sensitivity}\n\nOf the 63 positive-burden hosting basins, 62 have valid annual Aqueduct context. Withdrawal-based baseline water stress identifies 25 of those 62 basins (40.3\\%) as high context; 15 also meet the high on-site-burden criterion. Those high-stress hosting basins contain 36.24\\% of the modeled reference-scenario on-site pathway. Because common national PUE, WUE, utilization and consumptive-fraction settings make on-site burden proportional to hosted capacity within a scenario, the corresponding share of hosted capacity is also 36.24\\%.\n\nConsumption-based baseline water depletion identifies seven of 62 basins (11.3\\%) as high context, six of which also meet the burden criterion. Those high-depletion hosting basins contain 11.81\\% of modeled reference-scenario on-site water. Thus, replacing withdrawal-based stress with consumption-based depletion contracts the number of high-context basins by 72\\%, the burden-plus-context set by 60\\%, and the share of modeled on-site water lying in high-context basins from 36.24\\% to 11.81\\% (Table~\\ref{tab:aqueduct_results}). Indicator choice therefore changes the categorical screening geography materially while leaving modeled water volumes and concentration calculations unchanged.\n\n\\begin{table}[t]\n\\centering\n\\small\n\\begin{adjustbox}{max width=\\textwidth}\n\\begin{tabular}{@{}lrrrr@{}}\n\\toprule\n\\textbf{Annual Aqueduct context} & \\textbf{Valid-context basins} & \\textbf{High context} & \\textbf{High burden + context} & \\textbf{On-site water in high-context basins (\\%)} \\\\\n\\midrule\nBaseline water stress (\\texttt{bws\\_score}) & 62 & 25 & 15 & 36.24 \\\\\nBaseline water depletion (\\texttt{bwd\\_score}) & 62 & 7 & 6 & 11.81 \\\\\n\\bottomrule\n\\end{tabular}\n\\end{adjustbox}\n\\caption{Sensitivity of the on-site basin screen to the annual Aqueduct indicator. High context is score $\\geq3$; high burden is at least the median among positive-burden hosting basins. The final column is the share of the national reference-scenario on-site pathway assigned to hosting basins meeting the named context threshold. It is a descriptive hosting-basin screen, not a scarcity-weighted impact estimate.}\n\\label{tab:aqueduct_results}\n\\end{table}\n\n\\subsection{Pathway concentration}\n\nThe corrected aggregation contains 63 positive-burden facility-hosting HydroBASINS level-6 basins and 24 hosting BAs. Ten basins (15.9\\% of the hosting-basin set) are required to reach at least half of national on-site consumption, whereas three BAs (12.5\\% of hosting BAs) reach half of electricity-related consumption. The smallest sets reaching 80\\% of pathway burden contain 27 basins and eight BAs, respectively. The five leading hosting basins contain 34.74\\% of modeled on-site water and the ten leading basins 51.23\\%; the corresponding shares for electricity-related water are 72.36\\% and 87.65\\% across hosting BAs.\n\nWithin each pathway's own partition, the Gini coefficient is 0.539 for on-site basin burden and 0.666 for electricity-related BA burden; the corresponding Herfindahl--Hirschman indices calculated from regional pathway shares are 0.0393 and 0.1517 (Table~\\ref{tab:concentration_results}). These statistics consistently identify compact screening-priority sets, with a steeper electricity-attribution profile under the reference generation-water factor set. They are not used to claim that one physical pathway is intrinsically more spatially clustered: HydroBASINS polygons and BAs differ in number, area, topology and institutional function. Pathway-specific factor changes, particularly hydropower attribution, may also change BA ordering.\n\n\\begin{table}[t]\n\\centering\n\\scriptsize\n\\begin{adjustbox}{max width=\\textwidth}\n\\begin{tabular}{@{}lrrrrrrrr@{}}\n\\toprule\n\\textbf{Pathway / partition} & \\textbf{$R$} & \\textbf{$k_{50}$} & \\textbf{$k_{80}$} & \\textbf{Top 5 (\\%)} & \\textbf{Top 10 (\\%)} & \\textbf{HHI} & \\textbf{Gini} & \\textbf{$k_{50}/R$ (\\%)} \\\\\n\\midrule\nOn-site / hosting basin & 63 & 10 & 27 & 34.74 & 51.23 & 0.0393 & 0.539 & 15.9 \\\\\nElectricity-related / hosting BA & 24 & 3 & 8 & 72.36 & 87.65 & 0.1517 & 0.666 & 12.5 \\\\\n\\bottomrule\n\\end{tabular}\n\\end{adjustbox}\n\\caption{Within-partition concentration statistics for reference-scenario pathway burden. $R$ is the number of positive-burden hosting regions; $k_{50}$ and $k_{80}$ are the smallest ranked sets reaching at least 50\\% and 80\\% of the pathway total. HHI is $\\sum_r p_r^2$, where $p_r$ is a region's pathway share. Top-region shares, HHI and Gini are calculated within each pathway's own spatial partition and therefore support within-partition prioritization rather than a scale-free physical comparison between basins and BAs.}\n\\label{tab:concentration_results}\n\\end{table}\n\n\\subsection{Host-location profiles and fixed-facility-electricity comparisons}\n\nState and BA profiles demonstrate that hosted capacity and modeled water burden are not interchangeable. Regions with similar hosted capacity can have different reference pathway balances because on-site burden follows capacity under common parameters whereas electricity-related burden also reflects the selected BA production mix and factor set. The profiles are therefore useful decompositions of host-location burden, not physical source-water accounts. Exact electricity-side rankings are treated as conditional on hydropower and other generation-water factors rather than as invariant regional findings.\n\nThe regional LaTeX tables are generated directly from the same source-data tables as the plots; manual transcription is not used. State profiles, BA profiles, basin concentration and both fixed-reference comparison tables are created from the versioned machine-readable outputs listed in Table~\\ref{tab:output_crosswalk}. This design prevents a plotted subset from diverging from the national total. The fixed-facility-electricity outputs show where a uniform reference comparison produces the largest modeled difference; because the on-site comparison uses common parameters, its regional ranking is proportional to reference on-site burden; they do not identify feasible projects, achievable savings, cost-effective measures or effects attributable to a particular actor.\n\n\n\\section{Regional priority and robustness analyses}\n\\subsection{Named hosting regions}\n\\NWinput{generated/region_results.tex}\n\\NWinput{generated/regions_basins.tex}\n\\NWinput{generated/regions_ba.tex}\n\n\\subsection{Threshold and set agreement}\nThe context sweep uses thresholds 2, 2.5, 3, 3.5 and 4, on the intersection of basins with both valid scores. The high-burden median is calculated using all positive-burden hosting basins and held fixed across context thresholds. Missing-context burdens remain in the national volume denominator and are reported as a coverage limitation. The analysis exports each basin's membership, both joint criteria, captured volume shares, pairwise set inclusion and Jaccard overlap. A reduction in set size is not assumed to establish a nested subset or a reduction in water damage.\n\\NWinput{generated/threshold_results.tex}\n\\NWinput{generated/threshold_table.tex}\n\n\\subsection{Hydropower ranks and dependence on PJM}\nFor each BA $r$, write $H_r=\\sum_{i:r(i)=r}E^{\\mathrm{fac}}_i s_{r,\\mathrm{hydro}}$ in MWh and let $A_r$ be reference electricity-related water in m$^3$. The coefficient-specific amount in GL is\n\\begin{equation}\n A_r(h)=10^{-6}\\{A_r-H_r w_{\\mathrm{hydro,ref}}+H_rh\\},\\quad h\\in\\{0,6.1,8.0\\}.\n\\end{equation}\nMWh multiplied by L per kWh is numerically m$^3$; the factor $10^{-6}$ converts that result to GL. Generation shares, all other factors and electricity remain fixed. The common-support Gini and HHI include zero-water hosting BAs, with positive-burden counts reported separately. The without-PJM result normalizes to the remaining BAs and is not presented as the national total.\n\\NWinput{generated/hydro_results.tex}\n\\NWinput{generated/pjm_results.tex}\n\\NWinput{generated/hydro_table.tex}\n\n\\subsection{Common-partition concentration}\nBoth pathways are aggregated to the same hosting BAs and, separately, the same hosting states. All common hosting units remain in the support. Gini is calculated as $2\\sum_{j=1}^{R}j x_{(j)}/(R\\sum_j x_{(j)})-(R+1)/R$, with $x_{(j)}$ in ascending order; HHI is $\\sum_j(x_j/\\sum_k x_k)^2$. These diagnostics control the statistical partition without interpreting attributed generation water as physically occurring at hosted sites.\n\\NWinput{generated/common_partition.tex}\n\n\\subsection{Paired climate and cooling sensitivity}\nThe supplied paired-coefficient table must contain a unique facility--case row, PUE, WUE, an explicit consumption/withdrawal basis, a consumptive fraction and coefficient/assignment provenance. All analytic facilities must have a valid pair for each case; no coefficient is extrapolated or filled silently. Each paired case $k$ holds facility electricity fixed and calculates\n\\begin{equation}\nW^{(1)}_{i,k}=10^{-3}E_i^{\\mathrm{fac,kWh}}\\frac{\\mathrm{WUE}_{i,k}c_{i,k}}{\\mathrm{PUE}_{i,k}}.\n\\end{equation} Consumption-based WUE requires $c=1$. Scenarios do not estimate the prevalence of installed cooling systems. The workflow reports regional ranks, top-ten membership, continuous agreement, national volume and burden--context results, without presupposing that climate changes or preserves the priorities.\n\\NWinput{generated/climate_results.tex}\n\\NWinput{generated/climate_table.tex}\n\n\\section{Validation, reproducibility and analysis status}\n\n\\subsection{Canonical-table validation}\n\nFor a complete run of four scenarios, the canonical table has one row for each facility--scenario pair, giving an expected $472\\times4=1{,}888$ rows. The workflow stops before publishing results if a mandatory data contract fails. Core checks are listed in Table~\\ref{tab:validation}. National, state, BA and basin outputs are generated from the same table rather than from parallel calculations. A final build requires the empirical run to pass the equation identities, conservation tests and manuscript-number checks. Rounded values are compared using the displayed precision; the underlying unrounded values remain in the aggregate CSV files.\n\n\\begin{table}[t]\n\\centering\n\\small\n\\begin{tabularx}{\\textwidth}{@{}p{0.27\\textwidth}Y p{0.16\\textwidth}@{}}\n\\toprule\n\\textbf{Validation contract} & \\textbf{Pass condition} & \\textbf{Failure behavior} \\\\\n\\midrule\nInput identity & Required input exists; checksum and configuration hash recorded & Stop \\\\\nInventory uniqueness & One unique facility row after eligibility filtering & Stop \\\\\nSpatial assignment & At most one state, BA and hosting basin per facility; required coverage complete & Stop \\\\\nAnnual Aqueduct schema & Annual BWS/BWD fields; unique basin key; valid 0--5 scores after no-data removal & Stop \\\\\nBA mix & Seven non-negative shares; sum in 0.98--1.02 before rounding normalization & Stop \\\\\nCanonical uniqueness & One row per facility--scenario; expected scenario coverage & Stop \\\\\nNon-negativity & Electricity and both water pathways are finite and non-negative & Stop \\\\\nAdditive identity & $W^{\\mathrm{tot}}=W^{(1)}+W^{(2)}$ within numerical tolerance & Stop \\\\\nAggregation conservation & Relevant state, BA and basin sums reconcile to their national pathway totals & Stop \\\\\nSubset bound & Every ranked or top-region subset is no greater than its parent national total & Stop \\\\\nScenario isolation & No-hydropower case retains baseline activity and cooling parameters; only $w_{\\mathrm{hydro}}$ changes & Stop \\\\\n\\bottomrule\n\\end{tabularx}\n\\caption{Mandatory validation contracts applied by the analysis workflow. A successful figure-generating run can occur only after these checks pass.}\n\\label{tab:validation}\n\\end{table}\n\n\\subsection{Status of requested extensions}\n\nTable~\\ref{tab:status} distinguishes the core analysis design from descriptive diagnostics, conditional code paths and analyses that were not run. This distinction is important: code support is not treated as empirical evidence, and no unconfigured sensitivity is described as completed.\n\n\\begin{longtable}{@{}p{0.29\\textwidth}p{0.18\\textwidth}p{0.45\\textwidth}@{}}\n\\caption{Status and evidentiary boundary of analysis components.}\\label{tab:status}\\\\\n\\toprule\n\\textbf{Component} & \\textbf{Status} & \\textbf{Interpretation} \\\\\n\\midrule\n\\endfirsthead\n\\toprule\n\\textbf{Component} & \\textbf{Status} & \\textbf{Interpretation} \\\\\n\\midrule\n\\endhead\nCanonical hosting-BA production-mix model & Validated on final run & Facility-by-scenario accounting, BA attribution, national and regional reconciliation, and figure-source outputs \\\\\nInventory by announced/proposed stage & Descriptive only & Stage-specific counts/capacities where labels exist; no future water forecast \\\\\nTemporal persistence & Descriptive cohorts only & Current capacity is not back-cast; historical burden requires dated historical capacity and utilization \\\\\nHUC-8, HUC-10 and HUC-12 outputs & Not run & Polygon inputs were not configured; a host HUC would not establish utility source water \\\\\nBA versus eGRID-subregion attribution & Not run & Regional results may depend on spatial convention; no equivalence is claimed \\\\\nTechnology-factor probability analysis & Not run & Source-documented probability bounds were not configured; joint scenarios are not comprehensive uncertainty intervals \\\\\nHydropower spatial-ranking sensitivity & Generated by revision run & Complete regional ranks and amounts for coefficients 0, 6.1 and 8.0 L kWh$^{-1}$; acceptance requires empirical outputs \\\\\nModern plant/cooling factors & Not run & No complete validated plant, prime-mover, cooling and source-water layer was supplied \\\\\nMarket-based/PPA water accounting & Not run & Verified facility portfolio intensities, contractual quantities and residual-mix evidence were unavailable \\\\\nOperating-stage-only inventory & Not run & Stage field was incomplete or insufficiently standardized for a defensible operating-only subset \\\\\nIllustrative three-year ramp & Optional diagnostic & Commissioning-year proxy only; not observed utilization and not used for primary totals \\\\\nPlant/cooling-system derivation & Code path only; not run & Requires audited plant generation, prime mover, cooling, source-water and consumption data \\\\\nTouching-neighbour stress & Not used & Primary screen uses the host basin's own annual score; adjacency does not establish shared supply \\\\\nAWARE scarcity sensitivity & Not run & A licensed, validated and spatially matched characterization layer was not configured \\\\\nDistrict energy, CHP, waste heat and self-generation & Outside boundary & Facility-specific utility-flow and behind-the-meter metering are required; no proxy is imputed \\\\\nValue-chain/Scope 3 analogue & Outside boundary & Construction, equipment, semiconductors and fuels are not quantified \\\\\n\\bottomrule\n\\end{longtable}\n\n\\subsection{Output and claim crosswalk}\nLegacy filenames containing Scope1 or Scope2 are retained solely for compatibility with earlier code; their fields mean on-site and electricity-related water, not formal greenhouse-gas scopes. New regional diagnostics are in the revision's \\texttt{source\\_data} directory. The exact output manifest identifies the files used in the final build.\n\n\n\\begingroup\n\\scriptsize\n\\begin{longtable}{@{}p{0.31\\textwidth}p{0.27\\textwidth}p{0.34\\textwidth}@{}}\n\\caption{Machine-readable outputs supporting the manuscript.}\\label{tab:output_crosswalk}\\\\\n\\toprule\n\\textbf{Output} & \\textbf{Supports} & \\textbf{Key safeguard} \\\\\n\\midrule\n\\endfirsthead\n\\toprule\n\\textbf{Output} & \\textbf{Supports} & \\textbf{Key safeguard} \\\\\n\\midrule\n\\endhead\n\\path{Table_National_Scenarios.csv} & National electricity, pathway totals, shares and intensities & Derived from canonical rows; scenario identifiers immutable \\\\\n\\path{Table_State_Scenarios.csv} and \\path{PlotData_State_Scenario_Intervals.csv} & State pathway dominance and scenario profiles & State sums reconcile to national results \\\\\n\\path{Table_BA_Scenarios.csv} and \\path{PlotData_BA_Scenario_Intervals.csv} & BA pathway dominance and scenario profiles & BA sums reconcile; missing mix is not imputed \\\\\n\\path{Table_HydroBasin_Scenarios.csv} & On-site hosting-basin burdens & On-site pathway retained separately from electricity attribution \\\\\n\\path{Table_Scope1_Hotspots_AnnualBWS.csv} & Primary burden--stress intersection & Annual host-basin \\texttt{bws\\_score}; no adjacency in primary model \\\\\n\\path{Sensitivity_Aqueduct_BWS_vs_BWD_Summary.csv} & Stress-versus-depletion counts & Exact annual indicators; common threshold and burden definition \\\\\n\\path{PlotData_ExtendedData_Scope1_Concentration.csv} and \\path{PlotData_ExtendedData_Scope2_Concentration.csv} & Ranked concentration curves & Cumulative shares bounded by one, tied to parent totals and plotted against normalized within-partition rank \\\\\n\\path{Table_Scope1_FixedActivity_Benchmark.csv} & On-site fixed-reference comparison & Reference facility-input electricity fixed \\\\\n\\path{Table_Scope2_EmpiricalIntensity_Benchmark.csv} and \\path{scope2_benchmark.json} & Electricity fixed-reference comparison & 25th percentile recomputed from current hosting BAs \\\\\n\\path{canonical_validation.csv} and \\path{aggregation_conservation.csv} & Accounting integrity & Explicit pass/fail evidence for identities and geographic reconciliation \\\\\n\\path{inventory_attrition.csv}, \\path{geography_coverage.csv}, \\path{ba_mix_audit.csv} & Population and linkage audit & Missingness and exclusions visible rather than imputed \\\\\n\\path{manifest.json} and \\path{analysis_status.csv} & Reproducibility and scope & Configuration/input hashes and completed/not-run distinction \\\\\n\\bottomrule\n\\end{longtable}\n\n\\endgroup\n\n\\subsection{Submission metadata and reproducibility boundary}\n\n\\begin{table}[t]\n\\centering\n\\small\n\\begin{tabularx}{\\textwidth}{@{}p{0.25\\textwidth}Y@{}}\n\\toprule\n\\textbf{Item} & \\textbf{Version or access statement} \\\\\n\\midrule\nFacility provider and snapshot & \\InventoryProvider{}; commercially licensed file frozen 15 December 2025; provider-defined record units \\\\\nFacility access & The authors cannot redistribute restricted record-level data. Independent access requires lawful access obtained directly from Baxtel under its applicable commercial terms. Requests for non-restricted derived materials are answered by the corresponding authors within two weeks. \\\\\nElectricity resource mix & \\GridDataset{} BA-level table; calendar-year 2023 generation \\\\\nBA polygon geometry & Project-archived WattTime grid-region boundary file \\path{balancing_authorities_polygons/balancing_authorities_EPA.shp}; join identifier \\texttt{region\\_B\\_1}; exact file identified by checksum in the run manifest \\\\\nHydrologic geometry & HydroBASINS level 6, North America v1c \\\\\nWater context & WRI Aqueduct 4.0 annual baseline fields \\texttt{bws\\_score} and \\texttt{bwd\\_score} \\\\\nCode and non-restricted outputs & \\ReviewerArchive{}; review copy available to editors/referees on request; version corresponding to the accepted manuscript frozen publicly no later than publication \\\\\nRun identity & Input checksums, configuration hash and output manifest accompany the versioned analysis release \\\\\n\\bottomrule\n\\end{tabularx}\n\\caption{Submission metadata and reproducibility boundary for the analysis.}\n\\label{tab:submission_metadata}\n\\end{table}\n\nThe analysis package contains the model code, configuration schema, validation tests, plotting functions, input templates and a synthetic-input workflow. The version corresponding to the accepted manuscript will also contain the exact aggregate figure-source tables, non-restricted crosswalks, environment specification, validation reports, input checksums and configuration hash used for the reported results. Aggregate figure-source tables support numerical reproduction of the published displays and arithmetic from released derived values. The synthetic workflow validates software logic and invariants only; it does not reproduce the empirical facility-to-geography joins or validate proprietary records. The identified reviewer archive provides the assessment copy.\n\nThe underlying facility inventory is commercially licensed. Code and aggregate outputs cannot independently reconstruct provider-supplied records or provide a full end-to-end reproduction of the empirical analysis. Independent reconstruction therefore requires lawful access obtained directly from Baxtel. The corresponding authors will respond within two weeks to reasonable requests for non-restricted derived materials and, where permitted by the licence, will provide editors and referees with the applicable data-use documentation. This distinction between numerical reproduction of published aggregate products, code-path validation with synthetic data and reconstruction of the licensed source inventory is retained throughout the manuscript.\n\n\\section{Interpretive limitations}\n\nThe principal conclusions and their dependencies are separated here. The finding that BWS and BWD produce different categorical hosting-basin sets depends on the named indicators and threshold but not on the water-volume equations. The finding that electricity-related water remains larger than on-site water holds across the specified scenarios and hydropower endpoints. Absolute totals and reference regional rankings are less stable because they depend on the provider capacity boundary, common operating parameters, BA production mixes and technology factors.\n\nThe facility dataset is a provider-defined reported-current-capacity cross-section. It inherits the provider's classification, campus/building segmentation and data-collection procedures and does not independently establish metered load or full operating status. Common utilization, PUE, WUE and consumptive-fraction values collapse heterogeneity in ramp-up, climate, cooling topology, water chemistry, blowdown, reuse and return flows. The on-site geography is consequently capacity driven within each scenario, and the inventory does not consistently identify source-water category or utility intake basin.\n\nThe electricity pathway uses a calendar-year 2023 BA-average production mix and broad technology factors with a December 2025 facility snapshot. It does not resolve transfers, dispatch, prime mover, cooling system, source water or reporting-method uncertainty. Hydropower allocation is especially influential; a plant/cooling-system model, probability analysis and BA-versus-eGRID-subregion rerun were not completed. A BA is an attribution region, not a unitary decision maker or a physical source basin, and no market-based account is inferred from public procurement claims.\n\nHydroBASINS and Aqueduct provide chronic hosting-basin context, not seasonal drought coincidence, environmental-flow effects or causal ecological damage. Thresholds are descriptive conventions. Concentration statistics are valid within each partition but do not make basin and BA geometries commensurate. Construction, equipment manufacture, semiconductor fabrication, fuel-cycle water and other value-chain uses are excluded. The results should therefore be used to prioritize the next evidence needed, not as facility permits, engineering savings estimates or complete life-cycle footprints.\n\n\\section*{Use of generative AI}\n\nGenerative-AI assistance was used during revision for language editing and code drafting. It was not used as a scientific data source or to determine reported numerical results. The authors inspected all code, citations, calculations, outputs and interpretations and take responsibility for the work. A generative image model was used to explore an earlier conceptual draft, which is not used in this build. Scientific maps and plots are generated programmatically from the documented analysis; the main-text workflow figure is a deterministic text-and-equation flow chart and contains no generated imagery.\n\n\\clearpage\n\\begin{thebibliography}{99}\n\n\\bibitem[Baxtel(2025)]{baxtel2025}\nBaxtel. \\emph{United States data-center facility inventory: licensed snapshot delivered to Harvard, 15 December 2025} (proprietary dataset, 2025).\n\n\\bibitem[ISO/IEC(2022)]{iso30134}\nInternational Organization for Standardization and International Electrotechnical Commission. \\emph{ISO/IEC 30134-9: Data centres---Key performance indicators---Part 9: Water usage effectiveness (WUE)} (2022).\n\n\\bibitem[The Green Grid(2011)]{greengrid2011}\nThe Green Grid. \\emph{Water Usage Effectiveness (WUE): A Green Grid Data Center Sustainability Metric}. White Paper 35 (2011).\n\n\\bibitem[WRI and WBCSD(2015)]{ghgscope2}\nWorld Resources Institute and World Business Council for Sustainable Development. \\emph{GHG Protocol Scope 2 Guidance} (2015).\n\n\\bibitem[Mytton(2021)]{mytton2021}\nD. Mytton. Data centre water consumption. \\emph{npj Clean Water} \\textbf{4}, 11 (2021). doi:10.1038/s41545-021-00101-w.\n\n\\bibitem[Lei and Masanet(2022)]{lei2022}\nN. Lei and E. Masanet. Climate- and technology-specific PUE and WUE estimations for U.S. data centers using open data. \\emph{Resources, Conservation and Recycling} \\textbf{182}, 106323 (2022). doi:10.1016/j.resconrec.2022.106323.\n\n\\bibitem[U.S. EPA(2012)]{epa2012}\nU.S. Environmental Protection Agency. \\emph{WaterSense at Work: Best Management Practices for Commercial and Institutional Facilities} (2012).\n\n\\bibitem[U.S. EPA(2025)]{usepaegrid2023}\nU.S. Environmental Protection Agency. \\emph{eGRID2023 Data, Revision 2} (released 12 June 2025).\n\n\\bibitem[Macknick et al.(2012)]{macknick2012}\nJ. Macknick, R. Newmark, G. Heath and K. C. Hallett. Operational water consumption and withdrawal factors for electricity generating technologies: a review of existing literature. \\emph{Environmental Research Letters} \\textbf{7}, 045802 (2012). doi:10.1088/1748-9326/7/4/045802.\n\n\\bibitem[Meldrum et al.(2013)]{meldrum2013}\nJ. Meldrum, S. Nettles-Anderson, G. Heath and J. Macknick. Life cycle water use for electricity generation: a review and harmonization of literature estimates. \\emph{Environmental Research Letters} \\textbf{8}, 015031 (2013). doi:10.1088/1748-9326/8/1/015031.\n\n\\bibitem[Grubert(2016)]{grubert2016}\nE. Grubert. Water consumption from hydroelectricity in the United States. \\emph{Advances in Water Resources} \\textbf{96}, 88--94 (2016). doi:10.1016/j.advwatres.2016.07.004.\n\n\\bibitem[Mekonnen and Hoekstra(2012)]{mekonnen2012}\nM. M. Mekonnen and A. Y. Hoekstra. The blue water footprint of electricity from hydropower. \\emph{Hydrology and Earth System Sciences} \\textbf{16}, 179--187 (2012). doi:10.5194/hess-16-179-2012.\n\n\\bibitem[Sjostedt et al.(2025)]{sjostedt2025}\nE. Sjostedt \\emph{et al.} Impact of thermoelectric power plant operations and water use reporting methods on thermoelectric power plant water use. \\emph{Environmental Science \\& Technology} \\textbf{59}, 4482--4492 (2025). doi:10.1021/acs.est.4c02024.\n\n\\bibitem[Shehabi et al.(2024)]{lbnl2024}\nA. Shehabi \\emph{et al.} \\emph{2024 United States Data Center Energy Usage Report}. Lawrence Berkeley National Laboratory, LBNL-2001637 (2024). doi:10.71468/P1WC7Q.\n\n\\bibitem[Cooley and Shimabuku(2026)]{cooley2026}\nH. Cooley and M. Shimabuku. \\emph{How Much Water Do Data Centers Use? And Why Does It Matter?} Pacific Institute Issue Brief (2026). \\url{https://pacinst.org/publication/how-much-water-do-data-centers-use-and-why-does-it-matter/}.\n\n\\bibitem[Siddik et al.(2021)]{siddik2021}\nM. A. B. Siddik, A. Shehabi and L. Marston. The environmental footprint of data centers in the United States. \\emph{Environmental Research Letters} \\textbf{16}, 064017 (2021). doi:10.1088/1748-9326/abfba1.\n\n\\bibitem[WattTime(2026)]{watttimeBA}\nWattTime. \\emph{Grid Region Map Geometry and balancing-authority lookup documentation}. \\url{https://legacy-docs.watttime.org/} (accessed 10 September 2026).\n\n\\bibitem[Lehner and Grill(2013)]{lehner2013}\nB. Lehner and G. Grill. Global river hydrography and network routing: baseline data and new approaches to study the world's large river systems. \\emph{Hydrological Processes} \\textbf{27}, 2171--2186 (2013). doi:10.1002/hyp.9740.\n\n\\bibitem[Kuzma et al.(2023)]{kuzma2023}\nS. Kuzma \\emph{et al.} \\emph{Aqueduct 4.0: Updated Decision-Relevant Global Water Risk Indicators}. World Resources Institute Technical Note (2023). doi:10.46830/writn.23.00061.\n\n\\bibitem[Boulay et al.(2018)]{boulay2018}\nA.-M. Boulay \\emph{et al.} The WULCA consensus characterization model for water scarcity footprints: assessing impacts of water consumption based on available water remaining (AWARE). \\emph{International Journal of Life Cycle Assessment} \\textbf{23}, 368--378 (2018). doi:10.1007/s11367-017-1333-8.\n\n\\bibitem[Chini et al.(2018)]{chini2018}\nC. M. Chini, L. A. Djehdian, W. N. Lubega and A. S. Stillwell. Virtual water transfers of the U.S. electric grid. \\emph{Nature Energy} \\textbf{3}, 1115--1123 (2018). doi:10.1038/s41560-018-0266-1.\n\n\\bibitem[Grubert and Sanders(2018)]{grubert2018}\nE. Grubert and K. T. Sanders. Water use in the United States energy system: a national assessment and unit process inventory of water consumption and withdrawals. \\emph{Environmental Science \\& Technology} \\textbf{52}, 6695--6703 (2018). doi:10.1021/acs.est.8b00139.\n\n\\bibitem[Peer et al.(2019)]{peer2019}\nR. A. M. Peer, E. Grubert and K. T. Sanders. A regional assessment of the water embedded in the U.S. electricity system. \\emph{Environmental Research Letters} \\textbf{14} (2019). doi:10.1088/1748-9326/ab2daa.\n\\end{thebibliography}\n\n\\end{document}\n", 'tests/test_revision.py': "from pathlib import Path\nimport sys\nimport tempfile\nimport unittest\nimport json\nimport numpy as np\nimport pandas as pd\nsys.path.insert(0,str(Path(__file__).resolve().parents[1]/'code'))\nfrom nw_review import (ReviewError, concentration, validated_base, basin_table,\n    threshold_analysis, hydro_analysis, common_partition, stage_analysis,\n    climate_analysis, finalize, run_revision)\n\n\ndef fixture():\n    rows=[]\n    capacities=[4.,2.,1.,3.,2.,1.,3.,2.]\n    bas=['PJM','PJM','PJM','BA2','BA2','BA2','BA3','BA3']\n    basins=['101','101','102','102','103','103','104','104']\n    bws={'101':4.,'102':3.,'103':2.5,'104':np.nan}\n    bwd={'101':2.5,'102':3.5,'103':1.,'104':np.nan}\n    shares={'PJM':dict(gas=.6,hydro=.2,nuclear=.2), 'BA2':dict(hydro=1.), 'BA3':dict(coal=.4,gas=.6)}\n    params={'efficiency':(.55,1.15,.2,8.), 'baseline':(.66,1.25,.8,8.),\n            'high_load_water_intensive':(.85,1.4,1.5,8.),'no_hydropower':(.66,1.25,.8,0.)}\n    factors=dict(coal=1.9,gas=.7,nuclear=2.5,hydro=8.,wind=0.,solar=.1,other=1.)\n    for s,(u,p,w,h) in params.items():\n        f=dict(factors,hydro=h)\n        for i,cap in enumerate(capacities):\n            ba=bas[i]; basin=basins[i]; energy=cap*8760*u\n            sh={k:shares[ba].get(k,0.) for k in factors}\n            intensity=sum(sh[k]*f[k] for k in sh)\n            onsite=energy/p*w; grid=energy*intensity\n            row=dict(facility_id=f'SYN_{i}',scenario=s,capacity_mw=cap,ba_id=ba,hydrobasin_id=basin,\n                state_id='SYN_A' if i<4 else 'SYN_B',facility_electricity_MWh_y=energy,\n                it_electricity_MWh_y=energy/p,pue=p,utilization=u,wue_L_per_kWh_IT=w,\n                onsite_consumptive_fraction=1.,grid_water_L_per_kWh=intensity,\n                bws_score=bws[basin],bwd_score=bwd[basin],onsite_m3_y=onsite,electricity_m3_y=grid,\n                total_m3_y=onsite+grid,stage='operational' if i<4 else None)\n            row.update({'share_'+k:sh[k] for k in sh})\n            row.update({'factor_'+k+'_L_per_kWh':f[k] for k in f})\n            rows.append(row)\n    return pd.DataFrame(rows)\n\n\ndef pairs(base):\n    return pd.DataFrame(dict(facility_id=base.facility_id,case='SYN_paired',pue=1.25,\n        wue_L_per_kWh_IT=.8,water_basis='consumption',consumptive_fraction=1.,\n        coefficient_source='SYNTHETIC TEST ONLY',assignment_source='SYNTHETIC TEST ONLY'))\n\nclass Tests(unittest.TestCase):\n    def setUp(self):\n        self.canonical=fixture(); self.base,self.df=validated_base(self.canonical)\n\n    def test_equations(self):\n        self.assertEqual(len(self.base),8)\n        np.testing.assert_allclose(self.df.total_m3_y,self.df.onsite_m3_y+self.df.electricity_m3_y)\n\n    def test_duplicate_fails(self):\n        with self.assertRaises(ReviewError): validated_base(pd.concat([self.df,self.df.iloc[:1]]))\n\n    def test_equation_failure(self):\n        broken=self.df.copy(); broken.loc[0,'total_m3_y']+=1\n        with self.assertRaises(ReviewError): validated_base(broken)\n\n    def test_gini_zero_support(self):\n        self.assertAlmostEqual(concentration([1,1,1,1])['Gini'],0)\n        self.assertAlmostEqual(concentration([0,0,0,4],include_zeros=True)['Gini'],.75)\n        self.assertEqual(concentration([0,0,0,4],include_zeros=True)['R'],4)\n        self.assertEqual(concentration([0,0])['k50'],None)\n\n    def test_threshold_paired_and_nonnested(self):\n        b=basin_table(self.base); t,sets,context=threshold_analysis(b)\n        self.assertEqual(context['paired_basins'],3)\n        row=t[t.threshold.eq(3.5)].iloc[0]\n        self.assertFalse(row.b_subset_a)\n        self.assertTrue(t.bws_count.is_monotonic_decreasing)\n        self.assertGreater(context['excluded_context_water_share'],0)\n\n    def test_conflicting_basin_score_fails(self):\n        b=self.base.copy(); b.loc[0,'bws_score']=2\n        with self.assertRaises(ReviewError): basin_table(b)\n\n    def test_hydro_linearity_and_zero_regions(self):\n        h,m,c=hydro_analysis(self.base)\n        p=h.pivot(index='ba_id',columns='hydro_factor',values='electricity_GL')\n        np.testing.assert_allclose(p[6.1],p[0.]+(p[8.]-p[0.])*6.1/8)\n        zero=m[m.hydro_factor.eq(0)&m.population.eq('all_hosting_BAs')].iloc[0]\n        self.assertEqual(zero.R,3); self.assertEqual(zero.R_positive,2)\n        self.assertEqual(float(p.loc['BA2',0.]),0.)\n\n    def test_pjm_exclusion_and_partition(self):\n        table=common_partition(self.base)\n        g=table[table.partition.eq('ba_id')&table.quantity.eq('capacity_mw')]\n        self.assertEqual(int(g[g.population.eq('excluding_PJM')].iloc[0].R),2)\n        self.assertLess(g[g.population.eq('excluding_PJM')].iloc[0].total,g[g.population.eq('all_hosts')].iloc[0].total)\n\n    def test_stage_unknown(self):\n        stages,r=stage_analysis(self.base)\n        self.assertEqual(r['labelled_records'],4)\n        self.assertIsNone(r['operating_share_of_labelled_capacity'])\n        mapping=pd.DataFrame([dict(stage_label='operational',stage_class='operating',mapping_source='Synthetic dictionary')])\n        _,r=stage_analysis(self.base,mapping)\n        self.assertAlmostEqual(r['operating_share_of_labelled_capacity'],1)\n\n    def test_climate_identity_and_coverage(self):\n        b,s,t=climate_analysis(self.base,pairs(self.base))\n        np.testing.assert_allclose(b.reference_GL,b.onsite_GL)\n        self.assertAlmostEqual(float(s.iloc[0].spearman),1)\n        with self.assertRaises(ReviewError): climate_analysis(self.base,pairs(self.base).iloc[:-1])\n\n    def test_consumption_not_adjusted_twice(self):\n        p=pairs(self.base); p['consumptive_fraction']=.74\n        with self.assertRaises(ReviewError): climate_analysis(self.base,p)\n        p['water_basis']='withdrawal'\n        b,_,_=climate_analysis(self.base,p)\n        np.testing.assert_allclose(b.onsite_GL,b.reference_GL*.74)\n\n    def test_zero_water_climate_case(self):\n        p=pairs(self.base); p['wue_L_per_kWh_IT']=0.\n        b,s,t=climate_analysis(self.base,p)\n        self.assertEqual(float(s.iloc[0].total),0.)\n        self.assertTrue(pd.isna(s.iloc[0].k50))\n        self.assertEqual(int(s.iloc[0].n_b),0)\n        self.assertTrue(t.bws_national_water_share.isna().all())\n\n    def test_synthetic_finalization_forbidden(self):\n        with tempfile.TemporaryDirectory() as d:\n            p=Path(d)\n            (p/'RUN_STATUS.json').write_text(json.dumps({'synthetic':True}))\n            (p/'author.json').write_text('{}')\n            with self.assertRaises(ReviewError): finalize(p,p/'author.json',p)\n\nif __name__=='__main__': unittest.main(verbosity=2)\n", 'inputs/climate_pairs.csv': 'facility_id,case,pue,wue_L_per_kWh_IT,water_basis,consumptive_fraction,coefficient_source,assignment_source\n', 'inputs/basin_labels.csv': 'hydrobasin_id,basin_label,label_source\n', 'inputs/stage_mapping.csv': 'stage_label,stage_class,mapping_source\n', 'code/nw_plots.py': '"""Deterministic vector figures; only supplied analytical tables are plotted."""\nfrom __future__ import annotations\nfrom pathlib import Path\nimport numpy as np\nimport matplotlib\nmatplotlib.use(\'Agg\')\nimport matplotlib.pyplot as plt\nfrom matplotlib.patches import FancyBboxPatch\n\n\ndef save(fig, folder: Path, stem: str):\n    folder.mkdir(parents=True, exist_ok=True)\n    for ext in [\'pdf\',\'svg\',\'png\']:\n        fig.savefig(folder/f\'{stem}.{ext}\', dpi=600, bbox_inches=\'tight\', facecolor=\'white\')\n    plt.close(fig)\n\n\ndef clean(ax):\n    ax.spines[\'top\'].set_visible(False)\n    ax.spines[\'right\'].set_visible(False)\n    ax.tick_params(labelsize=8)\n    ax.xaxis.label.set_size(9)\n    ax.yaxis.label.set_size(9)\n\n\ndef design(folder: Path):\n    fig=plt.figure(figsize=(7.2,4.6))\n    ax=fig.add_axes([0,0,1,1]); ax.set_xlim(0,1); ax.set_ylim(0,1); ax.axis(\'off\')\n    def box(x,y,w,h,title,text):\n        ax.add_patch(FancyBboxPatch((x,y),w,h,boxstyle=\'round,pad=0.009,rounding_size=0.008\',\n                                   fill=False,linewidth=.7))\n        ax.text(x+.012,y+h-.034,title,fontsize=8.5,fontweight=\'bold\',va=\'top\')\n        ax.text(x+.012,y+h-.091,text,fontsize=7.5,va=\'top\',linespacing=1.45)\n    ax.text(.025,.965,\'a  Inputs\',fontsize=10,fontweight=\'bold\')\n    ax.text(.358,.965,\'b  Model\',fontsize=10,fontweight=\'bold\')\n    ax.text(.69,.965,\'c  Outputs\',fontsize=10,fontweight=\'bold\')\n    box(.025,.58,.275,.32,\'Facility and scenario inputs\',\n        \'Dated inventory; reported MW\\nUtilization, PUE and WUE\\nExplicit water-use basis\\nDocumented paired cooling case\')\n    box(.025,.18,.275,.33,\'Spatial inputs\',\n        \'Hosting basins and states\\nAnnual Aqueduct BWS / BWD\\nHosting balancing authorities\\nGeneration mix and water factors\')\n    box(.36,.71,.275,.19,\'Annual facility electricity\',\n        r\'$E_{\\rm fac}=P\\,8760\\,u$ (MWh yr$^{-1}$)\'+\'\\n\'+r\'$E_{\\rm kWh}=1000\\,E_{\\rm fac}$\')\n    box(.36,.44,.275,.20,\'On-site consumption\',\n        r\'$W_{\\rm site}=10^{-3}\\,\\frac{E_{\\rm kWh}}{\\mathrm{PUE}}\\,\\mathrm{WUE}\\,c$\'+\'\\n\'+r\'Output in m$^3$ yr$^{-1}$\')\n    box(.36,.18,.275,.20,\'Electricity pathway\',\n        r\'$W_{\\rm elec}=10^{-3}\\,E_{\\rm kWh}\\,\\sum_f s_{r,f}w_f$\'+\'\\n\'+r\'Output in m$^3$ yr$^{-1}$\')\n    box(.69,.59,.28,.31,\'Regional pathway results\',\n        \'On-site: hosting-basin totals\\nElectricity: hosting-BA attribution\\nNamed regional priority tables\\nWithin-partition concentration\')\n    box(.69,.18,.28,.34,\'Robustness diagnostics\',\n        \'Stress versus depletion thresholds\\nHydropower: 0, 6.1 and 8.0 L/kWh\\nPaired PUE/WUE sensitivity\\nCommon partitions; without PJM\')\n    for start,end in [((.304,.75),(.347,.79)),((.497,.70),(.497,.65)),((.30,.34),(.347,.28)),\n                      ((.64,.54),(.68,.7)),((.64,.28),(.68,.4))]:\n        ax.annotate(\'\',xy=end,xytext=start,arrowprops=dict(arrowstyle=\'-|>\',lw=.8))\n    ax.plot([.405,.405,.33,.33,.405],[.701,.681,.681,.412,.412],lw=.8,color=\'black\')\n    ax.annotate(\'\',xy=(.405,.389),xytext=(.405,.412),arrowprops=dict(arrowstyle=\'-|>\',lw=.8))\n    ax.text(.5,.08,\'One validated facility-by-scenario table; conserved regional and national sums\',\n            ha=\'center\',fontsize=8,fontweight=\'bold\')\n    ax.text(.5,.037,r\'1 GL = $10^6$ m$^3$. Hosting geography does not identify utility intakes or generating plants.\',\n            ha=\'center\',fontsize=7.1)\n    save(fig,folder,\'fig_1_NEW\')\n\n\ndef draw_revision_figures(basins,hydro,thresholds,climate,folder):\n    folder=Path(folder)\n    with plt.rc_context({\'font.family\':\'DejaVu Sans\',\'pdf.fonttype\':42,\'ps.fonttype\':42,\'svg.fonttype\':\'none\',\n                         \'axes.linewidth\':.7,\'lines.linewidth\':1.4}):\n        design(folder)\n        fig,ax=plt.subplots(figsize=(6.9,4.1))\n        ref=hydro[hydro.hydro_factor.eq(8)]\n        for table,field,name in [(basins,\'onsite_GL\',\'On-site / hosting basins\'),(ref,\'electricity_GL\',\'Electricity / hosting BAs\')]:\n            v=np.sort(table[field].to_numpy())[::-1]; v=v[v>0]\n            cum=np.cumsum(v)/v.sum(); k=int(np.searchsorted(cum,.5-1e-12)+1)\n            ax.step(np.r_[0,np.arange(1,len(v)+1)/len(v)*100],np.r_[0,cum*100],where=\'post\',\n                    label=f\'{name}: {k}/{len(v)} reach half\')\n            ax.scatter([100*k/len(v)],[100*cum[k-1]],s=24,marker=\'o\',facecolor=\'white\',zorder=5)\n        ax.plot([0,100],[0,100],\'--\',lw=.8,label=\'Equal distribution\')\n        ax.set(xlim=(0,100),ylim=(0,102),xlabel=\'Positive-burden hosting regions included (%)\',ylabel=\'Cumulative pathway share (%)\')\n        ax.legend(frameon=False,fontsize=8,loc=\'lower right\'); clean(ax)\n        save(fig,folder,\'Figure3_Normalized_Concentration\')\n        fig,ax=plt.subplots(figsize=(6.9,3.8))\n        for c,label,marker,ls in [(\'bws_count\',\'High stress\',\'o\',\'-\'),(\'bwd_count\',\'High depletion\',\'s\',\'-\'),\n            (\'bws_joint_count\',\'High stress + high burden\',\'o\',\'--\'),(\'bwd_joint_count\',\'High depletion + high burden\',\'s\',\'--\')]:\n            ax.plot(thresholds.threshold,thresholds[c],marker=marker,linestyle=ls,label=label)\n        ax.set(xlabel=\'Annual Aqueduct score threshold\',ylabel=\'Hosting basins meeting criterion\',xticks=thresholds.threshold)\n        ax.set_ylim(bottom=0); ax.legend(frameon=False,fontsize=8,loc=\'upper right\'); clean(ax)\n        save(fig,folder,\'ExtendedData_Threshold_Sweep\')\n        fig,ax=plt.subplots(figsize=(6.9,5.8))\n        pivot=hydro.pivot(index=\'ba_id\',columns=\'hydro_factor\',values=\'rank\')\n        for ba,row in pivot.iterrows():\n            ax.plot([0,1,2],row[[0.,6.1,8.]].to_numpy(),marker=\'o\',ms=2.5,lw=.8)\n            ax.text(2.08,row[8.],str(ba),fontsize=7,va=\'center\')\n        ax.set(xticks=[0,1,2],xticklabels=[\'0\',\'6.1\',\'8.0\'],xlabel=r\'Hydropower coefficient (L kWh$^{-1}$)\',\n               ylabel=\'Electricity-water rank (1 = largest)\',xlim=(-.1,2.65),ylim=(len(pivot)+.6,.4))\n        ax.set_yticks(np.arange(1,len(pivot)+1)); clean(ax)\n        save(fig,folder,\'ExtendedData_Hydro_Ranks\')\n        if climate is not None:\n            cb,cs,ct=climate\n            fig,ax=plt.subplots(figsize=(6.5,4.5))\n            for case,g in cb.groupby(\'case\'):\n                ax.scatter(g.reference_GL,g.onsite_GL,label=str(case),alpha=.75,s=20)\n            maximum=max(float(cb.reference_GL.max()),float(cb.onsite_GL.max()))*1.06\n            ax.plot([0,maximum],[0,maximum],\'--\',lw=.8)\n            ax.set(xlim=(0,maximum),ylim=(0,maximum),xlabel=r\'Uniform-reference on-site water (GL yr$^{-1}$)\',\n                   ylabel=r\'Paired-coefficient on-site water (GL yr$^{-1}$)\')\n            ax.legend(frameon=False,fontsize=8); clean(ax)\n            save(fig,folder,\'ExtendedData_Climate_Sensitivity\')\n', 'code/finalize_submission.py': '"""Package an inspected empirical run. Does not upload or submit anything."""\nfrom pathlib import Path\nimport argparse\nfrom nw_review import finalize\nif __name__ == \'__main__\':\n    parser=argparse.ArgumentParser(description=__doc__)\n    parser.add_argument(\'output\',type=Path)\n    parser.add_argument(\'signoff\',type=Path)\n    parser.add_argument(\'--bundle\',type=Path,default=Path(__file__).resolve().parents[1])\n    args=parser.parse_args()\n    print(finalize(args.output,args.signoff,args.bundle))\n', 'code/nw_review.py': '"""Reviewer-requested diagnostics, rebuilt from the recovered analysis interface.\n\nNo empirical inputs or results are bundled. The run consumes result[\'canonical\']\nfrom the recovered notebook. Facility rows stay in memory; exports are regional\nor national aggregates. Climate coefficients are never fabricated.\n"""\nfrom __future__ import annotations\n\nfrom pathlib import Path\nfrom itertools import combinations\nimport hashlib\nimport json\nimport math\nimport shutil\nimport zipfile\nimport re\nimport sys\nimport platform\nfrom typing import Any\n\nimport numpy as np\nimport pandas as pd\nfrom scipy.stats import spearmanr\n\nTECH = (\'coal\', \'gas\', \'nuclear\', \'hydro\', \'wind\', \'solar\', \'other\')\nSCENARIOS = (\'efficiency\', \'baseline\', \'high_load_water_intensive\', \'no_hydropower\')\nTHRESHOLDS = (2.0, 2.5, 3.0, 3.5, 4.0)\nHYDRO_FACTORS = (0.0, 6.1, 8.0)\nNAMES = {\'efficiency\': \'Lower-load/lower-water\', \'baseline\': \'Reference\',\n         \'high_load_water_intensive\': \'Higher-load/water-intensive\',\n         \'no_hydropower\': \'No hydropower attribution\'}\nWATER = [\'onsite_m3_y\', \'electricity_m3_y\', \'total_m3_y\']\n\nclass ReviewError(ValueError):\n    """A required data or reporting contract has not been met."""\n\n\ndef require(condition: bool, message: str) -> None:\n    if not bool(condition):\n        raise ReviewError(message)\n\n\ndef required(df: pd.DataFrame, names: list[str] | tuple[str, ...], label: str) -> None:\n    missing = [c for c in names if c not in df]\n    require(not missing, f\'{label}: missing columns {missing}\')\n\n\ndef identifier(s: pd.Series) -> pd.Series:\n    """Normalize ONLY whitespace and integer-valued .0 suffixes; never ID prefixes."""\n    return s.astype(\'string\').str.strip().str.replace(r\'(?<=\\d)\\.0$\', \'\', regex=True)\n\n\ndef digest(path: Path) -> str:\n    h = hashlib.sha256()\n    with Path(path).open(\'rb\') as f:\n        for chunk in iter(lambda: f.read(1024 * 1024), b\'\'):\n            h.update(chunk)\n    return h.hexdigest()\n\n\ndef json_safe(v: Any) -> Any:\n    if isinstance(v, dict):\n        return {str(k): json_safe(x) for k, x in v.items()}\n    if isinstance(v, (tuple, list)):\n        return [json_safe(x) for x in v]\n    if isinstance(v, (np.bool_,)):\n        return bool(v)\n    if isinstance(v, (np.integer,)):\n        return int(v)\n    if isinstance(v, (float, np.floating)):\n        return float(v) if math.isfinite(float(v)) else None\n    if pd.isna(v):\n        return None\n    return v\n\n\ndef write_json(v: Any, path: Path) -> None:\n    path.parent.mkdir(parents=True, exist_ok=True)\n    path.write_text(json.dumps(json_safe(v), indent=2, allow_nan=False), encoding=\'utf-8\')\n\n\ndef export(df: pd.DataFrame, path: Path) -> None:\n    forbidden = {\'facility_id\', \'id\', \'latitude\', \'longitude\', \'geometry\',\n                 \'company_name\', \'full_address\', \'address\', \'coordinates\'}\n    require(not (forbidden & set(df.columns)), f\'Restricted columns in aggregate export {path.name}\')\n    path.parent.mkdir(parents=True, exist_ok=True)\n    df.to_csv(path, index=False, float_format=\'%.12g\')\n\n\ndef validated_base(canonical: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:\n    df = pd.DataFrame(canonical).copy()\n    cols = [\'facility_id\', \'scenario\', \'capacity_mw\', \'ba_id\', \'state_id\', \'hydrobasin_id\',\n            \'facility_electricity_MWh_y\', \'it_electricity_MWh_y\', \'pue\', \'utilization\',\n            \'wue_L_per_kWh_IT\', \'onsite_consumptive_fraction\', \'grid_water_L_per_kWh\',\n            \'bws_score\', \'bwd_score\', *WATER,\n            *[\'share_\' + t for t in TECH], *[\'factor_\' + t + \'_L_per_kWh\' for t in TECH]]\n    required(df, cols, \'canonical\')\n    for col in [\'facility_id\', \'ba_id\', \'state_id\', \'hydrobasin_id\']:\n        df[col] = identifier(df[col])\n        require(df[col].notna().all() and df[col].ne(\'\').all(), f\'Missing {col}\')\n    require(not df.duplicated([\'facility_id\', \'scenario\']).any(), \'Duplicate facility-scenario rows\')\n    require(set(SCENARIOS).issubset(set(df[\'scenario\'])), \'Four original scenarios are required\')\n    for c in cols:\n        if c not in {\'facility_id\', \'scenario\', \'ba_id\', \'state_id\', \'hydrobasin_id\'}:\n            df[c] = pd.to_numeric(df[c], errors=\'raise\')\n    for c in [\'capacity_mw\', \'facility_electricity_MWh_y\', \'it_electricity_MWh_y\', \'pue\']:\n        require(np.isfinite(df[c]).all() and df[c].gt(0).all(), f\'Invalid {c}\')\n    require(df[\'pue\'].ge(1).all(), \'PUE below one\')\n    require(df[\'utilization\'].between(0, 1).all(), \'Invalid utilization\')\n    require(df[\'onsite_consumptive_fraction\'].between(0, 1).all(), \'Invalid consumptive fraction\')\n    for c in WATER + [\'wue_L_per_kWh_IT\', \'grid_water_L_per_kWh\']:\n        require(np.isfinite(df[c]).all() and df[c].ge(0).all(), f\'Invalid {c}\')\n    share = df[[\'share_\' + t for t in TECH]].to_numpy(float)\n    factor = df[[\'factor_\' + t + \'_L_per_kWh\' for t in TECH]].to_numpy(float)\n    require(np.isfinite(share).all() and (share >= 0).all(), \'Invalid generation shares\')\n    require(np.isfinite(factor).all() and (factor >= 0).all(), \'Invalid generation factors\')\n    require(np.allclose(share.sum(axis=1), 1, rtol=0, atol=1e-8), \'Shares do not sum to one\')\n    identities = {\n        \'facility electricity\': (df.capacity_mw * 8760 * df.utilization, df.facility_electricity_MWh_y),\n        \'IT electricity\': (df.facility_electricity_MWh_y / df.pue, df.it_electricity_MWh_y),\n        \'on-site\': (df.it_electricity_MWh_y * df.wue_L_per_kWh_IT * df.onsite_consumptive_fraction, df.onsite_m3_y),\n        \'grid intensity\': ((share * factor).sum(axis=1), df.grid_water_L_per_kWh),\n        \'electricity water\': (df.facility_electricity_MWh_y * df.grid_water_L_per_kWh, df.electricity_m3_y),\n        \'additivity\': (df.onsite_m3_y + df.electricity_m3_y, df.total_m3_y),\n    }\n    for key, (left, right) in identities.items():\n        require(np.allclose(left, right, rtol=1e-9, atol=1e-5), f\'Failed identity: {key}\')\n    for c in [\'bws_score\', \'bwd_score\']:\n        df.loc[df[c] < 0, c] = np.nan\n        require(df[c].dropna().between(0, 5).all(), f\'{c} outside 0-5\')\n    base = df[df.scenario.eq(\'baseline\')].sort_values(\'facility_id\').reset_index(drop=True)\n    for scenario in SCENARIOS:\n        s = df[df.scenario.eq(scenario)].sort_values(\'facility_id\').reset_index(drop=True)\n        require(s.facility_id.tolist() == base.facility_id.tolist(), f\'Population changed: {scenario}\')\n        for c in [\'ba_id\', \'state_id\', \'hydrobasin_id\']:\n            require(s[c].equals(base[c]), f\'Geography changed: {scenario}/{c}\')\n        require(np.allclose(s.capacity_mw, base.capacity_mw), f\'Capacity changed: {scenario}\')\n    nh = df[df.scenario.eq(\'no_hydropower\')].sort_values(\'facility_id\').reset_index(drop=True)\n    require(np.allclose(nh.onsite_m3_y, base.onsite_m3_y), \'Zero hydro changed on-site water\')\n    require(np.allclose(nh.facility_electricity_MWh_y, base.facility_electricity_MWh_y), \'Zero hydro changed electricity\')\n    for t in TECH:\n        if t != \'hydro\':\n            require(np.allclose(nh[\'factor_\' + t + \'_L_per_kWh\'], base[\'factor_\' + t + \'_L_per_kWh\']), \'Zero hydro changed another factor\')\n    require(np.allclose(nh.factor_hydro_L_per_kWh, 0), \'Zero-hydro factor is not zero\')\n    return base, df\n\n\ndef concentration(values: Any, *, include_zeros: bool = False) -> dict[str, Any]:\n    a = np.asarray(values, dtype=float)\n    require(np.isfinite(a).all() and (a >= 0).all(), \'Invalid concentration values\')\n    if not include_zeros:\n        a = a[a > 0]\n    n = len(a)\n    if n == 0 or a.sum() == 0:\n        return dict(R=n, R_positive=int((a > 0).sum()), total=0.0, k50=None, k80=None,\n                    top5_share=None, top10_share=None, HHI=None, Gini=None)\n    p = np.sort(a / a.sum())[::-1]\n    asc = np.sort(a)\n    gini = 2 * np.dot(np.arange(1, n + 1), asc) / (n * asc.sum()) - (n + 1) / n\n    return dict(R=n, R_positive=int((a > 0).sum()), total=float(a.sum()),\n                k50=int(np.searchsorted(np.cumsum(p), .5 - 1e-12) + 1),\n                k80=int(np.searchsorted(np.cumsum(p), .8 - 1e-12) + 1),\n                top5_share=float(p[:5].sum()), top10_share=float(p[:10].sum()),\n                HHI=float(np.dot(p, p)), Gini=float(gini))\n\n\ndef rho(x: Any, y: Any) -> float | None:\n    x, y = np.asarray(x, float), np.asarray(y, float)\n    valid = np.isfinite(x) & np.isfinite(y)\n    if valid.sum() < 3 or len(np.unique(x[valid])) < 2 or len(np.unique(y[valid])) < 2:\n        return None\n    result = spearmanr(x[valid], y[valid])\n    value = getattr(result, "statistic", None)\n    if value is None:\n        value = getattr(result, "correlation", None)\n    if value is None:\n        value = result[0]\n    value = float(value)\n    return value if np.isfinite(value) else None\n\n\ndef top_set(table: pd.DataFrame, key: str, value: str, n: int | None = None) -> set[str]:\n    ranked = table.loc[table[value].gt(0)].sort_values([value, key], ascending=[False, True])\n    if n is None:\n        n = concentration(ranked[value], include_zeros=True)[\'k50\'] or 0\n    return set(ranked.head(n)[key].astype(str))\n\n\ndef compare_sets(a: set[str], b: set[str]) -> dict[str, Any]:\n    return {\'n_a\': len(a), \'n_b\': len(b), \'intersection\': len(a & b),\n            \'jaccard\': len(a & b) / len(a | b) if a | b else None,\n            \'b_subset_a\': bool(b <= a), \'entered\': \'|\'.join(sorted(b - a)),\n            \'exited\': \'|\'.join(sorted(a - b))}\n\n\ndef grouped(base: pd.DataFrame, key: str) -> pd.DataFrame:\n    out = base.groupby(key, sort=True).agg(n_records=(\'facility_id\', \'size\'),\n        capacity_mw=(\'capacity_mw\', \'sum\'), electricity_MWh=(\'facility_electricity_MWh_y\', \'sum\'),\n        onsite_m3=(\'onsite_m3_y\', \'sum\'), electricity_m3=(\'electricity_m3_y\', \'sum\')).reset_index()\n    for source, target in [(\'onsite_m3\', \'onsite_GL\'), (\'electricity_m3\', \'electricity_GL\')]:\n        out[target] = out[source] / 1e6\n    out[\'capacity_GW\'] = out.capacity_mw / 1000\n    out[\'electricity_TWh\'] = out.electricity_MWh / 1e6\n    for col, original in [(\'onsite_m3\', \'onsite_m3_y\'), (\'electricity_m3\', \'electricity_m3_y\')]:\n        require(np.isclose(out[col].sum(), base[original].sum(), rtol=1e-10), f\'Conservation failed for {key}\')\n    return out\n\n\ndef basin_table(base: pd.DataFrame, labels: pd.DataFrame | None = None) -> pd.DataFrame:\n    out = grouped(base, \'hydrobasin_id\')\n    for indicator in [\'bws_score\', \'bwd_score\']:\n        # A basin must not be assigned different contextual scores at different facilities.\n        counts = base.groupby(\'hydrobasin_id\')[indicator].nunique(dropna=False)\n        require(counts.le(1).all(), f\'Non-unique basin {indicator}; check exact PFAF_ID join\')\n        score = base.groupby(\'hydrobasin_id\')[indicator].first().rename(indicator)\n        out = out.merge(score, left_on=\'hydrobasin_id\', right_index=True, validate=\'one_to_one\')\n    states = base.groupby(\'hydrobasin_id\').state_id.agg(lambda v: \', \'.join(sorted(set(v)))).rename(\'hosting_states\')\n    out = out.merge(states, left_on=\'hydrobasin_id\', right_index=True, validate=\'one_to_one\')\n    out[\'basin_label\'] = \'\'\n    out[\'label_source\'] = \'\'\n    if labels is not None and len(labels):\n        required(labels, [\'hydrobasin_id\', \'basin_label\', \'label_source\'], \'basin labels\')\n        labels = labels[[\'hydrobasin_id\', \'basin_label\', \'label_source\']].copy()\n        labels[\'hydrobasin_id\'] = identifier(labels.hydrobasin_id)\n        require(not labels.hydrobasin_id.duplicated().any(), \'Duplicate basin labels\')\n        require(labels[[\'basin_label\', \'label_source\']].notna().all().all(), \'Unverified basin labels\')\n        out = out.drop(columns=[\'basin_label\', \'label_source\']).merge(labels, on=\'hydrobasin_id\', how=\'left\', validate=\'one_to_one\').fillna({\'basin_label\': \'\', \'label_source\': \'\'})\n    out = out.sort_values([\'onsite_GL\', \'hydrobasin_id\'], ascending=[False, True]).reset_index(drop=True)\n    out[\'rank\'] = np.arange(1, len(out) + 1)\n    out[\'share\'] = out.onsite_GL / out.onsite_GL.sum()\n    out[\'cumulative_share\'] = out.share.cumsum()\n    return out\n\n\ndef threshold_analysis(basins: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame, dict]:\n    complete = basins.dropna(subset=[\'bws_score\', \'bwd_score\']).copy()\n    require(not complete.empty, \'No paired valid-context basins\')\n    burden_median = float(basins.loc[basins.onsite_GL.gt(0), \'onsite_GL\'].median())\n    national = float(basins.onsite_GL.sum())\n    rows, details = [], []\n    for threshold in THRESHOLDS:\n        a = set(complete.loc[complete.bws_score.ge(threshold), \'hydrobasin_id\'].astype(str))\n        b = set(complete.loc[complete.bwd_score.ge(threshold), \'hydrobasin_id\'].astype(str))\n        comparison = compare_sets(a, b)\n        row = dict(threshold=threshold, paired_basins=len(complete), **comparison)\n        for indicator, selected in [(\'bws\', a), (\'bwd\', b)]:\n            hit = complete.hydrobasin_id.astype(str).isin(selected)\n            joint = hit & complete.onsite_GL.ge(burden_median)\n            row.update({indicator + \'_count\': int(hit.sum()), indicator + \'_joint_count\': int(joint.sum()),\n                        indicator + \'_national_water_share\': float(complete.loc[hit, \'onsite_GL\'].sum() / national) if national > 0 else np.nan,\n                        indicator + \'_joint_water_share\': float(complete.loc[joint, \'onsite_GL\'].sum() / national) if national > 0 else np.nan})\n        row[\'count_reduction_fraction\'] = 1 - len(b) / len(a) if len(a) else None\n        rows.append(row)\n        detail = complete[[\'hydrobasin_id\', \'onsite_GL\', \'bws_score\', \'bwd_score\']].copy()\n        detail[\'threshold\'] = threshold\n        detail[\'bws_high\'] = detail.hydrobasin_id.astype(str).isin(a)\n        detail[\'bwd_high\'] = detail.hydrobasin_id.astype(str).isin(b)\n        details.append(detail)\n    report = dict(paired_basins=len(complete), positive_burden_basins=int(basins.onsite_GL.gt(0).sum()),\n                  spearman=rho(complete.bws_score, complete.bwd_score),\n                  burden_median_GL=burden_median,\n                  excluded_context_water_share=(1-float(complete.onsite_GL.sum())/national) if national > 0 else None)\n    return pd.DataFrame(rows), pd.concat(details, ignore_index=True), report\n\n\ndef hydro_analysis(base: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:\n    ba = grouped(base, \'ba_id\')\n    h = base.assign(hydro_energy_MWh=base.facility_electricity_MWh_y * base.share_hydro,\n                    nonhydro_m3=base.electricity_m3_y - base.facility_electricity_MWh_y * base.share_hydro * base.factor_hydro_L_per_kWh)\n    require(h.nonhydro_m3.ge(-1e-6).all(), \'Negative non-hydro water\')\n    component = h.groupby(\'ba_id\').agg(hydro_energy_MWh=(\'hydro_energy_MWh\', \'sum\'), nonhydro_m3=(\'nonhydro_m3\', \'sum\')).reset_index()\n    ba = ba.merge(component, on=\'ba_id\', validate=\'one_to_one\')\n    blocks, metrics = [], []\n    for factor in HYDRO_FACTORS:\n        block = ba.copy()\n        block[\'hydro_factor\'] = factor\n        block[\'electricity_GL\'] = (block.nonhydro_m3.clip(lower=0) + block.hydro_energy_MWh * factor) / 1e6\n        block[\'grid_L_per_kWh\'] = block.electricity_GL * 1e6 / block.electricity_MWh\n        block = block.sort_values([\'electricity_GL\', \'ba_id\'], ascending=[False, True]).reset_index(drop=True)\n        block[\'rank\'] = np.arange(1, len(block) + 1)\n        total = float(block.electricity_GL.sum())\n        block[\'share\'] = block.electricity_GL / total if total else np.nan\n        block[\'cumulative_share\'] = block.share.cumsum()\n        blocks.append(block)\n        for subset_name, frame in [(\'all_hosting_BAs\', block), (\'excluding_PJM\', block[block.ba_id.ne(\'PJM\')])]:\n            if subset_name == \'excluding_PJM\' and \'PJM\' not in set(block.ba_id):\n                continue\n            metrics.append(dict(hydro_factor=factor, population=subset_name,\n                                **concentration(frame.electricity_GL, include_zeros=True)))\n    full = pd.concat(blocks, ignore_index=True)\n    comparisons = []\n    for fa, fb in combinations(HYDRO_FACTORS, 2):\n        a = full[full.hydro_factor.eq(fa)].set_index(\'ba_id\').sort_index()\n        b = full[full.hydro_factor.eq(fb)].set_index(\'ba_id\').reindex(a.index)\n        for group, count in [(\'top5\', 5), (\'k50\', None)]:\n            comparisons.append(dict(factor_a=fa, factor_b=fb, set_definition=group,\n                                    spearman=rho(a.electricity_GL, b.electricity_GL),\n                                    **compare_sets(top_set(a.reset_index(), \'ba_id\', \'electricity_GL\', count),\n                                                   top_set(b.reset_index(), \'ba_id\', \'electricity_GL\', count))))\n    return full, pd.DataFrame(metrics), pd.DataFrame(comparisons)\n\n\ndef common_partition(base: pd.DataFrame) -> pd.DataFrame:\n    rows = []\n    for key in [\'ba_id\', \'state_id\']:\n        table = grouped(base, key)\n        for measure in [\'capacity_mw\', \'onsite_GL\', \'electricity_GL\']:\n            rows.append(dict(partition=key, quantity=measure, population=\'all_hosts\',\n                             **concentration(table[measure], include_zeros=True)))\n        if key == \'ba_id\' and \'PJM\' in set(table.ba_id):\n            for measure in [\'capacity_mw\', \'onsite_GL\', \'electricity_GL\']:\n                rows.append(dict(partition=key, quantity=measure, population=\'excluding_PJM\',\n                             **concentration(table.loc[table.ba_id.ne(\'PJM\'), measure], include_zeros=True)))\n    return pd.DataFrame(rows)\n\n\ndef stage_analysis(base: pd.DataFrame, mapping: pd.DataFrame | None = None) -> tuple[pd.DataFrame, dict]:\n    raw = base[\'stage\'].astype(\'string\') if \'stage\' in base else pd.Series(pd.NA, index=base.index, dtype=\'string\')\n    raw = raw.str.strip()\n    unknown = raw.isna() | raw.str.lower().isin([\'\', \'unknown\', \'nan\', \'none\', \'n/a\', \'na\', \'not available\'])\n    raw = raw.mask(unknown, \'Missing/unknown\')\n    frame = base[[\'capacity_mw\']].copy()\n    frame[\'stage_label\'] = raw\n    table = frame.groupby(\'stage_label\', sort=True).agg(n_records=(\'capacity_mw\', \'size\'), capacity_mw=(\'capacity_mw\', \'sum\')).reset_index()\n    table[\'record_share\'] = table.n_records / len(base)\n    table[\'capacity_share\'] = table.capacity_mw / base.capacity_mw.sum()\n    table[\'stage_class\'] = \'Unmapped label\'\n    table.loc[table.stage_label.eq(\'Missing/unknown\'), \'stage_class\'] = \'Missing/unknown\'\n    if mapping is not None and len(mapping):\n        required(mapping, [\'stage_label\', \'stage_class\', \'mapping_source\'], \'stage mapping\')\n        require(not mapping.stage_label.duplicated().any(), \'Duplicate stage mapping\')\n        require(mapping.mapping_source.fillna(\'\').str.strip().ne(\'\').all(), \'Stage mapping requires provenance\')\n        lookup = mapping.set_index(\'stage_label\').stage_class\n        table[\'stage_class\'] = table.stage_label.map(lookup).fillna(table.stage_class)\n    report = dict(n_records=len(base), labelled_records=int((~unknown).sum()),\n                  labelled_capacity_mw=float(base.loc[~unknown, \'capacity_mw\'].sum()),\n                  labelled_capacity_share=float(base.loc[~unknown, \'capacity_mw\'].sum()/base.capacity_mw.sum()),\n                  operating_share_of_labelled_capacity=None)\n    classified = table[~table.stage_class.isin([\'Missing/unknown\', \'Unmapped label\'])]\n    if len(classified) and report[\'labelled_capacity_mw\'] > 0:\n        report[\'operating_share_of_labelled_capacity\'] = float(table.loc[table.stage_class.eq(\'operating\'), \'capacity_mw\'].sum()/report[\'labelled_capacity_mw\'])\n    report[\'all_labels_classified\'] = bool(not table.stage_class.eq(\'Unmapped label\').any())\n    return table, report\n\n\ndef climate_analysis(base: pd.DataFrame, pairs: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:\n    cols = [\'facility_id\', \'case\', \'pue\', \'wue_L_per_kWh_IT\', \'water_basis\',\n            \'consumptive_fraction\', \'coefficient_source\', \'assignment_source\']\n    required(pairs, cols, \'climate pairs\')\n    require(len(pairs) > 0, \'Climate lookup is empty\')\n    pairs = pairs[cols].copy()\n    pairs.facility_id = identifier(pairs.facility_id)\n    require(not pairs.duplicated([\'facility_id\', \'case\']).any(), \'Duplicate climate pair\')\n    for col in [\'case\', \'coefficient_source\', \'assignment_source\', \'water_basis\']:\n        require(pairs[col].fillna(\'\').astype(str).str.strip().ne(\'\').all(), f\'Missing climate {col}\')\n    require(pairs.water_basis.isin([\'consumption\', \'withdrawal\']).all(), \'Climate water_basis must be consumption or withdrawal\')\n    for col in [\'pue\', \'wue_L_per_kWh_IT\', \'consumptive_fraction\']:\n        pairs[col] = pd.to_numeric(pairs[col], errors=\'raise\')\n        require(np.isfinite(pairs[col]).all(), f\'Invalid climate {col}\')\n    require(pairs.pue.ge(1).all() and pairs.wue_L_per_kWh_IT.ge(0).all(), \'Invalid paired PUE/WUE\')\n    require(pairs.consumptive_fraction.between(0, 1).all(), \'Invalid climate consumptive fraction\')\n    require(pairs.loc[pairs.water_basis.eq(\'consumption\'), \'consumptive_fraction\'].eq(1).all(),\n            \'Consumption-based WUE must have consumptive_fraction=1; no double return-flow adjustment\')\n    blocks, summary, threshold_blocks = [], [], []\n    reference = basin_table(base)\n    reference_set = top_set(reference, \'hydrobasin_id\', \'onsite_GL\', 10)\n    for case, values in pairs.groupby(\'case\', sort=True):\n        require(set(values.facility_id) == set(base.facility_id), f\'Incomplete or extra climate coverage in {case}\')\n        joined = base[[\'facility_id\', \'hydrobasin_id\', \'facility_electricity_MWh_y\', \'capacity_mw\', \'onsite_m3_y\', \'bws_score\', \'bwd_score\', \'state_id\']].merge(values, on=\'facility_id\', validate=\'one_to_one\')\n        # Fixed total facility electricity: paired coefficients test climate/topology, not IT workload.\n        joined[\'climate_m3\'] = joined.facility_electricity_MWh_y / joined.pue * joined.wue_L_per_kWh_IT * joined.consumptive_fraction\n        b = joined.groupby(\'hydrobasin_id\').agg(onsite_GL=(\'climate_m3\', lambda a: a.sum()/1e6)).reset_index()\n        b = reference.drop(columns=[\'onsite_GL\', \'rank\', \'share\', \'cumulative_share\']).merge(b, on=\'hydrobasin_id\', validate=\'one_to_one\')\n        b[\'case\'] = str(case)\n        b[\'reference_GL\'] = b.hydrobasin_id.map(reference.set_index(\'hydrobasin_id\').onsite_GL)\n        b[\'rank\'] = b.onsite_GL.rank(method=\'min\', ascending=False)\n        blocks.append(b)\n        profile = concentration(b.onsite_GL, include_zeros=True)\n        profile.update(case=str(case), spearman=rho(b.reference_GL, b.onsite_GL),\n                       **compare_sets(reference_set, top_set(b, \'hydrobasin_id\', \'onsite_GL\', 10)))\n        summary.append(profile)\n        thresholds, _, _ = threshold_analysis(b)\n        thresholds[\'case\'] = str(case)\n        threshold_blocks.append(thresholds)\n    return pd.concat(blocks, ignore_index=True), pd.DataFrame(summary), pd.concat(threshold_blocks, ignore_index=True)\n\n\ndef old_number_checks(base: pd.DataFrame, canonical: pd.DataFrame, basins: pd.DataFrame) -> pd.DataFrame:\n    ba = grouped(base, \'ba_id\')\n    bstats, estats = concentration(basins.onsite_GL), concentration(ba.electricity_GL)\n    actual = {\'records\': len(base), \'capacity_mw\': base.capacity_mw.sum(), \'basins\': len(basins), \'BAs\': len(ba),\n              \'context_basins\': len(basins.dropna(subset=[\'bws_score\', \'bwd_score\'])),\n              \'basin_k50\': bstats[\'k50\'], \'BA_k50\': estats[\'k50\'], \'basin_Gini\': bstats[\'Gini\'], \'BA_Gini\': estats[\'Gini\'],\n              \'basin_HHI\': bstats[\'HHI\'], \'BA_HHI\': estats[\'HHI\'],\n              \'basin_top5_percent\': 100*bstats[\'top5_share\'], \'BA_top5_percent\': 100*estats[\'top5_share\'],\n              \'basin_top10_percent\': 100*bstats[\'top10_share\'], \'BA_top10_percent\': 100*estats[\'top10_share\'],\n              \'basin_k80\': bstats[\'k80\'], \'BA_k80\': estats[\'k80\']}\n    expected = {\'records\': (472, 0), \'capacity_mw\': (20041, 0.6), \'basins\': (63, 0), \'BAs\': (24, 0),\n                \'context_basins\': (62, 0), \'basin_k50\': (10, 0), \'BA_k50\': (3, 0),\n                \'basin_Gini\': (.539, .0006), \'BA_Gini\': (.666, .0006), \'basin_HHI\': (.0393, .00006), \'BA_HHI\': (.1517, .00006),\n                \'basin_top5_percent\': (34.74, .011), \'BA_top5_percent\': (72.36, .011),\n                \'basin_top10_percent\': (51.23, .011), \'BA_top10_percent\': (87.65, .011),\n                \'basin_k80\': (27, 0), \'BA_k80\': (8, 0)}\n    published = {\'efficiency\': (96.56,16.79,188.11,204.91), \'baseline\': (115.87,74.16,225.74,299.89),\n                 \'high_load_water_intensive\': (149.22,159.88,290.72,450.60), \'no_hydropower\': (115.87,74.16,128.35,202.50)}\n    for s, vals in published.items():\n        frame = canonical[canonical.scenario.eq(s)]\n        for c, target in zip([\'facility_electricity_MWh_y\', *WATER], vals):\n            key = s + \'/\' + c\n            actual[key] = float(frame[c].sum()/1e6)\n            expected[key] = (target, .011)\n    thresholds, _, _ = threshold_analysis(basins)\n    row = thresholds[thresholds.threshold.eq(3)].iloc[0]\n    for c, target, tol in [(\'bws_count\',25,0), (\'bwd_count\',7,0), (\'bws_joint_count\',15,0), (\'bwd_joint_count\',6,0),\n                           (\'bws_national_water_share\',.3624,.0001), (\'bwd_national_water_share\',.1181,.0001)]:\n        actual[c] = row[c]; expected[c] = (target,tol)\n    return pd.DataFrame([dict(quantity=k, current=float(actual[k]), manuscript=v, tolerance=t,\n                              passed=abs(float(actual[k])-v)<=t+1e-12) for k,(v,t) in expected.items()])\n\n\ndef tex_escape(s: Any) -> str:\n    replacements = {\'\\\\\': r\'\\textbackslash{}\', \'&\': r\'\\&\', \'%\': r\'\\%\', \'$\': r\'\\$\', \'#\': r\'\\#\',\n                    \'_\': r\'\\_\\allowbreak{}\', \'{\': r\'\\{\', \'}\': r\'\\}\', \'~\': r\'\\textasciitilde{}\', \'^\': r\'\\textasciicircum{}\'}\n    return \'\'.join(replacements.get(c,c) for c in str(s))\n\n\ndef formatted(v: Any, digits: int = 2) -> str:\n    if v is None or (isinstance(v,(float,np.floating)) and not math.isfinite(v)):\n        return \'NA\'\n    if isinstance(v, (bool,np.bool_)):\n        return \'yes\' if v else \'no\'\n    if isinstance(v, (int,np.integer)):\n        return str(v)\n    if isinstance(v, (float,np.floating)):\n        return f\'{v:.{digits}f}\'\n    return tex_escape(v)\n\n\ndef latex_table(frame: pd.DataFrame, headers: dict[str, str], caption: str, label: str, *, long: bool=False) -> str:\n    cols = list(headers)\n    n = len(cols)\n    # Modest table widths with wrapped, ragged-right cells; no tiny resize-to-fit text.\n    width_spec = r\'\\dimexpr\\textwidth/\' + str(n) + r\'-2\\tabcolsep\\relax\'\n    spec = \'@{}\' + (\'>{\\\\raggedright\\\\arraybackslash}p{\' + width_spec + \'}\') * n + \'@{}\'\n    begin = r\'\\begin{longtable}{\' + spec + \'}\' if long else \'\\\\begin{table}[!htbp]\\n\\\\centering\\n\\\\footnotesize\\n\\\\begin{tabular}{\'+spec+\'}\'\n    text = \'{\\\\footnotesize\\n\\\\setlength{\\\\tabcolsep}{3pt}\\n\' + begin + \'\\n\'\n    head = \' & \'.join(headers.values()) + r\' \\\\\' + \'\\n\'\n    if long:\n        text += r\'\\caption{\'+caption+r\'}\\label{\'+label+\'}\' + r\'\\\\\' + \'\\n\'\n    text += \'\\\\toprule\\n\'+head+\'\\\\midrule\\n\'\n    if long:\n        text += \'\\\\endfirsthead\\n\\\\toprule\\n\'+head+\'\\\\midrule\\n\\\\endhead\\n\'\n    for _, row in frame[cols].iterrows():\n        text += \' & \'.join(formatted(row[c], 4 if c == \'HHI\' else 3 if c in {\'Gini\',\'spearman\'} else 2) for c in cols) + r\' \\\\\' + \'\\n\'\n    text += \'\\\\bottomrule\\n\'\n    if long:\n        text += \'\\\\end{longtable}\\n}\\n\'\n    else:\n        text += \'\\\\end{tabular}\\n\\\\caption{\'+caption+\'}\\\\label{\'+label+\'}\\n\\\\end{table}\\n}\\n\'\n    return text\n\n\ndef read_optional(folder: Path, name: str) -> pd.DataFrame | None:\n    p = folder/name\n    if not p.exists():\n        return None\n    df = pd.read_csv(p, dtype={\'facility_id\':\'string\',\'hydrobasin_id\':\'string\'})\n    return df if len(df) else None\n\n\ndef export_tex(basins, hydro, hm, hc, thresholds, context, common, stages, stage_report,\n               climate, output: Path) -> None:\n    g = output/\'generated\'; g.mkdir(parents=True, exist_ok=True)\n    (g/\'regions_basins.tex\').write_text(latex_table(basins.head(10),\n        {\'hydrobasin_id\':\'PFAF ID\', \'basin_label\':\'Basin label\', \'hosting_states\':\'Hosted states\',\n         \'capacity_GW\':\'GW\',\'onsite_GL\':r\'On-site GL yr$^{-1}$\',\'bws_score\':\'BWS\',\'bwd_score\':\'BWD\'},\n        \'Leading hosting basins under the uniform reference parameters. Basin labels are supplied only with a documented crosswalk; state labels locate the hosted records, not utility intakes.\', \'tab:priority_basins\'), encoding=\'utf-8\')\n    pivot = hydro.pivot(index=\'ba_id\', columns=\'hydro_factor\', values=\'electricity_GL\').rename(columns={0.0:\'hydro0\',6.1:\'hydro61\',8.0:\'hydro8\'}).reset_index()\n    ref = hydro[hydro.hydro_factor.eq(8)].sort_values(\'rank\')\n    pivot = ref[[\'ba_id\',\'capacity_GW\',\'grid_L_per_kWh\']].merge(pivot,on=\'ba_id\',validate=\'one_to_one\')\n    (g/\'regions_ba.tex\').write_text(latex_table(pivot.head(5), {\'ba_id\':\'BA code\',\'capacity_GW\':\'GW\',\n        \'grid_L_per_kWh\':r\'Ref. L kWh$^{-1}$\',\'hydro8\':\'Hydro 8.0\',\'hydro61\':\'Hydro 6.1\',\'hydro0\':\'Hydro 0\'},\n        r\'Leading reference-case hosting BAs. The last three columns report electricity-related water in GL yr$^{-1}$ under the named hydropower coefficient in L kWh$^{-1}$.\',\'tab:priority_ba\'), encoding=\'utf-8\')\n    (g/\'threshold_table.tex\').write_text(latex_table(thresholds,{\'threshold\':\'Score cut\',\'bws_count\':\'BWS high\',\'bwd_count\':\'BWD high\',\n        \'bws_joint_count\':\'BWS joint\',\'bwd_joint_count\':\'BWD joint\',\'intersection\':\'Overlap\',\'b_subset_a\':\'BWD subset?\'},\n        \'Paired-context basin screens. Joint means high context and on-site burden at least the median of all positive-burden hosting basins. Empty sets are retained explicitly in the source data.\',\'tab:threshold_sweep\'), encoding=\'utf-8\')\n    (g/\'hydro_table.tex\').write_text(latex_table(hydro[[\'ba_id\',\'hydro_factor\',\'rank\',\'electricity_GL\',\'share\']],\n        {\'ba_id\':\'BA\',\'hydro_factor\':r\'Hydro L kWh$^{-1}$\',\'rank\':\'Rank\',\'electricity_GL\':r\'GL yr$^{-1}$\',\'share\':\'Water share\'},\n        \'All hosting-BA rankings across hydropower conventions. Ties use ascending BA code; continuous rank correlations use average ranks.\',\'tab:hydro_ranks\',long=True),encoding=\'utf-8\')\n    (g/\'common_partition.tex\').write_text(latex_table(common,{\'partition\':\'Partition\',\'quantity\':\'Quantity\',\'population\':\'Population\',\'R\':\'Regions\',\'k50\':r\'$k_{50}$\',\'Gini\':\'Gini\',\'HHI\':\'HHI\'},\n        \'Within-common-partition concentration, including all hosting units (zero water included). Without-PJM statistics are normalized to the remaining population.\',\'tab:common_partition\',long=True),encoding=\'utf-8\')\n    (g/\'stages.tex\').write_text(latex_table(stages,{\'stage_label\':\'Provider stage label\',\'stage_class\':\'Mapped class\',\'n_records\':\'Records\',\'capacity_mw\':\'MW\',\'capacity_share\':\'Capacity share\'},\n        \'Descriptive provider-stage coverage in the analytic inventory. Unknown labels remain unknown; stage does not establish metered load.\',\'tab:stage_coverage\',long=True),encoding=\'utf-8\')\n    stats = []\n    for name, frame in [(\'On-site basins\',basins.onsite_GL),(\'Electricity BAs\',ref.electricity_GL)]:\n        stats.append(dict(pathway=name,**concentration(frame)))\n    (g/\'concentration_table.tex\').write_text(latex_table(pd.DataFrame(stats),{\'pathway\':\'Pathway\',\'R\':\'Regions\',\'k50\':r\'$k_{50}$\',\'k80\':r\'$k_{80}$\',\'HHI\':\'HHI\',\'Gini\':\'Gini\'},\n        \'Concentration under the reference factor set. Cross-partition metrics are descriptive, not a physical clustering test.\',\'tab:updated_concentration\'),encoding=\'utf-8\')\n    def put(name, text): (g/name).write_text(text+\'\\n\\n\',encoding=\'utf-8\')\n    b = basins.head(3)\n    ids = \', \'.join(tex_escape(v) for v in b.hydrobasin_id)\n    leading = \', \'.join(tex_escape(v) for v in ref.head(concentration(ref.electricity_GL)[\'k50\']).ba_id)\n    put(\'region_results.tex\', f\'The three leading on-site hosting basins have PFAF identifiers {ids}. Table~\\\\ref{{tab:priority_basins}} reports the first ten together with hosted-state context and both Aqueduct scores. The smallest reference electricity set reaching half of the pathway consists of {leading} (Table~\\\\ref{{tab:priority_ba}}). The basin table describes modeled on-site burden under common parameters; the BA table additionally reflects the generation mix.\')\n    comp = hc[(hc.factor_a.eq(0)) & (hc.factor_b.eq(8)) & hc.set_definition.eq(\'k50\')].iloc[0]\n    zero = hm[hm.hydro_factor.eq(0) & hm.population.eq(\'all_hosting_BAs\')].iloc[0]\n    put(\'hydro_results.tex\', f\'Assigning zero hydropower evaporation requires {formatted(zero.k50,0)} hosting BAs to reach half of electricity-related water, compared with {concentration(ref.electricity_GL)["k50"]} in the reference case. The sets share {int(comp.intersection)} BAs; the rank correlation across all hosting BAs is {formatted(comp.spearman)}. The supplementary hydropower-ranking table reports every rank and amount under 0, 6.1 and 8.0 L kWh$^{{-1}}$.\')\n    t = thresholds[thresholds.threshold.eq(3)].iloc[0]\n    put(\'threshold_results.tex\', f\'At score 3, {int(t.intersection)} basins are flagged by both indicators. The high-depletion set \'+(\'is\' if t.b_subset_a else \'is not\')+\' a subset of the high-stress set. Across paired hosting basins, the continuous-score Spearman correlation is \'+formatted(context[\'spearman\'])+\'. Thresholds from 2 to 4 are evaluated separately in the supplementary threshold table; the 72\\\\% figure refers specifically to the original score-3 count comparison, not a universal percentage of risk removed.\')\n    pjm = ref[ref.ba_id.eq(\'PJM\')]\n    if len(pjm):\n        pr = pjm.iloc[0]; ex = hm[hm.hydro_factor.eq(8) & hm.population.eq(\'excluding_PJM\')].iloc[0]\n        put(\'pjm_results.tex\', f\'PJM represents {100*pr.capacity_mw/ref.capacity_mw.sum():.1f}\\\\% of hosted capacity and {100*pr.share:.1f}\\\\% of reference electricity-related water. Excluding PJM and renormalizing to the remaining hosting BAs gives $k_{{50}}={int(ex.k50)}$, Gini {ex.Gini:.3f} and HHI {ex.HHI:.4f}. This is a dependence diagnostic, not an alternative national estimate.\')\n    else:\n        put(\'pjm_results.tex\',\'PJM is absent from this run; the expected inventory must be checked before submission.\')\n    put(\'stage_results.tex\',f\'{stage_report["labelled_records"]} of {stage_report["n_records"]} analytic records carry a non-missing provider stage label, representing {100*stage_report["labelled_capacity_share"]:.1f}\\\\% of capacity (supplementary stage-coverage table). The original labels and their explicit mapping, where supplied, are retained rather than inferring operating status from current capacity.\')\n    if climate is None:\n        put(\'climate_results.tex\', r\'\\NWpending{Climate-dependent paired PUE/WUE sensitivity has not been run: documented coefficient and assignment inputs are required.}\')\n        put(\'climate_table.tex\', r\'\\NWpending{Climate results are unavailable; no empirical climate sensitivity is claimed.}\')\n    else:\n        cb, cs, ct = climate\n        prose=[]\n        for _, r in cs.iterrows():\n            prose.append(f\'For cooling case {tex_escape(r["case"])}, on-site consumption is {r.total:.2f} GL yr$^{{-1}}$, $k_{{50}}={formatted(r.k50,0)}$, and Gini is {r.Gini:.3f}. The ten-basin sets share {int(r.intersection)} members with the uniform reference; the basin-rank correlation is {formatted(r.spearman)}.\')\n        put(\'climate_results.tex\',\' \'.join(prose)+\' These paired-coefficient comparisons hold facility electricity fixed; they do not identify each facility\\\'s installed cooling equipment.\')\n        put(\'climate_table.tex\',latex_table(cs,{\'case\':\'Cooling case\',\'total\':r\'On-site GL yr$^{-1}$\',\'k50\':r\'$k_{50}$\',\'Gini\':\'Gini\',\'spearman\':r\'Rank $\\rho$\',\'intersection\':\'Top-10 overlap\'},\'Climate/topology-specific paired-coefficient sensitivity at fixed facility electricity.\',\'tab:climate_summary\'))\n\n\ndef run_revision(result: dict, bundle: str | Path, *, synthetic: bool = False) -> dict:\n    bundle = Path(bundle).resolve()\n    cfg = result[\'config\']\n    root = Path(cfg.get(\'resolved_paths\',cfg.get(\'paths\',{}))[\'output_root\'])\n    out = root / (\'reviewer_revision_SYNTHETIC\' if synthetic else \'reviewer_revision\')\n    require(not (out/\'SUBMISSION_APPROVED.json\').exists(), \'Approved output exists. Move it aside before replacing a signed run.\')\n    if out.exists():\n        # Only files in this module-specific result directory are replaced.\n        shutil.rmtree(out)\n    out.mkdir(parents=True)\n    base, canonical = validated_base(result[\'canonical\'])\n    inp = bundle/\'inputs\'\n    basins = basin_table(base, read_optional(inp,\'basin_labels.csv\'))\n    thresholds, sets, context = threshold_analysis(basins)\n    hydro, hm, hc = hydro_analysis(base)\n    common = common_partition(base)\n    stages, stage_report = stage_analysis(base, read_optional(inp,\'stage_mapping.csv\'))\n    pairs = read_optional(inp,\'climate_pairs.csv\')\n    climate = climate_analysis(base,pairs) if pairs is not None else None\n    tables = {\'basin_priorities\':basins,\'threshold_sweep\':thresholds,\'threshold_membership\':sets,\n              \'BA_hydropower_rankings\':hydro,\'hydropower_concentration\':hm,\'hydropower_rank_agreement\':hc,\n              \'common_partition_concentration\':common,\'inventory_stage\':stages}\n    if climate is not None:\n        tables.update(climate_basin_results=climate[0],climate_concentration=climate[1],climate_thresholds=climate[2])\n    for name, data in tables.items(): export(data,out/\'source_data\'/f\'{name}.csv\')\n    checks = old_number_checks(base,canonical,basins)\n    export(checks,out/\'diagnostics\'/\'manuscript_number_checks.csv\')\n    write_json(context,out/\'diagnostics\'/\'context_coverage.json\')\n    write_json(stage_report,out/\'diagnostics\'/\'stage_coverage.json\')\n    sources={str(p.relative_to(bundle)):digest(p) for p in sorted(bundle.rglob(\'*\')) if p.is_file() and\n             \'__pycache__\' not in str(p) and \'.ipynb_checkpoints\' not in str(p) and p.suffix in {\'.py\',\'.tex\',\'.csv\'}}\n    canonical_identity_cols = sorted(c for c in canonical.columns if c!=\'geometry\')\n    frame_bytes = canonical.sort_values([\'scenario\',\'facility_id\'])[canonical_identity_cols].to_csv(index=False,float_format=\'%.17g\').encode()\n    identity = hashlib.sha256(frame_bytes+json.dumps(sources,sort_keys=True).encode()).hexdigest()\n    status = {\'identity\':identity, \'synthetic\':synthetic, \'core_identities_passed\':True,\n              \'baseline_matches_manuscript\':bool(checks.passed.all()),\'climate_run\':climate is not None,\n              \'basin_labels_complete_top10\':bool(basins.head(10).basin_label.str.strip().ne(\'\').all()),\n              \'stage_mapping_complete\':stage_report[\'all_labels_classified\'],\n              \'software\':{\'python\':platform.python_version(),\'numpy\':np.__version__,\'pandas\':pd.__version__},\n              \'sources\':sources,\'author_signoff\':False,\n              \'notice\':\'Local analysis build. Not submission-approved until finalization and author checks pass.\'}\n    export_tex(basins,hydro,hm,hc,thresholds,context,common,stages,stage_report,climate,out)\n    from nw_plots import draw_revision_figures\n    draw_revision_figures(basins,hydro,thresholds,climate,out/\'images\')\n    # Copy existing canonical figure products from THIS run. Source 1 and new diagnostics are rebuilt.\n    core_figs = root/\'super_defensible_figures\'/\'figures\'\n    for p in core_figs.glob(\'*\'):\n        if p.suffix.lower() in {\'.png\',\'.pdf\',\'.svg\'} and not (out/\'images\'/p.name).exists():\n            shutil.copy2(p,out/\'images\'/p.name)\n    for name in [\'main.tex\',\'supplement.tex\',\'cover_letter.tex\']:\n        if (bundle/name).exists(): shutil.copy2(bundle/name,out/name)\n    status[\'generated_hashes\']={str(p.relative_to(out)):digest(p) for p in sorted(out.rglob(\'*\')) if p.is_file()}\n    write_json(status,out/\'RUN_STATUS.json\')\n    write_json({\'status\':\'AWAITING_AUTHOR_CONFIRMATION\',\'run_identity\':identity,\n                \'funding_statement\':\'\',\'competing_interests_statement\':\'\',\'reviewer_archive_url\':\'\',\'related_manuscript_statement\':\'\',\n                \'confirm_data_release_rights\':False,\'confirm_basin_labels\':False,\n                \'confirm_coefficient_provenance\':False,\'confirm_stage_mapping\':False,\n                \'confirm_sources_and_figures\':False,\'confirm_related_manuscript_disclosure\':False,\n                \'confirm_author_approval\':False},out/\'author_signoff.template.json\')\n    print(\'Revision outputs:\',out)\n    print(\'Original-manuscript number checks:\',bool(checks.passed.all()))\n    print(\'Climate sensitivity executed:\',climate is not None)\n    print(\'No final submission approval has been issued.\')\n    return {\'output_root\':out,\'status\':status,\'tables\':tables}\n\n\ndef finalize(output: Path, signoff_file: Path, bundle: Path) -> Path:\n    output, bundle = Path(output), Path(bundle)\n    status=json.loads((output/\'RUN_STATUS.json\').read_text())\n    sign=json.loads(Path(signoff_file).read_text())\n    require(not status[\'synthetic\'],\'Synthetic runs cannot be finalized\')\n    for key in [\'core_identities_passed\',\'baseline_matches_manuscript\',\'climate_run\',\'basin_labels_complete_top10\',\'stage_mapping_complete\']:\n        require(status[key],f\'Unresolved requirement: {key}\')\n    require(sign.get(\'run_identity\')==status[\'identity\'],\'Sign-off does not match this run\')\n    for key in [\'confirm_data_release_rights\',\'confirm_basin_labels\',\'confirm_coefficient_provenance\',\'confirm_stage_mapping\',\n                \'confirm_sources_and_figures\',\'confirm_related_manuscript_disclosure\',\'confirm_author_approval\']:\n        require(sign.get(key) is True,f\'Author confirmation required: {key}\')\n    for key in [\'funding_statement\',\'competing_interests_statement\',\'reviewer_archive_url\',\'related_manuscript_statement\']:\n        require(isinstance(sign.get(key),str) and bool(sign[key].strip()),f\'Missing {key}\')\n    require(sign[\'reviewer_archive_url\'].startswith(\'https://\'),\'Reviewer archive needs an HTTPS access URL\')\n    for relative,sha in status[\'sources\'].items():\n        require((bundle/relative).exists() and digest(bundle/relative)==sha,\'Sources changed since run; rerun: \'+relative)\n    for relative,sha in status[\'generated_hashes\'].items():\n        require((output/relative).exists() and digest(output/relative)==sha,\'Generated outputs changed after run: \'+relative)\n    require(not list((output/\'generated\').glob(\'*.tmp\')), \'Incomplete generated files\')\n    required_images=[\'fig_1_NEW.pdf\',\'Figure2_Pathway_Burden_Maps.png\',\'Figure3_Normalized_Concentration.png\',\n                     \'Figure4_State_Pathway_Dominance.png\',\'Figure4_State_Scenario_Intervals.png\',\n                     \'Figure4_BA_Pathway_Dominance.png\',\'Figure4_BA_Scenario_Intervals.png\',\n                     \'ExtendedData_Threshold_Sweep.pdf\',\'ExtendedData_Hydro_Ranks.pdf\',\'ExtendedData_Climate_Sensitivity.pdf\']\n    for image in required_images:\n        require((output/\'images\'/image).is_file(),\'Missing empirical figure: \'+image)\n    statements = \'\\\\renewcommand{\\\\FundingStatement}{\'+tex_escape(sign[\'funding_statement\'])+\'}\\n\'\n    statements += \'\\\\renewcommand{\\\\ConflictsStatement}{\'+tex_escape(sign[\'competing_interests_statement\'])+\'}\\n\'\n    statements += \'\\\\renewcommand{\\\\ReviewerArchive}{\\\\url{\'+sign[\'reviewer_archive_url\'].replace(\'%\',r\'\\%\')+\'}}\\n\'\n    statements += \'\\\\renewcommand{\\\\RelatedManuscriptStatement}{\'+tex_escape(sign[\'related_manuscript_statement\'])+\'}\\n\'\n    (output/\'generated\'/\'author_statements.tex\').write_text(statements)\n    for name in [\'main.tex\',\'supplement.tex\',\'cover_letter.tex\']:\n        p=output/name\n        if p.exists(): p.write_text(p.read_text().replace(r\'\\NWFinalfalse\',r\'\\NWFinaltrue\'))\n    status[\'author_signoff\']=True\n    write_json(status,output/\'SUBMISSION_APPROVED.json\')\n    archive=output.parent/\'Nature_Water_Overleaf_after_local_verification.zip\'\n    with zipfile.ZipFile(archive,\'w\',zipfile.ZIP_DEFLATED) as z:\n        for p in sorted(output.rglob(\'*\')):\n            if p.is_file() and (p.suffix in {\'.tex\',\'.pdf\',\'.png\',\'.svg\'} or p.name==\'SUBMISSION_APPROVED.json\'):\n                z.write(p,p.relative_to(output))\n    return archive\n'}
CWD = Path.cwd().resolve()
BUNDLE_ROOT = CWD if (CWD / 'code' / 'nw_review.py').exists() else CWD / 'Nature_Water_revision_rebuilt'
for relative, content in PAYLOAD.items():
    target = BUNDLE_ROOT / relative
    target.parent.mkdir(parents=True, exist_ok=True)
    if not target.exists():
        target.write_text(content, encoding='utf-8')
sys.path.insert(0, str(BUNDLE_ROOT / 'code'))
print('Rebuilt bundle:', BUNDLE_ROOT)
print('Existing edited files were preserved. See START_HERE_DELIVERY.md.')


In [ ]:
from pathlib import Path
import importlib.util

REQUIRED_IMPORTS = {
    "numpy": "numpy",
    "pandas": "pandas",
    "yaml": "pyyaml",
    "matplotlib": "matplotlib",
    "geopandas": "geopandas",
    "openpyxl": "openpyxl",
}
missing_packages = [pip_name for module_name, pip_name in REQUIRED_IMPORTS.items() if importlib.util.find_spec(module_name) is None]
if missing_packages:
    raise ImportError(
        "Missing Python packages: " + ", ".join(missing_packages) +
        ". Install them in this notebook kernel, restart the kernel, and Run All."
    )

CWD = Path.cwd().expanduser().resolve()
root_candidates = [CWD, *CWD.parents]
ROOT = next((candidate for candidate in root_candidates if (candidate / "REPRO.md").exists()), None)
if ROOT is None:
    raise FileNotFoundError(
        "Run this notebook from the hyperscale-water-geography repository or one of its subdirectories."
    )

RUNTIME_ROOT = ROOT / "_nature_water_runtime"
RUNTIME_ROOT.mkdir(parents=True, exist_ok=True)
print("Repository root:", ROOT)
print("Outputs will be written to:", ROOT / "nature_water_outputs")


In [ ]:
# The complete tested analysis package is embedded here so this notebook is the only file you need.
# This cell writes a private runtime copy beside the notebook; it does not alter your input data.
import importlib
import sys

MODULE_SOURCES = {'__init__.py': '"""Pathway-resolved operational water accounting for hyperscale data centers."""\n\nfrom .config import load_config\nfrom .model import build_canonical_scenario_table\nfrom .validate import validate_canonical_table\n\n__all__ = ["load_config", "build_canonical_scenario_table", "validate_canonical_table"]\n__version__ = "0.2.0"\n', 'aggregate.py': 'from __future__ import annotations\n\nfrom typing import Iterable, Mapping\n\nimport numpy as np\nimport pandas as pd\n\nfrom .errors import DataContractError\n\n\nWATER_COLUMNS = ["onsite_m3_y", "electricity_m3_y", "total_m3_y"]\nENERGY_COLUMNS = ["facility_electricity_MWh_y", "it_electricity_MWh_y"]\n\n\ndef national_summary(canonical: pd.DataFrame) -> pd.DataFrame:\n    grouped = canonical.groupby("scenario", sort=False, dropna=False)\n    out = grouped.agg(\n        n_facilities=("facility_id", "nunique"),\n        capacity_mw=("capacity_mw", "sum"),\n        facility_electricity_MWh_y=("facility_electricity_MWh_y", "sum"),\n        it_electricity_MWh_y=("it_electricity_MWh_y", "sum"),\n        onsite_m3_y=("onsite_m3_y", "sum"),\n        electricity_m3_y=("electricity_m3_y", "sum"),\n        total_m3_y=("total_m3_y", "sum"),\n    ).reset_index()\n    for source in WATER_COLUMNS:\n        out[source.replace("m3", "GL")] = out[source] / 1e6\n    out["total_L_per_facility_kWh"] = (\n        out["total_m3_y"] * 1000.0 / (out["facility_electricity_MWh_y"] * 1000.0)\n    )\n    out["electricity_share_of_total"] = out["electricity_m3_y"] / out["total_m3_y"]\n    return out\n\n\ndef _weighted_mean(values: pd.Series, weights: pd.Series) -> float:\n    v = pd.to_numeric(values, errors="coerce").to_numpy(dtype=float)\n    w = pd.to_numeric(weights, errors="coerce").to_numpy(dtype=float)\n    keep = np.isfinite(v) & np.isfinite(w) & (w > 0)\n    return float(np.average(v[keep], weights=w[keep])) if keep.any() else np.nan\n\n\ndef aggregate_by_geography(\n    canonical: pd.DataFrame,\n    geography: str,\n    *,\n    require_complete: bool = True,\n) -> pd.DataFrame:\n    if geography not in canonical:\n        raise DataContractError(f"Canonical table has no {geography!r} geography.")\n    if require_complete and canonical[geography].isna().any():\n        count = int(canonical[geography].isna().sum())\n        raise DataContractError(f"Cannot claim conservation: {count} rows lack {geography}.")\n    data = canonical.dropna(subset=[geography]).copy()\n    keys = ["scenario", geography]\n    out = data.groupby(keys, dropna=False).agg(\n        n_facilities=("facility_id", "nunique"),\n        capacity_mw=("capacity_mw", "sum"),\n        facility_electricity_MWh_y=("facility_electricity_MWh_y", "sum"),\n        onsite_m3_y=("onsite_m3_y", "sum"),\n        electricity_m3_y=("electricity_m3_y", "sum"),\n        total_m3_y=("total_m3_y", "sum"),\n    ).reset_index()\n\n    descriptor_candidates = [\n        "bws_score",\n        "bwd_score",\n        "grid_water_L_per_kWh",\n        "share_coal_gas",\n    ]\n    for descriptor in descriptor_candidates:\n        if descriptor not in data:\n            continue\n        weighted = (\n            data.groupby(keys, dropna=False)[[descriptor, "capacity_mw"]]\n            .apply(lambda d, col=descriptor: _weighted_mean(d[col], d["capacity_mw"]))\n            .rename(f"capacity_weighted_{descriptor}")\n            .reset_index()\n        )\n        out = out.merge(weighted, on=keys, how="left", validate="one_to_one")\n    return out.sort_values(keys).reset_index(drop=True)\n\n\ndef concentration_table(\n    regional: pd.DataFrame,\n    *,\n    value_column: str,\n    geography: str,\n    scenario: str = "baseline",\n) -> pd.DataFrame:\n    data = regional.loc[regional["scenario"].eq(scenario), [geography, value_column]].copy()\n    data = data.loc[data[value_column].gt(0)].sort_values(value_column, ascending=False).reset_index(drop=True)\n    total = data[value_column].sum()\n    data["rank"] = np.arange(1, len(data) + 1)\n    data["share"] = data[value_column] / total\n    data["cumulative_share"] = data["share"].cumsum()\n    data["scenario"] = scenario\n    data["pathway"] = value_column\n    return data\n\n\ndef regions_to_reach_share(concentration: pd.DataFrame, target: float = 0.50) -> tuple[int, float]:\n    if concentration.empty:\n        return 0, np.nan\n    hit = concentration.loc[concentration["cumulative_share"].ge(target)]\n    row = hit.iloc[0] if not hit.empty else concentration.iloc[-1]\n    return int(row["rank"]), float(row["cumulative_share"])\n\n\ndef scope1_hotspot_table(\n    canonical: pd.DataFrame,\n    *,\n    basin_column: str = "hydrobasin_id",\n    stress_column: str = "bws_score",\n    neighbor_stress_by_basin: pd.DataFrame | None = None,\n    burden_quantile: float = 0.50,\n    high_stress_threshold: float = 3.0,\n    scenario: str = "baseline",\n) -> tuple[pd.DataFrame, dict[str, float | str]]:\n    base = canonical.loc[canonical["scenario"].eq(scenario)].copy()\n    if basin_column not in base or stress_column not in base:\n        raise DataContractError(f"Scope-1 hotspots require {basin_column} and {stress_column}.")\n    table = aggregate_by_geography(base.assign(scenario=scenario), basin_column)\n    stress_name = f"capacity_weighted_{stress_column}"\n    if neighbor_stress_by_basin is not None:\n        expected = {basin_column, "stress_with_neighbor_max"}\n        if expected - set(neighbor_stress_by_basin):\n            raise DataContractError(f"Neighbour stress table must contain {sorted(expected)}.")\n        table = table.merge(\n            neighbor_stress_by_basin[[basin_column, "stress_with_neighbor_max"]],\n            on=basin_column,\n            how="left",\n            validate="one_to_one",\n        )\n        stress_name = "stress_with_neighbor_max"\n    threshold = float(table.loc[table["onsite_m3_y"].gt(0), "onsite_m3_y"].quantile(burden_quantile))\n    table["high_onsite_burden"] = table["onsite_m3_y"].ge(threshold)\n    table["high_stress"] = table[stress_name].ge(high_stress_threshold)\n    table["scope1_hotspot"] = table["high_onsite_burden"] & table["high_stress"]\n    table["scope1_class"] = np.select(\n        [\n            table["scope1_hotspot"],\n            table["high_onsite_burden"],\n            table["high_stress"],\n        ],\n        ["High burden + high stress", "High burden only", "High stress only"],\n        default="Neither",\n    )\n    meta = {\n        "scenario": scenario,\n        "burden_quantile": burden_quantile,\n        "burden_threshold_m3_y": threshold,\n        "stress_indicator": stress_column,\n        "stress_field_used": stress_name,\n        "high_stress_threshold": high_stress_threshold,\n    }\n    return table, meta\n\n\ndef scope2_hotspot_table(\n    canonical: pd.DataFrame,\n    *,\n    region_column: str = "ba_id",\n    burden_quantile: float = 0.50,\n    coal_gas_threshold: float = 0.50,\n    hosted_site_stress_threshold: float = 3.0,\n    stress_column: str = "bws_score",\n    scenario: str = "baseline",\n) -> tuple[pd.DataFrame, dict[str, float | str]]:\n    base = canonical.loc[canonical["scenario"].eq(scenario)].copy()\n    table = aggregate_by_geography(base.assign(scenario=scenario), region_column)\n    burden_threshold = float(\n        table.loc[table["electricity_m3_y"].gt(0), "electricity_m3_y"].quantile(burden_quantile)\n    )\n    hosted_stress = f"capacity_weighted_{stress_column}"\n    table["high_electricity_burden"] = table["electricity_m3_y"].ge(burden_threshold)\n    table["coal_gas_heavy"] = table["capacity_weighted_share_coal_gas"].gt(coal_gas_threshold)\n    table["high_hosted_site_stress"] = table[hosted_stress].ge(hosted_site_stress_threshold)\n    table["scope2_hotspot"] = (\n        table["high_electricity_burden"]\n        & table["coal_gas_heavy"]\n        & table["high_hosted_site_stress"]\n    )\n    table["scope2_code"] = (\n        table["high_electricity_burden"].astype(int) * 4\n        + table["coal_gas_heavy"].astype(int) * 2\n        + table["high_hosted_site_stress"].astype(int)\n    )\n    meta = {\n        "scenario": scenario,\n        "burden_quantile": burden_quantile,\n        "burden_threshold_m3_y": burden_threshold,\n        "coal_gas_threshold": coal_gas_threshold,\n        "hosted_site_stress_indicator": stress_column,\n        "hosted_site_stress_threshold": hosted_site_stress_threshold,\n        "interpretation": "stress at hosted facilities; not the physical source basin of generation water",\n    }\n    return table, meta\n\n\ndef regional_scenario_intervals(\n    canonical: pd.DataFrame,\n    geography: str,\n    *,\n    rank_scenario: str = "baseline",\n    top_n: int = 15,\n) -> pd.DataFrame:\n    regional = aggregate_by_geography(canonical, geography)\n    leaders = (\n        regional.loc[regional["scenario"].eq(rank_scenario)]\n        .nlargest(top_n, "capacity_mw")[geography]\n        .tolist()\n    )\n    out = regional.loc[regional[geography].isin(leaders)].copy()\n    ranks = {region: i + 1 for i, region in enumerate(leaders)}\n    out["capacity_rank"] = out[geography].map(ranks)\n    return out.sort_values(["capacity_rank", "scenario"])\n', 'config.py': 'from __future__ import annotations\n\nimport hashlib\nimport json\nfrom copy import deepcopy\nfrom pathlib import Path\nfrom typing import Any, Mapping\n\nimport yaml\n\nfrom .errors import DataContractError\n\n\nREQUIRED_SCENARIO_FIELDS = {\n    "utilization",\n    "pue",\n    "wue_L_per_kWh_IT",\n    "onsite_consumptive_fraction",\n    "hydro_factor_L_per_kWh",\n}\n\n\ndef stable_hash(value: Any) -> str:\n    payload = json.dumps(value, sort_keys=True, separators=(",", ":"), default=str)\n    return hashlib.sha256(payload.encode("utf-8")).hexdigest()[:16]\n\n\ndef load_config(path: str | Path) -> dict[str, Any]:\n    """Load, validate, and resolve repository-relative paths."""\n    config_path = Path(path).expanduser().resolve()\n    with config_path.open("r", encoding="utf-8") as handle:\n        cfg = yaml.safe_load(handle)\n    if not isinstance(cfg, dict):\n        raise DataContractError("Configuration must be a YAML mapping.")\n\n    cfg = deepcopy(cfg)\n    cfg["_config_path"] = str(config_path)\n    cfg["_repo_root"] = str(config_path.parent.parent)\n    _validate_config(cfg)\n\n    root = Path(cfg["_repo_root"])\n    resolved = {}\n    for name, raw in cfg.get("paths", {}).items():\n        if raw is None:\n            resolved[name] = None\n            continue\n        p = Path(raw).expanduser()\n        resolved[name] = str(p if p.is_absolute() else (root / p).resolve())\n    cfg["resolved_paths"] = resolved\n    cfg["config_hash"] = stable_hash(_public_config(cfg))\n    return cfg\n\n\ndef _public_config(cfg: Mapping[str, Any]) -> dict[str, Any]:\n    return {k: v for k, v in cfg.items() if not str(k).startswith("_") and k != "resolved_paths"}\n\n\ndef _validate_config(cfg: Mapping[str, Any]) -> None:\n    project = cfg.get("project", {})\n    if project.get("capacity_basis") != "facility_input_nameplate_mw":\n        raise DataContractError(\n            "This revision assumes facility-input/nameplate MW. Set "\n            "project.capacity_basis=\'facility_input_nameplate_mw\' or change the equations explicitly."\n        )\n    if project.get("accounting_boundary") != "operational_water_consumption":\n        raise DataContractError("The model accepts operational water-consumption factors only.")\n\n    # The old analysis mixed a December inventory snapshot with a March study\n    # endpoint.  Force the two dates to agree unless the user updates both.\n    snapshot_date = cfg.get("inventory", {}).get("snapshot_date")\n    study_end_date = project.get("study_end_date")\n    if snapshot_date and study_end_date and str(snapshot_date) != str(study_end_date):\n        raise DataContractError(\n            "project.study_end_date must match inventory.snapshot_date. Update both when a newer "\n            "facility snapshot is acquired; do not relabel an older inventory as March 2026."\n        )\n\n    scenarios = cfg.get("scenarios")\n    if not isinstance(scenarios, dict) or not scenarios:\n        raise DataContractError("At least one scenario is required.")\n    required_scenarios = {"efficiency", "baseline", "high_load_water_intensive", "no_hydropower"}\n    if required_scenarios - set(scenarios):\n        raise DataContractError(f"Nature Water configuration is missing scenarios: {sorted(required_scenarios - set(scenarios))}")\n    if set(scenarios) - required_scenarios:\n        raise DataContractError(f"Figure 1 is locked to four scenarios; unexpected scenarios: {sorted(set(scenarios) - required_scenarios)}")\n    for name, values in scenarios.items():\n        missing = REQUIRED_SCENARIO_FIELDS - set(values or {})\n        if missing:\n            raise DataContractError(f"Scenario {name!r} is missing {sorted(missing)}.")\n        u = float(values["utilization"])\n        pue = float(values["pue"])\n        wue = float(values["wue_L_per_kWh_IT"])\n        consumptive_fraction = float(values["onsite_consumptive_fraction"])\n        hydro_factor = float(values["hydro_factor_L_per_kWh"])\n        if not 0 <= u <= 1:\n            raise DataContractError(f"Scenario {name!r}: utilization must be in [0, 1].")\n        if pue < 1:\n            raise DataContractError(f"Scenario {name!r}: PUE must be >=1.")\n        if wue < 0:\n            raise DataContractError(f"Scenario {name!r}: WUE cannot be negative.")\n        if not 0 <= consumptive_fraction <= 1:\n            raise DataContractError(f"Scenario {name!r}: onsite_consumptive_fraction must be in [0, 1].")\n        if hydro_factor < 0:\n            raise DataContractError(f"Scenario {name!r}: hydropower factor cannot be negative.")\n\n    baseline = scenarios["baseline"]\n    no_hydro = scenarios["no_hydropower"]\n    unchanged = ["utilization", "pue", "wue_L_per_kWh_IT", "onsite_consumptive_fraction"]\n    if any(float(no_hydro[key]) != float(baseline[key]) for key in unchanged):\n        raise DataContractError("The no-hydropower sensitivity must differ from baseline only in its hydropower factor.")\n    if float(no_hydro["hydro_factor_L_per_kWh"]) != 0:\n        raise DataContractError("The no-hydropower sensitivity must set hydropower to zero.")\n\n    factors = cfg.get("technology_water_consumption_L_per_kWh", {})\n    required_factors = {"coal", "gas", "nuclear", "hydro", "wind", "solar", "other"}\n    if required_factors - set(factors):\n        raise DataContractError(\n            "Technology factor set is incomplete: " + ", ".join(sorted(required_factors - set(factors)))\n        )\n    if any(float(v) < 0 for v in factors.values()):\n        raise DataContractError("Water-consumption factors cannot be negative.")\n\n    aq = cfg.get("aqueduct", {})\n    if aq.get("annual_only", True) and aq.get("primary_indicator") != "bws_score":\n        raise DataContractError(\n            "The primary annual Aqueduct 4.0 indicator must be \'bws_score\'; "\n            "monthly fields such as bws_01_score are not annual baseline stress."\n        )\n    if aq.get("sensitivity_indicator") != "bwd_score":\n        raise DataContractError("Use annual Aqueduct \'bwd_score\' for the depletion sensitivity.")\n\n\ndef scenario_frame(cfg: Mapping[str, Any]):\n    import pandas as pd\n\n    rows = []\n    for name, values in cfg["scenarios"].items():\n        row = {"scenario": name, **values}\n        row["scenario_hash"] = stable_hash(row)\n        rows.append(row)\n    return pd.DataFrame(rows)\n', 'errors.py': 'class DataContractError(ValueError):\n    """Raised when an input would make the analysis ambiguous or mislabeled."""\n\n\nclass ValidationError(AssertionError):\n    """Raised when a model or aggregation invariant fails."""\n', 'geospatial.py': 'from __future__ import annotations\n\nfrom typing import Iterable\n\nimport numpy as np\nimport pandas as pd\n\nfrom .errors import DataContractError\n\n\ndef facilities_to_geodataframe(facilities: pd.DataFrame):\n    import geopandas as gpd\n\n    if facilities[["latitude", "longitude"]].isna().any().any():\n        raise DataContractError("Facility coordinates must be complete before spatial assignment.")\n    return gpd.GeoDataFrame(\n        facilities.copy(),\n        geometry=gpd.points_from_xy(facilities["longitude"], facilities["latitude"]),\n        crs="EPSG:4326",\n    )\n\n\ndef _clean_id(series: pd.Series) -> pd.Series:\n    out = series.astype("string").str.strip().str.upper()\n    return out.mask(out.isin(["", "NAN", "NONE", "NULL", "<NA>"]))\n\n\ndef assign_polygon_id(\n    points,\n    polygons,\n    *,\n    polygon_source_id: str,\n    output_id: str,\n    label: str,\n    require_complete: bool = True,\n):\n    """Assign exactly zero or one polygon to each point, preserving row count."""\n    import geopandas as gpd\n\n    if "facility_id" not in points:\n        raise DataContractError("Point layer must contain facility_id.")\n    if polygon_source_id not in polygons:\n        raise DataContractError(f"{label} polygons lack {polygon_source_id!r}.")\n    if polygons.crs is None or points.crs is None:\n        raise DataContractError(f"{label} spatial layers must declare a CRS.")\n\n    left = points.copy()\n    right = polygons[[polygon_source_id, "geometry"]].copy().to_crs(left.crs)\n    right[polygon_source_id] = _clean_id(right[polygon_source_id])\n    if right[polygon_source_id].isna().any():\n        raise DataContractError(f"{label} polygon IDs contain missing values.")\n\n    joined = gpd.sjoin(left, right, how="left", predicate="within")\n    counts = joined.groupby("facility_id", dropna=False)[polygon_source_id].count()\n    ambiguous = counts[counts > 1]\n    if not ambiguous.empty:\n        raise DataContractError(\n            f"{label} spatial join assigned multiple polygons to {len(ambiguous)} facilities; "\n            f"examples: {ambiguous.index[:10].tolist()}"\n        )\n    joined = joined.drop(columns=["index_right"], errors="ignore")\n    if len(joined) != len(left):\n        raise DataContractError(f"{label} spatial join changed facility row count.")\n    joined = joined.rename(columns={polygon_source_id: output_id})\n    joined[output_id] = _clean_id(joined[output_id])\n    if require_complete and joined[output_id].isna().any():\n        missing = joined.loc[joined[output_id].isna(), "facility_id"].head(10).tolist()\n        raise DataContractError(f"{label} assignment is missing for facilities: {missing}")\n    return joined\n\n\ndef attach_aqueduct_scores(\n    facilities,\n    aqueduct_annual,\n    *,\n    hydrobasin_id_column: str = "hydrobasin_id",\n    require_complete: bool = True,\n):\n    """Prefer exact point-in-polygon Aqueduct assignment; allow a unique pfaf table."""\n    import geopandas as gpd\n\n    required_scores = ["bws_score", "bwd_score"]\n    missing = [c for c in required_scores if c not in aqueduct_annual]\n    if missing:\n        raise DataContractError(f"Aqueduct annual layer lacks {missing}.")\n\n    if isinstance(aqueduct_annual, gpd.GeoDataFrame):\n        right = aqueduct_annual[["bws_score", "bwd_score", "geometry"]].copy().to_crs(facilities.crs)\n        joined = gpd.sjoin(facilities, right, how="left", predicate="within")\n        counts = joined.loc[joined["index_right"].notna()].groupby("facility_id", dropna=False).size()\n        if (counts > 1).any():\n            raise DataContractError(\n                "Aqueduct geometry assigns multiple annual polygons to a facility. "\n                "Resolve the documented Aqueduct string_id/admin/aquifer geometry explicitly."\n            )\n        joined = joined.drop(columns=["index_right"], errors="ignore")\n        if len(joined) != len(facilities):\n            raise DataContractError("Aqueduct spatial join changed the facility row count.")\n    else:\n        if hydrobasin_id_column not in facilities:\n            raise DataContractError("A tabular Aqueduct join requires a prior hydrobasin_id assignment.")\n        aq = aqueduct_annual[["pfaf_id", *required_scores]].copy()\n        if aq["pfaf_id"].duplicated().any():\n            raise DataContractError("Tabular Aqueduct scores are not unique by pfaf_id.")\n        aq["pfaf_id"] = _clean_id(aq["pfaf_id"])\n        joined = facilities.merge(\n            aq,\n            how="left",\n            left_on=hydrobasin_id_column,\n            right_on="pfaf_id",\n            validate="many_to_one",\n        ).drop(columns="pfaf_id")\n        joined = gpd.GeoDataFrame(joined, geometry="geometry", crs=facilities.crs)\n\n    for score in required_scores:\n        joined[score] = pd.to_numeric(joined[score], errors="coerce")\n    if require_complete and joined[required_scores].isna().any().any():\n        missing_ids = joined.loc[joined[required_scores].isna().any(axis=1), "facility_id"].head(10).tolist()\n        raise DataContractError(f"Annual Aqueduct scores are missing for facilities: {missing_ids}")\n    return joined\n\n\ndef add_neighbor_stress(\n    polygons,\n    *,\n    polygon_id: str,\n    stress_column: str,\n    output_column: str = "neighbor_max_stress",\n):\n    """Calculate the maximum stress in touching/intersecting neighbours; never alter burden."""\n    import geopandas as gpd\n\n    base = polygons[[polygon_id, stress_column, "geometry"]].copy()\n    if base[polygon_id].duplicated().any():\n        raise DataContractError(f"{polygon_id} must be unique for neighbour analysis.")\n    pairs = gpd.sjoin(base, base, how="left", predicate="intersects", lsuffix="self", rsuffix="neighbor")\n    self_id = f"{polygon_id}_self"\n    neighbor_id = f"{polygon_id}_neighbor"\n    neighbor_score = f"{stress_column}_neighbor"\n    pairs = pairs.loc[pairs[self_id] != pairs[neighbor_id]].copy()\n    max_neighbor = pairs.groupby(self_id, as_index=True)[neighbor_score].max()\n    out = polygons.copy()\n    out[output_column] = out[polygon_id].map(max_neighbor)\n    out["stress_with_neighbor_max"] = out[[stress_column, output_column]].max(axis=1)\n    return out\n\n\ndef geography_coverage_table(facilities: pd.DataFrame, columns: Iterable[str]) -> pd.DataFrame:\n    rows = []\n    total = len(facilities)\n    for column in columns:\n        if column not in facilities:\n            rows.append({"geography": column, "assigned": 0, "total": total, "coverage": 0.0})\n        else:\n            assigned = int(facilities[column].notna().sum())\n            rows.append({"geography": column, "assigned": assigned, "total": total, "coverage": assigned / total if total else np.nan})\n    return pd.DataFrame(rows)\n', 'io.py': 'from __future__ import annotations\n\nimport re\nfrom pathlib import Path\nfrom typing import Any, Iterable, Mapping\n\nimport numpy as np\nimport pandas as pd\n\nfrom .errors import DataContractError\n\n\nCANONICAL_TECHNOLOGIES = ("coal", "gas", "nuclear", "hydro", "wind", "solar", "other")\n\nEGRID_SHARE_COLUMNS: dict[str, tuple[str, ...]] = {\n    "coal": ("share_coal", "BA coal generation percent (resource mix)"),\n    "gas": ("share_gas", "BA gas generation percent (resource mix)"),\n    "nuclear": ("share_nuclear", "BA nuclear generation percent (resource mix)"),\n    "hydro": ("share_hydro", "BA hydro generation percent (resource mix)"),\n    "wind": ("share_wind", "BA wind generation percent (resource mix)"),\n    "solar": ("share_solar", "BA solar generation percent (resource mix)"),\n}\n\nEGRID_OTHER_COLUMNS = (\n    "share_oil",\n    "share_biomass",\n    "share_geothermal",\n    "share_other_fossil",\n    "share_other_unknown",\n    "share_other",\n    "BA oil generation percent (resource mix)",\n    "BA biomass generation percent (resource mix)",\n    "BA geothermal generation percent (resource mix)",\n    "BA other fossil generation percent (resource mix)",\n    "BA other unknown/ purchased fuel generation percent (resource mix)",\n)\n\n\ndef read_table(path: str | Path, *, sheet_name: str | int = 0) -> pd.DataFrame:\n    path = Path(path)\n    if not path.exists():\n        raise FileNotFoundError(path)\n    suffix = path.suffix.lower()\n    if suffix == ".csv":\n        return pd.read_csv(path)\n    if suffix in {".xlsx", ".xls"}:\n        return pd.read_excel(path, sheet_name=sheet_name)\n    if suffix in {".parquet", ".pq"}:\n        return pd.read_parquet(path)\n    if suffix in {".feather", ".arrow"}:\n        return pd.read_feather(path)\n    raise DataContractError(f"Unsupported tabular input: {path}")\n\n\ndef read_geodata(path: str | Path, *, layer: str | None = None):\n    try:\n        import geopandas as gpd\n    except ImportError as exc:\n        raise ImportError("Geospatial inputs require geopandas; install the project environment.") from exc\n    path = Path(path)\n    if not path.exists():\n        raise FileNotFoundError(path)\n    if path.suffix.lower() in {".parquet", ".pq"}:\n        return gpd.read_parquet(path)\n    return gpd.read_file(path, layer=layer) if layer else gpd.read_file(path)\n\n\ndef _require_columns(df: pd.DataFrame, columns: Iterable[str], label: str) -> None:\n    missing = [c for c in columns if c not in df.columns]\n    if missing:\n        raise DataContractError(f"{label} is missing required columns: {missing}")\n\n\ndef _clean_identifier(series: pd.Series) -> pd.Series:\n    out = series.astype("string").str.strip().str.upper()\n    return out.mask(out.isin(["", "NAN", "NONE", "NULL", "<NA>"]))\n\n\ndef standardize_facilities(raw: pd.DataFrame, cfg: Mapping[str, Any]) -> tuple[pd.DataFrame, pd.DataFrame]:\n    """Return a one-row-per-facility table and an explicit attrition audit."""\n    cols = cfg["columns"]\n    mapping = {\n        cols["facility_id"]: "facility_id",\n        cols["latitude"]: "latitude",\n        cols["longitude"]: "longitude",\n        cols["capacity_mw"]: "capacity_mw",\n        cols["company"]: "company",\n        cols["company_type"]: "company_type",\n        cols["stage"]: "stage",\n        cols["commissioning_year"]: "commissioning_year",\n        cols["construction_mw"]: "construction_mw",\n        cols["planned_mw"]: "planned_mw",\n    }\n    required_source = [cols[k] for k in ("facility_id", "latitude", "longitude", "capacity_mw")]\n    if cfg["inventory"].get("require_hyperscale_label", True):\n        required_source.append(cols["company_type"])\n    _require_columns(raw, required_source, "facility inventory")\n\n    available_mapping = {src: dst for src, dst in mapping.items() if src in raw.columns}\n    keep = list(dict.fromkeys([*available_mapping, *[c for c in raw.columns if c not in available_mapping]]))\n    out = raw.loc[:, keep].rename(columns=available_mapping).copy()\n    audit = []\n\n    def record(step: str, frame: pd.DataFrame) -> None:\n        audit.append(\n            {\n                "step": step,\n                "rows": int(len(frame)),\n                "unique_facilities": int(frame["facility_id"].nunique(dropna=True)) if "facility_id" in frame else np.nan,\n                "capacity_mw": float(pd.to_numeric(frame.get("capacity_mw"), errors="coerce").sum()),\n            }\n        )\n\n    record("raw_inventory", out)\n    out["facility_id"] = _clean_identifier(out["facility_id"])\n    if out["facility_id"].isna().any():\n        raise DataContractError("Facility IDs contain missing values.")\n    duplicated = out["facility_id"].duplicated(keep=False)\n    if duplicated.any():\n        ids = out.loc[duplicated, "facility_id"].drop_duplicates().head(10).tolist()\n        raise DataContractError(f"Facility IDs are not unique; examples: {ids}")\n\n    if cfg["inventory"].get("require_hyperscale_label", True):\n        pattern = cfg["inventory"]["hyperscale_regex"]\n        mask = out["company_type"].astype("string").str.contains(pattern, case=False, na=False, regex=True)\n        out = out.loc[mask].copy()\n        record("hyperscale_label", out)\n\n    for column in ["latitude", "longitude", "capacity_mw", "commissioning_year", "construction_mw", "planned_mw"]:\n        if column in out:\n            out[column] = pd.to_numeric(out[column], errors="coerce")\n    out = out.loc[out["latitude"].between(-90, 90) & out["longitude"].between(-180, 180)].copy()\n    record("valid_coordinates", out)\n    if cfg["inventory"].get("include_only_positive_current_capacity", True):\n        out = out.loc[out["capacity_mw"].gt(0) & np.isfinite(out["capacity_mw"])].copy()\n        record("positive_current_capacity", out)\n\n    if "stage" in out:\n        out["stage"] = out["stage"].astype("string").str.strip()\n    out["capacity_basis"] = cfg["project"]["capacity_basis"]\n    out = out.sort_values("facility_id").reset_index(drop=True)\n    return out, pd.DataFrame(audit)\n\n\ndef _first_existing(df: pd.DataFrame, candidates: Iterable[str]) -> str | None:\n    return next((c for c in candidates if c in df.columns), None)\n\n\ndef standardize_generation_mix(\n    raw: pd.DataFrame,\n    *,\n    region_id_candidates: Iterable[str],\n    share_sum_min: float = 0.98,\n    share_sum_max: float = 1.02,\n    allow_rounding_renormalization: bool = True,\n) -> tuple[pd.DataFrame, pd.DataFrame]:\n    """Map every generation source exactly once to seven canonical technologies."""\n    region_col = _first_existing(raw, region_id_candidates)\n    if region_col is None:\n        raise DataContractError(f"Could not find a region ID; tried {list(region_id_candidates)}")\n    df = raw.copy()\n    df["region_id"] = _clean_identifier(df[region_col])\n    df = df.loc[df["region_id"].notna() & ~df["region_id"].isin(["BACODE", "SUBRGN"])].copy()\n    if df["region_id"].duplicated().any():\n        ids = df.loc[df["region_id"].duplicated(keep=False), "region_id"].unique()[:10]\n        raise DataContractError(f"Generation mix has duplicate region IDs: {ids.tolist()}")\n\n    result = pd.DataFrame({"region_id": df["region_id"]})\n    canonical_source_columns = [f"share_{t}" for t in CANONICAL_TECHNOLOGIES]\n    if set(canonical_source_columns).issubset(df.columns):\n        # An already-standardized table is accepted exactly once; component columns are ignored.\n        for column in canonical_source_columns:\n            result[column] = pd.to_numeric(df[column], errors="coerce")\n    else:\n        used_source_columns: set[str] = set()\n        for technology, candidates in EGRID_SHARE_COLUMNS.items():\n            source = _first_existing(df, candidates)\n            if source is None:\n                raise DataContractError(f"Generation mix is missing the {technology!r} category; tried {candidates}.")\n            result[f"share_{technology}"] = pd.to_numeric(df[source], errors="coerce")\n            used_source_columns.add(source)\n\n        # Raw eGRID residual categories are mutually exclusive and are collapsed once into \'other\'.\n        other_sources = [c for c in EGRID_OTHER_COLUMNS if c in df.columns and c not in used_source_columns and c != "share_other"]\n        if not other_sources and "share_other" in df:\n            other_sources = ["share_other"]\n        if not other_sources:\n            raise DataContractError(\n                "Generation mix has no residual/other source category. Map oil, biomass, geothermal, "\n                "other fossil, and unknown generation explicitly rather than dropping them."\n            )\n        other_matrix = df[other_sources].apply(pd.to_numeric, errors="coerce")\n        result["share_other"] = other_matrix.sum(axis=1, min_count=1)\n\n    share_columns = [f"share_{t}" for t in CANONICAL_TECHNOLOGIES]\n    values = result[share_columns]\n    # eGRID source columns are percentages, while already-standardized inputs are fractions.\n    if float(values.max(skipna=True).max(skipna=True)) > 1.5:\n        result[share_columns] = values / 100.0\n    # A row with no reported shares is not a zero-water grid.  Exclude it from\n    # the mix table so that a hosting facility in that region fails explicitly\n    # during the facility-to-mix join.\n    no_data = result[share_columns].isna().all(axis=1) | result[share_columns].fillna(0).sum(axis=1).eq(0)\n    audit_rows = []\n    if no_data.any():\n        empty = result.loc[no_data, ["region_id"]].copy()\n        empty["raw_share_sum"] = np.nan\n        empty["issue"] = "no generation-share data; region excluded"\n        audit_rows.append(empty)\n        result = result.loc[~no_data].copy()\n    if result.empty:\n        raise DataContractError("Generation mix contains no region with a non-zero, complete resource mix.")\n    if (result[share_columns].fillna(0) < 0).any().any():\n        raise DataContractError("Generation shares cannot be negative.")\n    result["raw_share_sum"] = result[share_columns].sum(axis=1, min_count=len(share_columns))\n    result["share_sum_was_renormalized"] = False\n\n    bad = ~result["raw_share_sum"].between(share_sum_min, share_sum_max)\n    audit = result.loc[bad, ["region_id", "raw_share_sum"]].copy()\n    audit["issue"] = "generation shares outside allowed rounding range"\n    if audit_rows:\n        audit = pd.concat([*audit_rows, audit], ignore_index=True)\n    if bad.any():\n        examples = audit.head(10).to_dict("records")\n        raise DataContractError(\n            "Generation shares do not sum to approximately one. No category was silently dropped; "\n            f"examples: {examples}"\n        )\n    if allow_rounding_renormalization:\n        needs = ~np.isclose(result["raw_share_sum"], 1.0, rtol=0, atol=1e-12)\n        result.loc[needs, share_columns] = result.loc[needs, share_columns].div(\n            result.loc[needs, "raw_share_sum"], axis=0\n        )\n        result.loc[needs, "share_sum_was_renormalized"] = True\n    elif not np.allclose(result["raw_share_sum"], 1.0, atol=1e-12):\n        raise DataContractError("Generation shares require rounding normalization, but it is disabled.")\n\n    result["share_coal_gas"] = result["share_coal"] + result["share_gas"]\n    return result.sort_values("region_id").reset_index(drop=True), audit\n\n\ndef validate_aqueduct_contract(df: pd.DataFrame, *, require_depletion: bool = True) -> None:\n    """Forbid the old January-as-annual mistake and require named annual indicators."""\n    monthly = [c for c in df.columns if re.fullmatch(r"(?:bws|bwd)_(?:0[1-9]|1[0-2])_(?:raw|score|cat|label)", c)]\n    required = ["pfaf_id", "bws_score"] + (["bwd_score"] if require_depletion else [])\n    missing = [c for c in required if c not in df.columns]\n    if missing:\n        hint = ""\n        if monthly:\n            hint = (\n                f" Detected monthly fields such as {monthly[:3]}; for example, bws_01_score is January, "\n                "not annual baseline water stress."\n            )\n        raise DataContractError(f"Aqueduct baseline-annual input is missing {missing}.{hint}")\n    for indicator in ["bws_score", "bwd_score"]:\n        if indicator in df:\n            values = pd.to_numeric(df[indicator], errors="coerce")\n            invalid = values.notna() & ~values.between(0, 5)\n            if invalid.any():\n                raise DataContractError(f"Aqueduct {indicator} has values outside the documented 0–5 scale.")\n\n\ndef load_aqueduct_annual(\n    path: str | Path,\n    *,\n    duplicate_pfaf_policy: str = "error",\n    layer: str | None = None,\n):\n    path = Path(path)\n    geospatial = path.suffix.lower() not in {".csv", ".xlsx", ".xls", ".parquet", ".pq"}\n    frame = read_geodata(path, layer=layer) if geospatial else read_table(path)\n    frame = frame.rename(columns={"PFAF_ID": "pfaf_id"})\n\n    # Aqueduct distribution tables commonly encode unavailable observations\n    # with a negative numeric sentinel (for example -9999).  Those are not\n    # stress scores and must become missing values before enforcing the 0--5\n    # score contract.  Positive values above 5 remain hard errors.\n    for indicator in ["bws_score", "bwd_score"]:\n        if indicator in frame:\n            values = pd.to_numeric(frame[indicator], errors="coerce")\n            frame[indicator] = values.mask(values.lt(0))\n\n    validate_aqueduct_contract(frame)\n    frame["pfaf_id"] = frame["pfaf_id"].astype("string").str.strip()\n    if not geospatial and frame["pfaf_id"].duplicated().any():\n        if duplicate_pfaf_policy == "error":\n            raise DataContractError(\n                "Aqueduct annual table has multiple rows per pfaf_id. Use the geospatial Aqueduct layer "\n                "and a point-in-polygon join; dropping duplicates would choose stress values arbitrarily."\n            )\n        if duplicate_pfaf_policy == "collapse_identical":\n            indicators = [c for c in ["bws_score", "bwd_score"] if c in frame.columns]\n            conflicts = (\n                frame.groupby("pfaf_id", dropna=False)[indicators]\n                .nunique(dropna=True)\n                .gt(1)\n                .any(axis=1)\n            )\n            if conflicts.any():\n                ids = conflicts[conflicts].index.astype(str).tolist()[:10]\n                raise DataContractError(\n                    "Aqueduct annual table has conflicting stress values within duplicate pfaf_id rows: "\n                    f"{ids}. Use the geospatial Aqueduct layer so facilities can be assigned unambiguously."\n                )\n            # GroupBy.first skips missing values, so a valid score is retained\n            # when another duplicate row contains only an Aqueduct sentinel.\n            scores = frame.groupby("pfaf_id", as_index=False, dropna=False)[indicators].first()\n            base = frame.drop(columns=indicators).drop_duplicates(subset=["pfaf_id"], keep="first")\n            frame = base.merge(scores, on="pfaf_id", how="left", validate="one_to_one").reset_index(drop=True)\n            return frame\n        raise DataContractError(f"Unsupported duplicate_pfaf_policy={duplicate_pfaf_policy!r}")\n    return frame\n\n\ndef write_public_table(df: pd.DataFrame, path: str | Path) -> None:\n    """Write aggregate output only; callers must not pass facility coordinates or identifiers."""\n    forbidden = {"facility_id", "latitude", "longitude", "geometry", "full_address", "site_name"}\n    present = forbidden & set(df.columns)\n    if present:\n        raise DataContractError(f"Refusing to publish sensitive facility fields: {sorted(present)}")\n    path = Path(path)\n    path.parent.mkdir(parents=True, exist_ok=True)\n    df.to_csv(path, index=False)\n', 'model.py': 'from __future__ import annotations\n\nfrom typing import Any, Mapping, Sequence\n\nimport numpy as np\nimport pandas as pd\n\nfrom .config import stable_hash\nfrom .errors import DataContractError\nfrom .io import CANONICAL_TECHNOLOGIES\n\n\nHOURS_PER_YEAR = 8760.0\n\n\ndef _require(df: pd.DataFrame, columns: Sequence[str], label: str) -> None:\n    missing = [c for c in columns if c not in df.columns]\n    if missing:\n        raise DataContractError(f"{label} is missing columns: {missing}")\n\n\ndef technology_factors_for_scenario(\n    base_factors: Mapping[str, float], scenario: Mapping[str, float]\n) -> dict[str, float]:\n    factors = {k: float(v) for k, v in base_factors.items()}\n    factors["hydro"] = float(scenario["hydro_factor_L_per_kWh"])\n    return factors\n\n\ndef add_grid_water_intensity(\n    region_mix: pd.DataFrame,\n    factors: Mapping[str, float],\n) -> pd.DataFrame:\n    """Calculate operational-consumption intensity without mutating the input mix."""\n    share_cols = [f"share_{t}" for t in CANONICAL_TECHNOLOGIES]\n    _require(region_mix, ["region_id", *share_cols, "raw_share_sum"], "region mix")\n    out = region_mix.copy()\n    factor_vector = np.array([float(factors[t]) for t in CANONICAL_TECHNOLOGIES], dtype=float)\n    shares = out[share_cols].apply(pd.to_numeric, errors="coerce").to_numpy(dtype=float)\n    out["grid_water_L_per_kWh"] = shares @ factor_vector\n    out["factor_hash"] = stable_hash({t: float(factors[t]) for t in CANONICAL_TECHNOLOGIES})\n    out["factor_set"] = "operational_consumption"\n    for technology in CANONICAL_TECHNOLOGIES:\n        out[f"factor_{technology}_L_per_kWh"] = float(factors[technology])\n    return out\n\n\ndef build_canonical_scenario_table(\n    facilities: pd.DataFrame,\n    region_mix: pd.DataFrame,\n    cfg: Mapping[str, Any],\n    *,\n    facility_region_column: str = "ba_id",\n    attribution_geography: str = "balancing_authority",\n    scenarios: Sequence[str] | None = None,\n) -> pd.DataFrame:\n    """\n    Build one immutable row per facility x scenario.\n\n    Assumptions locked by configuration:\n      E_fac[MWh/y] = capacity[MW] * 8760 * utilization\n      E_IT[MWh/y]  = E_fac / PUE\n      onsite[m3/y] = E_IT[MWh/y] * WUE[L/kWh_IT] * consumptive fraction\n      electricity[m3/y] = E_fac[MWh/y] * grid intensity[L/kWh]\n\n    MWh x L/kWh is numerically m3 because both conversion factors are 1,000.\n    """\n    if cfg["project"]["capacity_basis"] != "facility_input_nameplate_mw":\n        raise DataContractError("Equations are locked to facility-input/nameplate capacity.")\n    _require(\n        facilities,\n        ["facility_id", "capacity_mw", facility_region_column, "capacity_basis"],\n        "facilities",\n    )\n    if facilities["facility_id"].duplicated().any():\n        raise DataContractError("Facilities must be unique before scenario expansion.")\n    if facilities[facility_region_column].isna().any():\n        count = int(facilities[facility_region_column].isna().sum())\n        raise DataContractError(f"{count} facilities lack {facility_region_column}; no mix is imputed.")\n\n    requested = list(scenarios) if scenarios is not None else list(cfg["scenarios"])\n    unknown = sorted(set(requested) - set(cfg["scenarios"]))\n    if unknown:\n        raise DataContractError(f"Unknown scenarios: {unknown}")\n\n    pieces: list[pd.DataFrame] = []\n    base_factors = cfg["technology_water_consumption_L_per_kWh"]\n    for scenario_name in requested:\n        params = cfg["scenarios"][scenario_name]\n        factors = technology_factors_for_scenario(base_factors, params)\n        mix_s = add_grid_water_intensity(region_mix, factors)\n\n        sites = facilities.copy()\n        sites[facility_region_column] = sites[facility_region_column].astype("string").str.strip().str.upper()\n        joined = sites.merge(\n            mix_s,\n            how="left",\n            left_on=facility_region_column,\n            right_on="region_id",\n            validate="many_to_one",\n            suffixes=("", "_mix"),\n        )\n        if joined["grid_water_L_per_kWh"].isna().any():\n            missing_regions = joined.loc[joined["grid_water_L_per_kWh"].isna(), facility_region_column].drop_duplicates().tolist()\n            raise DataContractError(f"Missing generation mix for regions: {missing_regions[:20]}")\n\n        u = float(params["utilization"])\n        pue = float(params["pue"])\n        wue = float(params["wue_L_per_kWh_IT"])\n        consumptive_fraction = float(params["onsite_consumptive_fraction"])\n        joined["scenario"] = scenario_name\n        joined["utilization"] = u\n        joined["pue"] = pue\n        joined["wue_L_per_kWh_IT"] = wue\n        joined["onsite_consumptive_fraction"] = consumptive_fraction\n        joined["hours_per_year"] = HOURS_PER_YEAR\n        joined["facility_electricity_MWh_y"] = joined["capacity_mw"].astype(float) * HOURS_PER_YEAR * u\n        joined["it_electricity_MWh_y"] = joined["facility_electricity_MWh_y"] / pue\n        joined["onsite_m3_y"] = joined["it_electricity_MWh_y"] * wue * consumptive_fraction\n        joined["electricity_m3_y"] = joined["facility_electricity_MWh_y"] * joined["grid_water_L_per_kWh"]\n        joined["total_m3_y"] = joined["onsite_m3_y"] + joined["electricity_m3_y"]\n        # Aliases are retained for manuscript continuity, but the physical labels above are primary.\n        joined["scope1_m3_y"] = joined["onsite_m3_y"]\n        joined["scope2_m3_y"] = joined["electricity_m3_y"]\n        joined["grid_attribution_method"] = "location_based"\n        joined["attribution_geography"] = attribution_geography\n        joined["accounting_boundary"] = cfg["project"]["accounting_boundary"]\n        joined["scenario_hash"] = stable_hash({"name": scenario_name, **params})\n        joined["config_hash"] = cfg["config_hash"]\n        joined["canonical_row_id"] = (\n            joined["facility_id"].astype("string") + "::" + scenario_name + "::" + attribution_geography\n        )\n        pieces.append(joined)\n\n    out = pd.concat(pieces, ignore_index=True)\n    out = out.sort_values(["scenario", "facility_id"]).reset_index(drop=True)\n    return out\n\n\ndef build_fixed_activity_cooling_benchmark(\n    canonical: pd.DataFrame,\n    cfg: Mapping[str, Any],\n    *,\n    baseline_scenario: str = "baseline",\n    efficiency_scenario: str = "efficiency",\n) -> pd.DataFrame:\n    """Hold baseline facility electricity fixed while changing only PUE and WUE."""\n    base = canonical.loc[canonical["scenario"].eq(baseline_scenario)].copy()\n    if base.empty:\n        raise DataContractError(f"No rows for {baseline_scenario!r}.")\n    efficient = cfg["scenarios"][efficiency_scenario]\n    benchmark = base["facility_electricity_MWh_y"] / float(efficient["pue"])\n    benchmark = (\n        benchmark\n        * float(efficient["wue_L_per_kWh_IT"])\n        * float(efficient["onsite_consumptive_fraction"])\n    )\n    out = base[[c for c in ["facility_id", "hydrobasin_id", "huc8", "huc10", "huc12", "state_id"] if c in base]].copy()\n    out["baseline_onsite_m3_y"] = base["onsite_m3_y"].to_numpy()\n    out["benchmark_onsite_m3_y"] = benchmark.to_numpy()\n    out["potential_reduction_m3_y"] = (\n        out["baseline_onsite_m3_y"] - out["benchmark_onsite_m3_y"]\n    ).clip(lower=0)\n    out["benchmark_holds_activity_fixed"] = True\n    return out\n\n\ndef build_grid_intensity_benchmark(\n    canonical: pd.DataFrame,\n    *,\n    baseline_scenario: str = "baseline",\n    percentile: float = 0.25,\n) -> tuple[pd.DataFrame, float]:\n    """Hold electricity demand fixed and cap region intensity at an empirical benchmark."""\n    base = canonical.loc[canonical["scenario"].eq(baseline_scenario)].copy()\n    region_values = base[["region_id", "grid_water_L_per_kWh"]].drop_duplicates()\n    benchmark = float(region_values["grid_water_L_per_kWh"].quantile(percentile))\n    target = np.minimum(base["grid_water_L_per_kWh"].to_numpy(), benchmark)\n    target_water = base["facility_electricity_MWh_y"].to_numpy() * target\n    columns = [c for c in ["facility_id", "region_id", "ba_id", "egrid_subregion", "state_id"] if c in base]\n    out = base[columns].copy()\n    out["baseline_electricity_m3_y"] = base["electricity_m3_y"].to_numpy()\n    out["benchmark_grid_water_L_per_kWh"] = target\n    out["benchmark_electricity_m3_y"] = target_water\n    out["potential_reduction_m3_y"] = (\n        out["baseline_electricity_m3_y"] - out["benchmark_electricity_m3_y"]\n    ).clip(lower=0)\n    return out, benchmark\n', 'plotting.py': 'from __future__ import annotations\n\nfrom pathlib import Path\nfrom typing import Any, Mapping\n\nimport numpy as np\nimport pandas as pd\n\n\nSCOPE1 = "#0072B2"\nSCOPE1_LIGHT = "#DCEEF8"\nSCOPE2 = "#D55E00"\nSCOPE2_LIGHT = "#FBE5D5"\nNAVY = "#17324D"\nSTRESS = "#A61B29"\nGREY = "#E9EDF0"\nMID_GREY = "#5D6872"\nGREEN = "#009E73"\nPURPLE = "#7A5195"\n\n\ndef set_publication_style() -> None:\n    import matplotlib as mpl\n\n    mpl.rcParams.update(\n        {\n            "font.family": "DejaVu Sans",\n            "font.size": 8,\n            "axes.titlesize": 9,\n            "axes.labelsize": 8,\n            "xtick.labelsize": 7,\n            "ytick.labelsize": 7,\n            "legend.fontsize": 7,\n            "pdf.fonttype": 42,\n            "ps.fonttype": 42,\n            "svg.fonttype": "none",\n            "savefig.facecolor": "white",\n            "figure.facecolor": "white",\n        }\n    )\n\n\ndef save_figure(fig, output_stem: str | Path, cfg: Mapping[str, Any]) -> list[str]:\n    import matplotlib.pyplot as plt\n\n    stem = Path(output_stem)\n    stem.parent.mkdir(parents=True, exist_ok=True)\n    paths = []\n    for fmt in cfg["plots"].get("formats", ["pdf", "png"]):\n        path = stem.with_suffix(f".{fmt}")\n        kwargs = {"bbox_inches": "tight", "facecolor": "white"}\n        if fmt.lower() == "png":\n            kwargs["dpi"] = int(cfg["plots"].get("dpi", 600))\n        fig.savefig(path, **kwargs)\n        paths.append(str(path))\n    plt.close(fig)\n    return paths\n\n\ndef _box(ax, xy, width, height, text, *, fc="white", ec=NAVY, lw=0.8, fontsize=6.8, align="center"):\n    from matplotlib.patches import FancyBboxPatch\n\n    patch = FancyBboxPatch(\n        xy,\n        width,\n        height,\n        boxstyle="round,pad=0.012,rounding_size=0.015",\n        transform=ax.transAxes,\n        facecolor=fc,\n        edgecolor=ec,\n        linewidth=lw,\n    )\n    ax.add_patch(patch)\n    x = xy[0] + (width / 2 if align == "center" else 0.025)\n    ax.text(\n        x,\n        xy[1] + height / 2,\n        text,\n        transform=ax.transAxes,\n        ha=align,\n        va="center",\n        fontsize=fontsize,\n        color=NAVY,\n        linespacing=1.18,\n    )\n    return patch\n\n\ndef _arrow(ax, start, end, *, color=NAVY, lw=0.8):\n    from matplotlib.patches import FancyArrowPatch\n\n    arrow = FancyArrowPatch(\n        start,\n        end,\n        transform=ax.transAxes,\n        arrowstyle="-|>",\n        mutation_scale=8,\n        linewidth=lw,\n        color=color,\n        shrinkA=2,\n        shrinkB=2,\n    )\n    ax.add_patch(arrow)\n\n\ndef make_figure1(cfg: Mapping[str, Any]):\n    """Fully reproducible vector study-design figure; no generated imagery or hard-coded totals."""\n    import matplotlib.pyplot as plt\n    from matplotlib.patches import Rectangle\n\n    set_publication_style()\n    fig = plt.figure(figsize=(7.15, 5.35))\n    gs = fig.add_gridspec(2, 12, height_ratios=[1.48, 1.0], hspace=0.25, wspace=0.45)\n    axa = fig.add_subplot(gs[0, :4])\n    axb = fig.add_subplot(gs[0, 4:])\n    axc = fig.add_subplot(gs[1, :])\n    for ax in (axa, axb, axc):\n        ax.set_xlim(0, 1)\n        ax.set_ylim(0, 1)\n        ax.set_axis_off()\n\n    # Panel a: inputs and linkage.\n    axa.text(0.0, 1.04, "a", transform=axa.transAxes, fontsize=11, fontweight="bold", color=NAVY)\n    axa.text(0.10, 1.04, "Inputs and linkage", transform=axa.transAxes, fontsize=8.8, fontweight="bold", color=NAVY)\n    _box(axa, (0.05, 0.74), 0.90, 0.16, "Facility inventory\\ncoordinates; facility-input\\nnameplate capacity $P_i$", fc=GREY, fontsize=6.25)\n    _arrow(axa, (0.50, 0.74), (0.50, 0.66))\n    _box(\n        axa,\n        (0.05, 0.47),\n        0.90,\n        0.18,\n        "One-to-one spatial assignments\\nstate • HydroBASINS L6 / HUC\\nBA • eGRID subregion",\n        fc="white",\n        fontsize=6.15,\n    )\n    _arrow(axa, (0.50, 0.47), (0.50, 0.39))\n    _box(\n        axa,\n        (0.05, 0.06),\n        0.90,\n        0.32,\n        "Model and context inputs\\nAqueduct 4.0 annual bws_score\\n(withdrawal-based stress; 0–5)\\nannual bwd_score sensitivity\\neGRID shares $s_{r,f}$\\nconsumption factors $w_f$",\n        fc="#F7F9FA",\n        fontsize=5.8,\n    )\n\n    # Panel b: accounting equations and spatial meaning.\n    axb.text(0.0, 1.04, "b", transform=axb.transAxes, fontsize=11, fontweight="bold", color=NAVY)\n    axb.text(0.055, 1.04, "Pathway-resolved accounting", transform=axb.transAxes, fontsize=8.8, fontweight="bold", color=NAVY)\n    _box(axb, (0.29, 0.80), 0.42, 0.11, r"$E^{fac}_{i,s}=P_i\\times8760\\times u_s$", fc=GREY, fontsize=8)\n    _arrow(axb, (0.50, 0.80), (0.50, 0.72))\n    _box(axb, (0.34, 0.62), 0.32, 0.09, r"$E^{IT}_{i,s}=E^{fac}_{i,s}/PUE_s$", fc="white", fontsize=7.5)\n    _arrow(axb, (0.45, 0.62), (0.25, 0.54), color=SCOPE1)\n    # Electricity-related water is calculated from total facility electricity;\n    # it does not pass through the IT-energy/PUE conversion.\n    _arrow(axb, (0.71, 0.84), (0.75, 0.54), color=SCOPE2)\n    _box(\n        axb,\n        (0.02, 0.19),\n        0.46,\n        0.34,\n        "On-site cooling consumption\\n(Scope 1 analogue)\\n\\n$W^{(1)}_{i,s}=E^{IT}_{i,s} WUE_s c_s$\\n\\n$c_s$ = consumptive fraction\\nHost basin/HUC aggregation\\nBurden and stress stay separate",\n        fc=SCOPE1_LIGHT,\n        ec=SCOPE1,\n        fontsize=5.6,\n    )\n    _box(\n        axb,\n        (0.52, 0.19),\n        0.46,\n        0.34,\n        "Electricity-related consumption\\n(Scope 2 analogue; location-based)\\n\\n$I_{r,s}=Σ_f s_{r,f}w_{f,s}$\\n$W^{(2)}_{i,s}=E^{fac}_{i,s}I_{r(i),s}$\\n\\nPrimary attribution: BA\\nSensitivity: eGRID subregion",\n        fc=SCOPE2_LIGHT,\n        ec=SCOPE2,\n        fontsize=5.65,\n    )\n    _box(\n        axb,\n        (0.10, 0.025),\n        0.80,\n        0.12,\n        "National total only: Σᵢ(W⁽¹⁾+W⁽²⁾); spatial pathways remain separate.\\nBA/eGRID are attribution geographies—not physical locations\\nof generating-plant water consumption.",\n        fc="#F7F9FA",\n        ec=MID_GREY,\n        fontsize=5.15,\n    )\n\n    # Panel c: scenarios and decision geographies.\n    axc.text(0.0, 1.04, "c", transform=axc.transAxes, fontsize=11, fontweight="bold", color=NAVY)\n    axc.text(0.04, 1.04, "Scenarios and decision geographies", transform=axc.transAxes, fontsize=8.8, fontweight="bold", color=NAVY)\n    x0, y0, w, h = 0.01, 0.28, 0.59, 0.61\n    axc.add_patch(Rectangle((x0, y0), w, h, transform=axc.transAxes, facecolor="white", edgecolor=NAVY, linewidth=0.7))\n    headers = ["Scenario", "$u$", "PUE", "WUE", "Hydro"]\n    x_positions = [0.025, 0.34, 0.42, 0.50, 0.565]\n    for x, header in zip(x_positions, headers):\n        axc.text(x, 0.82, header, transform=axc.transAxes, fontsize=6.5, fontweight="bold", color=NAVY, ha="left" if header == "Scenario" else "center")\n    axc.text(0.50, 0.755, "L kWh$_{IT}^{-1}$", transform=axc.transAxes, fontsize=4.9, color=MID_GREY, ha="center")\n    axc.text(0.565, 0.755, "L kWh$^{-1}$", transform=axc.transAxes, fontsize=4.9, color=MID_GREY, ha="center")\n    colors = {"efficiency": GREEN, "baseline": "#4D4D4D", "high_load_water_intensive": SCOPE2, "no_hydropower": PURPLE}\n    labels = {\n        "efficiency": "Efficiency",\n        "baseline": "Baseline",\n        "high_load_water_intensive": "High-load / water-intensive",\n        "no_hydropower": "No hydropower",\n    }\n    ys = [0.67, 0.55, 0.43, 0.31]\n    for (name, params), y in zip(cfg["scenarios"].items(), ys):\n        axc.plot([0.02, 0.59], [y - 0.045, y - 0.045], transform=axc.transAxes, color="#E1E5E8", lw=0.45)\n        axc.scatter([0.025], [y], transform=axc.transAxes, s=14, color=colors.get(name, NAVY), zorder=3)\n        axc.text(0.043, y, labels.get(name, name), transform=axc.transAxes, va="center", fontsize=5.9, color=NAVY)\n        vals = [params["utilization"], params["pue"], params["wue_L_per_kWh_IT"], params["hydro_factor_L_per_kWh"]]\n        for x, value in zip(x_positions[1:], vals):\n            axc.text(x, y, f"{float(value):.2f}", transform=axc.transAxes, ha="center", va="center", fontsize=5.9, color=NAVY)\n\n    _box(\n        axc,\n        (0.64, 0.56),\n        0.35,\n        0.31,\n        "Basin / HUC decisions\\n\\nCooling technology; water source;\\npermits; reclaimed water; drought planning",\n        fc=SCOPE1_LIGHT,\n        ec=SCOPE1,\n        fontsize=6.4,\n    )\n    _box(\n        axc,\n        (0.64, 0.20),\n        0.35,\n        0.31,\n        "BA / eGRID-region decisions\\n\\nGeneration mix; procurement;\\nutility resource planning; transmission",\n        fc=SCOPE2_LIGHT,\n        ec=SCOPE2,\n        fontsize=6.4,\n    )\n    axc.text(\n        0.01,\n        0.095,\n        "Outputs: validated totals • burden and stress overlays • concentration • hotspot and benchmark sensitivities",\n        transform=axc.transAxes,\n        fontsize=5.8,\n        color=NAVY,\n    )\n    axc.text(\n        0.01,\n        0.005,\n        "Boundary: operational consumption only. Primary $c_s$ = 1; withdrawals, embodied/supply-chain water and unsupported market claims are excluded.",\n        transform=axc.transAxes,\n        fontsize=5.55,\n        color=MID_GREY,\n    )\n    return fig\n\n\ndef plot_scenario_totals(national: pd.DataFrame):\n    import matplotlib.pyplot as plt\n\n    set_publication_style()\n    data = national.copy()\n    order = [s for s in ["efficiency", "baseline", "high_load_water_intensive", "no_hydropower"] if s in set(data["scenario"])]\n    data = data.set_index("scenario").loc[order].reset_index()\n    fig, ax = plt.subplots(figsize=(7.1, 3.5))\n    x = np.arange(len(data))\n    ax.bar(x, data["onsite_GL_y"], color=SCOPE1, label="On-site cooling")\n    ax.bar(x, data["electricity_GL_y"], bottom=data["onsite_GL_y"], color=SCOPE2, label="Electricity-related")\n    ax.set_xticks(x, [s.replace("_", " ") for s in data["scenario"]], rotation=0)\n    ax.set_ylabel("Operational water consumption (GL yr$^{-1}$)")\n    ax.legend(frameon=False, ncol=2, loc="upper left")\n    ax.spines[["top", "right"]].set_visible(False)\n    ax.grid(axis="y", alpha=0.18)\n    fig.tight_layout()\n    return fig\n\n\ndef plot_concentration(scope1: pd.DataFrame, scope2: pd.DataFrame):\n    import matplotlib.pyplot as plt\n\n    set_publication_style()\n    fig, axes = plt.subplots(1, 2, figsize=(7.1, 3.0), sharey=True)\n    for ax, data, color, title in [\n        (axes[0], scope1, SCOPE1, "a  On-site cooling by hosting basin"),\n        (axes[1], scope2, SCOPE2, "b  Electricity-related by BA"),\n    ]:\n        ax.step(data["rank"], data["cumulative_share"], where="post", color=color, lw=1.8)\n        ax.fill_between(data["rank"], 0, data["cumulative_share"], step="post", color=color, alpha=0.12)\n        ax.axhline(0.5, color=MID_GREY, ls="--", lw=0.8)\n        ax.set_title(title, loc="left")\n        ax.set_xlabel("Ranked hosting regions")\n        ax.set_ylim(0, 1.02)\n        ax.set_xlim(1, max(len(data), 2))\n        ax.grid(alpha=0.15)\n        ax.spines[["top", "right"]].set_visible(False)\n    axes[0].set_ylabel("Cumulative share of pathway total")\n    fig.tight_layout()\n    return fig\n\n\ndef plot_scope2_quad(scope2_hotspots: pd.DataFrame):\n    """Reviewer-requested quadrants: grid intensity versus hosted-site stress."""\n    import matplotlib.pyplot as plt\n\n    set_publication_style()\n    d = scope2_hotspots.copy()\n    x = "capacity_weighted_bws_score"\n    y = "capacity_weighted_grid_water_L_per_kWh"\n    fig, ax = plt.subplots(figsize=(5.2, 4.3))\n    sizes = 25 + 260 * np.sqrt(d["capacity_mw"] / d["capacity_mw"].max())\n    sc = ax.scatter(d[x], d[y], s=sizes, c=d["electricity_m3_y"] / 1e6, cmap="Oranges", edgecolor=NAVY, lw=0.35, alpha=0.85)\n    ax.axvline(3.0, color=STRESS, ls="--", lw=0.9)\n    ax.axhline(d[y].median(), color=MID_GREY, ls="--", lw=0.9)\n    ax.set_xlabel("MW-weighted annual baseline water stress at hosted sites (0–5)")\n    ax.set_ylabel("BA grid water-consumption intensity (L kWh$^{-1}$)")\n    cbar = fig.colorbar(sc, ax=ax, pad=0.02)\n    cbar.set_label("Electricity-related water (GL yr$^{-1}$)")\n    ax.grid(alpha=0.12)\n    ax.spines[["top", "right"]].set_visible(False)\n    fig.tight_layout()\n    return fig\n\n\ndef plot_stress_sensitivity(stress_tables: pd.DataFrame, basin_column: str = "hydrobasin_id"):\n    import matplotlib.pyplot as plt\n\n    set_publication_style()\n    indicator_order = ["bws_score", "bwd_score"]\n    summary = stress_tables.groupby("stress_indicator").agg(\n        high_stress=("high_stress", "sum"), hotspots=("scope1_hotspot", "sum")\n    ).reindex(indicator_order).dropna(how="all").reset_index()\n    fig, ax = plt.subplots(figsize=(4.8, 3.4))\n    x = np.arange(len(summary))\n    width = 0.34\n    ax.bar(x - width / 2, summary["high_stress"], width, color=STRESS, alpha=0.55, label="High-stress hosting basins")\n    ax.bar(x + width / 2, summary["hotspots"], width, color=SCOPE1, label="High burden + high stress")\n    label_map = {\n        "bws_score": "Withdrawal-based\\nbws_score",\n        "bwd_score": "Consumption-based\\nbwd_score",\n    }\n    ax.set_xticks(x, [label_map[value] for value in summary["stress_indicator"]])\n    ax.set_ylabel("Number of hosting basins")\n    ax.legend(frameon=False)\n    ax.spines[["top", "right"]].set_visible(False)\n    fig.tight_layout()\n    return fig\n\n\ndef plot_attribution_sensitivity(summary: pd.DataFrame):\n    import matplotlib.pyplot as plt\n\n    set_publication_style()\n    d = summary.copy()\n    fig, ax = plt.subplots(figsize=(6.2, 3.4))\n    x = np.arange(len(d))\n    width = 0.36\n    ax.bar(x - width / 2, d["primary_electricity_GL_y"], width, color=SCOPE2, label="Balancing authority")\n    ax.bar(x + width / 2, d["alternative_electricity_GL_y"], width, color=PURPLE, label="eGRID subregion")\n    ax.set_xticks(x, d["scenario"].str.replace("_", " "))\n    ax.set_ylabel("Electricity-related water (GL yr$^{-1}$)")\n    ax.legend(frameon=False)\n    ax.spines[["top", "right"]].set_visible(False)\n    fig.tight_layout()\n    return fig\n\n\ndef plot_factor_uncertainty(draws: pd.DataFrame):\n    import matplotlib.pyplot as plt\n\n    set_publication_style()\n    fig, axes = plt.subplots(1, 2, figsize=(7.1, 3.1))\n    axes[0].hist(draws["electricity_GL_y"], bins=45, color=SCOPE2, alpha=0.82, edgecolor="white")\n    axes[0].set_xlabel("Electricity-related water (GL yr$^{-1}$)")\n    axes[0].set_ylabel("Monte Carlo draws")\n    axes[0].set_title("a  Technology-factor uncertainty", loc="left")\n    q = draws[["electricity_GL_y", "total_GL_y"]].quantile([0.025, 0.5, 0.975])\n    y = np.arange(2)\n    med = q.loc[0.5].to_numpy()\n    lower = med - q.loc[0.025].to_numpy()\n    upper = q.loc[0.975].to_numpy() - med\n    axes[1].errorbar(med, y, xerr=np.vstack([lower, upper]), fmt="o", color=NAVY, ecolor=NAVY, capsize=3)\n    axes[1].set_yticks(y, ["Electricity-related", "Total operational"])\n    axes[1].set_xlabel("Water consumption (GL yr$^{-1}$; median and 95% interval)")\n    axes[1].set_title("b  National uncertainty interval", loc="left")\n    for ax in axes:\n        ax.spines[["top", "right"]].set_visible(False)\n        ax.grid(axis="x", alpha=0.15)\n    fig.tight_layout()\n    return fig\n\n\ndef plot_regional_intervals(intervals: pd.DataFrame, geography: str):\n    """Plot scenario ranges without summing mutually exclusive scenario alternatives."""\n    import matplotlib.pyplot as plt\n\n    set_publication_style()\n    data = intervals.copy()\n    wide = data.pivot(index=[geography, "capacity_rank"], columns="scenario", values="total_m3_y").reset_index()\n    wide = wide.sort_values("capacity_rank", ascending=False)\n    low_col = "efficiency"\n    mid_col = "baseline"\n    high_col = "high_load_water_intensive"\n    missing = [c for c in [low_col, mid_col, high_col] if c not in wide]\n    if missing:\n        raise ValueError(f"Regional interval plot is missing scenarios: {missing}")\n    y = np.arange(len(wide))\n    low = wide[low_col] / 1e6\n    mid = wide[mid_col] / 1e6\n    high = wide[high_col] / 1e6\n    fig, ax = plt.subplots(figsize=(7.1, 4.8))\n    ax.hlines(y, low, high, color="#BBC2C7", lw=2.0, zorder=1)\n    ax.scatter(low, y, color=GREEN, s=16, label="Efficiency", zorder=2)\n    ax.scatter(mid, y, color=NAVY, s=18, label="Baseline", zorder=3)\n    ax.scatter(high, y, color=SCOPE2, s=16, label="High-load / water-intensive", zorder=2)\n    ax.set_yticks(y, wide[geography])\n    ax.set_xlabel("Total operational water consumption (GL yr$^{-1}$)")\n    ax.set_ylabel("Hosting region (ranked by baseline capacity)")\n    ax.legend(frameon=False, ncol=3, loc="lower center", bbox_to_anchor=(0.5, 1.01))\n    ax.spines[["top", "right", "left"]].set_visible(False)\n    ax.grid(axis="x", alpha=0.15)\n    fig.tight_layout()\n    return fig\n\n\ndef _map_layer(layer, target_crs: str):\n    if layer.crs is None:\n        raise ValueError("Map layer has no CRS.")\n    return layer.to_crs(target_crs)\n\n\ndef _polygon_ids(layer, source_id: str, output_id: str):\n    """Clean and dissolve polygon IDs before a one-to-one table merge."""\n    if source_id not in layer:\n        raise ValueError(f"Map layer is missing {source_id!r}.")\n    out = layer[[source_id, "geometry"]].copy()\n    out[output_id] = out[source_id].astype("string").str.strip().str.upper()\n    out = out.loc[out[output_id].notna() & ~out[output_id].isin(["", "NAN", "NONE", "NULL", "<NA>"])].copy()\n    return out[[output_id, "geometry"]].dissolve(by=output_id, as_index=False)\n\n\ndef _conus_states(states, cfg: Mapping[str, Any]):\n    out = states.copy()\n    state_col = cfg["columns"]["state_id"]\n    if state_col in out:\n        codes = out[state_col].astype("string").str.strip().str.upper()\n        out = out.loc[~codes.isin(cfg["inventory"]["exclude_state_codes"])].copy()\n    return out\n\n\ndef plot_pathway_hotspot_maps(\n    basin_shapes,\n    ba_shapes,\n    states,\n    scope1_table: pd.DataFrame,\n    scope2_table: pd.DataFrame,\n    cfg: Mapping[str, Any],\n):\n    import matplotlib.pyplot as plt\n    from matplotlib.patches import Patch\n\n    set_publication_style()\n    crs = cfg["plots"].get("conus_crs", "EPSG:5070")\n    basin_id_source = cfg["columns"]["hydrobasin_id"]\n    ba_id_source = cfg["columns"]["ba_polygon_id"]\n\n    bas = _polygon_ids(basin_shapes, basin_id_source, "hydrobasin_id")\n    bas = bas.merge(\n        scope1_table[["hydrobasin_id", "scope1_class", "scope1_hotspot", "onsite_m3_y"]],\n        on="hydrobasin_id",\n        how="left",\n        validate="one_to_one",\n    )\n    ba = _polygon_ids(ba_shapes, ba_id_source, "ba_id")\n    ba = ba.merge(\n        scope2_table[["ba_id", "scope2_code", "scope2_hotspot", "electricity_m3_y"]],\n        on="ba_id",\n        how="left",\n        validate="one_to_one",\n    )\n    # Plot only hosting regions; otherwise continent-wide HydroBASINS layers\n    # can determine the map extent and visually obscure the U.S. result.\n    bas = bas.loc[bas["scope1_class"].notna()].copy()\n    ba = ba.loc[ba["scope2_code"].notna()].copy()\n    st = _conus_states(states, cfg)\n    bas, ba, st = (_map_layer(g, crs) for g in (bas, ba, st))\n\n    fig, axes = plt.subplots(1, 2, figsize=(7.2, 3.75))\n    s1_colors = {\n        "Neither": "#E8ECEF",\n        "High stress only": "#F3C7C9",\n        "High burden only": "#9CCAE3",\n        "High burden + high stress": SCOPE1,\n    }\n    bas["_color"] = bas["scope1_class"].map(s1_colors).fillna("#F7F7F7")\n    bas.plot(ax=axes[0], color=bas["_color"], edgecolor="white", linewidth=0.08)\n    st.boundary.plot(ax=axes[0], color=NAVY, linewidth=0.28)\n    axes[0].set_title("a  On-site cooling: hosting-basin burden × annual stress", loc="left")\n    axes[0].legend(\n        handles=[Patch(facecolor=c, edgecolor="none", label=k) for k, c in s1_colors.items()],\n        frameon=False,\n        loc="lower left",\n        fontsize=6,\n    )\n\n    s2_colors = {\n        0: "#E8ECEF",\n        1: "#F4D0D2",\n        2: "#E9C9AA",\n        3: "#B78772",\n        4: "#F4B183",\n        5: "#B09BC8",\n        6: "#D97B37",\n        7: STRESS,\n    }\n    ba["_color"] = ba["scope2_code"].map(s2_colors).fillna("#F7F7F7")\n    ba.plot(ax=axes[1], color=ba["_color"], edgecolor="white", linewidth=0.12)\n    st.boundary.plot(ax=axes[1], color=NAVY, linewidth=0.28)\n    axes[1].set_title("b  Electricity-related: BA governance typology", loc="left")\n    legend_labels = {\n        0: "None",\n        1: "Hosted-site stress",\n        2: "Coal+gas heavy",\n        3: "Coal+gas + stress",\n        4: "High burden",\n        5: "High burden + stress",\n        6: "High burden + coal+gas",\n        7: "All three criteria",\n    }\n    axes[1].legend(\n        handles=[Patch(facecolor=s2_colors[k], edgecolor="none", label=legend_labels[k]) for k in range(8)],\n        frameon=False,\n        loc="lower left",\n        fontsize=5.4,\n        ncol=2,\n    )\n    for ax in axes:\n        ax.set_axis_off()\n    fig.text(\n        0.5,\n        0.015,\n        "Annual Aqueduct bws_score is withdrawal-based context; BA classes do not locate the physical source basin of generation water.",\n        ha="center",\n        fontsize=5.8,\n        color=MID_GREY,\n    )\n    fig.tight_layout(rect=[0, 0.035, 1, 1])\n    return fig\n\n\ndef plot_action_lever_maps(\n    basin_shapes,\n    ba_shapes,\n    states,\n    cooling_benchmark: pd.DataFrame,\n    grid_benchmark: pd.DataFrame,\n    cfg: Mapping[str, Any],\n):\n    import matplotlib.pyplot as plt\n\n    set_publication_style()\n    crs = cfg["plots"].get("conus_crs", "EPSG:5070")\n    bas = _polygon_ids(basin_shapes, cfg["columns"]["hydrobasin_id"], "hydrobasin_id")\n    bas = bas.merge(cooling_benchmark, on="hydrobasin_id", how="left", validate="one_to_one")\n    ba = _polygon_ids(ba_shapes, cfg["columns"]["ba_polygon_id"], "ba_id")\n    ba = ba.merge(grid_benchmark, on="ba_id", how="left", validate="one_to_one")\n    bas = bas.loc[bas["potential_reduction_m3_y"].notna()].copy()\n    ba = ba.loc[ba["potential_reduction_m3_y"].notna()].copy()\n    st = _conus_states(states, cfg)\n    bas, ba, st = (_map_layer(g, crs) for g in (bas, ba, st))\n\n    fig, axes = plt.subplots(1, 2, figsize=(7.2, 3.65))\n    for ax, layer, cmap, title in [\n        (axes[0], bas, "Blues", "a  Cooling-system benchmark\\n(activity held at baseline)"),\n        (axes[1], ba, "Oranges", "b  Grid-intensity benchmark\\n(activity held at baseline)"),\n    ]:\n        layer.plot(\n            column="potential_reduction_m3_y",\n            ax=ax,\n            cmap=cmap,\n            linewidth=0.08,\n            edgecolor="white",\n            legend=True,\n            legend_kwds={"label": "Potential reduction (m$^3$ yr$^{-1}$)", "shrink": 0.58},\n            missing_kwds={"color": "#F3F3F3"},\n        )\n        st.boundary.plot(ax=ax, color=NAVY, linewidth=0.28)\n        ax.set_title(title, loc="left")\n        ax.set_axis_off()\n    fig.tight_layout()\n    return fig\n\n\ndef plot_pathway_dominance_map(\n    shapes,\n    regional_baseline: pd.DataFrame,\n    *,\n    shape_id: str,\n    table_id: str,\n    states,\n    cfg: Mapping[str, Any],\n    title: str,\n):\n    import matplotlib.pyplot as plt\n    from matplotlib.patches import Patch\n\n    set_publication_style()\n    layer = _polygon_ids(shapes, shape_id, table_id)\n    data = regional_baseline.loc[regional_baseline["scenario"].eq("baseline")].copy()\n    data["dominance_ratio"] = data["electricity_m3_y"] / data["onsite_m3_y"].replace(0, np.nan)\n    data["pathway_dominance"] = np.select(\n        [data["dominance_ratio"].gt(1.10), data["dominance_ratio"].lt(1 / 1.10)],\n        ["Electricity-related dominant", "On-site dominant"],\n        default="Similar (within 10%)",\n    )\n    layer = layer.merge(data[[table_id, "pathway_dominance"]], on=table_id, how="left", validate="one_to_one")\n    if table_id != "state_id":\n        layer = layer.loc[layer["pathway_dominance"].notna()].copy()\n    else:\n        excluded = set(cfg["inventory"]["exclude_state_codes"])\n        layer = layer.loc[~layer[table_id].isin(excluded)].copy()\n    colors = {"On-site dominant": SCOPE1, "Electricity-related dominant": SCOPE2, "Similar (within 10%)": "#A6A6A6"}\n    layer["_color"] = layer["pathway_dominance"].map(colors).fillna("#F3F3F3")\n    crs = cfg["plots"].get("conus_crs", "EPSG:5070")\n    layer = _map_layer(layer, crs)\n    st = _map_layer(_conus_states(states, cfg), crs)\n    fig, ax = plt.subplots(figsize=(6.2, 3.8))\n    layer.plot(ax=ax, color=layer["_color"], edgecolor="white", linewidth=0.1)\n    st.boundary.plot(ax=ax, color=NAVY, linewidth=0.28)\n    ax.legend(handles=[Patch(facecolor=c, edgecolor="none", label=k) for k, c in colors.items()], frameon=False, loc="lower left")\n    ax.set_title(title, loc="left")\n    ax.set_axis_off()\n    fig.tight_layout()\n    return fig\n', 'sensitivity.py': 'from __future__ import annotations\n\nfrom typing import Any, Mapping, Sequence\n\nimport numpy as np\nimport pandas as pd\n\nfrom .aggregate import national_summary, scope1_hotspot_table, scope2_hotspot_table\nfrom .errors import DataContractError\nfrom .io import CANONICAL_TECHNOLOGIES\n\n\ndef _invalid_source(series: pd.Series) -> pd.Series:\n    source = series.astype("string").str.strip()\n    return source.isna() | source.eq("") | source.str.contains(r"REPLACE|DELETE|EXAMPLE|TODO", case=False, na=True)\n\n\ndef jaccard(left: set[str], right: set[str]) -> float:\n    union = left | right\n    return len(left & right) / len(union) if union else 1.0\n\n\ndef compare_attribution_geographies(\n    primary: pd.DataFrame,\n    alternative: pd.DataFrame,\n    *,\n    primary_label: str = "balancing_authority",\n    alternative_label: str = "egrid_subregion",\n) -> tuple[pd.DataFrame, pd.DataFrame]:\n    """Paired facility and national comparison for two location-based geographies."""\n    keys = ["facility_id", "scenario"]\n    keep = keys + ["electricity_m3_y", "grid_water_L_per_kWh"]\n    paired = primary[keep].merge(\n        alternative[keep], on=keys, how="inner", validate="one_to_one", suffixes=("_primary", "_alternative")\n    )\n    if len(paired) != len(primary) or len(paired) != len(alternative):\n        raise DataContractError("Attribution sensitivity requires the same complete facility×scenario universe.")\n    paired["electricity_difference_m3_y"] = (\n        paired["electricity_m3_y_alternative"] - paired["electricity_m3_y_primary"]\n    )\n    paired["electricity_percent_difference"] = (\n        100.0 * paired["electricity_difference_m3_y"] / paired["electricity_m3_y_primary"].replace(0, np.nan)\n    )\n    paired["primary_geography"] = primary_label\n    paired["alternative_geography"] = alternative_label\n\n    rows = []\n    for scenario, block in paired.groupby("scenario"):\n        x = block["electricity_m3_y_primary"]\n        y = block["electricity_m3_y_alternative"]\n        rows.append(\n            {\n                "scenario": scenario,\n                "primary_geography": primary_label,\n                "alternative_geography": alternative_label,\n                "primary_electricity_GL_y": x.sum() / 1e6,\n                "alternative_electricity_GL_y": y.sum() / 1e6,\n                "difference_GL_y": (y.sum() - x.sum()) / 1e6,\n                "percent_difference": 100.0 * (y.sum() - x.sum()) / x.sum(),\n                "facility_spearman": x.corr(y, method="spearman"),\n            }\n        )\n    return paired, pd.DataFrame(rows)\n\n\ndef stress_indicator_sensitivity(\n    canonical: pd.DataFrame,\n    cfg: Mapping[str, Any],\n    *,\n    basin_column: str = "hydrobasin_id",\n) -> tuple[pd.DataFrame, pd.DataFrame]:\n    """Compare annual withdrawal-based stress (`bws_score`) with depletion (`bwd_score`)."""\n    tables = []\n    sets: dict[str, set[str]] = {}\n    for indicator in ["bws_score", "bwd_score"]:\n        table, meta = scope1_hotspot_table(\n            canonical,\n            basin_column=basin_column,\n            stress_column=indicator,\n            burden_quantile=float(cfg["hotspots"]["scope1_burden_quantile"]),\n            high_stress_threshold=float(cfg["aqueduct"]["high_threshold"]),\n        )\n        table["stress_indicator"] = indicator\n        tables.append(table)\n        sets[indicator] = set(table.loc[table["scope1_hotspot"], basin_column].astype(str))\n    summary = pd.DataFrame(\n        [\n            {\n                "comparison": "bws_score_vs_bwd_score",\n                "bws_hotspots": len(sets["bws_score"]),\n                "bwd_hotspots": len(sets["bwd_score"]),\n                "intersection": len(sets["bws_score"] & sets["bwd_score"]),\n                "union": len(sets["bws_score"] | sets["bwd_score"]),\n                "jaccard": jaccard(sets["bws_score"], sets["bwd_score"]),\n            }\n        ]\n    )\n    return pd.concat(tables, ignore_index=True), summary\n\n\ndef hotspot_threshold_sensitivity(\n    canonical: pd.DataFrame,\n    cfg: Mapping[str, Any],\n    *,\n    basin_column: str = "hydrobasin_id",\n    ba_column: str = "ba_id",\n) -> pd.DataFrame:\n    """Prespecified burden-threshold robustness for both pathway typologies."""\n    grid = cfg["hotspots"]["threshold_grid"]\n    base_q1 = float(cfg["hotspots"]["scope1_burden_quantile"])\n    base_q2 = float(cfg["hotspots"]["scope2_burden_quantile"])\n    stress_threshold = float(cfg["aqueduct"]["high_threshold"])\n    base_s1, _ = scope1_hotspot_table(\n        canonical,\n        basin_column=basin_column,\n        burden_quantile=base_q1,\n        high_stress_threshold=stress_threshold,\n    )\n    base_s2, _ = scope2_hotspot_table(canonical, region_column=ba_column, burden_quantile=base_q2)\n    base_s1_set = set(base_s1.loc[base_s1["scope1_hotspot"], basin_column].astype(str))\n    base_s2_set = set(base_s2.loc[base_s2["scope2_hotspot"], ba_column].astype(str))\n    rows = []\n    for q in grid:\n        s1, _ = scope1_hotspot_table(\n            canonical,\n            basin_column=basin_column,\n            burden_quantile=float(q),\n            high_stress_threshold=stress_threshold,\n        )\n        s2, _ = scope2_hotspot_table(\n            canonical,\n            region_column=ba_column,\n            burden_quantile=float(q),\n            coal_gas_threshold=float(cfg["hotspots"]["scope2_coal_gas_threshold"]),\n            hosted_site_stress_threshold=float(cfg["hotspots"]["scope2_hosted_site_stress_threshold"]),\n        )\n        s1_set = set(s1.loc[s1["scope1_hotspot"], basin_column].astype(str))\n        s2_set = set(s2.loc[s2["scope2_hotspot"], ba_column].astype(str))\n        rows.extend(\n            [\n                {"pathway": "onsite", "burden_quantile": q, "hotspot_count": len(s1_set), "jaccard_vs_primary": jaccard(s1_set, base_s1_set)},\n                {"pathway": "electricity", "burden_quantile": q, "hotspot_count": len(s2_set), "jaccard_vs_primary": jaccard(s2_set, base_s2_set)},\n            ]\n        )\n    return pd.DataFrame(rows)\n\n\ndef factor_uncertainty_national(\n    canonical_baseline: pd.DataFrame,\n    factor_bounds: pd.DataFrame,\n    *,\n    draws: int = 5000,\n    seed: int = 20260901,\n) -> pd.DataFrame:\n    """Monte Carlo national Scope-2 sensitivity with globally coherent technology draws."""\n    required = {"technology", "lower_L_per_kWh", "mode_L_per_kWh", "upper_L_per_kWh", "source"}\n    if required - set(factor_bounds):\n        raise DataContractError(f"Factor-bounds table is missing {sorted(required - set(factor_bounds))}.")\n    bounds = factor_bounds.set_index("technology").reindex(CANONICAL_TECHNOLOGIES)\n    numeric = bounds[["lower_L_per_kWh", "mode_L_per_kWh", "upper_L_per_kWh"]].apply(pd.to_numeric, errors="coerce")\n    if numeric.isna().any().any() or _invalid_source(bounds["source"]).any():\n        raise DataContractError(\n            "Factor uncertainty is disabled until every technology has numeric bounds and an auditable source."\n        )\n    if not ((numeric["lower_L_per_kWh"] <= numeric["mode_L_per_kWh"]) & (numeric["mode_L_per_kWh"] <= numeric["upper_L_per_kWh"])).all():\n        raise DataContractError("Each factor must satisfy lower <= mode <= upper.")\n\n    base = canonical_baseline.loc[canonical_baseline["scenario"].eq("baseline")].copy()\n    shares = base[[f"share_{t}" for t in CANONICAL_TECHNOLOGIES]].to_numpy(float)\n    energy = base["facility_electricity_MWh_y"].to_numpy(float)\n    rng = np.random.default_rng(seed)\n    sampled_factors = np.column_stack(\n        [\n            rng.triangular(\n                numeric.loc[t, "lower_L_per_kWh"],\n                numeric.loc[t, "mode_L_per_kWh"],\n                numeric.loc[t, "upper_L_per_kWh"],\n                size=draws,\n            )\n            for t in CANONICAL_TECHNOLOGIES\n        ]\n    )\n    # Per-draw regional/facility intensity, then national energy-weighted volume.\n    scope2_m3 = np.einsum("i,ik,dk->d", energy, shares, sampled_factors, optimize=True)\n    onsite = float(base["onsite_m3_y"].sum())\n    out = pd.DataFrame(\n        {\n            "draw": np.arange(draws),\n            "electricity_GL_y": scope2_m3 / 1e6,\n            "onsite_GL_y": onsite / 1e6,\n            "total_GL_y": (scope2_m3 + onsite) / 1e6,\n        }\n    )\n    return out\n\n\ndef operational_and_ramp_facility_sets(\n    facilities: pd.DataFrame,\n    cfg: Mapping[str, Any],\n    *,\n    study_year: int | None = None,\n) -> dict[str, pd.DataFrame]:\n    """Create inventory sensitivities without pretending facility-specific utilization is observed."""\n    if study_year is None:\n        study_year = int(str(cfg["project"]["study_end_date"])[:4])\n    sets = {"inventory_current_capacity": facilities.copy()}\n    stage_col = "stage"\n    if stage_col in facilities:\n        pattern = cfg["inventory"]["operational_stage_regex"]\n        operational = facilities.loc[\n            facilities[stage_col].astype("string").str.contains(pattern, case=False, na=False, regex=True)\n        ].copy()\n        if not operational.empty:\n            sets["operational_stage_only"] = operational\n    if "commissioning_year" in facilities and facilities["commissioning_year"].notna().any():\n        years = float(cfg["inventory"].get("ramp_up_years", 3))\n        ramp = facilities.copy()\n        age = (study_year - pd.to_numeric(ramp["commissioning_year"], errors="coerce") + 1).clip(lower=0)\n        ramp["ramp_factor"] = (age / years).clip(lower=0, upper=1).fillna(1.0)\n        ramp["unadjusted_capacity_mw"] = ramp["capacity_mw"]\n        ramp["capacity_mw"] = ramp["capacity_mw"] * ramp["ramp_factor"]\n        # Future commissioning dates have zero effective capacity at the\n        # configured snapshot and do not count as active facilities.\n        ramp = ramp.loc[ramp["ramp_factor"].gt(0)].copy()\n        if not ramp.empty:\n            sets["illustrative_three_year_ramp"] = ramp\n    return sets\n\n\ndef derive_region_technology_factors_from_plants(plant_data: pd.DataFrame) -> pd.DataFrame:\n    """Derive generation-weighted consumption factors from audited plant/cooling inputs."""\n    required = {"plant_id", "region_id", "technology", "net_generation_MWh", "water_consumption_m3", "source"}\n    if required - set(plant_data):\n        raise DataContractError(f"Plant table is missing {sorted(required - set(plant_data))}.")\n    data = plant_data.copy()\n    data["region_id"] = data["region_id"].astype("string").str.strip().str.upper()\n    data["technology"] = data["technology"].astype("string").str.strip().str.lower()\n    unknown = sorted(set(data["technology"].dropna()) - set(CANONICAL_TECHNOLOGIES))\n    if unknown:\n        raise DataContractError(f"Unknown plant technologies: {unknown}")\n    data["net_generation_MWh"] = pd.to_numeric(data["net_generation_MWh"], errors="coerce")\n    data["water_consumption_m3"] = pd.to_numeric(data["water_consumption_m3"], errors="coerce")\n    if _invalid_source(data["source"]).any():\n        raise DataContractError("Every plant-water record needs an auditable source.")\n    data = data.loc[data["net_generation_MWh"].gt(0) & data["water_consumption_m3"].ge(0)].copy()\n    data["plant_consumption_L_per_kWh"] = data["water_consumption_m3"] / data["net_generation_MWh"]\n    out = data.groupby(["region_id", "technology"], as_index=False).agg(\n        net_generation_MWh=("net_generation_MWh", "sum"),\n        water_consumption_m3=("water_consumption_m3", "sum"),\n        n_plants=("plant_id", "nunique"),\n        source=("source", lambda values: "; ".join(sorted(set(map(str, values))))),\n    )\n    out["consumption_L_per_kWh"] = out["water_consumption_m3"] / out["net_generation_MWh"]\n    return out\n\n\ndef regional_factor_sensitivity(\n    canonical: pd.DataFrame,\n    regional_factors: pd.DataFrame,\n    *,\n    scenario: str = "baseline",\n) -> pd.DataFrame:\n    """Re-estimate electricity water using source-documented region×technology factors."""\n    required = {"region_id", "technology", "consumption_L_per_kWh", "source"}\n    if required - set(regional_factors):\n        raise DataContractError(f"Regional factors are missing {sorted(required - set(regional_factors))}.")\n    factors = regional_factors.copy()\n    factors["region_id"] = factors["region_id"].astype("string").str.strip().str.upper()\n    factors["technology"] = factors["technology"].astype("string").str.strip().str.lower()\n    unknown = sorted(set(factors["technology"].dropna()) - set(CANONICAL_TECHNOLOGIES))\n    if unknown:\n        raise DataContractError(f"Unknown regional-factor technologies: {unknown}")\n    factors["consumption_L_per_kWh"] = pd.to_numeric(factors["consumption_L_per_kWh"], errors="coerce")\n    if factors["consumption_L_per_kWh"].isna().any() or factors["consumption_L_per_kWh"].lt(0).any():\n        raise DataContractError("Regional consumption factors must be finite and non-negative.")\n    if _invalid_source(factors["source"]).any():\n        raise DataContractError("Every regional consumption factor needs an auditable source.")\n    if factors.duplicated(["region_id", "technology"]).any():\n        raise DataContractError("Regional factors must be unique by region_id and technology.")\n\n    base = canonical.loc[canonical["scenario"].eq(scenario)].copy()\n    wide = factors.pivot(index="region_id", columns="technology", values="consumption_L_per_kWh")\n    needed_regions = sorted(set(base["region_id"]))\n    missing_pairs = []\n    for region in needed_regions:\n        for technology in CANONICAL_TECHNOLOGIES:\n            share = base.loc[base["region_id"].eq(region), f"share_{technology}"].max()\n            if float(share) > 0 and (region not in wide.index or technology not in wide or pd.isna(wide.loc[region, technology])):\n                missing_pairs.append(f"{region}:{technology}")\n    if missing_pairs:\n        raise DataContractError(\n            "Regional factors do not cover every technology with positive generation share; "\n            f"examples: {missing_pairs[:20]}"\n        )\n\n    factor_columns = {technology: f"regional_factor_{technology}_L_per_kWh" for technology in CANONICAL_TECHNOLOGIES}\n    wide = wide.reindex(columns=list(CANONICAL_TECHNOLOGIES)).rename(columns=factor_columns).reset_index()\n    out = base.merge(wide, on="region_id", how="left", validate="many_to_one")\n    intensity = np.zeros(len(out), dtype=float)\n    for technology in CANONICAL_TECHNOLOGIES:\n        share = out[f"share_{technology}"].to_numpy(float)\n        factor = out[factor_columns[technology]].fillna(0).to_numpy(float)\n        intensity += share * factor\n    out["regional_factor_grid_water_L_per_kWh"] = intensity\n    out["regional_factor_electricity_m3_y"] = out["facility_electricity_MWh_y"] * intensity\n    out["difference_from_primary_m3_y"] = out["regional_factor_electricity_m3_y"] - out["electricity_m3_y"]\n    return out\n\n\ndef market_based_sensitivity(canonical: pd.DataFrame, portfolio: pd.DataFrame) -> pd.DataFrame:\n    """Use verified contractual/residual-mix intensity; never proxy it from renewable claims alone."""\n    required = {"facility_id", "portfolio_water_L_per_kWh", "portfolio_year", "evidence_source"}\n    if required - set(portfolio):\n        raise DataContractError(\n            "Market-based analysis needs facility-level portfolio intensity, year, and evidence source. "\n            "PPAs or renewable percentages alone are insufficient."\n        )\n    portfolio = portfolio.copy()\n    portfolio["facility_id"] = portfolio["facility_id"].astype("string").str.strip().str.upper()\n    if portfolio["facility_id"].duplicated().any():\n        raise DataContractError("Market-based portfolio must have one record per facility for the study year.")\n    portfolio["portfolio_water_L_per_kWh"] = pd.to_numeric(\n        portfolio["portfolio_water_L_per_kWh"], errors="coerce"\n    )\n    if portfolio["portfolio_water_L_per_kWh"].isna().any() or portfolio["portfolio_water_L_per_kWh"].lt(0).any():\n        raise DataContractError("Portfolio water intensities must be finite and non-negative.")\n    if _invalid_source(portfolio["evidence_source"]).any():\n        raise DataContractError("Every portfolio intensity requires an evidence source.")\n    base = canonical.loc[canonical["scenario"].eq("baseline")].copy()\n    out = base.merge(portfolio[list(required)], on="facility_id", how="left", validate="one_to_one")\n    if out["portfolio_water_L_per_kWh"].isna().any():\n        raise DataContractError("Market-based portfolio coverage is incomplete; no values are imputed.")\n    out["market_based_electricity_m3_y"] = (\n        out["facility_electricity_MWh_y"] * out["portfolio_water_L_per_kWh"]\n    )\n    out["difference_from_location_based_m3_y"] = (\n        out["market_based_electricity_m3_y"] - out["electricity_m3_y"]\n    )\n    return out\n', 'validate.py': 'from __future__ import annotations\n\nfrom typing import Any, Iterable, Mapping\n\nimport numpy as np\nimport pandas as pd\n\nfrom .errors import ValidationError\nfrom .io import CANONICAL_TECHNOLOGIES\n\n\ndef _check(rows: list[dict[str, Any]], name: str, passed: bool, detail: str = "") -> None:\n    rows.append({"check": name, "passed": bool(passed), "detail": detail})\n    if not passed:\n        raise ValidationError(f"{name}: {detail}")\n\n\ndef validate_canonical_table(df: pd.DataFrame, cfg: Mapping[str, Any]) -> pd.DataFrame:\n    """Fail fast on all model invariants and return a machine-readable audit."""\n    audit: list[dict[str, Any]] = []\n    required = [\n        "facility_id",\n        "scenario",\n        "canonical_row_id",\n        "capacity_mw",\n        "utilization",\n        "pue",\n        "wue_L_per_kWh_IT",\n        "onsite_consumptive_fraction",\n        "facility_electricity_MWh_y",\n        "it_electricity_MWh_y",\n        "grid_water_L_per_kWh",\n        "onsite_m3_y",\n        "electricity_m3_y",\n        "total_m3_y",\n        "factor_hash",\n        "scenario_hash",\n    ]\n    missing = [c for c in required if c not in df]\n    _check(audit, "required_columns", not missing, f"missing={missing}")\n\n    _check(audit, "canonical_key_unique", not df["canonical_row_id"].duplicated().any())\n    n_facilities = int(df["facility_id"].nunique())\n    n_scenarios = len(cfg["scenarios"])\n    expected = n_facilities * n_scenarios\n    _check(audit, "complete_facility_scenario_grid", len(df) == expected, f"rows={len(df)}, expected={expected}")\n    _check(\n        audit,\n        "facility_scenario_pair_unique",\n        not df.duplicated(["facility_id", "scenario"]).any(),\n    )\n    _check(\n        audit,\n        "scenario_set_exact",\n        set(df["scenario"]) == set(cfg["scenarios"]),\n        f"observed={sorted(set(df[\'scenario\']))}",\n    )\n    per_facility = df.groupby("facility_id")["scenario"].nunique()\n    _check(\n        audit,\n        "every_facility_has_every_scenario",\n        bool(per_facility.eq(n_scenarios).all()),\n    )\n\n    identifier_columns = [\n        column\n        for column in [\n            "facility_id",\n            "region_id",\n            "ba_id",\n            "state_id",\n            "hydrobasin_id",\n            "huc8",\n            "huc10",\n            "huc12",\n            "egrid_subregion",\n            "canonical_row_id",\n        ]\n        if column in df\n    ]\n    literal_nan = any(\n        df[column].astype("string").str.upper().isin(["NAN", "NONE", "NULL", "<NA>"]).any()\n        for column in identifier_columns\n    )\n    _check(audit, "no_literal_nan_identifiers", not literal_nan)\n\n    finite_columns = [\n        "capacity_mw",\n        "utilization",\n        "pue",\n        "wue_L_per_kWh_IT",\n        "onsite_consumptive_fraction",\n        "facility_electricity_MWh_y",\n        "it_electricity_MWh_y",\n        "grid_water_L_per_kWh",\n        "onsite_m3_y",\n        "electricity_m3_y",\n        "total_m3_y",\n    ]\n    values = df[finite_columns].apply(pd.to_numeric, errors="coerce").to_numpy(dtype=float)\n    _check(audit, "finite_numeric_outputs", np.isfinite(values).all())\n    _check(audit, "nonnegative_outputs", (values >= 0).all())\n    _check(\n        audit,\n        "parameter_ranges",\n        df["utilization"].between(0, 1).all()\n        and df["pue"].ge(1).all()\n        and df["onsite_consumptive_fraction"].between(0, 1).all(),\n    )\n\n    share_columns = [f"share_{t}" for t in CANONICAL_TECHNOLOGIES]\n    share_sum = df[share_columns].sum(axis=1)\n    _check(audit, "generation_shares_sum_to_one", np.allclose(share_sum, 1.0, atol=1e-10))\n\n    factor_columns = [f"factor_{t}_L_per_kWh" for t in CANONICAL_TECHNOLOGIES]\n    active_min = df[factor_columns].min(axis=1)\n    active_max = df[factor_columns].max(axis=1)\n    convex = df["grid_water_L_per_kWh"].between(active_min - 1e-10, active_max + 1e-10)\n    _check(audit, "grid_intensity_in_factor_convex_hull", bool(convex.all()))\n\n    identity = np.allclose(\n        df["onsite_m3_y"] + df["electricity_m3_y"],\n        df["total_m3_y"],\n        rtol=1e-11,\n        atol=1e-6,\n    )\n    _check(audit, "water_pathways_sum_to_total", identity)\n\n    expected_facility_energy = df["capacity_mw"] * 8760.0 * df["utilization"]\n    expected_it_energy = expected_facility_energy / df["pue"]\n    _check(\n        audit,\n        "facility_energy_equation",\n        np.allclose(df["facility_electricity_MWh_y"], expected_facility_energy, rtol=1e-12),\n    )\n    _check(\n        audit,\n        "it_energy_equation",\n        np.allclose(df["it_electricity_MWh_y"], expected_it_energy, rtol=1e-12),\n    )\n    expected_onsite = (\n        df["it_electricity_MWh_y"]\n        * df["wue_L_per_kWh_IT"]\n        * df["onsite_consumptive_fraction"]\n    )\n    expected_electricity = df["facility_electricity_MWh_y"] * df["grid_water_L_per_kWh"]\n    expected_grid_intensity = sum(\n        df[f"share_{technology}"] * df[f"factor_{technology}_L_per_kWh"]\n        for technology in CANONICAL_TECHNOLOGIES\n    )\n    _check(\n        audit,\n        "onsite_water_equation",\n        np.allclose(df["onsite_m3_y"], expected_onsite, rtol=1e-12, atol=1e-6),\n    )\n    _check(\n        audit,\n        "electricity_water_equation",\n        np.allclose(df["electricity_m3_y"], expected_electricity, rtol=1e-12, atol=1e-6),\n    )\n    _check(\n        audit,\n        "grid_intensity_equation",\n        np.allclose(df["grid_water_L_per_kWh"], expected_grid_intensity, rtol=1e-12, atol=1e-12),\n    )\n\n    hash_counts = df.groupby("scenario").agg(factor_hashes=("factor_hash", "nunique"), scenario_hashes=("scenario_hash", "nunique"))\n    _check(audit, "one_factor_and_scenario_hash_per_scenario", (hash_counts.eq(1)).all().all())\n\n    if "baseline" in set(df["scenario"]):\n        baseline = df.loc[df["scenario"].eq("baseline")].set_index("facility_id")\n        for scenario_name in ["efficiency", "high_load_water_intensive"]:\n            if scenario_name not in set(df["scenario"]):\n                continue\n            other = df.loc[df["scenario"].eq(scenario_name)].set_index("facility_id").loc[baseline.index]\n            s = cfg["scenarios"][scenario_name]\n            b = cfg["scenarios"]["baseline"]\n            expected_scope2_ratio = float(s["utilization"]) / float(b["utilization"])\n            expected_scope1_ratio = (\n                float(s["utilization"])\n                * float(s["wue_L_per_kWh_IT"])\n                * float(s["onsite_consumptive_fraction"])\n                / float(s["pue"])\n            ) / (\n                float(b["utilization"])\n                * float(b["wue_L_per_kWh_IT"])\n                * float(b["onsite_consumptive_fraction"])\n                / float(b["pue"])\n            )\n            _check(\n                audit,\n                f"{scenario_name}_scope2_scaling",\n                np.allclose(other["electricity_m3_y"], baseline["electricity_m3_y"] * expected_scope2_ratio, rtol=1e-10),\n            )\n            _check(\n                audit,\n                f"{scenario_name}_scope1_scaling",\n                np.allclose(other["onsite_m3_y"], baseline["onsite_m3_y"] * expected_scope1_ratio, rtol=1e-10),\n            )\n\n        if "no_hydropower" in set(df["scenario"]):\n            no_hydro = df.loc[df["scenario"].eq("no_hydropower")].set_index("facility_id").loc[baseline.index]\n            _check(audit, "no_hydro_onsite_unchanged", np.allclose(no_hydro["onsite_m3_y"], baseline["onsite_m3_y"], rtol=1e-12))\n            expected_reduction = (\n                baseline["facility_electricity_MWh_y"]\n                * baseline["share_hydro"]\n                * baseline["factor_hydro_L_per_kWh"]\n            )\n            actual_reduction = baseline["electricity_m3_y"] - no_hydro["electricity_m3_y"]\n            _check(audit, "no_hydro_scope2_exact_reduction", np.allclose(actual_reduction, expected_reduction, rtol=1e-10, atol=1e-6))\n\n    return pd.DataFrame(audit)\n\n\ndef validate_aggregation_conservation(\n    canonical: pd.DataFrame,\n    grouped: pd.DataFrame,\n    *,\n    value_columns: Iterable[str],\n    label: str,\n) -> pd.DataFrame:\n    """Verify a mutually exclusive complete geography sums back to national totals."""\n    rows = []\n    national = canonical.groupby("scenario")[list(value_columns)].sum(min_count=1)\n    aggregate = grouped.groupby("scenario")[list(value_columns)].sum(min_count=1)\n    aligned = national.join(aggregate, lsuffix="_national", rsuffix="_grouped", how="outer")\n    for scenario, row in aligned.iterrows():\n        for value in value_columns:\n            a = row[f"{value}_national"]\n            b = row[f"{value}_grouped"]\n            passed = bool(np.isclose(a, b, rtol=1e-10, atol=1e-6))\n            rows.append(\n                {\n                    "check": f"{label}_{value}_conservation",\n                    "scenario": scenario,\n                    "national": a,\n                    "grouped": b,\n                    "passed": passed,\n                }\n            )\n            if not passed:\n                raise ValidationError(\n                    f"{label} {value} does not conserve the {scenario} national total: {b} != {a}"\n                )\n    return pd.DataFrame(rows)\n', 'workflow.py': 'from __future__ import annotations\n\nimport argparse\nimport hashlib\nimport json\nimport shutil\nfrom datetime import datetime, timezone\nfrom pathlib import Path\nfrom typing import Any, Mapping\n\nimport pandas as pd\n\nfrom .aggregate import (\n    aggregate_by_geography,\n    concentration_table,\n    national_summary,\n    regional_scenario_intervals,\n    scope1_hotspot_table,\n    scope2_hotspot_table,\n)\nfrom .config import load_config, scenario_frame\nfrom .errors import DataContractError, ValidationError\nfrom .geospatial import (\n    assign_polygon_id,\n    attach_aqueduct_scores,\n    facilities_to_geodataframe,\n    geography_coverage_table,\n)\nfrom .io import (\n    load_aqueduct_annual,\n    read_geodata,\n    read_table,\n    standardize_facilities,\n    standardize_generation_mix,\n    write_public_table,\n)\nfrom .model import (\n    build_canonical_scenario_table,\n    build_fixed_activity_cooling_benchmark,\n    build_grid_intensity_benchmark,\n)\nfrom .plotting import (\n    make_figure1,\n    plot_attribution_sensitivity,\n    plot_concentration,\n    plot_factor_uncertainty,\n    plot_action_lever_maps,\n    plot_regional_intervals,\n    plot_pathway_dominance_map,\n    plot_pathway_hotspot_maps,\n    plot_scenario_totals,\n    plot_scope2_quad,\n    plot_stress_sensitivity,\n    save_figure,\n)\nfrom .sensitivity import (\n    compare_attribution_geographies,\n    factor_uncertainty_national,\n    hotspot_threshold_sensitivity,\n    market_based_sensitivity,\n    operational_and_ramp_facility_sets,\n    regional_factor_sensitivity,\n    stress_indicator_sensitivity,\n)\nfrom .validate import validate_aggregation_conservation, validate_canonical_table\n\n\ndef _sha256(path: Path) -> str:\n    digest = hashlib.sha256()\n    with path.open("rb") as handle:\n        for block in iter(lambda: handle.read(1024 * 1024), b""):\n            digest.update(block)\n    return digest.hexdigest()\n\n\ndef _checksum_input(path: Path) -> tuple[str | None, list[str]]:\n    """Hash a file, or the complete same-stem sidecar set for a shapefile."""\n    if not path.exists() or not path.is_file():\n        return None, []\n    components = sorted(path.parent.glob(f"{path.stem}.*")) if path.suffix.lower() == ".shp" else [path]\n    digest = hashlib.sha256()\n    for component in components:\n        digest.update(component.name.encode("utf-8"))\n        digest.update(bytes.fromhex(_sha256(component)))\n    return digest.hexdigest(), [component.name for component in components]\n\n\ndef _display_path(path: Path, repo_root: Path) -> str:\n    """Record reproducible relative paths without leaking local usernames."""\n    try:\n        return str(path.resolve().relative_to(repo_root.resolve()))\n    except ValueError:\n        return f"<external>/{path.name}"\n\n\ndef _input_manifest(cfg: Mapping[str, Any]) -> list[dict[str, Any]]:\n    rows = []\n    repo_root = Path(cfg["_repo_root"])\n    for name, raw in cfg["resolved_paths"].items():\n        if raw is None:\n            rows.append({"input": name, "path": None, "exists": None, "sha256": None, "components": []})\n            continue\n        path = Path(raw)\n        checksum, components = _checksum_input(path)\n        rows.append(\n            {\n                "input": name,\n                "path": _display_path(path, repo_root),\n                "exists": path.exists(),\n                "sha256": checksum,\n                "components": components,\n            }\n        )\n    return rows\n\n\ndef _prepare_output_dirs(cfg: Mapping[str, Any]) -> dict[str, Path]:\n    root = Path(cfg["resolved_paths"]["output_root"])\n    dirs = {\n        "root": root,\n        "figures": root / "figures",\n        "tables": root / "tables",\n        "diagnostics": root / "diagnostics",\n        "private": root / "private_data",\n    }\n    for path in dirs.values():\n        path.mkdir(parents=True, exist_ok=True)\n    return dirs\n\n\ndef _archive_generated_outputs(cfg: Mapping[str, Any]) -> Path | None:\n    """Move prior/partial generated outputs aside so runs cannot mix vintages."""\n    root = Path(cfg["resolved_paths"]["output_root"])\n    owned = [root / name for name in ["figures", "tables", "diagnostics", "private_data", "manifest.json"]]\n    existing = [path for path in owned if path.exists()]\n    if not existing:\n        return None\n    stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")\n    destination = root / "archive" / stamp\n    suffix = 1\n    while destination.exists():\n        destination = root / "archive" / f"{stamp}_{suffix}"\n        suffix += 1\n    destination.mkdir(parents=True, exist_ok=False)\n    for path in existing:\n        shutil.move(str(path), str(destination / path.name))\n    return destination\n\n\ndef make_study_design_figure(config_path: str | Path) -> list[str]:\n    cfg = load_config(config_path)\n    dirs = _prepare_output_dirs(cfg)\n    return save_figure(make_figure1(cfg), dirs["figures"] / "Figure1_StudyDesign_NatureWater", cfg)\n\n\ndef _load_and_assign_facilities(cfg: Mapping[str, Any]):\n    paths = cfg["resolved_paths"]\n    raw = read_table(paths["facilities"])\n    facilities, attrition = standardize_facilities(raw, cfg)\n    sites = facilities_to_geodataframe(facilities)\n\n    states = read_geodata(paths["states"])\n    sites = assign_polygon_id(\n        sites,\n        states,\n        polygon_source_id=cfg["columns"]["state_id"],\n        output_id="state_id",\n        label="state",\n    )\n    excluded = set(cfg["inventory"]["exclude_state_codes"])\n    sites = sites.loc[~sites["state_id"].isin(excluded)].copy()\n    attrition = pd.concat(\n        [\n            attrition,\n            pd.DataFrame(\n                [\n                    {\n                        "step": "contiguous_us_state_assignment",\n                        "rows": len(sites),\n                        "unique_facilities": sites["facility_id"].nunique(),\n                        "capacity_mw": sites["capacity_mw"].sum(),\n                    }\n                ]\n            ),\n        ],\n        ignore_index=True,\n    )\n\n    ba_polygons = read_geodata(paths["balancing_authority_polygons"])\n    sites = assign_polygon_id(\n        sites,\n        ba_polygons,\n        polygon_source_id=cfg["columns"]["ba_polygon_id"],\n        output_id="ba_id",\n        label="balancing authority",\n    )\n    basins = read_geodata(paths["hydrobasins_level6"])\n    sites = assign_polygon_id(\n        sites,\n        basins,\n        polygon_source_id=cfg["columns"]["hydrobasin_id"],\n        output_id="hydrobasin_id",\n        label="HydroBASINS level 6",\n    )\n\n    aqueduct = load_aqueduct_annual(\n        paths["aqueduct_baseline_annual"],\n        duplicate_pfaf_policy=cfg["aqueduct"]["duplicate_pfaf_policy"],\n        layer=cfg.get("input_options", {}).get("aqueduct_layer"),\n    )\n    sites = attach_aqueduct_scores(sites, aqueduct)\n\n    geography_layers: dict[str, Any] = {"states": states, "ba": ba_polygons, "hydrobasins": basins}\n    optional = [\n        ("huc8_polygons", "huc8_id", "huc8", "HUC8"),\n        ("huc10_polygons", "huc10_id", "huc10", "HUC10"),\n        ("huc12_polygons", "huc12_id", "huc12", "HUC12"),\n        ("egrid_subregion_polygons", "egrid_subregion_id", "egrid_subregion", "eGRID subregion"),\n    ]\n    for path_key, source_key, output_id, label in optional:\n        if paths.get(path_key):\n            layer = read_geodata(paths[path_key])\n            sites = assign_polygon_id(\n                sites,\n                layer,\n                polygon_source_id=cfg["columns"][source_key],\n                output_id=output_id,\n                label=label,\n            )\n            geography_layers[output_id] = layer\n    return sites, attrition, geography_layers\n\n\ndef _standardize_mix(raw: pd.DataFrame, cfg: Mapping[str, Any], region_candidates: list[str]):\n    mix_cfg = cfg["generation_mix"]\n    return standardize_generation_mix(\n        raw,\n        region_id_candidates=region_candidates,\n        share_sum_min=float(mix_cfg["acceptable_share_sum_min"]),\n        share_sum_max=float(mix_cfg["acceptable_share_sum_max"]),\n        allow_rounding_renormalization=bool(mix_cfg["allow_rounding_renormalization"]),\n    )\n\n\ndef _write(df: pd.DataFrame, path: Path) -> None:\n    write_public_table(df, path)\n\n\ndef _siddik_comparison(national: pd.DataFrame) -> pd.DataFrame:\n    current = national.loc[national["scenario"].eq("baseline")].iloc[0]\n    return pd.DataFrame(\n        [\n            {\n                "study": "Siddik et al. (2021)",\n                "fleet_and_year": "All U.S. data-center types, 2018",\n                "facility_electricity_TWh_y": 72.15,\n                "onsite_GL_y": 130.0,\n                "electricity_GL_y": 383.0,\n                "total_GL_y": 513.0,\n                "electricity_share": 383.0 / 513.0,\n                "comparison_status": "published comparator; different fleet, year, and spatial attribution",\n            },\n            {\n                "study": "Siddik et al. (2021)",\n                "fleet_and_year": "Hyperscale subset, 2018",\n                "facility_electricity_TWh_y": 22.85,\n                "onsite_GL_y": pd.NA,\n                "electricity_GL_y": pd.NA,\n                "total_GL_y": pd.NA,\n                "electricity_share": pd.NA,\n                "comparison_status": "published energy comparator only; do not pair with the all-data-center water total to claim decoupling",\n            },\n            {\n                "study": "This study — clean rerun",\n                "fleet_and_year": "Hyperscale inventory; configured study window",\n                "facility_electricity_TWh_y": current["facility_electricity_MWh_y"] / 1e6,\n                "onsite_GL_y": current["onsite_GL_y"],\n                "electricity_GL_y": current["electricity_GL_y"],\n                "total_GL_y": current["total_GL_y"],\n                "electricity_share": current["electricity_share_of_total"],\n                "comparison_status": "not directly comparable; used to establish prior baseline, not novelty of 75/25 split",\n            },\n        ]\n    )\n\n\ndef run_pipeline(config_path: str | Path) -> dict[str, Any]:\n    cfg = load_config(config_path)\n    # Do not disturb a prior valid run until all required inputs and spatial\n    # assignments have passed their contracts.\n    validate_inputs(config_path)\n    _archive_generated_outputs(cfg)\n    dirs = _prepare_output_dirs(cfg)\n    statuses: list[dict[str, str]] = []\n    figures: list[str] = []\n\n    figures.extend(save_figure(make_figure1(cfg), dirs["figures"] / "Figure1_StudyDesign_NatureWater", cfg))\n    statuses.append({"analysis": "Figure 1 study design", "status": "completed", "note": "generated from scenario config"})\n\n    sites, attrition, geography_layers = _load_and_assign_facilities(cfg)\n    _write(attrition, dirs["diagnostics"] / "inventory_attrition.csv")\n    coverage = geography_coverage_table(\n        sites,\n        ["state_id", "ba_id", "hydrobasin_id", "huc8", "huc10", "huc12", "egrid_subregion", "bws_score", "bwd_score"],\n    )\n    _write(coverage, dirs["diagnostics"] / "geography_coverage.csv")\n\n    inventory_frame = pd.DataFrame(sites.drop(columns="geometry", errors="ignore"))\n    if "stage" in inventory_frame:\n        inventory_frame["stage_reporting"] = inventory_frame["stage"].astype("string").fillna("Missing/unknown")\n        stage_aggregations: dict[str, tuple[str, str]] = {\n            "n_facilities": ("facility_id", "nunique"),\n            "current_capacity_mw": ("capacity_mw", "sum"),\n        }\n        for source, output in [("construction_mw", "construction_capacity_mw"), ("planned_mw", "planned_capacity_mw")]:\n            if source in inventory_frame:\n                stage_aggregations[output] = (source, "sum")\n        stage_summary = inventory_frame.groupby("stage_reporting", as_index=False).agg(**stage_aggregations)\n        _write(stage_summary, dirs["tables"] / "Table_Inventory_by_Stage.csv")\n        statuses.append({"analysis": "announced/proposed inventory", "status": "descriptive only", "note": "stage-specific counts and capacities; no future water forecast without dated, non-overlapping capacity fields"})\n    else:\n        statuses.append({"analysis": "announced/proposed inventory", "status": "not run", "note": "stage field unavailable"})\n\n    if "commissioning_year" in inventory_frame and inventory_frame["commissioning_year"].notna().any():\n        cohort = inventory_frame.loc[inventory_frame["commissioning_year"].notna()].copy()\n        cohort["commissioning_year"] = cohort["commissioning_year"].astype(int)\n        cohort_summary = cohort.groupby("commissioning_year", as_index=False).agg(\n            n_facilities=("facility_id", "nunique"),\n            current_snapshot_capacity_mw=("capacity_mw", "sum"),\n        )\n        _write(cohort_summary, dirs["tables"] / "Table_Commissioning_Cohorts.csv")\n        statuses.append({"analysis": "temporal burden persistence", "status": "descriptive cohorts only", "note": "historical regional burdens require historical capacity/utilization; current capacity is not back-cast"})\n    else:\n        statuses.append({"analysis": "temporal burden persistence", "status": "not run", "note": "commissioning year unavailable"})\n\n    ba_raw = read_table(\n        cfg["resolved_paths"]["balancing_authority_mix"],\n        sheet_name=cfg.get("input_options", {}).get("balancing_authority_mix_sheet", 0),\n    )\n    ba_mix, mix_issues = _standardize_mix(\n        ba_raw,\n        cfg,\n        ["Balancing Authority Code", "BA Code", "ba_id", "subregion_id", "region_id"],\n    )\n    _write(ba_mix.drop(columns=[c for c in ba_mix if c.startswith("factor_")], errors="ignore"), dirs["diagnostics"] / "ba_mix_audit.csv")\n    if not mix_issues.empty:\n        _write(mix_issues, dirs["diagnostics"] / "ba_mix_issues.csv")\n\n    canonical = build_canonical_scenario_table(sites, ba_mix, cfg)\n    validation = validate_canonical_table(canonical, cfg)\n    _write(validation, dirs["diagnostics"] / "canonical_validation.csv")\n    statuses.append({"analysis": "canonical BA model", "status": "completed", "note": "one facility×scenario table"})\n\n    if cfg["project"].get("write_private_facility_output", False):\n        private = canonical.drop(columns=["geometry"], errors="ignore")\n        private.to_parquet(dirs["private"] / "facility_scenario_private.parquet", index=False)\n\n    national = national_summary(canonical)\n    basin = aggregate_by_geography(canonical, "hydrobasin_id")\n    ba = aggregate_by_geography(canonical, "ba_id")\n    state = aggregate_by_geography(canonical, "state_id")\n    _write(national, dirs["tables"] / "Table_National_Scenarios.csv")\n    parameters = scenario_frame(cfg)\n    parameters["assumption_status"] = cfg["scenario_metadata"]["status"]\n    parameters["wue_definition"] = cfg["scenario_metadata"]["wue_definition"]\n    parameters["onsite_consumption_boundary"] = cfg["scenario_metadata"]["onsite_consumption_boundary"]\n    parameters["source_citations"] = "; ".join(cfg["scenario_metadata"]["citations"])\n    _write(parameters, dirs["tables"] / "Table_Model_Parameters.csv")\n    factors = pd.DataFrame(\n        [\n            {\n                "technology": technology,\n                "consumption_L_per_kWh": value,\n                "metric": cfg["technology_factor_metadata"]["metric"],\n                "source_citations": "; ".join(cfg["technology_factor_metadata"]["source_citations"]),\n                "statistic_note": cfg["technology_factor_metadata"]["statistic"],\n            }\n            for technology, value in cfg["technology_water_consumption_L_per_kWh"].items()\n        ]\n    )\n    _write(factors, dirs["tables"] / "Table_Technology_WaterConsumption_Factors.csv")\n    _write(basin, dirs["tables"] / "Table_HydroBasin_Scenarios.csv")\n    _write(ba, dirs["tables"] / "Table_BA_Scenarios.csv")\n    _write(state, dirs["tables"] / "Table_State_Scenarios.csv")\n    conservation = pd.concat(\n        [\n            validate_aggregation_conservation(canonical, basin, value_columns=["onsite_m3_y"], label="hydrobasin"),\n            validate_aggregation_conservation(canonical, ba, value_columns=["electricity_m3_y"], label="balancing_authority"),\n            validate_aggregation_conservation(canonical, state, value_columns=[*[\'onsite_m3_y\', \'electricity_m3_y\', \'total_m3_y\']], label="state"),\n        ],\n        ignore_index=True,\n    )\n    _write(conservation, dirs["diagnostics"] / "aggregation_conservation.csv")\n\n    s1_hot, s1_meta = scope1_hotspot_table(\n        canonical,\n        burden_quantile=float(cfg["hotspots"]["scope1_burden_quantile"]),\n        high_stress_threshold=float(cfg["aqueduct"]["high_threshold"]),\n    )\n    s2_hot, s2_meta = scope2_hotspot_table(\n        canonical,\n        burden_quantile=float(cfg["hotspots"]["scope2_burden_quantile"]),\n        coal_gas_threshold=float(cfg["hotspots"]["scope2_coal_gas_threshold"]),\n        hosted_site_stress_threshold=float(cfg["hotspots"]["scope2_hosted_site_stress_threshold"]),\n    )\n    s1_public_columns = [\n        column\n        for column in [\n            "scenario",\n            "hydrobasin_id",\n            "n_facilities",\n            "capacity_mw",\n            "onsite_m3_y",\n            "capacity_weighted_bws_score",\n            "capacity_weighted_bwd_score",\n            "high_onsite_burden",\n            "high_stress",\n            "scope1_hotspot",\n            "scope1_class",\n        ]\n        if column in s1_hot\n    ]\n    s2_public_columns = [\n        column\n        for column in [\n            "scenario",\n            "ba_id",\n            "n_facilities",\n            "capacity_mw",\n            "facility_electricity_MWh_y",\n            "electricity_m3_y",\n            "capacity_weighted_grid_water_L_per_kWh",\n            "capacity_weighted_share_coal_gas",\n            "capacity_weighted_bws_score",\n            "high_electricity_burden",\n            "coal_gas_heavy",\n            "high_hosted_site_stress",\n            "scope2_hotspot",\n            "scope2_code",\n        ]\n        if column in s2_hot\n    ]\n    _write(s1_hot[s1_public_columns], dirs["tables"] / "Table_Scope1_Hotspots_AnnualBWS.csv")\n    _write(s2_hot[s2_public_columns], dirs["tables"] / "Table_Scope2_BA_Typology.csv")\n    (dirs["diagnostics"] / "hotspot_definitions.json").write_text(json.dumps({"scope1": s1_meta, "scope2": s2_meta}, indent=2), encoding="utf-8")\n    figures.extend(\n        save_figure(\n            plot_pathway_hotspot_maps(\n                geography_layers["hydrobasins"],\n                geography_layers["ba"],\n                geography_layers["states"],\n                s1_hot,\n                s2_hot,\n                cfg,\n            ),\n            dirs["figures"] / "Figure2_Pathway_Hotspots",\n            cfg,\n        )\n    )\n\n    s1_conc = concentration_table(basin, value_column="onsite_m3_y", geography="hydrobasin_id")\n    s2_conc = concentration_table(ba, value_column="electricity_m3_y", geography="ba_id")\n    _write(s1_conc, dirs["tables"] / "PlotData_ExtendedData_Scope1_Concentration.csv")\n    _write(s2_conc, dirs["tables"] / "PlotData_ExtendedData_Scope2_Concentration.csv")\n    figures.extend(save_figure(plot_concentration(s1_conc, s2_conc), dirs["figures"] / "ExtendedData_Pathway_Concentration", cfg))\n    figures.extend(save_figure(plot_scope2_quad(s2_hot), dirs["figures"] / "Figure3_Scope2_Quadrant", cfg))\n\n    figures.extend(save_figure(plot_scenario_totals(national), dirs["figures"] / "ExtendedData_Scenario_Totals", cfg))\n    state_intervals = regional_scenario_intervals(canonical, "state_id")\n    ba_intervals = regional_scenario_intervals(canonical, "ba_id")\n    _write(state_intervals, dirs["tables"] / "PlotData_State_Scenario_Intervals.csv")\n    _write(ba_intervals, dirs["tables"] / "PlotData_BA_Scenario_Intervals.csv")\n    figures.extend(save_figure(plot_regional_intervals(state_intervals, "state_id"), dirs["figures"] / "Figure4_State_Scenario_Intervals", cfg))\n    figures.extend(save_figure(plot_regional_intervals(ba_intervals, "ba_id"), dirs["figures"] / "Figure4_BA_Scenario_Intervals", cfg))\n    figures.extend(\n        save_figure(\n            plot_pathway_dominance_map(\n                geography_layers["states"],\n                state,\n                shape_id=cfg["columns"]["state_id"],\n                table_id="state_id",\n                states=geography_layers["states"],\n                cfg=cfg,\n                title="State pathway dominance (baseline)",\n            ),\n            dirs["figures"] / "Figure4_State_Pathway_Dominance",\n            cfg,\n        )\n    )\n    figures.extend(\n        save_figure(\n            plot_pathway_dominance_map(\n                geography_layers["ba"],\n                ba,\n                shape_id=cfg["columns"]["ba_polygon_id"],\n                table_id="ba_id",\n                states=geography_layers["states"],\n                cfg=cfg,\n                title="Balancing-authority pathway dominance (baseline)",\n            ),\n            dirs["figures"] / "Figure4_BA_Pathway_Dominance",\n            cfg,\n        )\n    )\n\n    cooling_benchmark = build_fixed_activity_cooling_benchmark(canonical, cfg)\n    grid_benchmark, empirical_p25 = build_grid_intensity_benchmark(\n        canonical, percentile=float(cfg["benchmarks"]["scope2_grid_intensity_percentile"])\n    )\n    cooling_benchmark_by_basin = cooling_benchmark.groupby("hydrobasin_id", as_index=False)[["baseline_onsite_m3_y", "benchmark_onsite_m3_y", "potential_reduction_m3_y"]].sum()\n    grid_benchmark_by_ba = grid_benchmark.groupby("ba_id", as_index=False)[["baseline_electricity_m3_y", "benchmark_electricity_m3_y", "potential_reduction_m3_y"]].sum()\n    _write(cooling_benchmark_by_basin, dirs["tables"] / "Table_Scope1_FixedActivity_Benchmark.csv")\n    _write(grid_benchmark_by_ba, dirs["tables"] / "Table_Scope2_EmpiricalIntensity_Benchmark.csv")\n    (dirs["diagnostics"] / "scope2_benchmark.json").write_text(\n        json.dumps({"empirical_p25_L_per_kWh": empirical_p25, "computed_not_hard_coded": True}, indent=2), encoding="utf-8"\n    )\n    figures.extend(\n        save_figure(\n            plot_action_lever_maps(\n                geography_layers["hydrobasins"],\n                geography_layers["ba"],\n                geography_layers["states"],\n                cooling_benchmark_by_basin,\n                grid_benchmark_by_ba,\n                cfg,\n            ),\n            dirs["figures"] / "Figure5_Action_Levers",\n            cfg,\n        )\n    )\n\n    stress_tables, stress_summary = stress_indicator_sensitivity(canonical, cfg)\n    stress_public_columns = [\n        column\n        for column in [\n            "scenario",\n            "hydrobasin_id",\n            "n_facilities",\n            "capacity_mw",\n            "onsite_m3_y",\n            "capacity_weighted_bws_score",\n            "capacity_weighted_bwd_score",\n            "stress_indicator",\n            "high_onsite_burden",\n            "high_stress",\n            "scope1_hotspot",\n            "scope1_class",\n        ]\n        if column in stress_tables\n    ]\n    _write(stress_tables[stress_public_columns], dirs["tables"] / "Sensitivity_Aqueduct_BWS_vs_BWD_Basins.csv")\n    _write(stress_summary, dirs["tables"] / "Sensitivity_Aqueduct_BWS_vs_BWD_Summary.csv")\n    figures.extend(save_figure(plot_stress_sensitivity(stress_tables), dirs["figures"] / "ExtendedData_Aqueduct_BWS_vs_BWD", cfg))\n    threshold = hotspot_threshold_sensitivity(canonical, cfg)\n    _write(threshold, dirs["tables"] / "Sensitivity_Hotspot_Thresholds.csv")\n\n    comparator = _siddik_comparison(national)\n    _write(comparator, dirs["tables"] / "Table_Comparison_Siddik2021.csv")\n\n    # Optional HUC tables.\n    for huc in ["huc8", "huc10", "huc12"]:\n        if huc in canonical and canonical[huc].notna().all():\n            table = aggregate_by_geography(canonical, huc)\n            onsite_columns = [\n                column\n                for column in [\n                    "scenario",\n                    huc,\n                    "n_facilities",\n                    "capacity_mw",\n                    "facility_electricity_MWh_y",\n                    "onsite_m3_y",\n                    "capacity_weighted_bws_score",\n                    "capacity_weighted_bwd_score",\n                ]\n                if column in table\n            ]\n            _write(table[onsite_columns], dirs["tables"] / f"Sensitivity_{huc.upper()}_Onsite.csv")\n            membership = (\n                pd.DataFrame(sites.drop(columns="geometry", errors="ignore"))\n                .groupby([huc, "hydrobasin_id"], as_index=False)\n                .agg(n_facilities=("facility_id", "nunique"), capacity_mw=("capacity_mw", "sum"))\n            )\n            _write(membership, dirs["tables"] / f"Crosswalk_{huc.upper()}_to_HydroBASINS_L6_HostingFacilities.csv")\n            statuses.append(\n                {\n                    "analysis": huc.upper(),\n                    "status": "completed",\n                    "note": "on-site hosting-watershed aggregation plus facility-membership crosswalk to HydroBASINS L6",\n                }\n            )\n        else:\n            statuses.append({"analysis": huc.upper(), "status": "not run", "note": "polygon path not configured"})\n\n    # Optional eGRID-subregion location-based rerun.\n    if cfg["resolved_paths"].get("egrid_subregion_mix") and "egrid_subregion" in sites:\n        eg_raw = read_table(\n            cfg["resolved_paths"]["egrid_subregion_mix"],\n            sheet_name=cfg.get("input_options", {}).get("egrid_subregion_mix_sheet", 0),\n        )\n        eg_mix, eg_issues = _standardize_mix(\n            eg_raw, cfg, ["SUBRGN", "eGRID subregion", "egrid_subregion", "region_id"]\n        )\n        alternative = build_canonical_scenario_table(\n            sites,\n            eg_mix,\n            cfg,\n            facility_region_column="egrid_subregion",\n            attribution_geography="egrid_subregion",\n        )\n        validate_canonical_table(alternative, cfg)\n        paired, attr_summary = compare_attribution_geographies(canonical, alternative)\n        paired_public = paired.groupby("scenario", as_index=False).agg(\n            n_facilities=("facility_id", "nunique"),\n            mean_difference_m3_y=("electricity_difference_m3_y", "mean"),\n            median_difference_m3_y=("electricity_difference_m3_y", "median"),\n            p05_difference_m3_y=("electricity_difference_m3_y", lambda x: x.quantile(0.05)),\n            p95_difference_m3_y=("electricity_difference_m3_y", lambda x: x.quantile(0.95)),\n            median_percent_difference=("electricity_percent_difference", "median"),\n        )\n        _write(paired_public, dirs["tables"] / "Sensitivity_BA_vs_eGRID_FacilityDistribution.csv")\n        if cfg["project"].get("write_private_facility_output", False):\n            paired.to_parquet(dirs["private"] / "ba_vs_egrid_facility_private.parquet", index=False)\n        _write(attr_summary, dirs["tables"] / "Sensitivity_BA_vs_eGRID_Summary.csv")\n        figures.extend(save_figure(plot_attribution_sensitivity(attr_summary), dirs["figures"] / "ExtendedData_BA_vs_eGRID", cfg))\n        statuses.append({"analysis": "BA versus eGRID attribution", "status": "completed", "note": "paired location-based models"})\n    else:\n        statuses.append({"analysis": "BA versus eGRID attribution", "status": "not run", "note": "subregion polygons/mix not configured"})\n\n    # Optional factor uncertainty; template deliberately contains no invented bounds.\n    bounds_path = cfg["resolved_paths"].get("technology_factor_bounds")\n    try:\n        if not bounds_path:\n            raise FileNotFoundError("technology-factor bounds are not configured")\n        bounds = read_table(bounds_path)\n        draws = factor_uncertainty_national(\n            canonical,\n            bounds,\n            draws=int(cfg["uncertainty"]["factor_draws"]),\n            seed=int(cfg["uncertainty"]["random_seed"]),\n        )\n        _write(draws, dirs["tables"] / "Sensitivity_TechnologyFactor_MonteCarlo.csv")\n        figures.extend(save_figure(plot_factor_uncertainty(draws), dirs["figures"] / "ExtendedData_TechnologyFactor_Uncertainty", cfg))\n        statuses.append({"analysis": "technology-factor uncertainty", "status": "completed", "note": "source-documented triangular bounds"})\n    except (FileNotFoundError, DataContractError) as exc:\n        statuses.append({"analysis": "technology-factor uncertainty", "status": "not run", "note": str(exc)})\n\n    # Optional modern, region-specific operational-consumption factors.\n    regional_factor_path = cfg["resolved_paths"].get("regional_technology_factors")\n    if regional_factor_path:\n        regional_factors = read_table(regional_factor_path)\n        regional_result = regional_factor_sensitivity(canonical, regional_factors)\n        regional_public = regional_result.groupby("region_id", as_index=False).agg(\n            n_facilities=("facility_id", "nunique"),\n            facility_electricity_MWh_y=("facility_electricity_MWh_y", "sum"),\n            primary_electricity_m3_y=("electricity_m3_y", "sum"),\n            regional_factor_electricity_m3_y=("regional_factor_electricity_m3_y", "sum"),\n            difference_m3_y=("difference_from_primary_m3_y", "sum"),\n        )\n        _write(regional_public, dirs["tables"] / "Sensitivity_RegionalTechnologyFactors_by_BA.csv")\n        national_regional = pd.DataFrame(\n            [\n                {\n                    "scenario": "baseline",\n                    "primary_electricity_GL_y": regional_result["electricity_m3_y"].sum() / 1e6,\n                    "regional_factor_electricity_GL_y": regional_result["regional_factor_electricity_m3_y"].sum() / 1e6,\n                    "difference_GL_y": regional_result["difference_from_primary_m3_y"].sum() / 1e6,\n                }\n            ]\n        )\n        _write(national_regional, dirs["tables"] / "Sensitivity_RegionalTechnologyFactors_National.csv")\n        _write(regional_factors, dirs["diagnostics"] / "regional_technology_factor_provenance.csv")\n        statuses.append({"analysis": "regional technology factors", "status": "completed", "note": "source-documented BA×technology factors"})\n    else:\n        statuses.append({"analysis": "regional technology factors", "status": "not run", "note": "validated modern regional-factor table not configured"})\n\n    # Optional market-based comparison; exact facility pairs remain private.\n    market_path = cfg["resolved_paths"].get("market_based_portfolio")\n    if market_path:\n        portfolio = read_table(market_path)\n        market = market_based_sensitivity(canonical, portfolio)\n        market_summary = pd.DataFrame(\n            [\n                {\n                    "scenario": "baseline",\n                    "n_facilities": market["facility_id"].nunique(),\n                    "location_based_electricity_GL_y": market["electricity_m3_y"].sum() / 1e6,\n                    "market_based_electricity_GL_y": market["market_based_electricity_m3_y"].sum() / 1e6,\n                    "difference_GL_y": market["difference_from_location_based_m3_y"].sum() / 1e6,\n                    "median_facility_percent_difference": (\n                        100.0\n                        * market["difference_from_location_based_m3_y"]\n                        / market["electricity_m3_y"].replace(0, pd.NA)\n                    ).median(),\n                }\n            ]\n        )\n        _write(market_summary, dirs["tables"] / "Sensitivity_MarketBased_National.csv")\n        if cfg["project"].get("write_private_facility_output", False):\n            market.to_parquet(dirs["private"] / "market_based_facility_private.parquet", index=False)\n        statuses.append({"analysis": "market-based/PPA water accounting", "status": "completed", "note": "verified portfolio and residual-mix intensities supplied"})\n    else:\n        statuses.append({"analysis": "market-based/PPA water accounting", "status": "not run", "note": "verified facility portfolio intensity and residual-mix evidence not configured"})\n\n    # Operational-stage and transparent ramp-up sensitivity.\n    inventory_sets = operational_and_ramp_facility_sets(pd.DataFrame(sites.drop(columns="geometry")), cfg)\n    inventory_results = []\n    for label, facility_set in inventory_sets.items():\n        table = build_canonical_scenario_table(facility_set, ba_mix, cfg)\n        validate_canonical_table(table, cfg)\n        summary = national_summary(table)\n        summary["inventory_sensitivity"] = label\n        inventory_results.append(summary)\n    inventory_summary = pd.concat(inventory_results, ignore_index=True)\n    _write(inventory_summary, dirs["tables"] / "Sensitivity_OperationalStage_and_Ramp.csv")\n    statuses.append(\n        {\n            "analysis": "operational-stage-only inventory",\n            "status": "completed" if "operational_stage_only" in inventory_sets else "not run",\n            "note": "exact normalized stage labels required" if "operational_stage_only" in inventory_sets else "stage field missing or no accepted operational rows",\n        }\n    )\n    statuses.append(\n        {\n            "analysis": "illustrative three-year ramp",\n            "status": "completed" if "illustrative_three_year_ramp" in inventory_sets else "not run",\n            "note": "commissioning-year proxy; not observed utilization" if "illustrative_three_year_ramp" in inventory_sets else "commissioning year unavailable",\n        }\n    )\n\n    statuses.extend(\n        [\n            {"analysis": "plant/cooling-system factors", "status": "code provided; not run", "note": "requires audited plant generation and consumption data"},\n            {"analysis": "touching-neighbour stress", "status": "not run", "note": "helper only; requires a validated one-row-per-basin annual stress layer; primary uses in-basin annual stress"},\n            {"analysis": "AWARE scarcity sensitivity", "status": "not run", "note": "licensed/validated AWARE characterization layer not configured; annual Aqueduct BWS and BWD are reported separately"},\n            {"analysis": "district energy, CHP, waste-heat and self-generation", "status": "outside current data boundary", "note": "requires facility-specific utility-flow and behind-the-meter metering; no proxy is imputed"},\n        ]\n    )\n    status_frame = pd.DataFrame(statuses)\n    _write(status_frame, dirs["diagnostics"] / "analysis_status.csv")\n\n    manifest = {\n        "created_utc": datetime.now(timezone.utc).isoformat(),\n        "config_hash": cfg["config_hash"],\n        "model_version": "0.2.0",\n        "capacity_basis": cfg["project"]["capacity_basis"],\n        "accounting_boundary": cfg["project"]["accounting_boundary"],\n        "primary_stress_indicator": cfg["aqueduct"]["primary_indicator"],\n        "stress_sensitivity_indicator": cfg["aqueduct"]["sensitivity_indicator"],\n        "inputs": _input_manifest(cfg),\n        "figures": [_display_path(Path(p), Path(cfg["_repo_root"])) for p in figures],\n        "tables": sorted(_display_path(p, Path(cfg["_repo_root"])) for p in dirs["tables"].glob("*.csv")),\n        "diagnostics": sorted(_display_path(p, Path(cfg["_repo_root"])) for p in dirs["diagnostics"].glob("*")),\n    }\n    (dirs["root"] / "manifest.json").write_text(json.dumps(manifest, indent=2), encoding="utf-8")\n    return {"config": cfg, "canonical": canonical, "national": national, "manifest": manifest}\n\n\ndef validate_inputs(config_path: str | Path) -> pd.DataFrame:\n    cfg = load_config(config_path)\n    manifest = pd.DataFrame(_input_manifest(cfg))\n    required = [\n        "facilities",\n        "balancing_authority_polygons",\n        "balancing_authority_mix",\n        "hydrobasins_level6",\n        "aqueduct_baseline_annual",\n        "states",\n    ]\n    exists = manifest["exists"].astype("boolean").fillna(False)\n    missing = manifest.loc[manifest["input"].isin(required) & ~exists, "input"].tolist()\n    if missing:\n        raise FileNotFoundError(f"Missing required configured inputs: {missing}")\n    configured_missing = manifest.loc[\n        manifest["path"].notna()\n        & manifest["input"].ne("output_root")\n        & ~exists,\n        "input",\n    ].tolist()\n    if configured_missing:\n        raise FileNotFoundError(f"Configured optional inputs do not exist: {configured_missing}")\n\n    # This is intentionally more than a path check: run the actual inventory,\n    # CRS, one-to-one spatial-assignment, annual-Aqueduct and BA-mix contracts.\n    sites, _, _ = _load_and_assign_facilities(cfg)\n    if sites.empty:\n        raise DataContractError("No facility remains after the configured inventory and geography filters.")\n    ba_raw = read_table(\n        cfg["resolved_paths"]["balancing_authority_mix"],\n        sheet_name=cfg.get("input_options", {}).get("balancing_authority_mix_sheet", 0),\n    )\n    ba_mix, _ = _standardize_mix(\n        ba_raw,\n        cfg,\n        ["Balancing Authority Code", "BA Code", "ba_id", "subregion_id", "region_id"],\n    )\n    missing_ba = sorted(set(sites["ba_id"].dropna()) - set(ba_mix["region_id"]))\n    if missing_ba:\n        raise DataContractError(f"Hosting balancing authorities lack a valid generation mix: {missing_ba[:20]}")\n\n    if cfg["resolved_paths"].get("egrid_subregion_mix"):\n        eg_raw = read_table(\n            cfg["resolved_paths"]["egrid_subregion_mix"],\n            sheet_name=cfg.get("input_options", {}).get("egrid_subregion_mix_sheet", 0),\n        )\n        eg_mix, _ = _standardize_mix(\n            eg_raw,\n            cfg,\n            ["SUBRGN", "eGRID subregion", "egrid_subregion", "region_id"],\n        )\n        missing_eg = sorted(set(sites["egrid_subregion"].dropna()) - set(eg_mix["region_id"]))\n        if missing_eg:\n            raise DataContractError(f"Hosting eGRID subregions lack a valid generation mix: {missing_eg[:20]}")\n\n    manifest["validation"] = "path verified"\n    validated_inputs = {\n        "facilities",\n        "balancing_authority_polygons",\n        "balancing_authority_mix",\n        "hydrobasins_level6",\n        "aqueduct_baseline_annual",\n        "states",\n        "egrid_subregion_polygons",\n        "egrid_subregion_mix",\n        "huc8_polygons",\n        "huc10_polygons",\n        "huc12_polygons",\n    }\n    manifest.loc[manifest["input"].isin(validated_inputs) & manifest["path"].notna(), "validation"] = "contract validated"\n    manifest.loc[manifest["path"].isna(), "validation"] = "not configured"\n    manifest.loc[manifest["input"].eq("output_root"), "validation"] = "created on run"\n    return manifest\n\n\ndef main(argv: list[str] | None = None) -> int:\n    parser = argparse.ArgumentParser(description=__doc__)\n    sub = parser.add_subparsers(dest="command", required=True)\n    for command in ["validate", "figure1", "run"]:\n        child = sub.add_parser(command)\n        child.add_argument("--config", default="config/nature_water.yml")\n    args = parser.parse_args(argv)\n    try:\n        if args.command == "validate":\n            print(validate_inputs(args.config).to_string(index=False))\n        elif args.command == "figure1":\n            for path in make_study_design_figure(args.config):\n                print(path)\n        else:\n            result = run_pipeline(args.config)\n            print(json.dumps(result["manifest"], indent=2))\n    except (FileNotFoundError, DataContractError, ValidationError) as exc:\n        parser.exit(2, f"ERROR: {exc}\\n")\n    return 0\n\n\nif __name__ == "__main__":\n    raise SystemExit(main())\n'}
PACKAGE_DIR = RUNTIME_ROOT / 'src' / 'hyperscale_water_geography'
PACKAGE_DIR.mkdir(parents=True, exist_ok=True)
for filename, source in MODULE_SOURCES.items():
    (PACKAGE_DIR / filename).write_text(source, encoding='utf-8')
sys.path.insert(0, str(RUNTIME_ROOT / 'src'))
for module_name in list(sys.modules):
    if module_name == 'hyperscale_water_geography' or module_name.startswith('hyperscale_water_geography.'):
        del sys.modules[module_name]
importlib.invalidate_caches()
print(f'Embedded analysis code ready: {PACKAGE_DIR}')


## Inputs and exact paths

The following cell preserves the paths and field names in the submitted notebook. It also performs the only automatic input preparation required: caching the Census state ZIP locally and finding the annual Aqueduct table next to the configured monthly table.


In [ ]:
from pathlib import Path
import shutil
import urllib.request

import geopandas as gpd
import pandas as pd
import yaml

# Public/repository-relative input locations. The Baxtel facility file is licensed and is not distributed.
FACILITIES = ROOT / "data" / "raw" / "facilities.csv"
BA_POLYGONS = ROOT / "data" / "raw" / "balancing_authorities_polygons" / "balancing_authorities_EPA.shp"
BA_MIX = ROOT / "data" / "raw" / "BA_egrid.xlsx"
HYDROBASINS_L6 = ROOT / "data" / "raw" / "hydrobasins" / "hybas_lake_na_lev06_v1c.shp"
AQUEDUCT_ANNUAL = ROOT / "data" / "raw" / "aqueduct" / "Aqueduct40_baseline_annual_y2023m07d05.csv"
STATES_URL = "https://www2.census.gov/geo/tiger/GENZ2023/shp/cb_2023_us_state_20m.zip"

def local_census_states() -> Path:
    destination = ROOT / "data" / "raw" / "cb_2023_us_state_20m.zip"
    if destination.exists() and destination.stat().st_size > 0:
        return destination
    destination.parent.mkdir(parents=True, exist_ok=True)
    temporary = destination.with_suffix(".zip.part")
    request = urllib.request.Request(STATES_URL, headers={"User-Agent": "Mozilla/5.0"})
    try:
        with urllib.request.urlopen(request, timeout=120) as response, temporary.open("wb") as handle:
            shutil.copyfileobj(response, handle)
        temporary.replace(destination)
    except Exception as exc:
        temporary.unlink(missing_ok=True)
        raise RuntimeError(
            f"Could not download Census state boundaries. Download them manually from {STATES_URL}. Original error: {exc}"
        ) from exc
    return destination

STATES = local_census_states()

required_inputs = {
    "facilities": FACILITIES,
    "balancing_authority_polygons": BA_POLYGONS,
    "balancing_authority_mix": BA_MIX,
    "hydrobasins_level6": HYDROBASINS_L6,
    "aqueduct_baseline_annual": AQUEDUCT_ANNUAL,
    "states": STATES,
}
missing_inputs = {name: str(path) for name, path in required_inputs.items() if not path.exists()}
if missing_inputs:
    raise FileNotFoundError(
        "Missing required local inputs:\n" + "\n".join(f"  - {name}: {path}" for name, path in missing_inputs.items())
    )

annual_columns = {str(c).strip().lower() for c in pd.read_csv(AQUEDUCT_ANNUAL, nrows=0).columns}
if not {"bws_score", "bwd_score"}.issubset(annual_columns):
    raise ValueError("Aqueduct input must be the annual baseline table containing bws_score and bwd_score.")

display(pd.DataFrame([{
    "input": key, "path": str(value), "exists": value.exists()
} for key, value in required_inputs.items()]))


## Locked model configuration

This is the single source of truth for the rerun. The baseline and sensitivity assumptions are explicit, the high-load PUE is 1.40, hydropower is 8 L kWh⁻¹ except in the named zero-attribution sensitivity, Scope 1 uses IT electricity, and Scope 2 uses facility-input electricity. The annual inventory endpoint remains 15 December 2025 because that is the date of the actual inventory file.


In [ ]:
CONFIG = {
    "project": {
        "title": "Pathway and indicator choices alter water-screening priorities for U.S. hyperscale data centers",
        "study_window": "annualized operating year at the 2025-12-15 facility-inventory snapshot",
        "study_end_date": "2025-12-15",
        "capacity_basis": "facility_input_nameplate_mw",
        "accounting_boundary": "operational_water_consumption",
        "primary_scope2_method": "location_based",
        "write_private_facility_output": False,
    },
    "paths": {
        "facilities": str(FACILITIES),
        "balancing_authority_polygons": str(BA_POLYGONS),
        "balancing_authority_mix": str(BA_MIX),
        "hydrobasins_level6": str(HYDROBASINS_L6),
        "aqueduct_baseline_annual": str(AQUEDUCT_ANNUAL),
        "states": str(STATES),
        "egrid_subregion_polygons": None,
        "egrid_subregion_mix": None,
        "huc8_polygons": None,
        "huc10_polygons": None,
        "huc12_polygons": None,
        "market_based_portfolio": None,
        "regional_technology_factors": None,
        "technology_factor_bounds": None,
        "output_root": str(ROOT / "nature_water_outputs"),
    },
    "input_options": {
        "balancing_authority_mix_sheet": 0,
        "egrid_subregion_mix_sheet": 0,
        "aqueduct_layer": None,
    },
    "columns": {
        "facility_id": "id",
        "latitude": "latitude",
        "longitude": "longitude",
        "capacity_mw": "current_mw",
        "company": "company_name",
        "company_type": "company_type",
        "stage": "stage",
        "commissioning_year": "year_built",
        "construction_mw": "construction_mw",
        "planned_mw": "planned_mw",
        "ba_polygon_id": "region_B_1",
        "egrid_subregion_id": "SUBRGN",
        "state_id": "STUSPS",
        "hydrobasin_id": "PFAF_ID",
        "huc8_id": "huc8",
        "huc10_id": "huc10",
        "huc12_id": "huc12",
    },
    "inventory": {
        "snapshot_date": "2025-12-15",
        "reporting_geography": "contiguous_United_States",
        "main_stage_policy": "all inventory rows with positive current_mw; stage-tested separately",
        "hyperscale_regex": r"^(?:hyperscale|hyperscaler|hyper\s*scale)$",
        "require_hyperscale_label": True,
        "include_only_positive_current_capacity": True,
        "exclude_state_codes": ["AK", "HI", "PR", "GU", "VI", "MP", "AS"],
        "operational_stage_regex": r"^(?:operational|in service|active)$",
        "ramp_up_years": 3,
    },
    "aqueduct": {
        "primary_indicator": "bws_score",
        "sensitivity_indicator": "bwd_score",
        "high_threshold": 3.0,
        "annual_only": True,
        "duplicate_pfaf_policy": "collapse_identical",
        "source_kind": "baseline_annual",
    },
    "generation_mix": {
        "allow_rounding_renormalization": True,
        "acceptable_share_sum_min": 0.98,
        "acceptable_share_sum_max": 1.02,
        "primary_region": "balancing_authority",
        "sensitivity_region": "egrid_subregion",
    },
    "technology_water_consumption_L_per_kWh": {
        "coal": 1.9,
        "gas": 0.7,
        "nuclear": 2.5,
        "hydro": 8.0,
        "wind": 0.0,
        "solar": 0.1,
        "other": 1.0,
    },
    "technology_factor_metadata": {
        "metric": "operational water consumption",
        "unit": "L_per_kWh",
        "source_citations": ["Macknick et al. 2012", "Meldrum et al. 2013"],
        "statistic": "manuscript-selected central factors; verify against cited table before submission",
        "hydropower_note": "reservoir-evaporation allocation is contested; baseline and zero-attribution sensitivity reported",
        "renewables_note": "wind, solar and hydropower shares are included explicitly",
    },
    "scenario_metadata": {
        "status": "national scenario assumptions, not facility measurements",
        "wue_definition": "litres of site water use per kWh of IT electricity (ISO/IEC 30134-9)",
        "onsite_consumption_boundary": "primary model assumes onsite_consumptive_fraction=1.0; replace or sensitivity-test with documented discharge/blowdown evidence",
        "pue_role": "used only to convert facility electricity to IT electricity",
        "citations": ["ISO/IEC 30134-9", "Mytton 2021", "Shehabi et al. 2016", "LBNL data-center energy report"],
    },
    "scenarios": {
        "efficiency": {
            "utilization": 0.55,
            "pue": 1.15,
            "wue_L_per_kWh_IT": 0.20,
            "onsite_consumptive_fraction": 1.0,
            "hydro_factor_L_per_kWh": 8.0,
        },
        "baseline": {
            "utilization": 0.66,
            "pue": 1.25,
            "wue_L_per_kWh_IT": 0.80,
            "onsite_consumptive_fraction": 1.0,
            "hydro_factor_L_per_kWh": 8.0,
        },
        "high_load_water_intensive": {
            "utilization": 0.85,
            "pue": 1.40,
            "wue_L_per_kWh_IT": 1.50,
            "onsite_consumptive_fraction": 1.0,
            "hydro_factor_L_per_kWh": 8.0,
        },
        "no_hydropower": {
            "utilization": 0.66,
            "pue": 1.25,
            "wue_L_per_kWh_IT": 0.80,
            "onsite_consumptive_fraction": 1.0,
            "hydro_factor_L_per_kWh": 0.0,
        },
    },
    "hotspots": {
        "scope1_burden_quantile": 0.50,
        "scope1_neighbor_stress": False,
        "scope2_burden_quantile": 0.50,
        "scope2_coal_gas_threshold": 0.50,
        "scope2_hosted_site_stress_threshold": 3.0,
        "threshold_grid": [0.50, 0.60, 0.70, 0.75, 0.80, 0.85, 0.90],
    },
    "benchmarks": {
        "scope1_scenario": "efficiency",
        "scope2_grid_intensity_percentile": 0.25,
    },
    "uncertainty": {
        "factor_draws": 5000,
        "random_seed": 20260901,
    },
    "plots": {
        "dpi": 600,
        "formats": ["pdf", "png", "svg"],
        "conus_crs": "EPSG:5070",
        "color_scope1": "#0072B2",
        "color_scope2": "#D55E00",
        "color_stress": "#A61B29",
        "color_neutral": "#17324D",
    },
}

CONFIG_DIR = RUNTIME_ROOT / "config"
CONFIG_DIR.mkdir(parents=True, exist_ok=True)
CONFIG_PATH = CONFIG_DIR / "nature_water_local.yml"
CONFIG_PATH.write_text(yaml.safe_dump(CONFIG, sort_keys=False), encoding="utf-8")

from hyperscale_water_geography.config import load_config, scenario_frame

cfg = load_config(CONFIG_PATH)
display(scenario_frame(cfg))
print("Local run configuration:", CONFIG_PATH)


## Run the complete revised analysis

This single call performs the strict input and spatial-join checks before modelling, archives an earlier generated run so vintages cannot mix, builds the canonical facility × scenario table once, checks all conservation identities, then writes every aggregate table and figure.


In [ ]:
from hyperscale_water_geography.workflow import run_pipeline

result = run_pipeline(CONFIG_PATH)
national = result["national"]
display(national)
print("Complete. Output folder:", ROOT / "nature_water_outputs")


## Required validation checks

The next cell fails if any accounting identity or geographic aggregation does not reproduce the canonical national totals. Do not copy manuscript numbers unless this cell passes.


In [ ]:
OUTPUT_ROOT = ROOT / "nature_water_outputs"
validation = pd.read_csv(OUTPUT_ROOT / "diagnostics" / "canonical_validation.csv")
conservation = pd.read_csv(OUTPUT_ROOT / "diagnostics" / "aggregation_conservation.csv")
if not validation["passed"].all():
    raise AssertionError("One or more canonical accounting checks failed.")
if not conservation["passed"].all():
    raise AssertionError("One or more geographic aggregation checks failed.")
display(validation)
display(conservation)


## Generate the existing canonical figure suite

The historical plotting source is retained with sparse logarithmic colorbar ticks and relocated quadrant annotations. The new layer below replaces the concentration plot and creates Figure 1 from programmatic text, equations and connectors. Actual exported plots must still be inspected at publication size.


In [ ]:
from __future__ import annotations
FAIL_ON_MANUSCRIPT_MISMATCH = False

"""Create the exact figure files referenced by the reviewer-hardened Nature Water LaTeX.

This cell is designed to run after the validated pipeline cells in
water_consumption_hyp_nature_water_PUBLICATION_READY.ipynb.  It uses the in-memory
ROOT, cfg and result objects and preserves the same input paths.  It exports only
regional/national aggregate data, never facility-level identifiers or coordinates.
"""

import hashlib
import json
import math
import re
import shutil
from pathlib import Path
from typing import Iterable, Mapping

import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.colors import LinearSegmentedColormap, LogNorm, Normalize, PowerNorm
from matplotlib.lines import Line2D
from matplotlib.patches import FancyArrowPatch, FancyBboxPatch, Patch
from matplotlib.ticker import FuncFormatter, MaxNLocator

try:
    import geopandas as gpd
except ImportError as exc:  # pragma: no cover - the setup cell checks this first
    raise ImportError("geopandas is required for the map figures") from exc

from hyperscale_water_geography.aggregate import (
    aggregate_by_geography,
    concentration_table,
    national_summary,
)
from hyperscale_water_geography.io import read_geodata


# -----------------------------------------------------------------------------
# Publication controls
# -----------------------------------------------------------------------------
# Nature's research-figure guidance uses approximately 183 mm for double-column
# figures and 89 mm for single-column figures.  These sizes are used directly so
# that 5-7 pt text remains legible after placement.
DOUBLE_COLUMN_IN = 183.0 / 25.4
SINGLE_COLUMN_IN = 89.0 / 25.4
PANEL_LABEL_PT = 8.0
BASE_TEXT_PT = 6.4
PNG_DPI = 600
EXTENDED_DATA_JPEG_DPI = 300

# The current defensive main.tex contains these values.  Figures are always made
# from the canonical run; this table is only a submission-blocking cross-check.
EXPECTED_MANUSCRIPT_VALUES = {
    "n_facilities": 472,
    "capacity_mw_rounded": 20041,
    "baseline_total_GL_y": 299.89,
    "baseline_onsite_GL_y": 74.16,
    "baseline_electricity_GL_y": 225.74,
    "n_hosting_basins": 63,
    "n_hosting_BAs": 24,
    "n_basins_for_50_percent": 10,
    "n_BAs_for_50_percent": 3,
    "BWS_high_context_basins": 25,
    "BWS_high_burden_and_context": 15,
    "BWD_high_context_basins": 7,
    "BWD_high_burden_and_context": 6,
}

# Keep False while drafting so every file is still written.  Change to True for
# the release run; the final cell will then stop if the manuscript and canonical
# outputs disagree.
FAIL_ON_MANUSCRIPT_MISMATCH = bool(globals().get("FAIL_ON_MANUSCRIPT_MISMATCH", False))

# Colour-blind-safe palette; colour is reinforced by line/marker shape where it
# carries categorical meaning.
INK = "#1A1A1A"
NAVY = "#1B365D"
BLUE = "#0072B2"
BLUE_LIGHT = "#D9EAF4"
ORANGE = "#D55E00"
ORANGE_LIGHT = "#F6DFD2"
GREEN = "#009E73"
PURPLE = "#7A5195"
ROSE = "#A23B5A"
GREY = "#6F7780"
MID_GREY = "#AAB0B6"
LIGHT_GREY = "#E7EAED"
PALE_GREY = "#F5F6F7"
WHITE = "#FFFFFF"

SCENARIO_ORDER = [
    "efficiency",
    "baseline",
    "high_load_water_intensive",
    "no_hydropower",
]
SCENARIO_LABELS = {
    "efficiency": "Lower-load / lower-water",
    "baseline": "Reference",
    "high_load_water_intensive": "Higher-load / water-intensive",
    "no_hydropower": "No hydropower attribution",
}
PROFILE_SCENARIOS = ["efficiency", "baseline", "high_load_water_intensive"]
PROFILE_STYLE = {
    "efficiency": {"label": "Lower-load / lower-water", "color": GREEN, "marker": "o"},
    "baseline": {"label": "Reference", "color": NAVY, "marker": "D"},
    "high_load_water_intensive": {"label": "Higher-load / water-intensive", "color": ORANGE, "marker": "s"},
}


def _set_nature_style() -> None:
    """Set consistent, editable, reduction-safe artwork defaults."""
    mpl.rcParams.update(
        {
            "font.family": "sans-serif",
            "font.sans-serif": ["Arial", "Helvetica", "Liberation Sans", "DejaVu Sans"],
            "font.size": BASE_TEXT_PT,
            "axes.titlesize": 7.0,
            "axes.labelsize": 6.8,
            "xtick.labelsize": 5.9,
            "ytick.labelsize": 5.9,
            "legend.fontsize": 5.8,
            "legend.title_fontsize": 6.0,
            "axes.linewidth": 0.70,
            "lines.linewidth": 1.0,
            "patch.linewidth": 0.70,
            "xtick.major.width": 0.65,
            "ytick.major.width": 0.65,
            "xtick.major.size": 2.7,
            "ytick.major.size": 2.7,
            "xtick.direction": "out",
            "ytick.direction": "out",
            "pdf.fonttype": 42,
            "ps.fonttype": 42,
            "svg.fonttype": "none",
            "figure.facecolor": WHITE,
            "axes.facecolor": WHITE,
            "savefig.facecolor": WHITE,
            "savefig.transparent": False,
        }
    )


def _despine(ax: mpl.axes.Axes) -> None:
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)


def _panel_label(ax: mpl.axes.Axes, label: str, *, x: float = -0.06, y: float = 1.03) -> None:
    ax.text(
        x,
        y,
        label,
        transform=ax.transAxes,
        ha="right",
        va="bottom",
        fontsize=PANEL_LABEL_PT,
        fontweight="bold",
        color=INK,
        clip_on=False,
    )


def _normalise_id_value(value: object) -> str | None:
    if value is None or pd.isna(value):
        return None
    text = str(value).strip()
    if not text or text.upper() in {"NAN", "NONE", "NULL", "<NA>"}:
        return None
    if re.fullmatch(r"[-+]?\d+\.0+", text):
        text = text.split(".", 1)[0]
    return text.upper()


def _normalise_id(series: pd.Series) -> pd.Series:
    return series.map(_normalise_id_value).astype("string")


def _shape_ids(layer: gpd.GeoDataFrame, source_id: str, output_id: str) -> gpd.GeoDataFrame:
    if source_id not in layer.columns:
        raise KeyError(f"Shape layer is missing required identifier {source_id!r}")
    out = layer[[source_id, "geometry"]].copy()
    out[output_id] = _normalise_id(out[source_id])
    out = out.dropna(subset=[output_id, "geometry"])[[output_id, "geometry"]]
    out = out.loc[~out.geometry.is_empty].copy()
    try:
        out.geometry = out.geometry.make_valid()
    except Exception:
        out.geometry = out.buffer(0)
    return out.dissolve(by=output_id, as_index=False)


def _conus_states(states: gpd.GeoDataFrame, cfg: Mapping[str, object]) -> gpd.GeoDataFrame:
    state_id = cfg["columns"]["state_id"]
    out = states.copy()
    out[state_id] = _normalise_id(out[state_id])
    excluded = set(cfg["inventory"]["exclude_state_codes"])
    out = out.loc[~out[state_id].isin(excluded)].copy()
    return _shape_ids(out, state_id, "state_id")


def _set_conus_extent(ax: mpl.axes.Axes, states: gpd.GeoDataFrame) -> None:
    x0, y0, x1, y1 = states.total_bounds
    dx = (x1 - x0) * 0.012
    dy = (y1 - y0) * 0.020
    ax.set_xlim(x0 - dx, x1 + dx)
    ax.set_ylim(y0 - dy, y1 + dy)
    ax.set_axis_off()


def _truncate_cmap(name: str, low: float = 0.16, high: float = 0.95) -> LinearSegmentedColormap:
    base = mpl.colormaps[name]
    return LinearSegmentedColormap.from_list(
        f"{name}_{low:.2f}_{high:.2f}", base(np.linspace(low, high, 256))
    )


def _positive_log_norm(values: pd.Series) -> LogNorm:
    arr = pd.to_numeric(values, errors="coerce")
    arr = arr[np.isfinite(arr) & (arr > 0)]
    if arr.empty:
        return LogNorm(vmin=0.1, vmax=1.0)
    vmin = float(arr.min())
    vmax = float(arr.max())
    if math.isclose(vmin, vmax):
        vmin = max(vmin / 2.0, 1e-6)
        vmax = vmax * 2.0
    return LogNorm(vmin=vmin, vmax=vmax)


def _format_gl(value: float, _position: int | None = None) -> str:
    if value >= 100:
        return f"{value:.0f}"
    if value >= 10:
        return f"{value:.0f}"
    if value >= 1:
        return f"{value:.1f}"
    if value >= 0.1:
        return f"{value:.1f}"
    return f"{value:.2g}"


def _compact_log_ticks(norm: LogNorm, *, max_ticks: int = 3) -> list[float]:
    """Three well-separated log-space ticks; never append crowded endpoints."""
    vmin, vmax = float(norm.vmin), float(norm.vmax)
    if vmin <= 0 or not vmax > vmin:
        return [vmin]
    return [float(v) for v in np.geomspace(vmin, vmax, max_ticks)]


def _sha256(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def _public_csv(frame: pd.DataFrame, path: Path) -> None:
    forbidden = {
        "facility_id",
        "latitude",
        "longitude",
        "geometry",
        "full_address",
        "site_name",
        "baxtel_site_url",
    }
    present = forbidden.intersection(frame.columns)
    if present:
        raise ValueError("Refusing to export facility-sensitive columns: " + ", ".join(sorted(present)))
    path.parent.mkdir(parents=True, exist_ok=True)
    frame.to_csv(path, index=False)


def _save_figure(
    fig: mpl.figure.Figure,
    stem: str,
    *,
    figure_dir: Path,
    latex_targets: Iterable[Path],
    extended_data_number: int | None = None,
) -> list[Path]:
    """Save editable and review formats, then copy the exact LaTeX filename."""
    figure_dir.mkdir(parents=True, exist_ok=True)
    outputs: list[Path] = []
    for suffix, kwargs in [
        ("pdf", {}),
        ("svg", {}),
        ("png", {"dpi": PNG_DPI}),
    ]:
        path = figure_dir / f"{stem}.{suffix}"
        fig.savefig(path, bbox_inches="tight", pad_inches=0.025, facecolor=WHITE, **kwargs)
        outputs.append(path)

    png_path = figure_dir / f"{stem}.png"
    pdf_path = figure_dir / f"{stem}.pdf"
    for target in latex_targets:
        target.mkdir(parents=True, exist_ok=True)
        for source in [png_path, pdf_path]:
            destination = target / source.name
            if source.resolve() != destination.resolve():
                shutil.copy2(source, destination)

    if extended_data_number is not None:
        jpg = figure_dir / f"Guidi_EDfig{extended_data_number}.jpg"
        fig.savefig(
            jpg,
            dpi=EXTENDED_DATA_JPEG_DPI,
            bbox_inches="tight",
            pad_inches=0.025,
            facecolor=WHITE,
            pil_kwargs={"quality": 95, "subsampling": 0},
        )
        outputs.append(jpg)

    plt.close(fig)
    return outputs


def _ranked_concentration(
    regional: pd.DataFrame, *, value: str, geography: str, scenario: str = "baseline"
) -> pd.DataFrame:
    table = concentration_table(
        regional,
        value_column=value,
        geography=geography,
        scenario=scenario,
    ).copy()
    if table.empty:
        raise ValueError(f"No positive {value} values for {geography}")
    table["fraction_regions"] = table["rank"] / len(table)
    return table


def _n_to_share(table: pd.DataFrame, target: float = 0.50) -> int:
    hit = table.loc[table["cumulative_share"].ge(target)]
    return int(hit.iloc[0]["rank"]) if not hit.empty else int(len(table))


def _dominance_table(base: pd.DataFrame, geography: str) -> pd.DataFrame:
    out = base.copy()
    out[geography] = _normalise_id(out[geography])
    onsite = pd.to_numeric(out["onsite_m3_y"], errors="coerce").fillna(0.0)
    electricity = pd.to_numeric(out["electricity_m3_y"], errors="coerce").fillna(0.0)
    total = onsite + electricity
    out["electricity_to_onsite_ratio"] = electricity / onsite.replace(0, np.nan)
    out["pathway_balance_index"] = (electricity - onsite) / total.replace(0, np.nan)
    out["pathway_class"] = np.select(
        [electricity > 1.10 * onsite, onsite > 1.10 * electricity],
        ["Electricity-related dominant", "On-site dominant"],
        default="Similar (within 10%)",
    )
    out.loc[total.le(0), "pathway_class"] = "No modeled burden"
    return out[[
        geography,
        "n_facilities",
        "capacity_mw",
        "onsite_m3_y",
        "electricity_m3_y",
        "total_m3_y",
        "electricity_to_onsite_ratio",
        "pathway_balance_index",
        "pathway_class",
    ]]


# -----------------------------------------------------------------------------
# Figure 1: optional standalone production file matching the LaTeX TikZ concept
# -----------------------------------------------------------------------------
def _draw_study_design(figure_dir: Path, latex_targets: Iterable[Path]) -> list[Path]:
    fig, ax = plt.subplots(figsize=(DOUBLE_COLUMN_IN, 4.20))
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    ax.axis("off")

    def box(xy, width, height, text, *, face, edge, fontsize=6.2, weight="normal"):
        x, y = xy
        patch = FancyBboxPatch(
            (x, y), width, height,
            boxstyle="round,pad=0.012,rounding_size=0.012",
            linewidth=0.8, edgecolor=edge, facecolor=face,
        )
        ax.add_patch(patch)
        ax.text(x + width / 2, y + height / 2, text, ha="center", va="center",
                fontsize=fontsize, color=INK, fontweight=weight, linespacing=1.25)
        return patch

    box((0.17, 0.84), 0.66, 0.10,
        "Dated current-capacity inventory\n472 contiguous-U.S. hyperscale records; reported capacity and coordinates",
        face=PALE_GREY, edge=MID_GREY, weight="bold")
    box((0.17, 0.66), 0.66, 0.10,
        "Annualized facility electricity\n$E^{fac}_{i,s}=P_i\\,8760\\,u_s$; PUE partitions facility and IT electricity",
        face=PALE_GREY, edge=MID_GREY, weight="bold")

    box((0.05, 0.40), 0.40, 0.16,
        "On-site pathway\n$W^{(1)}_{i,s}=10^{-3}E^{IT,kWh}_{i,s}\\,WUE_s\\,c_s$\nHosting HydroBASINS level-6 basin",
        face=BLUE_LIGHT, edge=BLUE, weight="bold")
    box((0.55, 0.40), 0.40, 0.16,
        "Electricity-related pathway\n$I_{r,s}=\\sum_f s_{r,f}w_{f,s}$;  $W^{(2)}_{i,s}=10^{-3}E^{fac,kWh}_{i,s}I_{r(i),s}$\nHosting-BA production-mix attribution",
        face=ORANGE_LIGHT, edge=ORANGE, weight="bold")

    box((0.05, 0.14), 0.40, 0.15,
        "Hosting-basin context\nAnnual Aqueduct BWS/BWD\nFollow-up: cooling, source water, permits, reuse and drought planning",
        face=WHITE, edge=BLUE)
    box((0.55, 0.14), 0.40, 0.15,
        "BA attribution context\nAverage annual production mix\nFollow-up: utility planning, generation, transmission, procurement and load",
        face=WHITE, edge=ORANGE)

    arrows = [
        ((0.50, 0.84), (0.50, 0.76)),
        ((0.50, 0.66), (0.25, 0.56)),
        ((0.50, 0.66), (0.75, 0.56)),
        ((0.25, 0.40), (0.25, 0.29)),
        ((0.75, 0.40), (0.75, 0.29)),
    ]
    for start, end in arrows:
        ax.add_patch(FancyArrowPatch(start, end, arrowstyle="-|>", mutation_scale=8,
                                     linewidth=0.8, color=GREY))
    ax.text(0.50, 0.045,
            "National accounting may add pathway volumes; spatial interpretation remains pathway-specific.",
            ha="center", va="center", fontsize=6.0, color=GREY)
    return _save_figure(fig, "Figure1_Pathway_Resolved_Study_Design",
                        figure_dir=figure_dir, latex_targets=latex_targets)


# -----------------------------------------------------------------------------
# Figure 2: pathway-specific burden and context maps
# -----------------------------------------------------------------------------
def _draw_pathway_maps(
    *,
    states: gpd.GeoDataFrame,
    basins: gpd.GeoDataFrame,
    bas: gpd.GeoDataFrame,
    ba_shapes: gpd.GeoDataFrame,
    ba_base: pd.DataFrame,
    figure_dir: Path,
    table_dir: Path,
    latex_targets: Iterable[Path],
) -> tuple[list[Path], dict[str, float]]:
    base_basin = bas.copy()
    base_basin["onsite_GL_y"] = base_basin["onsite_m3_y"] / 1e6
    positive = base_basin["onsite_m3_y"].gt(0)
    burden_threshold = float(base_basin.loc[positive, "onsite_m3_y"].median())
    base_basin["high_burden"] = base_basin["onsite_m3_y"].ge(burden_threshold)
    base_basin["high_context"] = base_basin["capacity_weighted_bws_score"].ge(3.0)
    base_basin["high_burden_and_context"] = base_basin["high_burden"] & base_basin["high_context"]

    base_ba = ba_base.copy()
    base_ba["electricity_GL_y"] = base_ba["electricity_m3_y"] / 1e6
    intensity_col = "capacity_weighted_grid_water_L_per_kWh"
    valid_intensity = base_ba.loc[base_ba["electricity_m3_y"].gt(0), intensity_col].dropna()
    if valid_intensity.empty:
        raise ValueError("No finite hosting-BA grid-water intensities were available")
    intensity_median = float(valid_intensity.median())
    base_ba["above_median_grid_intensity"] = base_ba[intensity_col].gt(intensity_median)

    _public_csv(
        base_basin[[
            "hydrobasin_id", "n_facilities", "capacity_mw", "onsite_GL_y",
            "capacity_weighted_bws_score", "high_burden", "high_context",
            "high_burden_and_context",
        ]],
        table_dir / "Figure2_Onsite_Basins.csv",
    )
    _public_csv(
        base_ba[[
            "ba_id", "n_facilities", "capacity_mw", "electricity_GL_y", intensity_col,
            "above_median_grid_intensity",
        ]],
        table_dir / "Figure2_Electricity_BAs.csv",
    )

    crs = "EPSG:5070"
    st = states.to_crs(crs)
    basin_map = basins.merge(base_basin, on="hydrobasin_id", how="inner", validate="one_to_one").to_crs(crs)
    basin_map = basin_map.loc[basin_map["onsite_GL_y"].gt(0)].copy()
    ba_map = ba_shapes.merge(base_ba, on="ba_id", how="inner", validate="one_to_one").to_crs(crs)
    ba_map = ba_map.loc[ba_map["electricity_GL_y"].gt(0)].copy()

    fig, axes = plt.subplots(1, 2, figsize=(DOUBLE_COLUMN_IN, 3.08), constrained_layout=False)
    blue_cmap = _truncate_cmap("Blues", 0.18, 0.96)
    orange_cmap = _truncate_cmap("Oranges", 0.14, 0.96)

    for ax in axes:
        st.plot(ax=ax, color=PALE_GREY, edgecolor=WHITE, linewidth=0.20, zorder=0)

    norm_basin = _positive_log_norm(basin_map["onsite_GL_y"])
    basin_map.plot(
        ax=axes[0], column="onsite_GL_y", cmap=blue_cmap, norm=norm_basin,
        edgecolor=WHITE, linewidth=0.10, zorder=1,
    )
    high_context = basin_map.loc[basin_map["high_context"]]
    intersection = basin_map.loc[basin_map["high_burden_and_context"]]
    if not high_context.empty:
        high_context.boundary.plot(ax=axes[0], color=ROSE, linewidth=0.75, zorder=3)
    if not intersection.empty:
        intersection.boundary.plot(ax=axes[0], color=INK, linewidth=1.25, zorder=4)
    st.boundary.plot(ax=axes[0], color=NAVY, linewidth=0.25, zorder=5)
    _set_conus_extent(axes[0], st)
    _panel_label(axes[0], "a", x=-0.035, y=1.015)
    axes[0].text(0.015, 1.015, "On-site consumption in hosting basins",
                 transform=axes[0].transAxes, ha="left", va="bottom", fontsize=7.0, color=INK)
    axes[0].legend(
        handles=[
            Line2D([0], [0], color=ROSE, lw=1.1, label="Annual BWS ≥ 3"),
            Line2D([0], [0], color=INK, lw=1.7, label="High burden + annual BWS ≥ 3"),
        ],
        frameon=False, loc="lower left", bbox_to_anchor=(0.01, 0.01), borderaxespad=0,
        handlelength=2.2, labelspacing=0.35,
    )
    sm1 = mpl.cm.ScalarMappable(norm=norm_basin, cmap=blue_cmap)
    sm1.set_array([])
    cb1 = fig.colorbar(sm1, ax=axes[0], orientation="horizontal", fraction=0.038, pad=0.025, aspect=35)
    cb1.set_label("On-site consumption (GL yr$^{-1}$; logarithmic scale)", labelpad=2)
    cb1.set_ticks(_compact_log_ticks(norm_basin))
    cb1.ax.xaxis.set_major_formatter(FuncFormatter(_format_gl))
    cb1.ax.tick_params(labelsize=5.5, pad=1)

    norm_ba = _positive_log_norm(ba_map["electricity_GL_y"])
    ba_map.plot(
        ax=axes[1], column="electricity_GL_y", cmap=orange_cmap, norm=norm_ba,
        edgecolor=WHITE, linewidth=0.14, zorder=1,
    )
    high_intensity = ba_map.loc[ba_map["above_median_grid_intensity"]]
    if not high_intensity.empty:
        high_intensity.boundary.plot(ax=axes[1], color=INK, linewidth=1.10, zorder=4)
    st.boundary.plot(ax=axes[1], color=NAVY, linewidth=0.25, zorder=5)
    _set_conus_extent(axes[1], st)
    _panel_label(axes[1], "b", x=-0.035, y=1.015)
    axes[1].text(0.015, 1.015, "Electricity-related consumption by hosting BA",
                 transform=axes[1].transAxes, ha="left", va="bottom", fontsize=7.0, color=INK)
    axes[1].legend(
        handles=[Line2D(
            [0], [0], color=INK, lw=1.6,
            label=f"Grid-water intensity > median ({intensity_median:.2f} L kWh$^{{-1}}$)",
        )],
        frameon=False, loc="lower left", bbox_to_anchor=(0.01, 0.01), borderaxespad=0,
        handlelength=2.2,
    )
    sm2 = mpl.cm.ScalarMappable(norm=norm_ba, cmap=orange_cmap)
    sm2.set_array([])
    cb2 = fig.colorbar(sm2, ax=axes[1], orientation="horizontal", fraction=0.038, pad=0.025, aspect=35)
    cb2.set_label("Electricity-related consumption (GL yr$^{-1}$; logarithmic scale)", labelpad=2)
    cb2.set_ticks(_compact_log_ticks(norm_ba))
    cb2.ax.xaxis.set_major_formatter(FuncFormatter(_format_gl))
    cb2.ax.tick_params(labelsize=5.5, pad=1)

    fig.subplots_adjust(left=0.015, right=0.995, bottom=0.16, top=0.93, wspace=0.045)
    outputs = _save_figure(
        fig, "Figure2_Pathway_Burden_Maps",
        figure_dir=figure_dir, latex_targets=latex_targets,
    )
    return outputs, {
        "onsite_burden_median_m3_y": burden_threshold,
        "BA_grid_intensity_median_L_per_kWh": intensity_median,
    }


# -----------------------------------------------------------------------------
# Figure 3: normalized concentration curves
# -----------------------------------------------------------------------------
def _draw_concentration(
    basin_all: pd.DataFrame,
    ba_all: pd.DataFrame,
    *,
    figure_dir: Path,
    table_dir: Path,
    latex_targets: Iterable[Path],
) -> tuple[list[Path], dict[str, int]]:
    onsite = _ranked_concentration(
        basin_all, value="onsite_m3_y", geography="hydrobasin_id"
    )
    electricity = _ranked_concentration(
        ba_all, value="electricity_m3_y", geography="ba_id"
    )
    onsite["pathway_label"] = "On-site; hosting basins"
    electricity["pathway_label"] = "Electricity-related; hosting BAs"
    _public_csv(onsite, table_dir / "Figure3_Concentration_Onsite.csv")
    _public_csv(electricity, table_dir / "Figure3_Concentration_Electricity.csv")

    n50_on = _n_to_share(onsite, 0.50)
    n50_el = _n_to_share(electricity, 0.50)
    x50_on = 100.0 * n50_on / len(onsite)
    x50_el = 100.0 * n50_el / len(electricity)
    y50_on = 100.0 * float(onsite.loc[onsite["rank"].eq(n50_on), "cumulative_share"].iloc[0])
    y50_el = 100.0 * float(electricity.loc[electricity["rank"].eq(n50_el), "cumulative_share"].iloc[0])

    fig, ax = plt.subplots(figsize=(5.55, 3.55))
    for data, color, label, n50, x50, y50, yoffset in [
        (onsite, BLUE, f"On-site: {n50_on}/{len(onsite)} basins", n50_on, x50_on, y50_on, 5.5),
        (electricity, ORANGE, f"Electricity-related: {n50_el}/{len(electricity)} BAs", n50_el, x50_el, y50_el, -8.0),
    ]:
        x = np.r_[0.0, data["fraction_regions"].to_numpy(dtype=float) * 100.0]
        y = np.r_[0.0, data["cumulative_share"].to_numpy(dtype=float) * 100.0]
        ax.step(x, y, where="post", color=color, linewidth=1.75, label=label, zorder=3)
        ax.scatter([x50], [y50], s=24, facecolor=WHITE, edgecolor=color, linewidth=1.15, zorder=4)
        ax.plot([x50, x50], [0, y50], color=color, linewidth=0.65, linestyle=(0, (2, 2)), zorder=1)
        ax.annotate(
            f"{n50} regions\n({x50:.1f}% of hosts)",
            xy=(x50, y50), xytext=(7, yoffset), textcoords="offset points",
            ha="left", va="center", fontsize=5.6, color=color,
            arrowprops={"arrowstyle": "-", "lw": 0.45, "color": color},
        )

    ax.plot([0, 100], [0, 100], color=MID_GREY, linewidth=0.8, linestyle=(0, (3, 3)),
            label="Equal distribution", zorder=0)
    ax.axhline(50, color=LIGHT_GREY, linewidth=0.7, zorder=0)
    ax.set_xlim(0, 100)
    ax.set_ylim(0, 102)
    ax.set_xlabel("Positive-burden hosting regions included (%)")
    ax.set_ylabel("Cumulative share of pathway total (%)")
    ax.xaxis.set_major_locator(MaxNLocator(6))
    ax.yaxis.set_major_locator(MaxNLocator(6))
    ax.grid(axis="y", color=LIGHT_GREY, linewidth=0.45, zorder=0)
    _despine(ax)
    ax.legend(frameon=False, loc="lower right", handlelength=2.6)
    fig.subplots_adjust(left=0.12, right=0.98, bottom=0.20, top=0.90)
    outputs = _save_figure(
        fig, "Figure3_Normalized_Concentration",
        figure_dir=figure_dir, latex_targets=latex_targets,
    )
    return outputs, {
        "n_hosting_basins": int(len(onsite)),
        "n_hosting_BAs": int(len(electricity)),
        "n_basins_for_50_percent": int(n50_on),
        "n_BAs_for_50_percent": int(n50_el),
    }


# -----------------------------------------------------------------------------
# Figure 4: maps and scenario intervals, with exact component filenames
# -----------------------------------------------------------------------------
DOMINANCE_COLORS = {
    "On-site dominant": BLUE,
    "Electricity-related dominant": ORANGE,
    "Similar (within 10%)": MID_GREY,
    "No modeled burden": PALE_GREY,
}


def _plot_dominance_axis(
    ax: mpl.axes.Axes,
    *,
    states: gpd.GeoDataFrame,
    region_shapes: gpd.GeoDataFrame,
    dominance: pd.DataFrame,
    region_id: str,
) -> None:
    st = states.to_crs("EPSG:5070")
    layer = region_shapes.merge(dominance, on=region_id, how="left", validate="one_to_one").to_crs("EPSG:5070")
    layer["pathway_class"] = layer["pathway_class"].fillna("No modeled burden")
    for category in ["No modeled burden", "Similar (within 10%)", "On-site dominant", "Electricity-related dominant"]:
        subset = layer.loc[layer["pathway_class"].eq(category)]
        if not subset.empty:
            subset.plot(
                ax=ax, color=DOMINANCE_COLORS[category], edgecolor=WHITE,
                linewidth=0.16, zorder=1,
            )
    st.boundary.plot(ax=ax, color=NAVY, linewidth=0.25, zorder=3)
    _set_conus_extent(ax, st)


def _dominance_component(
    stem: str,
    *,
    states: gpd.GeoDataFrame,
    region_shapes: gpd.GeoDataFrame,
    dominance: pd.DataFrame,
    region_id: str,
    show_legend: bool,
    figure_dir: Path,
    latex_targets: Iterable[Path],
) -> list[Path]:
    fig = plt.figure(figsize=(SINGLE_COLUMN_IN, 3.55))
    ax = fig.add_axes([0.025, 0.15 if show_legend else 0.08, 0.95, 0.79])
    _plot_dominance_axis(
        ax, states=states, region_shapes=region_shapes, dominance=dominance, region_id=region_id
    )
    if show_legend:
        handles = [
            Patch(facecolor=BLUE, edgecolor="none", label="On-site dominant"),
            Patch(facecolor=ORANGE, edgecolor="none", label="Electricity-related dominant"),
            Patch(facecolor=MID_GREY, edgecolor="none", label="Similar (within 10%)"),
        ]
        fig.legend(handles=handles, frameon=False, ncol=1, loc="lower left",
                   bbox_to_anchor=(0.045, 0.012), handlelength=1.1, labelspacing=0.25)
    return _save_figure(fig, stem, figure_dir=figure_dir, latex_targets=latex_targets)


def _profile_source(
    all_data: pd.DataFrame,
    geography: str,
    *,
    top_n: int = 15,
) -> tuple[pd.DataFrame, list[str]]:
    all_data = all_data.copy()
    all_data[geography] = _normalise_id(all_data[geography])
    baseline = all_data.loc[all_data["scenario"].eq("baseline")].copy()
    leaders = baseline.nlargest(top_n, "capacity_mw")[geography].dropna().astype(str).tolist()
    subset = all_data.loc[
        all_data[geography].astype(str).isin(leaders)
        & all_data["scenario"].isin(PROFILE_SCENARIOS)
    ].copy()
    rank_map = {region: i + 1 for i, region in enumerate(leaders)}
    subset["capacity_rank"] = subset[geography].astype(str).map(rank_map)
    subset["total_GL_y"] = subset["total_m3_y"] / 1e6
    return subset.sort_values(["capacity_rank", "scenario"]), leaders


def _plot_profile_axis(
    ax: mpl.axes.Axes,
    *,
    source: pd.DataFrame,
    geography: str,
    leaders: list[str],
    show_legend: bool,
) -> None:
    y = np.arange(len(leaders))
    value_map = source.pivot(index=geography, columns="scenario", values="total_GL_y")
    capacity_map = (
        source.loc[source["scenario"].eq("baseline")]
        .set_index(geography)["capacity_mw"]
        .to_dict()
    )
    labels = []
    for region in leaders:
        vals = value_map.loc[region, PROFILE_SCENARIOS].astype(float)
        yy = leaders.index(region)
        ax.hlines(yy, float(vals.min()), float(vals.max()), color=LIGHT_GREY, linewidth=1.6, zorder=1)
        for scenario in PROFILE_SCENARIOS:
            style = PROFILE_STYLE[scenario]
            face = WHITE if scenario == "baseline" else style["color"]
            edge = style["color"]
            ax.scatter(
                float(vals[scenario]), yy, s=20 if scenario == "baseline" else 15,
                marker=style["marker"], facecolor=face, edgecolor=edge,
                linewidth=0.9, zorder=3,
            )
        cap = float(capacity_map.get(region, np.nan))
        labels.append(f"{region}  ·  {cap/1000:.2f} GW" if np.isfinite(cap) else str(region))

    ax.set_yticks(y)
    ax.set_yticklabels(labels)
    ax.invert_yaxis()
    ax.set_xlabel("Total host-location operational water (GL yr$^{-1}$)")
    ax.grid(axis="x", color=LIGHT_GREY, linewidth=0.45, zorder=0)
    ax.xaxis.set_major_locator(MaxNLocator(5))
    _despine(ax)
    if show_legend:
        handles = [
            Line2D([0], [0], marker=PROFILE_STYLE[s]["marker"], linestyle="none",
                   markerfacecolor=WHITE if s == "baseline" else PROFILE_STYLE[s]["color"],
                   markeredgecolor=PROFILE_STYLE[s]["color"], markeredgewidth=0.9,
                   markersize=4.5, label=PROFILE_STYLE[s]["label"])
            for s in PROFILE_SCENARIOS
        ]
        ax.legend(handles=handles, frameon=False, ncol=1, loc="lower right",
                  bbox_to_anchor=(1.0, 1.0), borderaxespad=0, handletextpad=0.5, labelspacing=0.25)


def _profile_component(
    stem: str,
    *,
    source: pd.DataFrame,
    geography: str,
    leaders: list[str],
    show_legend: bool,
    figure_dir: Path,
    latex_targets: Iterable[Path],
) -> list[Path]:
    fig, ax = plt.subplots(figsize=(SINGLE_COLUMN_IN, 3.55))
    _plot_profile_axis(ax, source=source, geography=geography, leaders=leaders, show_legend=show_legend)
    fig.subplots_adjust(left=0.27, right=0.985, bottom=0.14, top=0.91)
    return _save_figure(fig, stem, figure_dir=figure_dir, latex_targets=latex_targets)


def _draw_figure4(
    *,
    states: gpd.GeoDataFrame,
    state_shapes: gpd.GeoDataFrame,
    ba_shapes: gpd.GeoDataFrame,
    state_all: pd.DataFrame,
    ba_all: pd.DataFrame,
    figure_dir: Path,
    table_dir: Path,
    latex_targets: Iterable[Path],
) -> list[Path]:
    state_base = state_all.loc[state_all["scenario"].eq("baseline")].copy()
    ba_base = ba_all.loc[ba_all["scenario"].eq("baseline")].copy()
    state_dom = _dominance_table(state_base, "state_id")
    ba_dom = _dominance_table(ba_base, "ba_id")
    state_profile, state_leaders = _profile_source(state_all, "state_id", top_n=15)
    ba_profile, ba_leaders = _profile_source(ba_all, "ba_id", top_n=15)

    _public_csv(state_dom, table_dir / "Figure4_State_Pathway_Dominance.csv")
    _public_csv(ba_dom, table_dir / "Figure4_BA_Pathway_Dominance.csv")
    _public_csv(state_profile, table_dir / "Figure4_State_Scenario_Intervals.csv")
    _public_csv(ba_profile, table_dir / "Figure4_BA_Scenario_Intervals.csv")

    outputs: list[Path] = []
    outputs += _dominance_component(
        "Figure4_State_Pathway_Dominance", states=states, region_shapes=state_shapes,
        dominance=state_dom, region_id="state_id", show_legend=True,
        figure_dir=figure_dir, latex_targets=latex_targets,
    )
    outputs += _profile_component(
        "Figure4_State_Scenario_Intervals", source=state_profile, geography="state_id",
        leaders=state_leaders, show_legend=True,
        figure_dir=figure_dir, latex_targets=latex_targets,
    )
    outputs += _dominance_component(
        "Figure4_BA_Pathway_Dominance", states=states, region_shapes=ba_shapes,
        dominance=ba_dom, region_id="ba_id", show_legend=False,
        figure_dir=figure_dir, latex_targets=latex_targets,
    )
    outputs += _profile_component(
        "Figure4_BA_Scenario_Intervals", source=ba_profile, geography="ba_id",
        leaders=ba_leaders, show_legend=False,
        figure_dir=figure_dir, latex_targets=latex_targets,
    )

    # Standalone composite for final production upload.  The LaTeX uses the four
    # exact component PNGs above, but Nature production normally expects one file
    # per numbered figure.
    fig = plt.figure(figsize=(DOUBLE_COLUMN_IN, 6.80))
    gs = fig.add_gridspec(2, 2, width_ratios=[1.0, 1.0], height_ratios=[1.0, 1.0],
                          wspace=0.30, hspace=0.28)
    ax_a = fig.add_subplot(gs[0, 0])
    ax_b = fig.add_subplot(gs[0, 1])
    ax_c = fig.add_subplot(gs[1, 0])
    ax_d = fig.add_subplot(gs[1, 1])
    _plot_dominance_axis(ax_a, states=states, region_shapes=state_shapes,
                         dominance=state_dom, region_id="state_id")
    _plot_profile_axis(ax_b, source=state_profile, geography="state_id",
                       leaders=state_leaders, show_legend=False)
    _plot_dominance_axis(ax_c, states=states, region_shapes=ba_shapes,
                         dominance=ba_dom, region_id="ba_id")
    _plot_profile_axis(ax_d, source=ba_profile, geography="ba_id",
                       leaders=ba_leaders, show_legend=False)
    for ax, label in zip([ax_a, ax_b, ax_c, ax_d], ["a", "b", "c", "d"]):
        _panel_label(ax, label, x=-0.04, y=1.02)
    ax_a.set_title("Host-state pathway balance", loc="left", fontsize=7.0, pad=3)
    ax_b.set_title("Host-state scenario range", loc="left", fontsize=7.0, pad=3)
    ax_c.set_title("Hosting-BA pathway balance", loc="left", fontsize=7.0, pad=3)
    ax_d.set_title("Hosting-BA scenario range", loc="left", fontsize=7.0, pad=3)
    class_handles = [
        Patch(facecolor=BLUE, edgecolor="none", label="On-site dominant"),
        Patch(facecolor=ORANGE, edgecolor="none", label="Electricity-related dominant"),
        Patch(facecolor=MID_GREY, edgecolor="none", label="Similar (within 10%)"),
    ]
    scenario_handles = [
        Line2D([0], [0], marker=PROFILE_STYLE[s]["marker"], linestyle="none",
               markerfacecolor=WHITE if s == "baseline" else PROFILE_STYLE[s]["color"],
               markeredgecolor=PROFILE_STYLE[s]["color"], markeredgewidth=0.9,
               markersize=4.5, label=PROFILE_STYLE[s]["label"])
        for s in PROFILE_SCENARIOS
    ]
    fig.legend(class_handles + scenario_handles,
               [h.get_label() for h in class_handles + scenario_handles],
               frameon=False, ncol=3, loc="lower center", bbox_to_anchor=(0.5, 0.006),
               columnspacing=1.3, handletextpad=0.5)
    fig.subplots_adjust(left=0.08, right=0.99, bottom=0.09, top=0.97)
    outputs += _save_figure(
        fig, "Figure4_HostLocation_Profiles",
        figure_dir=figure_dir, latex_targets=latex_targets,
    )
    return outputs


# -----------------------------------------------------------------------------
# Extended Data figures referenced by main.tex
# -----------------------------------------------------------------------------
def _draw_scenario_totals(
    national: pd.DataFrame,
    *,
    figure_dir: Path,
    table_dir: Path,
    latex_targets: Iterable[Path],
) -> list[Path]:
    data = national.copy()
    data["order"] = data["scenario"].map({name: i for i, name in enumerate(SCENARIO_ORDER)})
    data = data.sort_values("order").reset_index(drop=True)
    data["label"] = data["scenario"].map(SCENARIO_LABELS)
    _public_csv(data.drop(columns="order"), table_dir / "ExtendedData_Scenario_Totals.csv")

    fig, ax = plt.subplots(figsize=(5.65, 3.25))
    y = np.arange(len(data))
    on = data["onsite_GL_y"].to_numpy(dtype=float)
    el = data["electricity_GL_y"].to_numpy(dtype=float)
    total = on + el
    ax.barh(y, on, color=BLUE, height=0.58, label="On-site")
    ax.barh(y, el, left=on, color=ORANGE, height=0.58, label="Electricity-related")
    for yy, value in zip(y, total):
        ax.text(value + total.max() * 0.014, yy, f"{value:.1f}", va="center", ha="left", fontsize=5.8)
    ax.set_yticks(y)
    ax.set_yticklabels(data["label"])
    ax.invert_yaxis()
    ax.set_xlabel("Annual operational water consumption (GL yr$^{-1}$)")
    ax.set_xlim(0, total.max() * 1.13)
    ax.grid(axis="x", color=LIGHT_GREY, linewidth=0.45, zorder=0)
    _despine(ax)
    ax.legend(frameon=False, ncol=2, loc="lower left", bbox_to_anchor=(0, 1.0), borderaxespad=0)
    fig.subplots_adjust(left=0.30, right=0.97, bottom=0.16, top=0.90)
    return _save_figure(
        fig, "ExtendedData_Scenario_Totals",
        figure_dir=figure_dir, latex_targets=latex_targets, extended_data_number=1,
    )


def _stress_counts(basin_base: pd.DataFrame) -> pd.DataFrame:
    positive = basin_base["onsite_m3_y"].gt(0)
    threshold = float(basin_base.loc[positive, "onsite_m3_y"].median())
    high_burden = basin_base["onsite_m3_y"].ge(threshold)
    rows = []
    for indicator, label in [
        ("capacity_weighted_bws_score", "Withdrawal-based BWS"),
        ("capacity_weighted_bwd_score", "Consumption-based BWD"),
    ]:
        high_context = basin_base[indicator].ge(3.0)
        rows.append({
            "indicator": indicator,
            "label": label,
            "n_hosting_basins": int(positive.sum()),
            "n_high_context": int(high_context.sum()),
            "n_high_burden_and_context": int((high_burden & high_context).sum()),
            "burden_threshold_m3_y": threshold,
            "context_threshold": 3.0,
        })
    return pd.DataFrame(rows)


def _draw_stress_sensitivity(
    basin_base: pd.DataFrame,
    *,
    figure_dir: Path,
    table_dir: Path,
    latex_targets: Iterable[Path],
) -> tuple[list[Path], pd.DataFrame]:
    counts = _stress_counts(basin_base)
    _public_csv(counts, table_dir / "ExtendedData_Aqueduct_BWS_vs_BWD.csv")
    x = np.arange(len(counts))
    width = 0.32
    fig, ax = plt.subplots(figsize=(4.55, 3.20))
    bars1 = ax.bar(x - width / 2, counts["n_high_context"], width,
                   color=MID_GREY, edgecolor=INK, linewidth=0.55, hatch="///",
                   label="High context")
    bars2 = ax.bar(x + width / 2, counts["n_high_burden_and_context"], width,
                   color=BLUE, edgecolor=INK, linewidth=0.55,
                   label="High burden + high context")
    for bars in [bars1, bars2]:
        for bar in bars:
            value = int(round(bar.get_height()))
            ax.text(bar.get_x() + bar.get_width() / 2, value + 0.45, str(value),
                    ha="center", va="bottom", fontsize=6.0)
    ax.set_xticks(x)
    ax.set_xticklabels(["Withdrawal-based\nBWS", "Consumption-based\nBWD"])
    ax.set_ylabel("Number of hosting basins")
    ax.set_ylim(0, max(counts["n_high_context"].max() * 1.20, 5))
    ax.yaxis.set_major_locator(MaxNLocator(integer=True))
    ax.grid(axis="y", color=LIGHT_GREY, linewidth=0.45, zorder=0)
    _despine(ax)
    ax.legend(frameon=False, loc="lower left", bbox_to_anchor=(0, 1.0), borderaxespad=0)
    fig.subplots_adjust(left=0.16, right=0.98, bottom=0.20, top=0.88)
    outputs = _save_figure(
        fig, "ExtendedData_Aqueduct_BWS_vs_BWD",
        figure_dir=figure_dir, latex_targets=latex_targets, extended_data_number=2,
    )
    return outputs, counts


def _deterministic_label_offsets(data: pd.DataFrame, ids: list[str]) -> dict[str, tuple[int, int]]:
    offsets: dict[str, tuple[int, int]] = {}
    ordered = data.loc[data["ba_id"].astype(str).isin(ids)].sort_values(
        ["electricity_GL_y", "capacity_weighted_grid_water_L_per_kWh"], ascending=False
    )
    choices = [(6, 8), (6, -9), (-7, 8), (-7, -9), (8, 14), (-8, 14), (8, -15), (-8, -15)]
    for idx, row in enumerate(ordered.itertuples(index=False)):
        x = float(getattr(row, "capacity_weighted_bws_score"))
        dx, dy = choices[idx % len(choices)]
        if x > 4.2 and dx > 0:
            dx = -abs(dx)
        if x < 0.5 and dx < 0:
            dx = abs(dx)
        offsets[str(getattr(row, "ba_id"))] = (dx, dy)
    return offsets


def _draw_ba_quadrant(
    ba_base: pd.DataFrame,
    *,
    figure_dir: Path,
    table_dir: Path,
    latex_targets: Iterable[Path],
) -> list[Path]:
    intensity_col = "capacity_weighted_grid_water_L_per_kWh"
    stress_col = "capacity_weighted_bws_score"
    data = ba_base.copy()
    data["electricity_GL_y"] = data["electricity_m3_y"] / 1e6
    data["capacity_GW"] = data["capacity_mw"] / 1000.0
    keep = (
        data["electricity_GL_y"].gt(0)
        & data[intensity_col].notna()
        & data[stress_col].notna()
        & data["capacity_GW"].gt(0)
    )
    data = data.loc[keep].copy()
    if data.empty:
        raise ValueError("No complete BA rows were available for the quadrant diagnostic")
    median_intensity = float(data[intensity_col].median())
    rho = float(data[stress_col].rank(method="average").corr(data[intensity_col].rank(method="average")))
    _public_csv(
        data[[
            "ba_id", "n_facilities", "capacity_GW", "electricity_GL_y",
            stress_col, "capacity_weighted_bwd_score", intensity_col,
        ]],
        table_dir / "Figure3_Scope2_Quadrant.csv",
    )

    max_cap = float(data["capacity_GW"].max())
    sizes = 22.0 + 165.0 * np.sqrt(data["capacity_GW"] / max_cap)
    color_norm = PowerNorm(gamma=0.55, vmin=0, vmax=float(data["electricity_GL_y"].max()))
    cmap = _truncate_cmap("YlOrBr", 0.18, 0.95)

    fig, ax = plt.subplots(figsize=(5.45, 4.05))
    scatter = ax.scatter(
        data[stress_col], data[intensity_col], s=sizes,
        c=data["electricity_GL_y"], cmap=cmap, norm=color_norm,
        edgecolor=NAVY, linewidth=0.55, alpha=0.93, zorder=3,
    )
    ax.axvline(3.0, color=ROSE, linewidth=0.9, linestyle=(0, (4, 2)), zorder=1)
    ax.axhline(median_intensity, color=GREY, linewidth=0.9, linestyle=(0, (4, 2)), zorder=1)
    ax.set_xlim(-0.15, 5.15)
    ax.set_ylim(0, max(float(data[intensity_col].max()) * 1.10, median_intensity * 1.3))
    ax.set_xlabel("Capacity-weighted annual baseline water stress at hosted sites (score, 0–5)")
    ax.set_ylabel("BA-average grid-water intensity (L kWh$^{-1}$)")
    ax.text(0.02, 1.04, f"Spearman $\\rho$ = {rho:.2f}; $n$ = {len(data)} hosting BAs",
            transform=ax.transAxes, ha="left", va="bottom", fontsize=6.0)
    ax.text(3.04, 0.03, "high-context hosted sites", transform=ax.get_xaxis_transform(),
            ha="left", va="bottom", fontsize=5.4, color=ROSE)
    ax.text(0.02, median_intensity + 0.04 * ax.get_ylim()[1], "median grid intensity",
            ha="left", va="bottom", fontsize=5.4, color=GREY)

    top_burden = data.nlargest(5, "electricity_GL_y")["ba_id"].astype(str).tolist()
    top_intensity = data.nlargest(2, intensity_col)["ba_id"].astype(str).tolist()
    label_ids = list(dict.fromkeys(top_burden + top_intensity))
    offsets = _deterministic_label_offsets(data, label_ids)
    for _, row in data.loc[data["ba_id"].astype(str).isin(label_ids)].iterrows():
        dx, dy = offsets[str(row["ba_id"])]
        ax.annotate(
            str(row["ba_id"]),
            xy=(row[stress_col], row[intensity_col]), xytext=(dx, dy),
            textcoords="offset points", ha="left" if dx >= 0 else "right", va="center",
            fontsize=5.5, color=INK,
            arrowprops={"arrowstyle": "-", "lw": 0.4, "color": MID_GREY},
        )

    cbar = fig.colorbar(scatter, ax=ax, fraction=0.048, pad=0.025)
    cbar.set_label("Electricity-related consumption (GL yr$^{-1}$)")
    cbar.ax.yaxis.set_major_formatter(FuncFormatter(_format_gl))

    q = np.array([0.25, 1.0, 4.0])
    handles = []
    for value in q:
        marker_area = 22.0 + 165.0 * math.sqrt(float(value) / max_cap)
        handles.append(Line2D(
            [0], [0], marker="o", linestyle="none", markersize=math.sqrt(marker_area),
            markerfacecolor=WHITE, markeredgecolor=NAVY, markeredgewidth=0.55,
            label=f"{value:.2f} GW",
        ))
    ax.legend(handles=handles, title="Hosted capacity", frameon=False,
              loc="upper center", bbox_to_anchor=(0.5, -0.27), ncol=3, labelspacing=0.55)
    _despine(ax)
    fig.subplots_adjust(left=0.13, right=0.88, bottom=0.20, top=0.90)
    return _save_figure(
        fig, "Figure3_Scope2_Quadrant",
        figure_dir=figure_dir, latex_targets=latex_targets, extended_data_number=3,
    )


# -----------------------------------------------------------------------------
# Reconciliation, manifest and LaTeX values
# -----------------------------------------------------------------------------
def _actual_values(
    *,
    canonical: pd.DataFrame,
    national: pd.DataFrame,
    concentration_values: Mapping[str, int],
    stress_counts: pd.DataFrame,
) -> dict[str, float | int]:
    baseline = national.loc[national["scenario"].eq("baseline")].iloc[0]
    bws = stress_counts.loc[stress_counts["indicator"].eq("capacity_weighted_bws_score")].iloc[0]
    bwd = stress_counts.loc[stress_counts["indicator"].eq("capacity_weighted_bwd_score")].iloc[0]
    return {
        "n_facilities": int(baseline["n_facilities"]),
        "capacity_mw_rounded": int(round(float(baseline["capacity_mw"]))),
        "baseline_total_GL_y": float(baseline["total_GL_y"]),
        "baseline_onsite_GL_y": float(baseline["onsite_GL_y"]),
        "baseline_electricity_GL_y": float(baseline["electricity_GL_y"]),
        "n_hosting_basins": int(concentration_values["n_hosting_basins"]),
        "n_hosting_BAs": int(concentration_values["n_hosting_BAs"]),
        "n_basins_for_50_percent": int(concentration_values["n_basins_for_50_percent"]),
        "n_BAs_for_50_percent": int(concentration_values["n_BAs_for_50_percent"]),
        "BWS_high_context_basins": int(bws["n_high_context"]),
        "BWS_high_burden_and_context": int(bws["n_high_burden_and_context"]),
        "BWD_high_context_basins": int(bwd["n_high_context"]),
        "BWD_high_burden_and_context": int(bwd["n_high_burden_and_context"]),
    }


def _compare_expected(actual: Mapping[str, float | int], expected: Mapping[str, float | int]) -> pd.DataFrame:
    rows = []
    for key, expected_value in expected.items():
        actual_value = actual.get(key, np.nan)
        if isinstance(expected_value, float):
            passed = bool(np.isclose(float(actual_value), float(expected_value), atol=0.01, rtol=0.0))
        else:
            passed = bool(int(actual_value) == int(expected_value))
        rows.append({
            "quantity": key,
            "expected_in_current_main_tex": expected_value,
            "canonical_run_value": actual_value,
            "passed": passed,
            "action": "none" if passed else "update manuscript/legend or diagnose the canonical run before submission",
        })
    return pd.DataFrame(rows)


def _write_latex_values(actual: Mapping[str, float | int], path: Path) -> None:
    commands = {
        "NFacilities": f"{int(actual['n_facilities'])}",
        "CapacityMW": f"{int(actual['capacity_mw_rounded']):,}",
        "BaselineTotalGL": f"{float(actual['baseline_total_GL_y']):.2f}",
        "BaselineOnsiteGL": f"{float(actual['baseline_onsite_GL_y']):.2f}",
        "BaselineElectricityGL": f"{float(actual['baseline_electricity_GL_y']):.2f}",
        "NHostingBasins": f"{int(actual['n_hosting_basins'])}",
        "NHostingBAs": f"{int(actual['n_hosting_BAs'])}",
        "NBasinsHalf": f"{int(actual['n_basins_for_50_percent'])}",
        "NBAsHalf": f"{int(actual['n_BAs_for_50_percent'])}",
        "NBWSHigh": f"{int(actual['BWS_high_context_basins'])}",
        "NBWSBoth": f"{int(actual['BWS_high_burden_and_context'])}",
        "NBWDHigh": f"{int(actual['BWD_high_context_basins'])}",
        "NBWDBoth": f"{int(actual['BWD_high_burden_and_context'])}",
    }
    text = "% Auto-generated from the canonical Nature Water figure run.\n"
    text += "% Do not edit by hand; regenerate after any data or configuration change.\n"
    text += "\n".join(f"\\newcommand{{\\{name}}}{{{value}}}" for name, value in commands.items())
    path.write_text(text + "\n", encoding="utf-8")


def _file_manifest(paths: Iterable[Path]) -> pd.DataFrame:
    rows = []
    for path in sorted({Path(p).resolve() for p in paths if Path(p).exists()}):
        rows.append({
            "file": str(path),
            "name": path.name,
            "suffix": path.suffix.lower(),
            "bytes": path.stat().st_size,
            "sha256": _sha256(path),
        })
    return pd.DataFrame(rows)


def generate_super_defensible_figures() -> dict[str, object]:
    _set_nature_style()

    if "ROOT" not in globals() or "cfg" not in globals() or "result" not in globals():
        raise RuntimeError(
            "Run the setup, configuration and pipeline cells above first; ROOT, cfg and result are required."
        )
    root = Path(ROOT).expanduser().resolve()
    canonical = result["canonical"].copy()
    cfg_local = result.get("config", cfg)

    # Canonical validations must have passed before any publication figure is made.
    validation_path = root / "nature_water_outputs" / "diagnostics" / "canonical_validation.csv"
    conservation_path = root / "nature_water_outputs" / "diagnostics" / "aggregation_conservation.csv"
    for path in [validation_path, conservation_path]:
        if not path.exists():
            raise FileNotFoundError(f"Required validation output is missing: {path}")
        table = pd.read_csv(path)
        if "passed" not in table.columns or not table["passed"].astype(bool).all():
            failed = table.loc[~table["passed"].astype(bool)] if "passed" in table else table
            raise AssertionError(f"Publication figures blocked by failed validation in {path.name}:\n{failed}")

    output_root = root / "nature_water_outputs" / "super_defensible_figures"
    figure_dir = output_root / "figures"
    table_dir = output_root / "source_data"
    diagnostics_dir = output_root / "diagnostics"
    for folder in [figure_dir, table_dir, diagnostics_dir]:
        folder.mkdir(parents=True, exist_ok=True)

    cwd = Path.cwd().expanduser().resolve()
    target_candidates = [
        root / "images",
        root / "Nature_Water_super_defensible_package" / "images",
    ]
    # Jupyter is sometimes launched from the filesystem root.  Do not create
    # accidental /images directories in that case; add cwd-based targets only
    # when cwd is visibly part of this analysis project.
    cwd_is_project = (
        cwd != Path(cwd.anchor)
        and (
            cwd == root
            or root in cwd.parents
            or cwd in root.parents
            or (cwd / "BA_egrid.xlsx").exists()
            or (cwd / "Nature_Water_super_defensible_package").exists()
        )
    )
    if cwd_is_project:
        target_candidates.extend([
            cwd / "images",
            cwd / "Nature_Water_super_defensible_package" / "images",
        ])
    for extra_target in globals().get("EXTRA_LATEX_IMAGE_DIRS", []):
        target_candidates.append(Path(extra_target).expanduser().resolve())
    latex_targets: list[Path] = []
    seen = set()
    for candidate in target_candidates:
        key = str(candidate)
        if key not in seen:
            seen.add(key)
            latex_targets.append(candidate)

    # Load public geography layers using the exact paths resolved by the source notebook.
    states_raw = read_geodata(cfg_local["resolved_paths"]["states"])
    basins_raw = read_geodata(cfg_local["resolved_paths"]["hydrobasins_level6"])
    ba_raw = read_geodata(cfg_local["resolved_paths"]["balancing_authority_polygons"])
    states = _conus_states(states_raw, cfg_local)
    basin_shapes = _shape_ids(
        basins_raw, cfg_local["columns"]["hydrobasin_id"], "hydrobasin_id"
    )
    ba_shapes = _shape_ids(
        ba_raw, cfg_local["columns"]["ba_polygon_id"], "ba_id"
    )
    state_shapes = states[["state_id", "geometry"]].copy()

    national = national_summary(canonical)
    basin_all = aggregate_by_geography(canonical, "hydrobasin_id")
    ba_all = aggregate_by_geography(canonical, "ba_id")
    state_all = aggregate_by_geography(canonical, "state_id")
    basin_base = basin_all.loc[basin_all["scenario"].eq("baseline")].copy()
    ba_base = ba_all.loc[ba_all["scenario"].eq("baseline")].copy()

    all_paths: list[Path] = []
    all_paths += _draw_study_design(figure_dir, latex_targets)
    map_paths, map_metrics = _draw_pathway_maps(
        states=states, basins=basin_shapes, bas=basin_base,
        ba_shapes=ba_shapes, ba_base=ba_base,
        figure_dir=figure_dir, table_dir=table_dir, latex_targets=latex_targets,
    )
    all_paths += map_paths
    concentration_paths, concentration_values = _draw_concentration(
        basin_all, ba_all, figure_dir=figure_dir, table_dir=table_dir,
        latex_targets=latex_targets,
    )
    all_paths += concentration_paths
    all_paths += _draw_figure4(
        states=states, state_shapes=state_shapes, ba_shapes=ba_shapes,
        state_all=state_all, ba_all=ba_all,
        figure_dir=figure_dir, table_dir=table_dir, latex_targets=latex_targets,
    )
    all_paths += _draw_scenario_totals(
        national, figure_dir=figure_dir, table_dir=table_dir,
        latex_targets=latex_targets,
    )
    stress_paths, stress_counts = _draw_stress_sensitivity(
        basin_base, figure_dir=figure_dir, table_dir=table_dir,
        latex_targets=latex_targets,
    )
    all_paths += stress_paths
    all_paths += _draw_ba_quadrant(
        ba_base, figure_dir=figure_dir, table_dir=table_dir,
        latex_targets=latex_targets,
    )

    # Core output tables and defensibility checks.
    _public_csv(national, table_dir / "National_Scenario_Results.csv")
    actual = _actual_values(
        canonical=canonical, national=national,
        concentration_values=concentration_values, stress_counts=stress_counts,
    )
    crosscheck = _compare_expected(actual, EXPECTED_MANUSCRIPT_VALUES)
    crosscheck.to_csv(diagnostics_dir / "MANUSCRIPT_NUMERICAL_CROSSCHECK.csv", index=False)
    _write_latex_values(actual, output_root / "generated_manuscript_values.tex")
    for target in latex_targets:
        target.mkdir(parents=True, exist_ok=True)
        shutil.copy2(output_root / "generated_manuscript_values.tex", target / "generated_manuscript_values.tex")

    manifest = _file_manifest(all_paths)
    manifest.to_csv(diagnostics_dir / "FIGURE_FILE_MANIFEST.csv", index=False)
    expected_pngs = [
        "Figure2_Pathway_Burden_Maps.png",
        "Figure3_Normalized_Concentration.png",
        "Figure4_State_Pathway_Dominance.png",
        "Figure4_State_Scenario_Intervals.png",
        "Figure4_BA_Pathway_Dominance.png",
        "Figure4_BA_Scenario_Intervals.png",
        "ExtendedData_Scenario_Totals.png",
        "ExtendedData_Aqueduct_BWS_vs_BWD.png",
        "Figure3_Scope2_Quadrant.png",
    ]
    copy_checks = []
    for target in latex_targets:
        for name in expected_pngs:
            path = target / name
            copy_checks.append({
                "target_directory": str(target),
                "file": name,
                "exists": path.exists(),
                "bytes": path.stat().st_size if path.exists() else 0,
            })
    copy_checks_df = pd.DataFrame(copy_checks)
    copy_checks_df.to_csv(diagnostics_dir / "LATEX_FIGURE_COPY_CHECK.csv", index=False)
    if not copy_checks_df["exists"].all():
        raise AssertionError("At least one exact LaTeX figure filename was not copied successfully")

    metadata = {
        "created_from_config_hash": cfg_local.get("config_hash"),
        "canonical_rows": int(len(canonical)),
        "output_root": str(output_root),
        "latex_image_targets": [str(path) for path in latex_targets],
        "figure_style": {
            "double_column_width_mm": 183,
            "single_column_width_mm": 89,
            "base_text_pt": BASE_TEXT_PT,
            "panel_label_pt": PANEL_LABEL_PT,
            "png_dpi": PNG_DPI,
            "vector_formats": ["pdf", "svg"],
            "colour_space": "RGB",
        },
        "actual_manuscript_values": actual,
        "map_metrics": map_metrics,
        "all_current_main_tex_values_match": bool(crosscheck["passed"].all()),
        "privacy": "regional and national aggregate outputs only",
    }
    (output_root / "figure_run_manifest.json").write_text(json.dumps(metadata, indent=2), encoding="utf-8")

    print("\nNature Water figure suite complete")
    print("Canonical output:", output_root)
    print("Exact LaTeX PNGs copied to:")
    for target in latex_targets:
        print(" -", target)
    print("\nManuscript numerical cross-check:")
    print(crosscheck.to_string(index=False))
    if not crosscheck["passed"].all():
        print(
            "\nSUBMISSION BLOCKER: at least one number hard-coded in the current main.tex "
            "does not match this canonical run. The figure files were still generated. "
            "Use generated_manuscript_values.tex and MANUSCRIPT_NUMERICAL_CROSSCHECK.csv "
            "to update or diagnose the manuscript before submission."
        )
        if FAIL_ON_MANUSCRIPT_MISMATCH:
            raise AssertionError("Current main.tex values do not all match the canonical figure run")

    return {
        "output_root": output_root,
        "figure_dir": figure_dir,
        "table_dir": table_dir,
        "diagnostics_dir": diagnostics_dir,
        "latex_targets": latex_targets,
        "actual_values": actual,
        "crosscheck": crosscheck,
        "manifest": manifest,
    }


nature_water_final_figures = generate_super_defensible_figures()


## New reviewer-requested analyses

Populate `inputs/climate_pairs.csv`, `inputs/basin_labels.csv` and `inputs/stage_mapping.csv` using documented sources. Missing paired climate data are explicitly reported as not run; they are never replaced with made-up multipliers.

The calculation below uses the same in-memory canonical result. It writes aggregate CSVs and a draft Overleaf folder. Check `RUN_STATUS.json` and `diagnostics/manuscript_number_checks.csv` before using any output.


In [ ]:
import importlib
import nw_review, nw_plots
importlib.reload(nw_plots)
importlib.reload(nw_review)
revision = nw_review.run_revision(result, BUNDLE_ROOT)
REVISION_OUTPUT = Path(revision['output_root'])
display(pd.read_csv(REVISION_OUTPUT / 'diagnostics' / 'manuscript_number_checks.csv'))
display(pd.read_csv(REVISION_OUTPUT / 'source_data' / 'threshold_sweep.csv'))
display(pd.read_csv(REVISION_OUTPUT / 'source_data' / 'BA_hydropower_rankings.csv'))
print('Draft Overleaf project:', REVISION_OUTPUT)


## Finalization - only after inspection and author confirmation

Compile and inspect the empirical draft. Copy `author_signoff.template.json` to `author_signoff.json`, supply factual declarations and a verified reviewer-access URL, and confirm only checks that you actually completed. The routine refuses a missing climate result, unmatched baseline numbers, unverified regional names, incomplete stage mapping or changed run files.

The finalization step packages files; it does not upload a repository, issue a DOI or submit the paper. The response to Nature Communications must also match the completed run.


In [ ]:
FINALIZE = False  # Set True only after all checks in START_HERE_DELIVERY.md are complete.
if FINALIZE:
    from nw_review import finalize
    final_archive = finalize(REVISION_OUTPUT, REVISION_OUTPUT / 'author_signoff.json', BUNDLE_ROOT)
    print('Verified local source package:', final_archive)
else:
    print('Finalization not requested. Current outputs remain a draft.')
